# Full Thesis Project — Actual Source Code

This notebook contains the **real Python implementation from the repository**, grouped by pipeline stage. Each source cell begins with its original file path. The files in the repository remain the source of truth; this notebook is a portable explanation/reference copy.

> Source cells are tagged `project-source`. Do not execute a source cell in isolation: many scripts use command-line arguments and upstream artifact gates. Use the launcher cells below to run them from the repository root.

## Notebook launcher

This helper runs the original project scripts safely from the repository root and preserves their normal command-line behavior. Heavy stages still require CheXpert, approved manifests, and GPU resources.

In [ ]:
from pathlib import Path
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'scripts').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from the project checkout.')

REPO = find_repo_root()

def run_script(relative_path, *args, env=None):
    command = [sys.executable, str(REPO / relative_path), *map(str, args)]
    print('Running:', subprocess.list2cmdline(command))
    return subprocess.run(command, cwd=REPO, env=env, check=True)

print('Repository:', REPO)

## Quick local engineering validation

These switches default to `False` to prevent accidental long runs. The smoke fixture is explicitly non-scientific and does not represent thesis results.

In [ ]:
RUN_TESTS = False
RUN_LOCAL_SMOKE = False

if RUN_TESTS:
    run_script('tests/run_all.py')
if RUN_LOCAL_SMOKE:
    run_script('scripts/smoke/run_smoke_pipeline.py', '--phase', 'local')
if not (RUN_TESTS or RUN_LOCAL_SMOKE):
    print('Nothing executed. Change a switch to True when ready.')

# Shared utilities


## `scripts/utils/artifact_contracts.py`


In [ ]:
# SOURCE FILE: scripts/utils/artifact_contracts.py
"""Namespace-safe Stage 2/3 artifact paths and strict provenance contracts."""
from __future__ import annotations

from pathlib import Path
from typing import Any

from omegaconf import OmegaConf

from scripts.utils.manifest import code_identity, hash_dict, read_json, sha256_file
from scripts.utils.splits import read_split_manifest, resolve_splits_dir


class ArtifactContractError(SystemExit):
    pass


def validate_namespace_run_id(namespace: str, namespace_class: str) -> None:
    prefix = f"{namespace_class}-"
    if namespace_class not in {"dev", "production"} or not namespace.startswith(prefix) or len(namespace) <= len(prefix):
        raise ArtifactContractError(
            f"Invalid immutable split run ID {namespace!r}: a {namespace_class!r} run must be versioned as {prefix}<run-or-version>"
        )


def namespace_identity(namespace: str) -> dict[str, Any]:
    manifest = read_split_manifest(namespace)
    namespace_class = manifest.get("namespace_class")
    if namespace_class not in {"dev", "production"}:
        raise ArtifactContractError(
            f"Split run {namespace!r} has no valid namespace_class; rebuild it with the current six-way builder"
        )
    validate_namespace_run_id(namespace, namespace_class)
    return {
        "split_namespace": namespace,
        "namespace_class": namespace_class,
        "split_manifest_version": manifest["manifest_version"],
        "split_manifest_hash": manifest["manifest_hash"],
        "split_manifest_file_sha256": sha256_file(resolve_splits_dir(namespace) / "split_manifest_v2.json"),
        "split_frozen": bool(manifest.get("frozen", False)),
    }


def stage2_paths(config, namespace: str) -> dict[str, Path]:
    root = Path(config.paths.synthetic_root) / namespace
    pilot = root / "pilot"
    return {
        "root": root,
        "images_dir": root / "images",
        "recipes_path": root / "label_recipes.csv",
        "recipe_decisions_path": root / "recipe_decisions.jsonl",
        "recipes_manifest": root / "recipes_manifest.json",
        "manifest_path": root / "generation_manifest.jsonl",
        "generation_provenance": root / "generation_provenance.json",
        "generation_completion": root / "generation_complete.json",
        "pilot_dir": pilot,
        "pilot_images": pilot / "images",
        "pilot_manifest": pilot / "pilot_generation_manifest.jsonl",
        "pilot_approval_manifest": pilot / "pilot_approval_manifest.json",
    }


def stage3_paths(config, namespace: str) -> dict[str, Path]:
    root = Path(config.paths.synthetic_root) / namespace
    asism = root / "asism"
    return {
        "root": root,
        "scores_dir": root / "scores",
        "asism_dir": asism,
        "gonogo_report": asism / "gonogo_report.json",
        "search_log": asism / "proxy_search_log.jsonl",
        "frozen_manifest": asism / "asism_frozen_manifest.json",
        "fold_manifest": asism / "asism_patient_folds.json",
        "selected_manifest": asism / "selected_manifest.jsonl",
        "rejected_log": asism / "rejected_log.jsonl",
    }


def auxiliary_checkpoint_path(config, namespace: str) -> Path:
    configured = Path(config.auxiliary_classifier.checkpoint_path)
    return configured.parent / namespace / configured.name


def stage4_paths(config, namespace: str) -> dict[str, Path]:
    checkpoints = Path(config.paths.checkpoints_dir) / namespace
    results = Path(config.paths.results_dir) / namespace
    return {
        "checkpoints_dir": checkpoints, "results_dir": results,
        "protocol_manifest": results / "frozen_experiment_protocol.json",
        "label_policy_manifest": results / "frozen_label_policy.json",
        "threshold_policy_manifest": results / "frozen_threshold_policy.json",
        "seed_plan_manifest": results / "frozen_seed_run_plan.json",
        "d_matching_policy_manifest": results / "frozen_d_matching_policy.json",
    }


def config_sha256(config_section) -> str:
    return hash_dict(OmegaConf.to_container(config_section, resolve=True), length=64)


def require_manifest_fields(path: Path, expected: dict[str, Any], label: str) -> dict[str, Any]:
    if not path.is_file():
        raise ArtifactContractError(f"Missing {label} manifest: {path}")
    actual = read_json(path)
    mismatches = {key: {"expected": value, "actual": actual.get(key)} for key, value in expected.items() if actual.get(key) != value}
    if mismatches:
        raise ArtifactContractError(f"Stale/incompatible {label} artifact at {path}: {mismatches}")
    return actual


def current_code_identity_hash() -> str:
    return code_identity()["source_tree_sha256"]


def require_generation_complete(config, namespace: str) -> tuple[dict[str, Path], dict[str, Any]]:
    paths = stage2_paths(config, namespace)
    identity = namespace_identity(namespace)
    expected = {"split_namespace": namespace, "namespace_class": identity["namespace_class"],
                "split_manifest_hash": identity["split_manifest_hash"], "status": "complete"}
    completion = require_manifest_fields(paths["generation_completion"], expected, "Stage 2 generation completion")
    if not paths["manifest_path"].is_file() or completion.get("generation_manifest_sha256") != sha256_file(paths["manifest_path"]):
        raise ArtifactContractError("Generation manifest is missing or differs from its completion record")
    if completion.get("code_identity_sha256") != current_code_identity_hash():
        raise ArtifactContractError("Generation artifacts were produced by a different source-tree identity")
    return paths, completion


def require_score_artifact(path: Path, signal: str, expected_provenance: dict[str, Any]) -> dict[str, Any]:
    sidecar = path.with_suffix(".provenance.json")
    expected = {"schema_version": 2, "signal": signal, **expected_provenance}
    manifest = require_manifest_fields(sidecar, expected, f"ASISM {signal} score")
    if not path.is_file() or manifest.get("parquet_sha256") != sha256_file(path):
        raise ArtifactContractError(f"ASISM {signal} Parquet hash mismatch: {path}")
    return manifest


def asism_score_provenance(stage3_config, stage2_config, namespace: str) -> dict[str, Any]:
    stage2, generation = require_generation_complete(stage2_config, namespace)
    return {
        **namespace_identity(namespace), "asism_config_sha256": config_sha256(stage3_config),
        "generation_manifest_sha256": sha256_file(stage2["manifest_path"]),
        "generation_completion_sha256": sha256_file(stage2["generation_completion"]),
        "lora_checkpoint_sha256": generation["lora_checkpoint_sha256"],
        "code_identity_sha256": current_code_identity_hash(),
    }


## `scripts/utils/caption_builder.py`


In [ ]:
# SOURCE FILE: scripts/utils/caption_builder.py
"""Structured label-to-text caption builder for CheXpert chest X-rays (docs/stage1_plan.md §7).

This module is the single source of truth for turning a CheXpert label row into a text prompt.
It is imported verbatim by both the Stage 1 training data pipeline (scripts/data/04_generate_captions.py)
and, later, Stage 2's generation pipeline — any drift between train-time and generation-time
phrasing directly degrades conditioning fidelity, so nothing caption-related should be
reimplemented elsewhere.

Design decisions encoded here (see docs/stage1_plan.md §7 for full rationale):
- Uncertain (-1) labels are OMITTED from the finding clause, never asserted present or absent.
- "No Finding" == 1 is authoritative over any spuriously co-positive pathology columns.
- "Support Devices" is a device, not a pathology, and gets its own clause.
- Age is decade-bucketed to reduce caption vocabulary sparsity.
- A small set of paraphrase templates is selected deterministically (hash of image id + epoch),
  not by an unseeded RNG, so caption generation is reproducible.
"""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, field

# Canonical order for serializing multi-label findings. Must match configs/dataset_config.yaml's
# `schema.pathology_columns` — kept as a plain constant here (rather than loaded from YAML) so this
# module has no config-parsing dependency and can be imported standalone by Stage 2.
PATHOLOGY_COLUMNS = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

DEVICE_COLUMN = "Support Devices"
NO_FINDING_COLUMN = "No Finding"

# Human-readable phrasing for each pathology column (lowercase, fit for "Findings: X, Y, and Z.")
FINDING_TEXT = {
    "Enlarged Cardiomediastinum": "an enlarged cardiomediastinum",
    "Cardiomegaly": "cardiomegaly",
    "Lung Opacity": "lung opacity",
    "Lung Lesion": "a lung lesion",
    "Edema": "pulmonary edema",
    "Consolidation": "consolidation",
    "Pneumonia": "pneumonia",
    "Atelectasis": "atelectasis",
    "Pneumothorax": "pneumothorax",
    "Pleural Effusion": "pleural effusion",
    "Pleural Other": "other pleural abnormality",
    "Fracture": "a fracture",
}

NO_FINDING_PHRASE = "no acute cardiopulmonary abnormality"
DEVICE_PHRASE = "Support devices are present."

# Paraphrase templates (plan §7: 3-5 variants, config default 4). All must use exactly the
# {view}, {age_bucket}, {sex}, {finding_clause} placeholders; {device_clause} is appended
# separately by build_caption(), never interpolated inside these.
TEMPLATES = [
    "A {view} chest X-ray of a {age_bucket} {sex} patient. Findings: {finding_clause}.",
    "Chest radiograph, {view} view, {age_bucket} {sex} patient. Findings: {finding_clause}.",
    "A {view} view chest X-ray from a {age_bucket} {sex} patient showing {finding_clause}.",
    "{age_bucket} {sex} patient, {view} chest X-ray. Findings: {finding_clause}.",
]


@dataclass(frozen=True)
class CaptionConfig:
    age_bucket_width_years: int = 10
    uncertain_label_policy: str = "omit"  # only "omit" is implemented (plan §7 decision)
    no_finding_overrides_positives: bool = True
    num_paraphrase_variants: int = 4
    template_version: str = "v1"


def bucket_age(age, bucket_width_years: int = 10) -> str:
    """Decade-bucket a numeric age into a phrase like '40-49-year-old'."""
    try:
        age_int = int(float(age))
    except (TypeError, ValueError):
        return "adult"
    if age_int <= 0:
        return "adult"
    low = (age_int // bucket_width_years) * bucket_width_years
    high = low + bucket_width_years - 1
    return f"{low}-{high}-year-old"


def sex_phrase(sex_value) -> str:
    if sex_value is None:
        return "patient"
    s = str(sex_value).strip().lower()
    if s.startswith("m"):
        return "male"
    if s.startswith("f"):
        return "female"
    return "patient"


def view_phrase(frontal_lateral_value, ap_pa_value=None) -> str:
    fl = str(frontal_lateral_value).strip().lower() if frontal_lateral_value is not None else "frontal"
    if fl.startswith("lat"):
        return "lateral"
    return "frontal"


def serialize_natural_list(items: list[str]) -> str:
    """Join items with natural 'X', 'X and Y', or 'X, Y, and Z' grammar (plan §7: more stable
    CLIP embeddings than a raw comma token-dump)."""
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return f"{items[0]} and {items[1]}"
    return ", ".join(items[:-1]) + f", and {items[-1]}"


def _label_value(row: dict, column: str):
    """Normalize a raw CheXpert label cell to one of {-1, 0, 1, None}."""
    value = row.get(column)
    if value is None or value == "":
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    if f != f:  # NaN
        return None
    return int(f)


def build_finding_clause(row: dict, config: CaptionConfig = CaptionConfig()) -> str:
    """Build the 'Findings: ...' clause per the uncertainty and No-Finding policies in plan §7."""
    if config.uncertain_label_policy != "omit":
        raise NotImplementedError(
            f"uncertain_label_policy={config.uncertain_label_policy!r} not implemented; "
            "the plan's chosen policy is 'omit' — see docs/stage1_plan.md §7."
        )

    no_finding = _label_value(row, NO_FINDING_COLUMN)
    positive_pathologies = [
        col
        for col in PATHOLOGY_COLUMNS
        if col not in (NO_FINDING_COLUMN, DEVICE_COLUMN) and _label_value(row, col) == 1
    ]

    if config.no_finding_overrides_positives and no_finding == 1:
        # A positive "No Finding" is treated as authoritative over any spuriously co-positive
        # pathology columns (known NLP-label-extraction artifact) — see plan §7.
        return NO_FINDING_PHRASE

    if not positive_pathologies:
        if no_finding == 1:
            return NO_FINDING_PHRASE
        # No positive findings and No Finding not asserted (e.g. only uncertain/negative labels):
        # still emit the normal-chest phrasing rather than an empty clause.
        return NO_FINDING_PHRASE

    phrases = [FINDING_TEXT[col] for col in positive_pathologies]
    return serialize_natural_list(phrases)


def build_device_clause(row: dict) -> str:
    if _label_value(row, DEVICE_COLUMN) == 1:
        return DEVICE_PHRASE
    return ""


def select_variant_index(image_id: str, epoch: int, num_variants: int) -> int:
    """Deterministic (not process-RNG-dependent) paraphrase variant selection, so caption
    generation is reproducible across runs given the same image id and epoch (plan §7)."""
    key = f"{image_id}:{epoch}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    return int(digest, 16) % num_variants


def build_caption(
    row: dict,
    config: CaptionConfig = CaptionConfig(),
    variant_index: int = 0,
) -> str:
    """Build a single caption string from a CheXpert label row.

    `row` is expected to expose (at minimum) the columns listed in
    configs/dataset_config.yaml `schema`: Sex, Age, Frontal/Lateral, AP/PA, and the 14
    pathology columns in PATHOLOGY_COLUMNS, with ternary values in {-1, 0, 1, NaN/None}.
    """
    template = TEMPLATES[variant_index % len(TEMPLATES)]
    view = view_phrase(row.get("Frontal/Lateral"), row.get("AP/PA"))
    age_bucket = bucket_age(row.get("Age"), config.age_bucket_width_years)
    sex = sex_phrase(row.get("Sex"))
    finding_clause = build_finding_clause(row, config)
    device_clause = build_device_clause(row)

    caption = template.format(
        view=view,
        age_bucket=age_bucket,
        sex=sex,
        finding_clause=finding_clause,
    )
    if device_clause:
        caption = f"{caption} {device_clause}"
    return caption


def build_caption_variants(row: dict, config: CaptionConfig = CaptionConfig()) -> list[str]:
    """Return all configured paraphrase variants for a row (useful for prompt-cache precomputation,
    plan §8, where every variant an image might use during training is embedded once up front)."""
    n = min(config.num_paraphrase_variants, len(TEMPLATES))
    return [build_caption(row, config, variant_index=i) for i in range(n)]


## `scripts/utils/chexpert_schema.py`


In [ ]:
# SOURCE FILE: scripts/utils/chexpert_schema.py
"""Shared CheXpert source-schema and path validation used by verification and split building."""
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

from scripts.utils.labels import validate_label_domain
from scripts.utils.manifest import hash_dict


def resolve_chexpert_image(raw_dir: Path, value: object) -> Path:
    relative = str(value).replace("\\", "/")
    for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
        if relative.startswith(prefix):
            relative = relative[len(prefix):]
    return raw_dir / relative


def validate_chexpert_frame(frame: pd.DataFrame, dataset_cfg, raw_dir: Path | None = None,
                            require_all_images: bool = False) -> dict:
    schema = dataset_cfg.schema
    required = [schema.path_column, schema.sex_column, schema.age_column,
                schema.frontal_lateral_column, *list(schema.pathology_columns)]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"missing required columns: {missing}")
    validate_label_domain(frame, list(schema.pathology_columns), str(schema.path_column))
    paths = frame[schema.path_column]
    if paths.isna().any() or paths.astype(str).str.strip().eq("").any():
        raise ValueError("blank image path")
    duplicates = paths[paths.duplicated()].astype(str).head().tolist()
    if duplicates:
        raise ValueError(f"duplicate image paths: {duplicates}")
    invalid_patient = [value for value in paths.astype(str) if not re.search(str(schema.patient_id_regex), value)]
    if invalid_patient:
        raise ValueError(f"invalid patient ID/path: {invalid_patient[:5]}")
    if require_all_images:
        if raw_dir is None:
            raise ValueError("raw_dir is required for full image-path validation")
        missing_images = [str(resolve_chexpert_image(raw_dir, value)) for value in paths if not resolve_chexpert_image(raw_dir, value).is_file()]
        if missing_images:
            raise ValueError(f"{len(missing_images)} image paths do not exist; examples={missing_images[:5]}")
    return {"source_schema_hash": hash_dict({"columns": list(frame.columns),
                                              "dtypes": {c: str(t) for c, t in frame.dtypes.items()}}, length=64)}


## `scripts/utils/classifier.py`


In [ ]:
# SOURCE FILE: scripts/utils/classifier.py
"""Shared multi-label CXR classifier: model, dataset, masked loss, training loop, inference.

One implementation, three consumers, so they cannot drift apart:
  - the auxiliary reference classifier (Stage 3 prerequisite, docs/stages2_to_5_plan.md §2);
  - the ASISM proxy classifier used to rank candidate configurations (§4.7);
  - the A-E headline experiment classifiers (§7).

Two design points are load-bearing rather than incidental:

MC Dropout (§4.3) requires dropout ACTIVE at inference. `model.eval()` disables it, so
`enable_mc_dropout()` re-enables only the dropout modules while leaving BatchNorm in eval mode —
switching the whole model back to train() would also make BatchNorm use batch statistics, which
would corrupt the predictions the uncertainty estimate is computed from.

Masked loss (§6) implements the frozen uncertainty policy: -1 and blank labels contribute to
neither the loss nor any metric, and are never converted into a 0 or 1 the model would be trained
to reproduce.
"""

from __future__ import annotations

import json
import os
import random
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

from scripts.utils.labels import CLASSIFIER_TARGET_LABELS, build_label_arrays


@dataclass
class TrainingBudget:
    """Frozen fairness protocol (§7.1): budget is expressed in OPTIMIZER STEPS, not epochs.

    At fixed epochs a larger dataset silently receives more gradient updates, which would conflate
    "more data" with "more training" — exactly the confound condition D exists to rule out.
    """

    max_steps: int
    batch_size: int = 32
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    warmup_steps: int = 0
    eval_every_n_steps: int = 200
    num_workers: int = 0
    seed: int = 42


@dataclass
class RunAccounting:
    """Per-run record required by §7.1 — what was actually consumed, not what was configured."""

    optimizer_steps: int = 0
    epochs_completed: float = 0.0
    effective_dataset_size: int = 0
    real_sample_exposures: int = 0
    synthetic_sample_exposures: int = 0
    sampling_policy: str = "shuffled_concat"
    model_seed: int = 0
    dataset_id: str = ""
    config_hash: str = ""
    checkpoint_hash: str = ""
    extra: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "optimizer_steps": self.optimizer_steps,
            "epochs_completed": round(self.epochs_completed, 4),
            "effective_dataset_size": self.effective_dataset_size,
            "real_sample_exposures": self.real_sample_exposures,
            "synthetic_sample_exposures": self.synthetic_sample_exposures,
            "sampling_policy": self.sampling_policy,
            "model_seed": self.model_seed,
            "dataset_id": self.dataset_id,
            "config_hash": self.config_hash,
            "checkpoint_hash": self.checkpoint_hash,
            **self.extra,
        }


def require_torch():
    """Import torch lazily with a clean, actionable message when the GPU stack is absent."""
    try:
        import torch  # noqa: F401
        import torchvision  # noqa: F401
    except ImportError as exc:
        raise SystemExit(
            f"UPSTREAM GATE: the training stack is not installed here ({exc}).\n"
            "Classifier training is GPU work — run it on the RunPod pod after:\n"
            "  pip install -r environment/requirements.txt"
        ) from exc
    import torch

    return torch


class CXRRecordDataset:
    """Multi-label CXR dataset over an explicit record list.

    Each record is {"image_path": str, "labels": {label: raw_value}, "is_synthetic": bool}. Keeping
    real and synthetic rows in one uniform record type is what lets conditions A-E differ only by
    which records they are handed, rather than by separate code paths per condition.
    """

    def __init__(self, records: list[dict], resolution: int = 320, augment: bool = False):
        self.records = records
        self.resolution = resolution
        self.augment = augment
        self._labels = CLASSIFIER_TARGET_LABELS

        frame = pd.DataFrame([record["labels"] for record in records])
        for label in self._labels:
            if label not in frame.columns:
                frame[label] = np.nan
        _, self.targets, self.masks = build_label_arrays(frame[self._labels], self._labels)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        import torch
        from PIL import Image
        from torchvision.transforms import functional as TF

        record = self.records[index]
        with Image.open(record["image_path"]) as image:
            image = image.convert("RGB")
            if image.size != (self.resolution, self.resolution):
                image = image.resize((self.resolution, self.resolution), Image.BICUBIC)
            tensor = TF.to_tensor(image)

        # Horizontal flip is deliberately NOT offered: it inverts CXR left/right anatomy
        # (heart position, aortic arch, gastric bubble) — same constraint as Stage 1's data config.
        tensor = TF.normalize(tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

        return {
            "image": tensor,
            "target": torch.from_numpy(self.targets[index]),
            "mask": torch.from_numpy(self.masks[index]),
            "index": index,
        }


def build_model(num_labels: int, dropout_p: float, pretrained_source: str, seed: int = 42):
    """DenseNet121 with a dropout layer before the classifier head.

    `pretrained_source` is recorded verbatim in run provenance. CheXpert-pretrained weights are
    rejected here rather than merely discouraged: most published CheXpert checkpoints do not
    publish patient-level provenance, so overlap with final_eval_heldout cannot be excluded (§2).
    """
    torch = require_torch()
    import torchvision

    if "chexpert" in str(pretrained_source).lower():
        raise SystemExit(
            f"Refusing CheXpert-pretrained initialization ({pretrained_source!r}).\n"
            "Its training patients cannot be shown disjoint from our evaluation splits, which "
            "would contaminate the headline comparison (docs/stages2_to_5_plan.md §2).\n"
            "Use 'imagenet', 'random', or a CXR checkpoint with published patient-level provenance."
        )

    torch.manual_seed(seed)
    if pretrained_source == "imagenet":
        weights = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
    elif pretrained_source == "random":
        weights = None
    else:
        raise SystemExit(
            f"Unsupported pretrained_source {pretrained_source!r}. "
            "Supported: 'imagenet', 'random'. A documented non-overlapping CXR checkpoint may be "
            "added here once its provenance is recorded (§2)."
        )

    model = torchvision.models.densenet121(weights=weights)
    in_features = model.classifier.in_features
    model.classifier = torch.nn.Sequential(
        torch.nn.Dropout(p=dropout_p),
        torch.nn.Linear(in_features, num_labels),
    )
    return model


def enable_mc_dropout(model) -> int:
    """Put ONLY dropout modules into train mode, leaving BatchNorm in eval mode.

    Calling model.train() instead would also switch BatchNorm to batch statistics, which changes
    the predictions themselves rather than just sampling over dropout masks — the resulting spread
    would not be the epistemic uncertainty §4.3 intends to measure.
    """
    import torch

    count = 0
    for module in model.modules():
        if isinstance(module, (torch.nn.Dropout, torch.nn.Dropout2d)):
            module.train()
            count += 1
    return count


def masked_bce_loss(logits, targets, masks):
    """Binary cross-entropy over confidently-labelled positions only (§6).

    Uncertain (-1) and blank labels are excluded from the loss entirely rather than mapped to 0/1,
    so the model is never trained to reproduce an assertion the radiologist declined to make.
    """
    import torch
    import torch.nn.functional as F

    per_element = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    masked = per_element * masks.float()
    denominator = masks.float().sum()
    if denominator.item() == 0:
        return torch.zeros((), device=logits.device, requires_grad=True)
    return masked.sum() / denominator


def train_classifier(
    train_records: list[dict],
    val_records: list[dict],
    budget: TrainingBudget,
    dropout_p: float,
    pretrained_source: str,
    resolution: int,
    device: str | None = None,
    progress_desc: str = "train",
    checkpoint_path: Path | None = None,
    resume_from: Path | None = None,
    checkpoint_metadata: dict | None = None,
) -> tuple[object, RunAccounting, list[dict]]:
    """Train to a FIXED optimizer-step budget. Returns (model, accounting, eval_history).

    Deliberately no early stopping inside this function: the ASISM proxy search forbids
    candidate-specific early stopping (§4.7), and A-E requires an identical protocol across
    conditions (§7.1). Checkpoint SELECTION (best-on-classifier_val) is applied by the caller that
    is entitled to do it, from the history returned here.
    """
    torch = require_torch()
    from torch.utils.data import DataLoader
    from tqdm.auto import tqdm

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    random.seed(budget.seed)
    torch.manual_seed(budget.seed)
    np.random.seed(budget.seed)

    train_dataset = CXRRecordDataset(train_records, resolution=resolution)
    sampler_generator = torch.Generator().manual_seed(budget.seed)
    loader = DataLoader(
        train_dataset,
        batch_size=budget.batch_size,
        shuffle=True,
        num_workers=budget.num_workers,
        drop_last=False,
        generator=sampler_generator,
    )

    model = build_model(len(CLASSIFIER_TARGET_LABELS), dropout_p, pretrained_source, budget.seed)
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=budget.learning_rate, weight_decay=budget.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, budget.max_steps))

    accounting = RunAccounting(
        effective_dataset_size=len(train_records),
        real_sample_exposures=0,
        synthetic_sample_exposures=0,
        model_seed=budget.seed,
    )
    n_synthetic = sum(1 for record in train_records if record.get("is_synthetic"))
    n_real = len(train_records) - n_synthetic

    history: list[dict] = []
    step = 0
    best_metric = float("-inf")
    best_checkpoint_identity = None
    expected_meta = checkpoint_metadata or {}
    best_checkpoint_path = (
        checkpoint_path.with_name(f"{checkpoint_path.stem}.best{checkpoint_path.suffix}")
        if checkpoint_path is not None else None
    )
    if resume_from is not None:
        payload = torch.load(resume_from, map_location=device, weights_only=False)
        if payload.get("checkpoint_schema_version") != 2:
            raise ValueError("incompatible classifier checkpoint schema")
        actual_meta = payload.get("provenance", {})
        mismatches = {key: (actual_meta.get(key), value) for key, value in expected_meta.items() if actual_meta.get(key) != value}
        if mismatches:
            raise ValueError(f"incompatible classifier checkpoint provenance: {mismatches}")
        model.load_state_dict(payload["model_state"])
        optimizer.load_state_dict(payload["optimizer_state"])
        scheduler.load_state_dict(payload["scheduler_state"])
        step = int(payload["global_optimizer_step"])
        history = list(payload.get("eval_history", []))
        best_metric = float(payload.get("best_validation_metric", float("-inf")))
        best_checkpoint_identity = payload.get("best_checkpoint_identity")
        random.setstate(payload["rng_states"]["python"])
        np.random.set_state(payload["rng_states"]["numpy"])
        # ``map_location=device`` also moves the saved CPU RNG tensor to CUDA on GPU runs.
        # PyTorch requires a CPU ByteTensor when restoring the default CPU generator.
        torch_cpu_rng = payload["rng_states"]["torch_cpu"].detach().cpu()
        torch.set_rng_state(torch_cpu_rng)
        if torch.cuda.is_available() and payload["rng_states"].get("torch_cuda"):
            torch.cuda.set_rng_state_all(
                [state.detach().cpu() for state in payload["rng_states"]["torch_cuda"]]
            )
        sampler_generator.set_state(payload["sampler_state"].detach().cpu())

    def save_resumable() -> None:
        nonlocal best_checkpoint_identity
        if checkpoint_path is None:
            return
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        best_checkpoint_identity = best_checkpoint_identity or f"step-{step}"
        payload = {
            "checkpoint_schema_version": 2, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
            "global_optimizer_step": step, "step": step,
            "best_validation_metric": best_metric, "best_checkpoint_identity": best_checkpoint_identity,
            "early_stopping_state": {"enabled": False, "bad_evaluations": 0},
            "rng_states": {"python": random.getstate(), "numpy": np.random.get_state(),
                           "torch_cpu": torch.get_rng_state(),
                           "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []},
            "sampler_state": sampler_generator.get_state(), "sampler_rule": "torch RandomSampler generator state",
            "eval_history": history, "provenance": expected_meta,
        }
        fd, temp_name = tempfile.mkstemp(prefix=f".{checkpoint_path.name}.", suffix=".tmp", dir=checkpoint_path.parent)
        os.close(fd)
        try:
            torch.save(payload, temp_name)
            os.replace(temp_name, checkpoint_path)
        finally:
            Path(temp_name).unlink(missing_ok=True)

    def save_best() -> None:
        if best_checkpoint_path is None:
            return
        best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "checkpoint_schema_version": 2, "checkpoint_role": "best_validation_model",
            "model_state": model.state_dict(), "best_validation_metric": best_metric,
            "best_checkpoint_identity": best_checkpoint_identity, "provenance": expected_meta,
            "model_config": {"dropout_p": dropout_p, "pretrained_source": pretrained_source,
                             "resolution": resolution, "num_labels": len(CLASSIFIER_TARGET_LABELS)},
        }
        fd, temp_name = tempfile.mkstemp(prefix=f".{best_checkpoint_path.name}.", suffix=".tmp", dir=best_checkpoint_path.parent)
        os.close(fd)
        try:
            torch.save(payload, temp_name)
            os.replace(temp_name, best_checkpoint_path)
        finally:
            Path(temp_name).unlink(missing_ok=True)
    model.train()
    progress = tqdm(total=budget.max_steps, initial=step, desc=progress_desc, unit="step")

    while step < budget.max_steps:
        for batch in loader:
            if step >= budget.max_steps:
                break
            images = batch["image"].to(device)
            targets = batch["target"].to(device)
            masks = batch["mask"].to(device)

            logits = model(images)
            loss = masked_bce_loss(logits, targets, masks)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()

            step += 1
            progress.update(1)
            progress.set_postfix(loss=f"{loss.item():.4f}", refresh=False)

            if val_records and budget.eval_every_n_steps and step % budget.eval_every_n_steps == 0:
                metrics = evaluate_classifier(model, val_records, resolution, device)
                metrics["step"] = step
                history.append(metrics)
                if np.isfinite(metrics["macro_auroc"]) and metrics["macro_auroc"] >= best_metric:
                    best_metric = metrics["macro_auroc"]
                    best_checkpoint_identity = f"step-{step}"
                    save_best()
                save_resumable()
                model.train()

    progress.close()

    accounting.optimizer_steps = step
    seen = step * budget.batch_size
    accounting.epochs_completed = seen / max(len(train_records), 1)
    total = max(len(train_records), 1)
    accounting.real_sample_exposures = int(seen * n_real / total)
    accounting.synthetic_sample_exposures = int(seen * n_synthetic / total)

    if val_records:
        final_metrics = evaluate_classifier(model, val_records, resolution, device)
        final_metrics["step"] = step
        history.append(final_metrics)
        if np.isfinite(final_metrics["macro_auroc"]) and final_metrics["macro_auroc"] >= best_metric:
            best_metric = final_metrics["macro_auroc"]
            best_checkpoint_identity = f"step-{step}"
            save_best()
    save_resumable()

    if val_records and (best_checkpoint_path is None or not best_checkpoint_path.is_file()):
        raise RuntimeError(
            "No finite validation macro-AUROC was observed; refusing to publish an arbitrary final-step model as best"
        )

    if best_checkpoint_path is not None and best_checkpoint_path.is_file():
        best_payload = torch.load(best_checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(best_payload["model_state"])
        accounting.extra["best_checkpoint_path"] = str(best_checkpoint_path)
        accounting.extra["best_validation_metric"] = best_metric

    return model, accounting, history


def predict_probabilities(
    model,
    records: list[dict],
    resolution: int,
    device: str | None = None,
    batch_size: int = 32,
    mc_dropout_passes: int = 0,
) -> tuple[np.ndarray, np.ndarray | None]:
    """Predict per-label probabilities.

    With mc_dropout_passes > 0, returns (mean, std) over stochastic forward passes — the §4.3
    uncertainty signal. With 0, returns (probabilities, None) from a single deterministic pass.
    """
    torch = require_torch()
    from torch.utils.data import DataLoader

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    dataset = CXRRecordDataset(records, resolution=resolution)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    n_passes = max(1, mc_dropout_passes)
    if mc_dropout_passes > 0:
        activated = enable_mc_dropout(model)
        if activated == 0:
            raise SystemExit(
                "MC Dropout requested but the model contains no dropout modules — the uncertainty "
                "signal would silently degenerate to a constant. Check classifier.dropout_p (§2)."
            )

    all_passes = []
    with torch.no_grad():
        for _ in range(n_passes):
            batch_outputs = []
            for batch in loader:
                logits = model(batch["image"].to(device))
                batch_outputs.append(torch.sigmoid(logits).cpu().numpy())
            all_passes.append(np.concatenate(batch_outputs, axis=0))

    stacked = np.stack(all_passes, axis=0)
    mean = stacked.mean(axis=0)
    if mc_dropout_passes > 0:
        return mean, stacked.std(axis=0)
    return mean, None


def evaluate_classifier(model, records: list[dict], resolution: int, device: str | None = None) -> dict:
    """Masked macro-AUROC over the primary endpoint labels, plus effective N (§5.3, §6)."""
    from scripts.utils.metrics import macro_auroc_masked

    probabilities, _ = predict_probabilities(model, records, resolution, device)

    frame = pd.DataFrame([record["labels"] for record in records])
    for label in CLASSIFIER_TARGET_LABELS:
        if label not in frame.columns:
            frame[label] = np.nan
    _, targets, masks = build_label_arrays(frame[CLASSIFIER_TARGET_LABELS], CLASSIFIER_TARGET_LABELS)

    return macro_auroc_masked(probabilities, targets, masks)


def records_from_split(
    frame: pd.DataFrame,
    images_dir: Path,
    path_column: str = "Path",
    require_exists: bool = True,
) -> list[dict]:
    """Build classifier records from a real-data split, using Stage 1's preprocessed images.

    Rows whose preprocessed image is absent (filtered as lateral/corrupt during Stage 1
    preprocessing) are skipped rather than silently substituted.
    """
    from scripts.utils.identifiers import sanitize_image_id

    records = []
    for row in frame.to_dict("records"):
        image_id = sanitize_image_id(row[path_column])
        image_path = images_dir / f"{image_id}.jpg"
        if require_exists and not image_path.is_file():
            continue
        records.append(
            {
                "image_id": image_id,
                "image_path": str(image_path),
                "patient_id": row.get("patient_id"),
                "labels": {label: row.get(label) for label in CLASSIFIER_TARGET_LABELS},
                "is_synthetic": False,
            }
        )
    return records


def records_from_synthetic_manifest(
    manifest_path: Path,
    images_dir: Path,
    image_ids: Iterable[str] | None = None,
    require_exists: bool = True,
) -> list[dict]:
    """Build classifier records from Stage 2 output.

    The synthetic label vector is the INTENDED vector (§3) — the generator's conditioning intent,
    which Stage 3 measures agreement against and which Stage 4 then trains on. It is labelled
    `intended_label_vector` in the manifest and is never described as ground truth.
    """
    wanted = set(image_ids) if image_ids is not None else None
    records = []
    with open(manifest_path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            image_id = entry["image_id"]
            if wanted is not None and image_id not in wanted:
                continue
            image_path = images_dir / f"{image_id}.jpg"
            if require_exists and not image_path.is_file():
                continue
            labels = {label: value for label, value in entry["intended_label_vector"].items()}
            labels["No Finding"] = 1 if entry.get("is_no_finding") else 0
            labels["Support Devices"] = int(entry.get("support_devices", 0))
            records.append(
                {
                    "image_id": image_id,
                    "image_path": str(image_path),
                    "patient_id": f"synthetic::{image_id}",
                    "labels": labels,
                    "is_synthetic": True,
                }
            )
    return records


## `scripts/utils/config.py`


In [ ]:
# SOURCE FILE: scripts/utils/config.py
"""Load and resolve the Stage 1 YAML configs (configs/stage1_lora_sdxl.yaml,
configs/dataset_config.yaml) via OmegaConf, with PROJECT_ROOT-relative path resolution.

Every script under scripts/ should load config through this module rather than parsing YAML
directly, so path resolution (PROJECT_ROOT env var, see docs/stage1_plan.md §4) stays consistent.
"""

from __future__ import annotations

from pathlib import Path
import os

from omegaconf import OmegaConf, DictConfig

REPO_ROOT = Path(__file__).resolve().parents[2]
CONFIGS_DIR = REPO_ROOT / "configs"


def _smoke_overlay(section: str) -> DictConfig | None:
    path = os.environ.get("THESIS_CONFIG_OVERLAY")
    if not path:
        return None
    overlay_path = Path(path)
    if not overlay_path.is_file():
        raise SystemExit(f"THESIS_CONFIG_OVERLAY does not exist: {overlay_path}")
    overlay = OmegaConf.load(overlay_path)
    return overlay.get(section)


def load_named_config(filename: str, section: str) -> DictConfig:
    config = OmegaConf.load(CONFIGS_DIR / filename)
    overlay = _smoke_overlay(section)
    if overlay is not None:
        config = OmegaConf.merge(config, overlay)
    OmegaConf.resolve(config)
    return config


def load_stage1_config(overrides: list[str] | None = None) -> DictConfig:
    """Load configs/stage1_lora_sdxl.yaml with PROJECT_ROOT-based interpolation resolved.

    `overrides` accepts OmegaConf dotlist entries, e.g. ["training.train_batch_size=4"], for
    CLI-driven experiment overrides without editing the checked-in YAML.
    """
    config = OmegaConf.load(CONFIGS_DIR / "stage1_lora_sdxl.yaml")
    overlay = _smoke_overlay("stage1")
    if overlay is not None:
        config = OmegaConf.merge(config, overlay)
    if overrides:
        config = OmegaConf.merge(config, OmegaConf.from_dotlist(overrides))
    OmegaConf.resolve(config)
    return config


def load_dataset_config() -> DictConfig:
    config = OmegaConf.load(CONFIGS_DIR / "dataset_config.yaml")
    overlay = _smoke_overlay("dataset")
    if overlay is not None:
        config = OmegaConf.merge(config, overlay)
    OmegaConf.resolve(config)
    return config


def ensure_dirs(config: DictConfig) -> None:
    """Create the directories under config.paths that scripts write into (idempotent)."""
    for key in (
        "raw_dir",
        "processed_dir",
        "images_dir",
        "splits_dir",
        "captions_dir",
        "checkpoints_dir",
        "logs_dir",
        "outputs_dir",
        "hf_cache_dir",
    ):
        Path(config.paths[key]).mkdir(parents=True, exist_ok=True)


## `scripts/utils/experiment_registry.py`


In [ ]:
# SOURCE FILE: scripts/utils/experiment_registry.py
"""Global, cross-stage experiment registry: one parquet row per run, so every Stage 1-5 run is
queryable in a single place (outputs/experiments/experiment_registry.parquet) and nothing
overwrites a previous run's record.

Generalizes the per-artifact manifest provenance already produced by every stage (see
scripts/utils/manifest.py: make_run_id, build_checkpoint_metadata) into something queryable
*across* runs, rather than only inspectable one manifest file at a time. `parent_experiment_id`
lets a Stage 4 classifier run point at the Stage 3 selection run it consumed, which points at the
Stage 2 generation run, which points at the Stage 1 checkpoint — the full provenance chain as one
table.

Concurrency note: appends/updates are implemented as read-existing -> modify -> atomic replace
(temp file + os.replace, matching the write_json idiom in manifest.py). This is correct and
sufficient for this project's actual execution model (a single researcher, a single GPU,
sequential runs) and deliberately does not add a locking layer or a real database, which would be
over-engineering for a use case with no concurrent writers.
"""

from __future__ import annotations

import json
import os
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from scripts.utils.manifest import get_git_commit_hash, hash_dict, make_run_id

DEFAULT_REGISTRY_RELATIVE_PATH = Path("outputs") / "experiments" / "experiment_registry.parquet"

REGISTRY_COLUMNS = [
    "experiment_id",
    "parent_experiment_id",
    "timestamp",
    "stage",
    "git_sha",
    "config_hash",
    "dataset_version",
    "checkpoint_path",
    "execution_time_seconds",
    "metrics",
    "status",
]


def resolve_registry_path(project_root: str | Path | None = None) -> Path:
    """Default registry location: <PROJECT_ROOT>/outputs/experiments/experiment_registry.parquet."""
    root = Path(project_root) if project_root is not None else Path(os.environ.get("PROJECT_ROOT", "."))
    return root / DEFAULT_REGISTRY_RELATIVE_PATH


def _atomic_write_parquet(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        df.to_parquet(temporary, index=False)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _append_row(registry_path: Path, row: dict[str, Any]) -> None:
    if registry_path.exists():
        existing = pd.read_parquet(registry_path)
        updated = pd.concat([existing, pd.DataFrame([row], columns=REGISTRY_COLUMNS)], ignore_index=True)
    else:
        updated = pd.DataFrame([row], columns=REGISTRY_COLUMNS)
    _atomic_write_parquet(registry_path, updated)


def _update_row(registry_path: Path, experiment_id: str, updates: dict[str, Any]) -> None:
    if not registry_path.exists():
        raise FileNotFoundError(f"Registry not found: {registry_path}")
    df = pd.read_parquet(registry_path)
    mask = df["experiment_id"] == experiment_id
    if not mask.any():
        raise KeyError(f"experiment_id {experiment_id!r} not found in {registry_path}")
    for key, value in updates.items():
        df.loc[mask, key] = json.dumps(value, sort_keys=True, default=str) if isinstance(value, (dict, list)) else value
    _atomic_write_parquet(registry_path, df)


def read_registry(registry_path: str | Path | None = None) -> pd.DataFrame:
    path = Path(registry_path) if registry_path is not None else resolve_registry_path()
    if not path.exists():
        return pd.DataFrame(columns=REGISTRY_COLUMNS)
    return pd.read_parquet(path)


class ExperimentRun:
    """Context manager registering one experiment-registry row per run.

    Usage:
        with ExperimentRun(stage="stage2_generation", config=cfg_dict,
                            parent_experiment_id=stage1_checkpoint_run_id) as run:
            ... do the stage's work ...
            run.set_checkpoint_path(output_dir)
            run.set_metrics({"num_images": 5000})

    On normal exit, status is recorded as "completed"; on any exception, "failed" (the exception
    still propagates — this context manager never swallows errors).
    """

    def __init__(
        self,
        stage: str,
        config: dict[str, Any],
        parent_experiment_id: str | None = None,
        dataset_version: str | None = None,
        experiment_id: str | None = None,
        registry_path: str | Path | None = None,
        repo_dir: str | Path = ".",
    ) -> None:
        self.stage = stage
        self.config = config
        self.parent_experiment_id = parent_experiment_id
        self.dataset_version = dataset_version
        self.experiment_id = experiment_id or make_run_id(config, repo_dir=repo_dir)
        self.registry_path = Path(registry_path) if registry_path is not None else resolve_registry_path()
        self.repo_dir = repo_dir
        self.checkpoint_path: str | None = None
        self.metrics: dict[str, Any] = {}
        self.status = "running"
        self._start_time: float | None = None

    def set_checkpoint_path(self, path: str | Path) -> None:
        self.checkpoint_path = str(path)

    def set_metrics(self, metrics: dict[str, Any]) -> None:
        self.metrics.update(metrics)

    def __enter__(self) -> "ExperimentRun":
        self._start_time = time.monotonic()
        row = {
            "experiment_id": self.experiment_id,
            "parent_experiment_id": self.parent_experiment_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "stage": self.stage,
            "git_sha": get_git_commit_hash(self.repo_dir),
            "config_hash": hash_dict(self.config),
            "dataset_version": self.dataset_version,
            "checkpoint_path": None,
            "execution_time_seconds": None,
            "metrics": "{}",
            "status": self.status,
        }
        _append_row(self.registry_path, row)
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        elapsed = time.monotonic() - self._start_time if self._start_time is not None else None
        self.status = "failed" if exc_type is not None else "completed"
        _update_row(
            self.registry_path,
            self.experiment_id,
            {
                "checkpoint_path": self.checkpoint_path,
                "execution_time_seconds": elapsed,
                "metrics": self.metrics,
                "status": self.status,
            },
        )
        return False

## `scripts/utils/identifiers.py`


In [ ]:
# SOURCE FILE: scripts/utils/identifiers.py
"""Shared image-id derivation, used identically by preprocessing (03) and caption generation (04)
so the two stay joinable on image_id."""

from __future__ import annotations

import re


def sanitize_image_id(raw_path: str) -> str:
    p = str(raw_path).replace("\\", "/")
    for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
        if p.startswith(prefix):
            p = p[len(prefix):]
    p = re.sub(r"\.(jpg|jpeg|png)$", "", p, flags=re.IGNORECASE)
    return p.replace("/", "__")


## `scripts/utils/labels.py`


In [ ]:
# SOURCE FILE: scripts/utils/labels.py
"""Frozen CheXpert label policy, shared by every Stage 2-5 script
(docs/stages2_to_5_plan.md §5 and §6).

Three label sets, deliberately not interchangeable:

  CLASSIFIER_TARGET_LABELS (14)  - the classifier's output space. Everything is predicted and
                                   reported.
  PRIMARY_ENDPOINT_LABELS  (11)  - the disease labels the PRIMARY endpoint (macro-AUROC) is
                                   computed over. Excludes "No Finding" (an absence-of-disease
                                   meta-label, not a pathology) and "Support Devices" (a device
                                   label, and the highest-prevalence/easiest column in CheXpert —
                                   including it would flatter the macro-average without measuring
                                   diagnostic performance). Also excludes "Pleural Other"
                                   (docs/stages2_to_5_plan.md §5.2 revision note): on the full
                                   production cohort only ~100 patients dataset-wide carry a
                                   confident negative label for it, so no split-fraction choice can
                                   give every decision-bearing split the frozen ≥50-negative-patient
                                   support rule (§1.3) — unlike "Lung Lesion"/"Atelectasis", which
                                   were fixed by revising configs/splits.yaml fractions instead. All
                                   three remain secondary outcomes.
  GENERATION_TARGET_LABELS (12)  - PRIMARY_ENDPOINT_LABELS plus INSUFFICIENT_SUPPORT_LABELS
                                   (docs/stages2_to_5_plan.md §3 revision note). Real-data support
                                   scarcity is a property of what can be reliably EVALUATED on the
                                   real cohort; it is not a reason to also stop generating or
                                   training on synthetic examples of the condition. Stage 2 recipe
                                   eligibility (co-occurrence mining, quotas, captions) keys off
                                   this list, not PRIMARY_ENDPOINT_LABELS. ASISM's agreement
                                   scoring (§4.5) and Stage 4 condition D's marginal matching (§7)
                                   deliberately stay scoped to PRIMARY_ENDPOINT_LABELS only.

Uncertainty policy (plan §6): raw labels are preserved; -1 and blank are MASKED out of loss and
metrics, never silently mapped to 0 or 1. Three quantities are always kept separate:
`raw_label`, `training_target`, and `loss_mask`/`eval_mask`.

`caption_builder.PATHOLOGY_COLUMNS` is the same 14 columns in the same canonical order; it is
imported here rather than re-typed so the two modules cannot drift.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from scripts.utils.caption_builder import (
    DEVICE_COLUMN,
    NO_FINDING_COLUMN,
    PATHOLOGY_COLUMNS,
)

# The classifier's full output space: all 14 CheXpert observations, canonical order.
CLASSIFIER_TARGET_LABELS: list[str] = list(PATHOLOGY_COLUMNS)

# Excluded from the primary endpoint due to insufficient patient-level negative support under the
# frozen §1.3 support rule (measured on the full production cohort: see module docstring). Reported
# as a secondary outcome with its support counts (plan §5.3), same treatment as No Finding /
# Support Devices.
INSUFFICIENT_SUPPORT_LABELS: list[str] = ["Pleural Other"]

# The 11 disease labels the primary endpoint is computed over (plan §5.2).
PRIMARY_ENDPOINT_LABELS: list[str] = [
    column
    for column in CLASSIFIER_TARGET_LABELS
    if column not in (NO_FINDING_COLUMN, DEVICE_COLUMN, *INSUFFICIENT_SUPPORT_LABELS)
]

# Reported, but never part of the primary macro-average.
SECONDARY_LABELS: list[str] = [NO_FINDING_COLUMN, DEVICE_COLUMN, *INSUFFICIENT_SUPPORT_LABELS]

# Every disease label worth intentionally synthesizing in Stage 2, whether or not it currently has
# enough REAL-data patient support to be scored as a primary endpoint (module docstring).
GENERATION_TARGET_LABELS: list[str] = [*PRIMARY_ENDPOINT_LABELS, *INSUFFICIENT_SUPPORT_LABELS]

# Sentinel used in integer label arrays for "uncertain or not mentioned" (i.e. masked).
MASKED = -1


def normalize_label(value) -> int | None:
    """Normalize one raw CheXpert cell to exactly one of {1, 0, -1, None}.

    None means blank/not-mentioned. -1 means the radiologist recorded uncertainty. Both are masked
    downstream, but they are distinct facts and are kept distinct in `raw_label`.
    """
    if value is None or value == "":
        return None
    try:
        numeric = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"invalid CheXpert label value {value!r}; expected -1, 0, 1, or blank") from exc
    if numeric != numeric:  # NaN
        return None
    if numeric not in (-1.0, 0.0, 1.0):
        raise ValueError(f"invalid CheXpert label value {value!r}; expected -1, 0, 1, or blank")
    return int(numeric)


def validate_label_domain(frame: pd.DataFrame, labels: list[str], identifier_column: str = "Path") -> None:
    """Validate labels with enough row context to diagnose a bad source cohort."""
    for label in labels:
        if label not in frame.columns:
            raise ValueError(f"missing required CheXpert label column {label!r}")
        for index, value in frame[label].items():
            try:
                normalize_label(value)
            except ValueError as exc:
                identifier = frame.at[index, identifier_column] if identifier_column in frame else index
                raise ValueError(f"label={label!r}, value={value!r}, row/image={identifier!r}: {exc}") from exc


def build_label_arrays(
    frame: pd.DataFrame,
    labels: list[str] | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (raw_label, training_target, mask) as three aligned (n_rows, n_labels) arrays.

    raw_label       int8, values in {1, 0, -1, -2}; -2 encodes blank/not-mentioned so the raw
                    array stays integer while remaining distinguishable from an explicit -1.
    training_target float32, values in {0.0, 1.0}; positions where mask is False are 0.0 and MUST
                    NOT be read (they are meaningless placeholders, not negatives).
    mask            bool, True only where the label is confidently 0 or 1.

    The mask is what enforces plan §6: -1 and blank contribute to neither loss nor metrics, and are
    never converted into a 0 or a 1 that the model would then be trained to reproduce.
    """
    labels = labels if labels is not None else CLASSIFIER_TARGET_LABELS
    n_rows, n_labels = len(frame), len(labels)

    raw = np.full((n_rows, n_labels), -2, dtype=np.int8)
    target = np.zeros((n_rows, n_labels), dtype=np.float32)
    mask = np.zeros((n_rows, n_labels), dtype=bool)

    for column_index, label in enumerate(labels):
        if label not in frame.columns:
            # Column entirely absent: leave as blank/masked rather than inventing negatives.
            continue
        values = [normalize_label(value) for value in frame[label].to_numpy()]
        for row_index, value in enumerate(values):
            if value is None:
                continue
            raw[row_index, column_index] = value
            if value in (0, 1):
                target[row_index, column_index] = float(value)
                mask[row_index, column_index] = True

    return raw, target, mask


def patient_level_support(
    frame: pd.DataFrame,
    labels: list[str] | None = None,
    patient_column: str = "patient_id",
) -> pd.DataFrame:
    """Per-label positive/negative/masked PATIENT counts (plan §1.3 — patient counts, not image
    counts, determine primary support eligibility).

    A patient counts as positive for a label if ANY of their studies carries a confident 1, and as
    negative if they are not positive and any study carries a confident 0. Patients whose every
    study is uncertain/blank for that label count as masked-only and support neither side.

    Returns one row per label with columns:
    label, positive_patients, negative_patients, masked_only_patients, positive_images,
    negative_images, total_patients.
    """
    labels = labels if labels is not None else PRIMARY_ENDPOINT_LABELS
    if patient_column not in frame.columns:
        raise KeyError(
            f"patient_level_support requires a {patient_column!r} column; got {list(frame.columns)[:8]}..."
        )

    records = []
    total_patients = frame[patient_column].nunique()

    for label in labels:
        if label not in frame.columns:
            records.append(
                {
                    "label": label,
                    "positive_patients": 0,
                    "negative_patients": 0,
                    "masked_only_patients": total_patients,
                    "positive_images": 0,
                    "negative_images": 0,
                    "total_patients": total_patients,
                }
            )
            continue

        normalized = frame[label].map(normalize_label)
        is_positive = normalized == 1
        is_negative = normalized == 0

        positive_patients = set(frame.loc[is_positive, patient_column].unique())
        negative_candidates = set(frame.loc[is_negative, patient_column].unique())
        # "Any positive" wins: a patient with one positive study is a positive patient.
        negative_patients = negative_candidates - positive_patients
        masked_only = total_patients - len(positive_patients) - len(negative_patients)

        records.append(
            {
                "label": label,
                "positive_patients": len(positive_patients),
                "negative_patients": len(negative_patients),
                "masked_only_patients": masked_only,
                "positive_images": int(is_positive.sum()),
                "negative_images": int(is_negative.sum()),
                "total_patients": total_patients,
            }
        )

    return pd.DataFrame.from_records(records)


def check_support_rule(
    support: pd.DataFrame,
    min_positive_patients: int,
    min_negative_patients: int,
) -> tuple[bool, list[dict]]:
    """Apply the frozen §1.3 support rule to a `patient_level_support` frame.

    Returns (passed, failures), where each failure records the label and both observed counts so
    the caller can emit a complete report rather than a first-failure message.
    """
    failures = []
    for row in support.to_dict("records"):
        positive_ok = row["positive_patients"] >= min_positive_patients
        negative_ok = row["negative_patients"] >= min_negative_patients
        if not (positive_ok and negative_ok):
            failures.append(
                {
                    "label": row["label"],
                    "positive_patients": row["positive_patients"],
                    "negative_patients": row["negative_patients"],
                    "min_positive_patients": min_positive_patients,
                    "min_negative_patients": min_negative_patients,
                    "positive_ok": positive_ok,
                    "negative_ok": negative_ok,
                }
            )
    return (len(failures) == 0), failures


def intended_vector_to_labels(intended: dict[str, int]) -> list[str]:
    """The primary disease labels a Stage 2 recipe intends to be positive.

    A `No Finding` recipe is the all-zero intended vector over the 11 primary labels (plan §3), so
    this correctly returns an empty list for it — agreement (§4.5) then scores it as "all 12
    predicted probabilities should be low."
    """
    return [label for label in PRIMARY_ENDPOINT_LABELS if int(intended.get(label, 0)) == 1]


## `scripts/utils/manifest.py`


In [ ]:
# SOURCE FILE: scripts/utils/manifest.py
"""Provenance helpers: run_id composition, git/config hashing, JSON manifest read/write.

Used throughout Stage 1 (dataset splits, captions, checkpoints) so every artifact records enough
to be traced back to the exact code + config + data that produced it (docs/stage1_plan.md:
reproducibility, checkpoint metadata, handoff to Stage 2).
"""

from __future__ import annotations

import hashlib
import json
import os
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def get_git_commit_hash(repo_dir: str | Path = ".", short: bool = True) -> str:
    try:
        args = ["git", "-C", str(repo_dir), "rev-parse"]
        args.append("--short" if short else "HEAD")
        if short:
            args.append("HEAD")
        result = subprocess.run(args, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception:
        return "nogit"


def hash_dict(data: dict[str, Any], length: int = 8) -> str:
    """Stable short hash of a (JSON-serializable) config dict, for use in run_ids and provenance."""
    serialized = json.dumps(data, sort_keys=True, default=str)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()[:length]


def make_run_id(config: dict[str, Any], repo_dir: str | Path = ".") -> str:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    git_sha = get_git_commit_hash(repo_dir)
    config_hash = hash_dict(config)
    return f"{timestamp}_{git_sha}_{config_hash}"


def get_library_versions() -> dict[str, str]:
    versions: dict[str, str] = {}
    for module_name in (
        "torch",
        "diffusers",
        "transformers",
        "accelerate",
        "peft",
        "bitsandbytes",
    ):
        try:
            module = __import__(module_name)
            versions[module_name] = getattr(module, "__version__", "unknown")
        except ImportError:
            versions[module_name] = "not_installed"
    return versions


def code_identity(repo_dir: str | Path = ".") -> dict[str, Any]:
    """Hash the executable/config source, including uncommitted and untracked files.

    A Git SHA alone is insufficient in a dirty research worktree. Data and generated artifacts are
    intentionally excluded; their identities are recorded by their own manifests.
    """
    root = Path(repo_dir).resolve()
    included_roots = ("scripts", "configs", "environment")
    files = []
    for relative_root in included_roots:
        directory = root / relative_root
        if not directory.exists():
            continue
        for path in sorted(p for p in directory.rglob("*") if p.is_file() and "__pycache__" not in p.parts):
            files.append({"path": path.relative_to(root).as_posix(), "sha256": sha256_file(path)})
    digest = hashlib.sha256(json.dumps(files, sort_keys=True).encode("utf-8")).hexdigest()
    return {"git_commit": get_git_commit_hash(root, short=False), "source_tree_sha256": digest, "files": files}


def sha256_directory(path: str | Path) -> str:
    root = Path(path)
    if not root.is_dir():
        raise FileNotFoundError(root)
    entries = [
        {"path": file.relative_to(root).as_posix(), "sha256": sha256_file(file)}
        for file in sorted(p for p in root.rglob("*") if p.is_file())
    ]
    return hashlib.sha256(json.dumps(entries, sort_keys=True).encode("utf-8")).hexdigest()


def sha256_file(path: str | Path) -> str:
    """Stream-hash a file's contents (used for provenance of split/checkpoint/manifest inputs)."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: str | Path, data: dict[str, Any]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, sort_keys=True, default=str)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temporary_name, path)
    finally:
        Path(temporary_name).unlink(missing_ok=True)


def write_frozen_json(path: str | Path, data: dict[str, Any]) -> None:
    """Create a freeze artifact exactly once; identical or changed overwrites are refused."""
    path = Path(path)
    if path.exists():
        raise FileExistsError(
            f"Frozen artifact already exists and is immutable: {path}. "
            "Use a new explicit protocol/run version and record its parent."
        )
    payload = dict(data)
    payload["frozen"] = True
    write_json(path, payload)


def read_json(path: str | Path) -> dict[str, Any]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def build_checkpoint_metadata(
    step: int,
    epoch: int,
    config: dict[str, Any],
    split_manifest_hash: str,
    seed: int,
    repo_dir: str | Path = ".",
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Metadata recorded alongside every checkpoint (docs/stage1_plan.md §10)."""
    metadata = {
        "step": step,
        "epoch": epoch,
        "wall_clock_utc": datetime.now(timezone.utc).isoformat(),
        "git_commit_hash": get_git_commit_hash(repo_dir),
        "config": config,
        "config_hash": hash_dict(config),
        "split_manifest_hash": split_manifest_hash,
        "seed": seed,
        "library_versions": get_library_versions(),
    }
    if extra:
        metadata.update(extra)
    return metadata


## `scripts/utils/metrics.py`


In [ ]:
# SOURCE FILE: scripts/utils/metrics.py
"""Masked metrics and patient-level inference for Stage 5 (docs/stages2_to_5_plan.md §5, §6, §8).

Every metric here is MASKED: uncertain (-1) and blank labels are excluded rather than mapped to 0
or 1 (§6), and every result carries its effective N so a metric computed over 40 usable patients is
never silently compared against one computed over 400.

Resampling is PATIENT-level (§8). Bootstrapping images would let a patient with many studies
dominate a confidence interval purely by study count, which understates the true uncertainty.

Pure NumPy — no torch, no GPU. This module is fully testable and runnable locally, which is what
makes the whole Stage 5 analysis layer developable off the pod.
"""

from __future__ import annotations

import numpy as np

from scripts.utils.labels import (
    CLASSIFIER_TARGET_LABELS,
    PRIMARY_ENDPOINT_LABELS,
)


def auroc(scores: np.ndarray, labels: np.ndarray) -> float:
    """AUROC via the rank (Mann-Whitney U) identity, with correct mid-rank tie handling.

    Returns NaN when either class is absent — an undefined AUROC, not a silent 0.5, so it can be
    excluded from a macro-average rather than dragging it toward chance.
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    n_positive = int((labels == 1).sum())
    n_negative = int((labels == 0).sum())
    if n_positive == 0 or n_negative == 0:
        return float("nan")

    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=np.float64)
    sorted_scores = scores[order]

    index = 0
    while index < len(sorted_scores):
        end = index
        while end + 1 < len(sorted_scores) and sorted_scores[end + 1] == sorted_scores[index]:
            end += 1
        mid_rank = (index + end) / 2.0 + 1.0
        ranks[order[index : end + 1]] = mid_rank
        index = end + 1

    positive_rank_sum = ranks[labels == 1].sum()
    u_statistic = positive_rank_sum - n_positive * (n_positive + 1) / 2.0
    return float(u_statistic / (n_positive * n_negative))


def average_precision(scores: np.ndarray, labels: np.ndarray) -> float:
    """Area under the precision-recall curve (step interpolation, the standard AP definition)."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    if (labels == 1).sum() == 0:
        return float("nan")

    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = labels[order]
    true_positives = np.cumsum(sorted_labels == 1)
    predicted_positives = np.arange(1, len(sorted_labels) + 1)
    precision = true_positives / predicted_positives
    total_positives = (labels == 1).sum()
    return float((precision * (sorted_labels == 1)).sum() / total_positives)


def brier_score(scores: np.ndarray, labels: np.ndarray) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    if len(scores) == 0:
        return float("nan")
    return float(np.mean((scores - labels) ** 2))


def expected_calibration_error(scores: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """ECE with equal-width bins."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    if len(scores) == 0:
        return float("nan")

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    error = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        in_bin = (scores > lower) & (scores <= upper) if lower > 0 else (scores >= lower) & (scores <= upper)
        count = int(in_bin.sum())
        if count == 0:
            continue
        error += (count / len(scores)) * abs(labels[in_bin].mean() - scores[in_bin].mean())
    return float(error)


def sensitivity_specificity_f1(
    scores: np.ndarray, labels: np.ndarray, threshold: float
) -> tuple[float, float, float]:
    predicted = (np.asarray(scores) >= threshold).astype(np.int64)
    labels = np.asarray(labels, dtype=np.int64)

    true_positive = int(((predicted == 1) & (labels == 1)).sum())
    false_positive = int(((predicted == 1) & (labels == 0)).sum())
    true_negative = int(((predicted == 0) & (labels == 0)).sum())
    false_negative = int(((predicted == 0) & (labels == 1)).sum())

    sensitivity = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else float("nan")
    specificity = true_negative / (true_negative + false_positive) if (true_negative + false_positive) else float("nan")
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else float("nan")
    if np.isnan(precision) or np.isnan(sensitivity) or (precision + sensitivity) == 0:
        f1 = float("nan")
    else:
        f1 = 2 * precision * sensitivity / (precision + sensitivity)
    return float(sensitivity), float(specificity), float(f1)


def macro_auroc_masked(
    probabilities: np.ndarray,
    targets: np.ndarray,
    masks: np.ndarray,
    labels: list[str] | None = None,
    primary_labels: list[str] | None = None,
    min_positive: int = 1,
    min_negative: int = 1,
) -> dict:
    """Per-label and macro AUROC over the PRIMARY endpoint labels, masked.

    Labels failing the minimum-support rule are excluded from the macro-average by the
    pre-specified rule (§5.3) and reported with their support counts rather than an unstable value.
    """
    labels = labels or CLASSIFIER_TARGET_LABELS
    primary_labels = primary_labels or PRIMARY_ENDPOINT_LABELS
    index_of = {label: position for position, label in enumerate(labels)}

    per_label = {}
    usable = []
    for label in primary_labels:
        if label not in index_of:
            continue
        column = index_of[label]
        mask = masks[:, column].astype(bool)
        y_true = targets[mask, column].astype(np.int64)
        y_score = probabilities[mask, column]

        n_positive = int((y_true == 1).sum())
        n_negative = int((y_true == 0).sum())
        eligible = n_positive >= min_positive and n_negative >= min_negative
        value = auroc(y_score, y_true) if eligible else float("nan")

        per_label[label] = {
            "auroc": value,
            "n_positive": n_positive,
            "n_negative": n_negative,
            "effective_n": int(mask.sum()),
            "eligible": bool(eligible and not np.isnan(value)),
        }
        if per_label[label]["eligible"]:
            usable.append(value)

    return {
        "macro_auroc": float(np.mean(usable)) if usable else float("nan"),
        "n_labels_in_macro": len(usable),
        "n_labels_excluded": len(primary_labels) - len(usable),
        "per_label": per_label,
    }


def full_metric_suite(
    probabilities: np.ndarray,
    targets: np.ndarray,
    masks: np.ndarray,
    thresholds: dict[str, float] | None = None,
    labels: list[str] | None = None,
    min_positive: int = 1,
    min_negative: int = 1,
) -> dict:
    """Every Stage 5 metric (§8) for one condition, per label, with effective N attached."""
    labels = labels or CLASSIFIER_TARGET_LABELS
    thresholds = thresholds or {}
    index_of = {label: position for position, label in enumerate(labels)}

    per_label = {}
    for label in labels:
        column = index_of[label]
        mask = masks[:, column].astype(bool)
        y_true = targets[mask, column].astype(np.int64)
        y_score = probabilities[mask, column]

        n_positive = int((y_true == 1).sum())
        n_negative = int((y_true == 0).sum())
        eligible = n_positive >= min_positive and n_negative >= min_negative
        threshold = float(thresholds.get(label, 0.5))
        sensitivity, specificity, f1 = (
            sensitivity_specificity_f1(y_score, y_true, threshold)
            if eligible
            else (float("nan"), float("nan"), float("nan"))
        )

        per_label[label] = {
            "auroc": auroc(y_score, y_true) if eligible else float("nan"),
            "auprc": average_precision(y_score, y_true) if eligible else float("nan"),
            "sensitivity": sensitivity,
            "specificity": specificity,
            "f1": f1,
            "brier": brier_score(y_score, y_true) if eligible else float("nan"),
            "ece": expected_calibration_error(y_score, y_true) if eligible else float("nan"),
            "threshold": threshold,
            "n_positive": n_positive,
            "n_negative": n_negative,
            "effective_n": int(mask.sum()),
            "eligible": bool(eligible),
        }

    def macro_over(metric: str, subset: list[str]) -> float:
        values = [
            per_label[label][metric]
            for label in subset
            if label in per_label and per_label[label]["eligible"]
            and not np.isnan(per_label[label][metric])
        ]
        return float(np.mean(values)) if values else float("nan")

    primary = [label for label in PRIMARY_ENDPOINT_LABELS if label in per_label]
    secondary = [label for label in labels if label not in PRIMARY_ENDPOINT_LABELS]

    # Micro-AUROC pools all confidently-labelled (row, primary-label) pairs into one ranking.
    micro_scores, micro_targets = [], []
    for label in primary:
        column = index_of[label]
        mask = masks[:, column].astype(bool)
        micro_scores.append(probabilities[mask, column])
        micro_targets.append(targets[mask, column].astype(np.int64))

    return {
        "macro_auroc_primary": macro_over("auroc", primary),
        "macro_auprc_primary": macro_over("auprc", primary),
        "micro_auroc_primary": (
            auroc(np.concatenate(micro_scores), np.concatenate(micro_targets))
            if micro_scores else float("nan")
        ),
        "macro_f1_primary": macro_over("f1", primary),
        "mean_brier_primary": macro_over("brier", primary),
        "mean_ece_primary": macro_over("ece", primary),
        "primary_labels": primary,
        "secondary_labels": secondary,
        "per_label": per_label,
    }


def patient_level_bootstrap(
    patient_ids: np.ndarray,
    metric_fn,
    n_resamples: int = 1000,
    seed: int = 42,
    alpha: float = 0.05,
) -> dict:
    """Patient-level bootstrap CI (§8).

    Resamples PATIENTS with replacement and re-evaluates on all rows belonging to the drawn
    patients, so a patient's whole set of studies moves in or out together — the correct unit,
    since studies from one patient are not independent observations.
    """
    rng = np.random.default_rng(seed)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    rows_by_patient = {patient: np.where(patient_ids == patient)[0] for patient in unique_patients}

    point_estimate = metric_fn(np.arange(len(patient_ids)))

    samples = []
    for _ in range(n_resamples):
        drawn = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        indices = np.concatenate([rows_by_patient[patient] for patient in drawn])
        value = metric_fn(indices)
        if value is not None and not np.isnan(value):
            samples.append(value)

    if not samples:
        return {
            "point_estimate": point_estimate,
            "ci_lower": float("nan"),
            "ci_upper": float("nan"),
            "n_resamples_valid": 0,
            "n_patients": len(unique_patients),
        }

    samples = np.array(samples)
    return {
        "point_estimate": float(point_estimate),
        "ci_lower": float(np.percentile(samples, 100 * alpha / 2)),
        "ci_upper": float(np.percentile(samples, 100 * (1 - alpha / 2))),
        "bootstrap_mean": float(samples.mean()),
        "bootstrap_std": float(samples.std(ddof=1)) if len(samples) > 1 else float("nan"),
        "n_resamples_valid": len(samples),
        "n_patients": len(unique_patients),
    }


def paired_bootstrap_difference(
    patient_ids: np.ndarray,
    metric_fn_a,
    metric_fn_b,
    n_resamples: int = 1000,
    seed: int = 42,
    alpha: float = 0.05,
) -> dict:
    """Paired patient-level bootstrap of (A - B), with an effect size and a two-sided p-value.

    Paired: both conditions are evaluated on the SAME resampled patients each iteration, which
    removes patient-composition variance from the comparison and is why this is more sensitive than
    comparing two independent CIs.

    The p-value is the standard bootstrap proportion of resamples whose difference crosses zero.
    It is always reported WITH the effect size and CI (§8.2) — significance alone is not a result.
    """
    rng = np.random.default_rng(seed)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    rows_by_patient = {patient: np.where(patient_ids == patient)[0] for patient in unique_patients}

    all_rows = np.arange(len(patient_ids))
    observed = metric_fn_a(all_rows) - metric_fn_b(all_rows)

    differences = []
    for _ in range(n_resamples):
        drawn = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        indices = np.concatenate([rows_by_patient[patient] for patient in drawn])
        value_a, value_b = metric_fn_a(indices), metric_fn_b(indices)
        if not (np.isnan(value_a) or np.isnan(value_b)):
            differences.append(value_a - value_b)

    if not differences:
        return {
            "observed_difference": float(observed),
            "ci_lower": float("nan"),
            "ci_upper": float("nan"),
            "p_value": float("nan"),
            "n_resamples_valid": 0,
        }

    differences = np.array(differences)
    proportion_crossing = float(np.mean(differences <= 0)) if observed > 0 else float(np.mean(differences >= 0))
    p_value = float(min(1.0, 2 * proportion_crossing))

    return {
        "observed_difference": float(observed),
        "effect_size": float(observed),
        "ci_lower": float(np.percentile(differences, 100 * alpha / 2)),
        "ci_upper": float(np.percentile(differences, 100 * (1 - alpha / 2))),
        "bootstrap_mean_difference": float(differences.mean()),
        "p_value": p_value,
        "n_resamples_valid": len(differences),
        "n_patients": len(unique_patients),
    }


def holm_bonferroni(p_values: dict[str, float], alpha: float = 0.05) -> dict[str, dict]:
    """Holm-Bonferroni step-down correction for the CONFIRMATORY family (§8.2).

    Uniformly more powerful than plain Bonferroni at the same family-wise error rate, which matters
    when the confirmatory family is small and each test is expensive to obtain.
    """
    valid = {name: value for name, value in p_values.items() if not np.isnan(value)}
    ordered = sorted(valid.items(), key=lambda item: item[1])
    n_tests = len(ordered)

    results: dict[str, dict] = {}
    previous_adjusted = 0.0
    still_rejecting = True

    for rank, (name, p_value) in enumerate(ordered):
        adjusted = min(1.0, (n_tests - rank) * p_value)
        adjusted = max(adjusted, previous_adjusted)  # enforce monotonicity
        previous_adjusted = adjusted
        if adjusted > alpha:
            still_rejecting = False
        results[name] = {
            "p_value": p_value,
            "adjusted_p_value": adjusted,
            "rejected": bool(still_rejecting and adjusted <= alpha),
            "rank": rank + 1,
            "n_tests": n_tests,
            "method": "holm_bonferroni",
            "family": "confirmatory",
            "alpha": alpha,
        }

    for name, value in p_values.items():
        if name not in results:
            results[name] = {
                "p_value": value,
                "adjusted_p_value": float("nan"),
                "rejected": False,
                "method": "holm_bonferroni",
                "family": "confirmatory",
                "note": "undefined p-value (insufficient support)",
            }
    return results


def benjamini_hochberg(p_values: dict[str, float], alpha: float = 0.05) -> dict[str, dict]:
    """Benjamini-Hochberg FDR control for EXPLORATORY analyses (§8.2).

    Exploratory results corrected this way must be labelled exploratory wherever reported — FDR
    controls the expected false-discovery proportion, not the family-wise error rate, and is not
    interchangeable with the confirmatory correction.
    """
    valid = {name: value for name, value in p_values.items() if not np.isnan(value)}
    ordered = sorted(valid.items(), key=lambda item: item[1])
    n_tests = len(ordered)

    adjusted_values = []
    for rank, (_, p_value) in enumerate(ordered, start=1):
        adjusted_values.append(min(1.0, p_value * n_tests / rank))

    # Enforce monotonicity from the largest p-value downward.
    for index in range(len(adjusted_values) - 2, -1, -1):
        adjusted_values[index] = min(adjusted_values[index], adjusted_values[index + 1])

    results: dict[str, dict] = {}
    for (name, p_value), adjusted in zip(ordered, adjusted_values):
        results[name] = {
            "p_value": p_value,
            "adjusted_p_value": adjusted,
            "rejected": bool(adjusted <= alpha),
            "n_tests": n_tests,
            "method": "benjamini_hochberg_fdr",
            "family": "exploratory",
            "alpha": alpha,
        }

    for name, value in p_values.items():
        if name not in results:
            results[name] = {
                "p_value": value,
                "adjusted_p_value": float("nan"),
                "rejected": False,
                "method": "benjamini_hochberg_fdr",
                "family": "exploratory",
                "note": "undefined p-value (insufficient support)",
            }
    return results


## `scripts/utils/provenance_gate.py`


In [ ]:
# SOURCE FILE: scripts/utils/provenance_gate.py
"""Generic upstream-artifact provenance gate.

Generalizes the staleness-check idiom already duplicated across
scripts/data/03_preprocess_images.py (`check_manifest`) and scripts/train/train_lora_sdxl.py
(`validate_training_inputs`): compare an "expected" provenance dict (computed from the *current*
config/inputs) against what was actually recorded the last time an upstream artifact was
produced, and fail loudly on a mismatch rather than silently mixing incompatible outputs.

New Stage 2-5 scripts should call `validate_upstream_artifact()` instead of writing their own
version of this comparison, satisfying "each downstream stage must validate upstream artifacts
before execution" with one shared implementation. `03_preprocess_images.py`'s and
`train_lora_sdxl.py`'s existing bespoke checks are left as-is — they already work and carry extra
script-specific behavior (e.g. legacy-manifest adoption) that doesn't belong in a generic helper.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from scripts.utils.manifest import read_json


class ProvenanceMismatch(SystemExit):
    """Raised (as a SystemExit subclass) when an upstream artifact's recorded provenance doesn't
    match what the current run expects — surfaces as a clean CLI failure, not a traceback."""


def validate_upstream_artifact(
    expected: dict[str, Any],
    manifest_path: str | Path,
    stage_name: str,
    allow_missing: bool = True,
) -> None:
    """Compare `expected` (the current run's provenance for some upstream artifact) against what
    was actually recorded at `manifest_path`.

    No-ops if they match, or if the manifest doesn't exist yet and `allow_missing=True` (first
    run — nothing to compare against). Raises `ProvenanceMismatch` otherwise.
    """
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        if allow_missing:
            return
        raise ProvenanceMismatch(
            f"[{stage_name}] Required upstream manifest is missing: {manifest_path}. "
            "Run the upstream stage first."
        )

    actual = read_json(manifest_path)
    if actual == expected:
        return

    raise ProvenanceMismatch(
        f"[{stage_name}] Upstream artifact at {manifest_path} was produced with different "
        "settings than this run expects — refusing to mix incompatible provenance.\n"
        f"Recorded: {json.dumps(actual, indent=2, sort_keys=True, default=str)}\n"
        f"Expected: {json.dumps(expected, indent=2, sort_keys=True, default=str)}\n"
        "Re-run the upstream stage with matching settings, or update this stage's config to match."
    )

## `scripts/utils/seed.py`


In [ ]:
# SOURCE FILE: scripts/utils/seed.py
"""Deterministic seeding helper, shared by every Stage 1 script (docs/stage1_plan.md: reproducibility)."""

from __future__ import annotations

import os
import random


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass

    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


## `scripts/utils/splits.py`


In [ ]:
# SOURCE FILE: scripts/utils/splits.py
"""Split loading, namespace resolution, and the programmatic `final_eval_heldout` access guard
(docs/stages2_to_5_plan.md §1.4 and §1.6).

The guard exists because the honest version of the leakage rule is not "final_eval_heldout is never
touched" — it IS touched, exactly once, during split construction, to assign patients, assert
disjointness, run the support check, and hash the manifest. What must never happen is an
outcome-bearing read before Stage 5: loading its images, loading its labels to compute outcome
statistics, or letting any model/method decision depend on it.

So access is mediated by an explicit purpose token. Non-outcome-bearing purposes are always
allowed; outcome-bearing purposes require a Stage 5 final-evaluation run id, which only the Stage 5
entry point supplies. Any other caller asking for an outcome-bearing read fails loudly.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from omegaconf import DictConfig, OmegaConf

from scripts.utils.config import load_named_config
from scripts.utils.manifest import hash_dict, read_json, sha256_file, write_frozen_json
import random

SPLIT_NAMES = [
    "gen_train",
    "gen_val",
    "classifier_train",
    "classifier_val",
    "asism_tuning_heldout",
    "final_eval_heldout",
]

FINAL_EVAL_SPLIT = "final_eval_heldout"

SPLIT_MANIFEST_FILENAME = "split_manifest_v2.json"

# Purposes that do NOT read outcomes: allowed at any time, including before Stage 5.
NON_OUTCOME_PURPOSES = {
    "split_construction",    # deterministic assignment, disjointness/partition assertions
    "support_check",         # positive/negative patient counts (a property of the split, not a result)
    "schema_validation",     # column/dtype checks
    "existence_check",       # does the file exist
    "hash_verification",     # manifest hashing / hash comparison
}

# Purposes that DO read outcomes: require an explicit Stage 5 final-evaluation run id.
OUTCOME_PURPOSES = {
    "load_images",
    "load_labels_for_evaluation",
    "compute_outcome_statistics",
    "model_selection",
}


class FinalEvalAccessViolation(SystemExit):
    """Raised (as SystemExit, so it surfaces as a clean CLI failure) when code attempts an
    outcome-bearing read of final_eval_heldout outside a declared Stage 5 evaluation run."""


def assert_final_eval_access_allowed(
    purpose: str,
    final_eval_run_id: str | None = None,
    caller: str = "unknown",
) -> None:
    """Gate every access to final_eval_heldout behind an explicit declared purpose.

    Non-outcome purposes (§1.6's enumerated split-construction activities) pass unconditionally.
    Outcome purposes require `final_eval_run_id` — supplied only by the Stage 5 entry point.
    """
    if purpose in NON_OUTCOME_PURPOSES:
        return

    if purpose not in OUTCOME_PURPOSES:
        raise FinalEvalAccessViolation(
            f"[{caller}] Unrecognized final_eval_heldout access purpose {purpose!r}. "
            f"Declare one of: non-outcome {sorted(NON_OUTCOME_PURPOSES)} "
            f"or outcome-bearing {sorted(OUTCOME_PURPOSES)}."
        )

    if not final_eval_run_id:
        raise FinalEvalAccessViolation(
            f"[{caller}] Refusing outcome-bearing access to {FINAL_EVAL_SPLIT} "
            f"(purpose={purpose!r}) without an explicit final-evaluation run id.\n"
            "final_eval_heldout may only be read for outcomes inside a declared Stage 5 run "
            "(docs/stages2_to_5_plan.md §1.6). If this is Stage 5, pass --final-eval-run-id."
        )


def load_splits_config() -> DictConfig:
    return load_named_config("splits.yaml", "splits")


def resolve_splits_dir(namespace: str, config: DictConfig | None = None) -> Path:
    """Namespaced split directory: <splits_root>/<namespace>/ (plan §1.4).

    Keeping `dev` and `production` in separate directories is what makes it structurally impossible
    for a dev-subset split to overwrite a production split.
    """
    config = config if config is not None else load_splits_config()
    if not namespace or namespace in {".", ".."} or any(ch in namespace for ch in "/\\"):
        raise ValueError(f"Invalid explicit split namespace/run id {namespace!r}")
    return Path(config.paths.splits_root) / namespace


def split_manifest_path(namespace: str, config: DictConfig | None = None) -> Path:
    return resolve_splits_dir(namespace, config) / SPLIT_MANIFEST_FILENAME


def read_split_manifest(namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    path = split_manifest_path(namespace, config)
    if not path.is_file():
        raise SystemExit(
            f"Missing split manifest: {path}\n"
            f"Build it first: python scripts/data/02b_build_sixway_splits.py --namespace {namespace}"
        )
    manifest = read_json(path)
    expected = int((config or load_splits_config()).manifest_version)
    if manifest.get("manifest_version") != expected:
        raise SystemExit(f"Split manifest version mismatch at {path}: expected {expected}, got {manifest.get('manifest_version')!r}")
    if manifest.get("split_namespace") != namespace:
        raise SystemExit(f"Split namespace mismatch at {path}: expected {namespace!r}, got {manifest.get('split_namespace')!r}")
    if manifest.get("manifest_hash_scope") == "full_manifest_without_manifest_hash_v2":
        declared = manifest.get("manifest_hash")
        unhashed = {key: value for key, value in manifest.items() if key != "manifest_hash"}
        actual = hash_dict(unhashed, length=64)
        if declared != actual:
            raise SystemExit(f"Split manifest integrity mismatch at {path}: declared {declared}, computed {actual}")
    return manifest


def split_metadata(split_name: str, namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    """Return non-outcome metadata without parsing a label-bearing CSV."""
    manifest = read_split_manifest(namespace, config)
    path = resolve_splits_dir(namespace, config) / f"{split_name}.csv"
    if not path.is_file():
        raise SystemExit(f"Missing split file: {path}")
    outputs = manifest.get("output_files", {})
    declared = outputs.get(split_name, {})
    actual_hash = sha256_file(path)
    expected_hash = declared.get("sha256")
    if expected_hash and actual_hash != expected_hash:
        raise SystemExit(f"Split hash mismatch for {split_name}: expected {expected_hash}, got {actual_hash}")
    return {
        "exists": True, "path": str(path), "sha256": actual_hash,
        "row_count": declared.get("row_count", manifest.get("images_per_split", {}).get(split_name)),
        "schema_hash": manifest.get("source_schema_hash"),
        "manifest_hash": manifest.get("manifest_hash"), "namespace": namespace,
    }


def validate_final_eval_context(context_path: str | Path, run_id: str, namespace: str) -> dict[str, Any]:
    path = Path(context_path)
    if not path.is_file():
        raise FinalEvalAccessViolation(f"Final-evaluation context is not registered: {path}")
    record = read_json(path)
    if record.get("final_eval_run_id") != run_id or record.get("namespace") != namespace:
        raise FinalEvalAccessViolation("Final-evaluation context run ID/namespace mismatch")
    if record.get("status") not in {"registered", "outcome_accessed", "in_progress", "complete"}:
        raise FinalEvalAccessViolation(f"Final-evaluation context has invalid status {record.get('status')!r}")
    required = ("split_manifest_hash", "asism_manifest_hash", "protocol_manifest_hash", "threshold_policy_hash", "checkpoint_hashes", "code_git_version", "code_identity_sha256")
    missing = [key for key in required if not record.get(key)]
    if missing:
        raise FinalEvalAccessViolation(f"Final-evaluation context is incomplete: {missing}")
    return record


def load_split(
    split_name: str,
    namespace: str,
    config: DictConfig | None = None,
    purpose: str = "load_labels_for_evaluation",
    final_eval_run_id: str | None = None,
    final_eval_context_path: str | Path | None = None,
    caller: str = "unknown",
) -> pd.DataFrame:
    """Load one split CSV, enforcing the final_eval_heldout access rule.

    `purpose` is only consulted for final_eval_heldout; the other five splits are ordinary
    development data with no such restriction.
    """
    if split_name not in SPLIT_NAMES:
        raise ValueError(f"Unknown split {split_name!r}; expected one of {SPLIT_NAMES}")

    if split_name == FINAL_EVAL_SPLIT:
        assert_final_eval_access_allowed(
            purpose=purpose, final_eval_run_id=final_eval_run_id, caller=caller
        )
        if purpose in NON_OUTCOME_PURPOSES:
            raise FinalEvalAccessViolation(
                "Metadata-only final-evaluation checks must use split_metadata(); load_split() "
                "never returns final_eval_heldout rows before a validated evaluation context"
            )
        if purpose in OUTCOME_PURPOSES:
            if not final_eval_context_path or not final_eval_run_id:
                raise FinalEvalAccessViolation("Outcome-bearing final data access requires a validated registered context")
            validate_final_eval_context(final_eval_context_path, final_eval_run_id, namespace)

    path = resolve_splits_dir(namespace, config) / f"{split_name}.csv"
    if not path.is_file():
        raise SystemExit(
            f"Missing split file: {path}\n"
            f"Build it first: python scripts/data/02b_build_sixway_splits.py --namespace {namespace}"
        )
    return pd.read_csv(path)


def split_provenance(namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    """The provenance block every downstream stage records and validates against, so a stage can
    never silently consume outputs built from a different split partition.

    Deliberately small: the manifest hash pins the entire partition (seed, fractions, patient
    assignment) without every downstream manifest having to embed all of it.
    """
    manifest = read_split_manifest(namespace, config)
    return {
        "split_namespace": namespace,
        "split_manifest_version": manifest.get("manifest_version"),
        "split_manifest_hash": manifest.get("manifest_hash"),
        "split_seed": manifest.get("split_seed"),
        "frozen": manifest.get("frozen", False),
    }


def build_patient_folds(patient_ids: list[str], fold_count: int, seed: int) -> dict[int, set[str]]:
    """Deterministic patient-level validation folds; disjoint with an exact union."""
    if fold_count < 2:
        raise ValueError("fold_count must be at least 2")
    ordered = sorted(set(map(str, patient_ids)))
    random.Random(seed).shuffle(ordered)
    folds = {index: set() for index in range(fold_count)}
    for index, patient_id in enumerate(ordered):
        folds[index % fold_count].add(patient_id)
    union = set().union(*folds.values())
    if union != set(ordered) or sum(len(v) for v in folds.values()) != len(union):
        raise AssertionError("fold construction is not an exact disjoint patient partition")
    return folds


def freeze_patient_folds(path: str | Path, patient_ids: list[str], fold_count: int, seed: int, split_manifest_hash: str) -> dict[str, Any]:
    folds = build_patient_folds(patient_ids, fold_count, seed)
    core = {
        "schema_version": 1, "assignment_unit": "patient_id", "fold_count": fold_count,
        "assignment_seed": seed, "split_manifest_hash": split_manifest_hash,
        "pool_patient_set_hash": hash_dict({"patient_ids": sorted(set(map(str, patient_ids)))}, length=64),
        "folds": {str(i): {"patient_ids": sorted(ids), "patient_set_hash": hash_dict({"patient_ids": sorted(ids)}, length=64)} for i, ids in folds.items()},
    }
    core["fold_manifest_hash"] = hash_dict(core, length=64)
    write_frozen_json(path, core)
    return {**core, "frozen": True}


def require_frozen_production_splits(config: DictConfig | None = None) -> dict[str, Any]:
    """Stage 5 precondition (plan §8): the production split manifest must exist AND be frozen."""
    manifest = read_split_manifest("production", config)
    if not manifest.get("frozen", False):
        raise SystemExit(
            "Production split manifest exists but is not frozen. "
            "Freeze it before any final evaluation: "
            "python scripts/data/02b_build_sixway_splits.py --namespace production --freeze"
        )
    if not manifest.get("support_check", {}).get("passed", False):
        raise SystemExit(
            "Production split manifest is frozen but its support check did not pass — "
            "final evaluation on splits that fail the frozen support rule is not permitted "
            "(docs/stages2_to_5_plan.md §1.3)."
        )
    return manifest


def require_frozen_production_split_run(namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    """Require production class, freeze state, and support for any versioned production run ID."""
    manifest = read_split_manifest(namespace, config)
    if manifest.get("namespace_class") != "production":
        raise SystemExit(f"Split run {namespace!r} is not a production namespace (namespace_class={manifest.get('namespace_class')!r})")
    if not manifest.get("frozen", False):
        raise SystemExit(f"Production split run {namespace!r} is not frozen")
    if not manifest.get("support_check", {}).get("passed", False):
        raise SystemExit(f"Production split run {namespace!r} failed the frozen patient-support rule")
    return manifest


# Data preparation


## `scripts/data/00_download_dataset.py`


In [ ]:
# SOURCE FILE: scripts/data/00_download_dataset.py
#!/usr/bin/env python3
"""Download CheXpert-v1.0-small via kagglehub and make it available at data/chexpert/raw/
(docs/stage1_plan.md §6 — this is the acquisition step feeding 01_verify_download.py).

- Downloads with `kagglehub.dataset_download("ashery/chexpert")` (kagglehub caches the extracted
  dataset outside the repo, typically under ~/.cache/kagglehub/).
- Locates the actual data root inside the download (the Kaggle archive may nest the CSVs/images
  under a "CheXpert-v1.0-small/" subfolder) and symlinks (falling back to copying, e.g. on Windows
  without Developer Mode / admin rights) each top-level entry into data/chexpert/raw/, so every
  other script in the pipeline can keep reading from that fixed path with no manual edits.
- Updates configs/dataset_config.yaml's `source.kaggle_dataset_slug` if it doesn't already match
  the slug used here.
- Always finishes by running scripts/data/01_verify_download.py and propagates its exit code.
- Idempotent: if data/chexpert/raw/ already has train.csv + valid.csv and they pass verification,
  the download is skipped entirely. Use --force to re-download and re-link anyway.

Usage:
    python scripts/data/00_download_dataset.py [--force] [--copy] [--sample-images N]
"""

from __future__ import annotations

import argparse
import re
import shutil
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import CONFIGS_DIR, ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402

VERIFY_SCRIPT = Path(__file__).resolve().parent / "01_verify_download.py"


def find_data_root(base: Path) -> Path:
    """Locate the directory that directly contains train.csv + a train/ subdir, searching a few
    likely nesting levels before falling back to a full recursive search."""
    candidates = [base, base / "CheXpert-v1.0-small", base / "CheXpert-v1.0"]
    for candidate in candidates:
        if (candidate / "train.csv").exists() and (candidate / "train").is_dir():
            return candidate

    for train_csv in base.rglob("train.csv"):
        if (train_csv.parent / "train").is_dir():
            return train_csv.parent

    raise FileNotFoundError(
        f"Could not locate a CheXpert 'train.csv' (with sibling 'train/' dir) anywhere under {base}"
    )


def link_or_copy(source: Path, target: Path, force_copy: bool) -> str:
    """Idempotent per-entry: never overwrites an existing target. Prefers a symlink (dataset stays
    out of the repo/persistent-volume duplication); falls back to copying if symlinking isn't
    permitted (e.g. Windows without Developer Mode / admin rights)."""
    if target.exists() or target.is_symlink():
        return "skipped (already present)"

    if not force_copy:
        try:
            target.symlink_to(source, target_is_directory=source.is_dir())
            return "symlinked"
        except (OSError, NotImplementedError) as e:
            print(
                f"  Symlink failed for {target.name} ({e}); falling back to copy. "
                "(On Windows, enable Developer Mode or run as Administrator to allow symlinks.)"
            )

    if source.is_dir():
        shutil.copytree(source, target)
    else:
        shutil.copy2(source, target)
    return "copied"


def populate_raw_dir(data_root: Path, raw_dir: Path, force_copy: bool) -> dict[str, str]:
    raw_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for entry in sorted(data_root.iterdir()):
        target = raw_dir / entry.name
        results[entry.name] = link_or_copy(entry, target, force_copy)
    return results


def patch_dataset_config_slug(slug: str) -> bool:
    """Update configs/dataset_config.yaml's kaggle_dataset_slug if it doesn't already match `slug`
    (idempotent text patch — preserves the rest of the file/comments, which OmegaConf round-tripping
    would not)."""
    config_path = CONFIGS_DIR / "dataset_config.yaml"
    text = config_path.read_text(encoding="utf-8")

    if f'kaggle_dataset_slug: "{slug}"' in text:
        return False

    pattern = re.compile(r'kaggle_dataset_slug:\s*".*?"')
    if not pattern.search(text):
        return False

    patched = pattern.sub(f'kaggle_dataset_slug: "{slug}"', text, count=1)
    config_path.write_text(patched, encoding="utf-8")
    return True


def run_verification(sample_images: int) -> int:
    result = subprocess.run([sys.executable, str(VERIFY_SCRIPT), "--sample-images", str(sample_images)])
    return result.returncode


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--force", action="store_true", help="Re-download and re-link even if raw_dir already looks valid")
    parser.add_argument("--copy", action="store_true", help="Copy instead of symlinking into data/chexpert/raw/")
    parser.add_argument("--sample-images", type=int, default=200, help="Passed through to 01_verify_download.py")
    args = parser.parse_args()

    stage1_cfg = load_stage1_config()
    ensure_dirs(stage1_cfg)
    raw_dir = Path(stage1_cfg.paths.raw_dir)

    if not args.force and (raw_dir / "train.csv").exists() and (raw_dir / "valid.csv").exists():
        print(f"Found existing train.csv/valid.csv in {raw_dir}; verifying before deciding whether to download...")
        if run_verification(args.sample_images) == 0:
            print("\nAlready downloaded and verified — skipping download (idempotent). Use --force to redo it anyway.")
            return 0
        print("\nExisting data did not pass verification; proceeding to (re)download.\n")

    try:
        import kagglehub
    except ImportError:
        print("kagglehub is not installed. Run: pip install kagglehub")
        return 1

    dataset_cfg = load_dataset_config()
    slug = dataset_cfg.source.kaggle_dataset_slug
    print(f"Downloading Kaggle dataset '{slug}' via kagglehub (this can take a while on first run)...")
    download_path = Path(kagglehub.dataset_download(slug))
    print(f"kagglehub cached the dataset at: {download_path}")

    data_root = find_data_root(download_path)
    print(f"Located dataset root: {data_root}")

    results = populate_raw_dir(data_root, raw_dir, force_copy=args.copy)
    for name, action in results.items():
        print(f"  {action}: {raw_dir / name}")

    if patch_dataset_config_slug(slug):
        print(f"Updated configs/dataset_config.yaml kaggle_dataset_slug to '{slug}'")

    print("\nRunning scripts/data/01_verify_download.py to confirm the download is usable...")
    rc = run_verification(args.sample_images)
    if rc != 0:
        print("\nDownload completed but verification FAILED — see errors above.")
        return rc

    print("\nOK: dataset downloaded, linked into data/chexpert/raw/, and verified.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/01_verify_download.py`


In [ ]:
# SOURCE FILE: scripts/data/01_verify_download.py
#!/usr/bin/env python3
"""Verify the extracted CheXpert-v1.0-small download before anything else runs.

Since the exact Kaggle mirror isn't pinned with a checksum (docs/stage1_plan.md §6), this checks
*structural* integrity instead: expected files, row counts, column schema, label-value domain,
a random image-openability spot-check, and a coarse label-prevalence sanity print for manual
comparison against published CheXpert statistics.

Usage:
    python scripts/data/01_verify_download.py [--sample-images N]

Exits non-zero (and prints every failure, not just the first) if a hard check fails.
"""

from __future__ import annotations

import argparse
import random
import sys
from pathlib import Path

import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_dataset_config, load_stage1_config  # noqa: E402

VALID_LABEL_VALUES = {-1.0, 0.0, 1.0}
ROW_COUNT_TOLERANCE = 0.01  # 1%


def check_files_exist(raw_dir: Path) -> tuple[bool, list[str]]:
    errors = []
    train_csv = raw_dir / "train.csv"
    valid_csv = raw_dir / "valid.csv"
    if not train_csv.exists():
        errors.append(f"Missing {train_csv}")
    if not valid_csv.exists():
        errors.append(f"Missing {valid_csv}")
    return (len(errors) == 0, errors)


def check_row_counts(df_train: pd.DataFrame, df_valid: pd.DataFrame, dataset_cfg) -> tuple[bool, list[str]]:
    errors = []
    expected_train = dataset_cfg.source.expected_train_rows
    expected_valid = dataset_cfg.source.expected_valid_rows

    train_diff = abs(len(df_train) - expected_train) / expected_train
    if train_diff > ROW_COUNT_TOLERANCE:
        errors.append(
            f"train.csv has {len(df_train)} rows, expected ~{expected_train} "
            f"(off by {train_diff:.1%}, tolerance {ROW_COUNT_TOLERANCE:.0%})"
        )

    if len(df_valid) != expected_valid:
        errors.append(f"valid.csv has {len(df_valid)} rows, expected exactly {expected_valid}")

    return (len(errors) == 0, errors)


def check_schema(df: pd.DataFrame, dataset_cfg, csv_name: str) -> tuple[bool, list[str]]:
    from scripts.utils.chexpert_schema import validate_chexpert_frame
    try:
        validate_chexpert_frame(df, dataset_cfg)
        return True, []
    except ValueError as exc:
        return False, [f"{csv_name}: {exc}"]
    """Legacy body retained below for source-history readability; shared validator returns above."""
    errors = []
    schema = dataset_cfg.schema
    required_columns = [
        schema.path_column,
        schema.sex_column,
        schema.age_column,
        schema.frontal_lateral_column,
    ] + list(schema.pathology_columns)

    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        errors.append(f"{csv_name}: missing expected columns: {missing}")

    for col in schema.pathology_columns:
        if col not in df.columns:
            continue
        observed = set(df[col].dropna().unique().tolist())
        invalid = {v for v in observed if v not in VALID_LABEL_VALUES}
        if invalid:
            errors.append(f"{csv_name}: column {col!r} has out-of-domain values: {invalid}")

    return (len(errors) == 0, errors)


def check_image_spotcheck(df: pd.DataFrame, raw_dir: Path, dataset_cfg, n: int, seed: int) -> tuple[bool, list[str]]:
    errors = []
    rng = random.Random(seed)
    path_col = dataset_cfg.schema.path_column
    n = min(n, len(df))
    sample_rows = df.sample(n=n, random_state=seed)

    for _, row in sample_rows.iterrows():
        raw_path = str(row[path_col]).replace("\\", "/")
        # Paths in train.csv are typically like "CheXpert-v1.0-small/train/patient.../view1_frontal.jpg"
        for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
            if raw_path.startswith(prefix):
                raw_path = raw_path[len(prefix):]
        image_path = raw_dir / raw_path
        if not image_path.exists():
            errors.append(f"Spot-check: image not found: {image_path}")
            continue
        try:
            with Image.open(image_path) as im:
                im.verify()
            with Image.open(image_path) as im:
                w, h = im.size
                if w < 50 or h < 50:
                    errors.append(f"Spot-check: implausibly small image {image_path} ({w}x{h})")
        except Exception as e:
            errors.append(f"Spot-check: failed to open {image_path}: {e}")

    return (len(errors) == 0, errors)


def print_label_prevalence(df: pd.DataFrame, dataset_cfg) -> None:
    print("\nLabel prevalence in train.csv (compare against published CheXpert statistics manually --")
    print("this script does not hard-fail on prevalence, since exact published rates aren't asserted here):")
    for col in dataset_cfg.schema.pathology_columns:
        if col not in df.columns:
            continue
        counts = df[col].value_counts(dropna=False)
        total = len(df)
        pos = counts.get(1.0, 0)
        neg = counts.get(0.0, 0)
        unc = counts.get(-1.0, 0)
        blank = total - pos - neg - unc
        print(
            f"  {col:<28} positive={pos / total:6.1%}  negative={neg / total:6.1%}  "
            f"uncertain={unc / total:6.1%}  blank={blank / total:6.1%}"
        )


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--sample-images", type=int, default=200, help="Number of images to spot-check")
    args = parser.parse_args()

    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    raw_dir = Path(stage1_cfg.paths.raw_dir)
    seed = stage1_cfg.run.seed

    all_errors: list[str] = []

    ok, errors = check_files_exist(raw_dir)
    all_errors += errors
    if not ok:
        print("FAILED: required files missing, cannot continue further checks.")
        for e in all_errors:
            print(f"  - {e}")
        return 1

    df_train = pd.read_csv(raw_dir / "train.csv")
    df_valid = pd.read_csv(raw_dir / "valid.csv")

    for check_fn, args_ in (
        (check_row_counts, (df_train, df_valid, dataset_cfg)),
        (check_schema, (df_train, dataset_cfg, "train.csv")),
        (check_schema, (df_valid, dataset_cfg, "valid.csv")),
        (check_image_spotcheck, (df_train, raw_dir, dataset_cfg, args.sample_images, seed)),
    ):
        _, errors = check_fn(*args_)
        all_errors += errors

    print_label_prevalence(df_train, dataset_cfg)

    print()
    if all_errors:
        print(f"FAILED: {len(all_errors)} check(s) did not pass:")
        for e in all_errors:
            print(f"  - {e}")
        return 1

    print("OK: all structural integrity checks passed.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/01b_build_dev_subset.py`


In [ ]:
# SOURCE FILE: scripts/data/01b_build_dev_subset.py
#!/usr/bin/env python3
"""Build a configurable, patient-safe development subset of CheXpert's train.csv.

Purpose: validate the entire pipeline quickly on ~10,000 images (default, never hardcoded —
see configs/stage1_lora_sdxl.yaml: dev_subset.target_size) before running it on the full
~224,316-image dataset. The full dataset remains the source of the thesis's final results;
this subset only exists for fast iteration (docs/proposal.md, README's development guidance).

Sampling is a deterministic, patient-level, rarity-first greedy quota heuristic: no patient is
ever split across the subset boundary (a patient's images are either all included or all
excluded), and rarer pathologies are given priority when choosing which patients to draw, so the
subset stays "reasonably representative" of the full label distribution without requiring a
heavier formal multi-label stratification algorithm (see docs/thesis plan discussion — this is a
dev-convenience feature, not the mechanism that produces final thesis results).

02_build_patient_splits.py is unaware this script exists: it just reads whatever CSV
`split.input_csv` points at and reruns its own unmodified 70/10/20 patient-level split on it.
Pointing `split.input_csv` back at the full train.csv (the checked-in default) requires no code
change anywhere in the pipeline.

Usage:
    python scripts/data/01b_build_dev_subset.py

No-ops (prints a message, exits 0) when configs/stage1_lora_sdxl.yaml: dev_subset.enabled=false.
"""

from __future__ import annotations

import importlib
import random
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.manifest import (  # noqa: E402
    get_git_commit_hash,
    hash_dict,
    sha256_file,
    write_json,
)

ALGORITHM_VERSION = 1


def compute_prevalence(df: pd.DataFrame, pathology_columns: list[str]) -> pd.Series:
    return (df[pathology_columns] == 1).mean()


def select_dev_subset_patients(
    df: pd.DataFrame,
    patient_column: str,
    pathology_columns: list[str],
    target_size: int,
    seed: int,
) -> set[str]:
    """Rarity-first greedy quota sampling. Returns the set of selected patient IDs.

    Sorts pathology columns ascending by full-dataset image-level prevalence (rarest first),
    and for each, greedily draws whole patients carrying that finding until its proportional
    quota of `target_size` is met (or the eligible pool is exhausted). Because CheXpert
    pathologies co-occur, later quotas are frequently partially satisfied "for free" by earlier
    draws. Any shortfall against `target_size` after processing all columns is filled by
    uniform random patient draws. All randomness comes from one seeded `random.Random` stream.
    """
    prevalence = compute_prevalence(df, pathology_columns)
    rarity_order = prevalence.sort_values(ascending=True).index.tolist()
    quotas = (prevalence * target_size).round().astype(int)

    positive_counts_per_patient = (df[pathology_columns] == 1).groupby(df[patient_column]).sum()
    patient_image_counts = df.groupby(patient_column).size()
    positive_patients_by_col = {
        col: sorted(positive_counts_per_patient.index[positive_counts_per_patient[col] > 0].tolist())
        for col in pathology_columns
    }
    all_patients = sorted(patient_image_counts.index.tolist())

    rng = random.Random(seed)
    selected: set[str] = set()
    running_count = pd.Series(0, index=pathology_columns, dtype="int64")

    for col in rarity_order:
        quota = int(quotas[col])
        if quota <= 0:
            continue
        candidates = [p for p in positive_patients_by_col[col] if p not in selected]
        rng.shuffle(candidates)
        for patient in candidates:
            if running_count[col] >= quota:
                break
            selected.add(patient)
            running_count = running_count.add(positive_counts_per_patient.loc[patient], fill_value=0)

    total_images = int(patient_image_counts[list(selected)].sum()) if selected else 0
    if total_images < target_size:
        remaining = [p for p in all_patients if p not in selected]
        rng.shuffle(remaining)
        for patient in remaining:
            if total_images >= target_size:
                break
            selected.add(patient)
            total_images += int(patient_image_counts[patient])

    return selected


def build_prevalence_report(
    df_full: pd.DataFrame,
    df_subset: pd.DataFrame,
    pathology_columns: list[str],
    tolerance_pct: float,
) -> dict:
    full_pos = compute_prevalence(df_full, pathology_columns)
    subset_pos = compute_prevalence(df_subset, pathology_columns)
    full_unc = (df_full[pathology_columns] == -1).mean()
    subset_unc = (df_subset[pathology_columns] == -1).mean()

    per_column = {}
    num_exceeding = 0
    for col in pathology_columns:
        deviation_pct = abs(float(subset_pos[col]) - float(full_pos[col])) * 100.0
        within = deviation_pct <= tolerance_pct
        if not within:
            num_exceeding += 1
        per_column[col] = {
            "full_dataset_positive_rate": float(full_pos[col]),
            "subset_positive_rate": float(subset_pos[col]),
            "absolute_deviation_pct": deviation_pct,
            "within_tolerance": within,
            "full_dataset_uncertain_rate": float(full_unc[col]),
            "subset_uncertain_rate": float(subset_unc[col]),
        }

    return {
        "tolerance_pct": tolerance_pct,
        "num_columns_exceeding_tolerance": num_exceeding,
        "all_within_tolerance": num_exceeding == 0,
        "columns": per_column,
    }


def main() -> int:
    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    dev_cfg = stage1_cfg.dev_subset

    if not bool(dev_cfg.enabled):
        print("dev_subset.enabled=false — 01b_build_dev_subset.py is a no-op. "
              "split.input_csv should point at the full train.csv.")
        return 0

    raw_dir = Path(stage1_cfg.paths.raw_dir)
    train_csv_path = raw_dir / "train.csv"
    if not train_csv_path.is_file():
        raise SystemExit(f"Missing {train_csv_path} (run 00_download_dataset.py / 01_verify_download.py first).")

    schema = dataset_cfg.schema
    pathology_columns = list(schema.pathology_columns)
    target_size = int(dev_cfg.target_size)
    if target_size <= 0:
        raise SystemExit(f"dev_subset.target_size must be positive, got {target_size}")

    df_full = pd.read_csv(train_csv_path)
    original_columns = list(df_full.columns)

    patient_splits_module = importlib.import_module("scripts.data.02_build_patient_splits")
    extract_patient_id = patient_splits_module.extract_patient_id

    df_full = df_full.copy()
    df_full["__patient_id"] = df_full[schema.path_column].apply(
        lambda p: extract_patient_id(p, schema.patient_id_regex)
    )

    selected_patients = select_dev_subset_patients(
        df_full,
        patient_column="__patient_id",
        pathology_columns=pathology_columns,
        target_size=target_size,
        seed=int(dev_cfg.seed),
    )

    df_subset = df_full[df_full["__patient_id"].isin(selected_patients)][original_columns].reset_index(drop=True)

    output_csv = Path(dev_cfg.output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df_subset.to_csv(output_csv, index=False)

    report = build_prevalence_report(
        df_full, df_subset, pathology_columns, float(dev_cfg.prevalence_tolerance_pct)
    )
    write_json(Path(dev_cfg.prevalence_report_path), report)

    manifest = {
        "algorithm_version": ALGORITHM_VERSION,
        "algorithm": "rarity_first_greedy_quota_patient_sampling",
        "target_size": target_size,
        "actual_num_images": len(df_subset),
        "actual_num_patients": len(selected_patients),
        "seed": int(dev_cfg.seed),
        "source_train_csv_path": str(train_csv_path),
        "source_train_csv_hash": sha256_file(train_csv_path),
        "source_num_images": len(df_full),
        "source_num_patients": int(df_full["__patient_id"].nunique()),
        "dev_subset_config_hash": hash_dict(
            {
                "target_size": target_size,
                "seed": int(dev_cfg.seed),
                "prevalence_tolerance_pct": float(dev_cfg.prevalence_tolerance_pct),
            }
        ),
        "git_commit_hash": get_git_commit_hash(),
        "all_within_tolerance": report["all_within_tolerance"],
    }
    write_json(Path(dev_cfg.manifest_path), manifest)

    print(f"Source: {len(df_full)} images / {df_full['__patient_id'].nunique()} patients")
    print(f"Dev subset: {len(df_subset)} images / {len(selected_patients)} patients (target {target_size})")
    if report["all_within_tolerance"]:
        print(f"Prevalence check: all {len(pathology_columns)} pathology columns within "
              f"{dev_cfg.prevalence_tolerance_pct}pp tolerance.")
    else:
        print(f"WARNING: {report['num_columns_exceeding_tolerance']} column(s) exceed "
              f"{dev_cfg.prevalence_tolerance_pct}pp prevalence-deviation tolerance — see "
              f"{dev_cfg.prevalence_report_path} (non-fatal).")
    print(f"Wrote {output_csv}, {dev_cfg.manifest_path}, {dev_cfg.prevalence_report_path}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## `scripts/data/02_build_patient_splits.py`


In [ ]:
# SOURCE FILE: scripts/data/02_build_patient_splits.py
#!/usr/bin/env python3
"""Patient-level train/val/heldout split for CheXpert (docs/stage1_plan.md §6 — the single most
consequence-laden decision in Stage 1).

Splits by patient ID, never by image ID, into three disjoint groups:
  - gen_train:          used for LoRA gradient updates.
  - gen_val:             held out from training, used for validation loss / qualitative monitoring.
  - classifier_heldout:  NEVER touched by Stage 1 at all — reserved so Stage 5's eventual
                          real-vs-synthetic comparison isn't contaminated by generator exposure.
                          Further sub-split (patient-level, separately seeded) into:
                            - asism_tuning_heldout: used only by Stage 3's ASISM weight/threshold
                              search, so that search never touches Stage 5's final report data.
                            - final_eval_heldout:   untouched until Stage 5's final comparison.

The official CheXpert valid.csv (234 curated images) is left untouched in raw/ and is not
consumed by this script at all, for the same reason.

Reads its input from `split.input_csv` (configs/stage1_lora_sdxl.yaml), which defaults to the
full CheXpert train.csv but can be pointed at the dev-subset CSV produced by
01b_build_dev_subset.py — a config-only switch, no code change required here.

Usage:
    python scripts/data/02_build_patient_splits.py
"""

from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.manifest import hash_dict, read_json, write_json  # noqa: E402


def extract_patient_id(path_value: str, patient_id_regex: str) -> str:
    match = re.search(patient_id_regex, str(path_value))
    if not match:
        raise ValueError(f"Could not extract patient id from path: {path_value!r}")
    return match.group(0)


def split_patients(
    patient_ids: list[str],
    gen_train_fraction: float,
    gen_val_fraction: float,
    seed: int,
) -> tuple[set[str], set[str], set[str]]:
    import random

    rng = random.Random(seed)
    shuffled = list(patient_ids)
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(round(n * gen_train_fraction))
    n_val = int(round(n * gen_val_fraction))

    train_ids = set(shuffled[:n_train])
    val_ids = set(shuffled[n_train : n_train + n_val])
    heldout_ids = set(shuffled[n_train + n_val :])

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(heldout_ids)
    assert val_ids.isdisjoint(heldout_ids)
    assert len(train_ids) + len(val_ids) + len(heldout_ids) == n

    return train_ids, val_ids, heldout_ids


def split_heldout_for_asism(
    heldout_patient_ids: list[str],
    asism_tuning_fraction: float,
    seed: int,
) -> tuple[set[str], set[str]]:
    """Sub-split classifier_heldout so Stage 3's ASISM weight/threshold search (tuned on
    asism_tuning_heldout) never touches the same data Stage 5 reports final numbers on
    (final_eval_heldout) — avoiding optimistic-bias leakage between tuning and evaluation."""
    import random

    rng = random.Random(seed)
    shuffled = list(heldout_patient_ids)
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_tuning = int(round(n * asism_tuning_fraction))

    tuning_ids = set(shuffled[:n_tuning])
    final_eval_ids = set(shuffled[n_tuning:])

    assert tuning_ids.isdisjoint(final_eval_ids)
    assert len(tuning_ids) + len(final_eval_ids) == n

    return tuning_ids, final_eval_ids


def main() -> int:
    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    ensure_dirs(stage1_cfg)

    splits_dir = Path(stage1_cfg.paths.splits_dir)
    schema = dataset_cfg.schema
    split_cfg = stage1_cfg.split

    input_csv_path = Path(split_cfg.input_csv)
    if not input_csv_path.is_file():
        raise SystemExit(f"Missing {input_csv_path} (run 00/01, and 01b if using the dev subset, first).")

    df = pd.read_csv(input_csv_path)
    df["patient_id"] = df[schema.path_column].apply(
        lambda p: extract_patient_id(p, schema.patient_id_regex)
    )

    unique_patients = sorted(df["patient_id"].unique().tolist())
    train_ids, val_ids, heldout_ids = split_patients(
        unique_patients,
        gen_train_fraction=split_cfg.gen_train_fraction,
        gen_val_fraction=split_cfg.gen_val_fraction,
        seed=split_cfg.split_seed,
    )
    asism_tuning_ids, final_eval_ids = split_heldout_for_asism(
        sorted(heldout_ids),
        asism_tuning_fraction=split_cfg.asism_tuning_fraction,
        seed=split_cfg.heldout_split_seed,
    )

    df_train = df[df["patient_id"].isin(train_ids)].reset_index(drop=True)
    df_val = df[df["patient_id"].isin(val_ids)].reset_index(drop=True)
    df_heldout = df[df["patient_id"].isin(heldout_ids)].reset_index(drop=True)
    df_asism_tuning = df[df["patient_id"].isin(asism_tuning_ids)].reset_index(drop=True)
    df_final_eval = df[df["patient_id"].isin(final_eval_ids)].reset_index(drop=True)

    # Belt-and-braces: verify no patient leaked across the image-level dataframes too.
    assert set(df_train["patient_id"]).isdisjoint(set(df_val["patient_id"]))
    assert set(df_train["patient_id"]).isdisjoint(set(df_heldout["patient_id"]))
    assert set(df_val["patient_id"]).isdisjoint(set(df_heldout["patient_id"]))
    assert set(df_asism_tuning["patient_id"]).isdisjoint(set(df_final_eval["patient_id"]))
    assert set(df_asism_tuning["patient_id"]) | set(df_final_eval["patient_id"]) == set(df_heldout["patient_id"])

    df_train.to_csv(splits_dir / "gen_train.csv", index=False)
    df_val.to_csv(splits_dir / "gen_val.csv", index=False)
    df_heldout.to_csv(splits_dir / "classifier_heldout.csv", index=False)
    df_asism_tuning.to_csv(splits_dir / "asism_tuning_heldout.csv", index=False)
    df_final_eval.to_csv(splits_dir / "final_eval_heldout.csv", index=False)

    dev_subset_manifest_hash = None
    dev_cfg = getattr(stage1_cfg, "dev_subset", None)
    if dev_cfg is not None and Path(dev_cfg.output_csv) == input_csv_path:
        dev_manifest_path = Path(dev_cfg.manifest_path)
        if dev_manifest_path.is_file():
            dev_subset_manifest_hash = hash_dict(read_json(dev_manifest_path))

    manifest = {
        "input_csv_path": str(input_csv_path),
        "dev_subset_manifest_hash": dev_subset_manifest_hash,
        "split_unit": split_cfg.split_unit,
        "split_seed": split_cfg.split_seed,
        "gen_train_fraction": split_cfg.gen_train_fraction,
        "gen_val_fraction": split_cfg.gen_val_fraction,
        "classifier_heldout_fraction": split_cfg.classifier_heldout_fraction,
        "asism_tuning_fraction": split_cfg.asism_tuning_fraction,
        "heldout_split_seed": split_cfg.heldout_split_seed,
        "num_patients_total": len(unique_patients),
        "num_patients_gen_train": len(train_ids),
        "num_patients_gen_val": len(val_ids),
        "num_patients_classifier_heldout": len(heldout_ids),
        "num_patients_asism_tuning_heldout": len(asism_tuning_ids),
        "num_patients_final_eval_heldout": len(final_eval_ids),
        "num_images_gen_train": len(df_train),
        "num_images_gen_val": len(df_val),
        "num_images_classifier_heldout": len(df_heldout),
        "num_images_asism_tuning_heldout": len(df_asism_tuning),
        "num_images_final_eval_heldout": len(df_final_eval),
        "official_valid_csv": "reserved untouched at data/chexpert/raw/valid.csv — not read by this script",
        "source_csv_hash": hash_dict({"n_rows": len(df), "n_patients": len(unique_patients)}),
    }
    write_json(splits_dir / "split_manifest.json", manifest)

    print(f"Input CSV: {input_csv_path}")
    print(f"Total patients: {len(unique_patients)}")
    print(f"  gen_train:             {len(train_ids):>6} patients, {len(df_train):>7} images")
    print(f"  gen_val:               {len(val_ids):>6} patients, {len(df_val):>7} images")
    print(f"  classifier_heldout:    {len(heldout_ids):>6} patients, {len(df_heldout):>7} images")
    print(f"    asism_tuning_heldout: {len(asism_tuning_ids):>6} patients, {len(df_asism_tuning):>7} images")
    print(f"    final_eval_heldout:   {len(final_eval_ids):>6} patients, {len(df_final_eval):>7} images")
    print(f"Wrote split CSVs and split_manifest.json to {splits_dir}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/02b_build_sixway_splits.py`


In [ ]:
# SOURCE FILE: scripts/data/02b_build_sixway_splits.py
#!/usr/bin/env python3
"""Build the six-way patient-level split (docs/stages2_to_5_plan.md §1) — FROZEN v1 policy.

Supersedes scripts/data/02_build_patient_splits.py's three-way split (which derived
asism_tuning_heldout/final_eval_heldout as children of classifier_heldout and left no population
for a classifier development split). That script and its artifacts are retained for provenance and
are never read by Stage 2-5 code.

Six DIRECT top-level partitions from one deterministic patient list and one recorded seed:
    gen_train 0.60 | gen_val 0.10 | classifier_train 0.15
    classifier_val 0.05 | asism_tuning_heldout 0.05 | final_eval_heldout 0.05

Namespaced (§1.4) so a dev-subset split can never overwrite a production split:
    production - requires the full-cohort integrity contract; these are the thesis splits.
    dev        - same policy on the development subset, for pipeline smoke testing; rare-label
                 support insufficiency is recorded in the manifest rather than being fatal.

final_eval_heldout IS accessed here (§1.6) — for deterministic assignment, disjointness assertions,
the support check, and hashing. Every such access goes through
scripts/utils/splits.assert_final_eval_access_allowed() with a non-outcome purpose, so the
distinction between construction-time access and outcome-bearing access is enforced in code, not
just documented.

Usage:
    python scripts/data/02b_build_sixway_splits.py --namespace dev
    python scripts/data/02b_build_sixway_splits.py --namespace production [--freeze]
"""

from __future__ import annotations

import argparse
import os
import random
import re
import shutil
import sys
import tempfile
import uuid
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.artifact_contracts import validate_namespace_run_id  # noqa: E402
from scripts.utils.chexpert_schema import validate_chexpert_frame  # noqa: E402
from scripts.utils.labels import (  # noqa: E402
    CLASSIFIER_TARGET_LABELS,
    PRIMARY_ENDPOINT_LABELS,
    check_support_rule,
    patient_level_support,
    normalize_label,
    validate_label_domain,
)
from scripts.utils.manifest import get_git_commit_hash, get_library_versions, hash_dict, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import (  # noqa: E402
    FINAL_EVAL_SPLIT,
    SPLIT_MANIFEST_FILENAME,
    SPLIT_NAMES,
    assert_final_eval_access_allowed,
    load_splits_config,
    resolve_splits_dir,
)

SUPERSEDED_NOTE_FILENAME = "SUPERSEDED_v1.md"

SUPERSEDED_NOTE = """# Superseded schema-v1 split artifacts

The files in this directory's parent (`gen_train.csv`, `gen_val.csv`, `classifier_heldout.csv`,
`asism_tuning_heldout.csv`, `final_eval_heldout.csv`, `split_manifest.json`) were produced by
`scripts/data/02_build_patient_splits.py` under the schema-v1 three-way split
(gen_train 0.70 / gen_val 0.10 / classifier_heldout 0.20, with the ASISM-tuning and final-eval
splits derived as children of classifier_heldout).

That architecture is superseded by the six-way partition in
`scripts/data/02b_build_sixway_splits.py` (docs/stages2_to_5_plan.md §1), because it left no
patient population for a classifier development split, and deriving one from `gen_train` would
have meant the A-E classifiers' early-stopping and threshold decisions were made on patients the
SDXL generator had itself trained on.

These files are RETAINED for provenance and reproducibility of anything already built against
them. They are NOT read by any Stage 2-5 code. Current splits live under
`splits/production/` and `splits/dev/`.
"""


def extract_patient_id(path_value: str, patient_id_regex: str) -> str:
    match = re.search(patient_id_regex, str(path_value))
    if not match:
        raise ValueError(f"Could not extract patient id from path: {path_value!r}")
    return match.group(0)


def partition_patients(
    patient_ids: list[str],
    fractions: dict[str, float],
    seed: int,
) -> dict[str, set[str]]:
    """Six direct partitions from ONE shuffle of ONE deterministically ordered patient list.

    Sorting before shuffling is what makes this reproducible regardless of the row order the source
    CSV happens to arrive in. Remainder patients (from rounding) go to the largest split so the
    partition is exact rather than dropping anyone.
    """
    ordered = sorted(patient_ids)
    rng = random.Random(seed)
    rng.shuffle(ordered)

    total = len(ordered)
    counts = {name: int(total * fraction) for name, fraction in fractions.items()}

    # Assign rounding remainder to the largest split, keeping the partition exact.
    assigned = sum(counts.values())
    if assigned < total:
        largest = max(fractions, key=lambda name: fractions[name])
        counts[largest] += total - assigned

    result: dict[str, set[str]] = {}
    cursor = 0
    for name in SPLIT_NAMES:
        take = counts[name]
        result[name] = set(ordered[cursor : cursor + take])
        cursor += take

    assert cursor == total, f"Partition consumed {cursor} of {total} patients"
    return result


def patient_label_vectors(frame: pd.DataFrame, labels: list[str] = PRIMARY_ENDPOINT_LABELS) -> dict[str, tuple[int, ...]]:
    """Collapse image labels to deterministic patient-level any-positive vectors."""
    vectors = {}
    for patient_id, rows in frame.groupby("patient_id", sort=True):
        vectors[str(patient_id)] = tuple(
            int(any(normalize_label(value) == 1 for value in rows[label])) for label in labels
        )
    return vectors


def patient_label_negative_vectors(
    frame: pd.DataFrame, labels: list[str] = PRIMARY_ENDPOINT_LABELS
) -> dict[str, tuple[int, ...]]:
    """Collapse image labels to deterministic patient-level confident-negative vectors.

    Mirrors patient_level_support's negative definition (scripts/utils/labels.py): a patient is
    negative for a label if none of their studies is a confident 1 and at least one is a confident
    0. Kept separate from patient_label_vectors (positive-only) so multilabel_partition_patients can
    target both sides of the §1.3 support rule without changing prevalence_report's positive-only
    semantics.
    """
    vectors = {}
    for patient_id, rows in frame.groupby("patient_id", sort=True):
        vectors[str(patient_id)] = tuple(
            int(
                not any(normalize_label(value) == 1 for value in rows[label])
                and any(normalize_label(value) == 0 for value in rows[label])
            )
            for label in labels
        )
    return vectors


def multilabel_partition_patients(
    vectors: dict[str, tuple[int, ...]],
    neg_vectors: dict[str, tuple[int, ...]],
    fractions: dict[str, float],
    seed: int,
) -> dict[str, set[str]]:
    """Deterministic greedy iterative multilabel allocation.

    Targets both positive AND confident-negative patient counts per label (the §1.3 support rule
    needs both sides, but only positives drove allocation before this fix). Deficits are normalized
    by each split's own target so a small split (e.g. a 5% heldout) is judged by how much of its own
    quota is unmet, not by raw patient counts — otherwise a large split's much bigger absolute target
    (e.g. gen_train at 60%) always looks more "urgent" and starves rare labels out of the small
    splits entirely. Rarest labels (by either their positive or negative population) are placed
    first; each patient goes to the non-full split with the largest normalized deficit for that
    patient's positive+negative labels, then capacity deficit, then a seeded stable tie rank. This
    uses one recorded seed and never retries seeds.
    """
    names = list(SPLIT_NAMES)
    patients = sorted(vectors)
    total = len(patients)
    n_labels = len(next(iter(vectors.values()), ()))
    capacities = {name: int(total * fractions[name]) for name in names}
    capacities[max(fractions, key=fractions.get)] += total - sum(capacities.values())

    pos_totals = [sum(v[j] for v in vectors.values()) for j in range(n_labels)]
    neg_totals = [sum(v[j] for v in neg_vectors.values()) for j in range(n_labels)]
    pos_targets = {name: [pos_totals[j] * fractions[name] for j in range(n_labels)] for name in names}
    neg_targets = {name: [neg_totals[j] * fractions[name] for j in range(n_labels)] for name in names}

    rng = random.Random(seed)
    tie_order = patients[:]
    rng.shuffle(tie_order)
    tie_rank = {patient: rank for rank, patient in enumerate(tie_order)}

    def rarest_signal(patient: str) -> int:
        counts = [pos_totals[j] for j, value in enumerate(vectors[patient]) if value]
        counts += [neg_totals[j] for j, value in enumerate(neg_vectors[patient]) if value]
        return min(counts, default=total + 1)

    ordered = sorted(
        patients,
        key=lambda p: (
            rarest_signal(p),
            -(sum(vectors[p]) + sum(neg_vectors[p])),
            tie_rank[p],
            p,
        ),
    )
    result = {name: set() for name in names}
    pos_observed = {name: [0] * n_labels for name in names}
    neg_observed = {name: [0] * n_labels for name in names}
    for patient in ordered:
        pos_vector = vectors[patient]
        neg_vector = neg_vectors[patient]
        candidates = [name for name in names if len(result[name]) < capacities[name]]

        def score(name):
            pos_deficit = sum(
                max(0.0, (pos_targets[name][j] - pos_observed[name][j]) / max(pos_targets[name][j], 1e-9))
                for j, value in enumerate(pos_vector) if value
            )
            neg_deficit = sum(
                max(0.0, (neg_targets[name][j] - neg_observed[name][j]) / max(neg_targets[name][j], 1e-9))
                for j, value in enumerate(neg_vector) if value
            )
            capacity_deficit = (capacities[name] - len(result[name])) / max(capacities[name], 1)
            return (pos_deficit + neg_deficit, capacity_deficit, -names.index(name))

        chosen = max(candidates, key=score)
        result[chosen].add(patient)
        pos_observed[chosen] = [a + b for a, b in zip(pos_observed[chosen], pos_vector)]
        neg_observed[chosen] = [a + b for a, b in zip(neg_observed[chosen], neg_vector)]
    return result


def prevalence_report(vectors: dict[str, tuple[int, ...]], groups: dict[str, set[str]]) -> dict:
    labels = list(PRIMARY_ENDPOINT_LABELS)
    def rates(ids):
        n = max(len(ids), 1)
        return {label: sum(vectors[p][j] for p in ids) / n for j, label in enumerate(labels)}
    source = rates(set(vectors))
    per_split = {}
    for name, ids in groups.items():
        split_rates = rates(ids)
        per_split[name] = {label: {
            "prevalence": split_rates[label],
            "absolute_deviation": abs(split_rates[label] - source[label]),
            "relative_deviation": (abs(split_rates[label] - source[label]) / source[label] if source[label] else None),
        } for label in labels}
    return {"source": source, "per_split": per_split, "labels_unable_to_meet_tolerance": []}


def assert_disjoint_and_exact(groups: dict[str, set[str]], all_patients: set[str]) -> None:
    """Pairwise disjointness + exact partition. Hard assertions, not warnings — a leak here
    invalidates every downstream comparison."""
    names = list(groups.keys())
    for i, left in enumerate(names):
        for right in names[i + 1 :]:
            overlap = groups[left] & groups[right]
            if overlap:
                raise SystemExit(
                    f"SPLIT LEAK: {len(overlap)} patient(s) in both {left} and {right}; "
                    f"examples: {sorted(overlap)[:5]}"
                )

    union = set().union(*groups.values())
    if union != all_patients:
        missing = all_patients - union
        extra = union - all_patients
        raise SystemExit(
            f"SPLIT NOT EXACT: {len(missing)} patient(s) unassigned, {len(extra)} unknown. "
            f"missing examples: {sorted(missing)[:5]}"
        )


def verify_full_cohort(frame: pd.DataFrame, dataset_cfg, tolerance_pct: float) -> dict:
    """Production namespace gate (§1.4): the source must be the complete intended CheXpert cohort,
    checked against the same expected-row contract 01_verify_download.py uses."""
    expected = int(dataset_cfg.source.expected_train_rows)
    actual = len(frame)
    deviation_pct = abs(actual - expected) / expected * 100.0
    passed = deviation_pct <= tolerance_pct
    return {
        "expected_train_rows": expected,
        "actual_rows": actual,
        "deviation_pct": round(deviation_pct, 4),
        "tolerance_pct": tolerance_pct,
        "passed": passed,
    }


def build_support_report(
    frames: dict[str, pd.DataFrame],
    decision_bearing_splits: list[str],
    min_positive: int,
    min_negative: int,
) -> dict:
    """Run the frozen §1.3 support rule over every decision-bearing split and return a COMPLETE
    report (all splits, all labels, all failures) — not a first-failure abort, because the caller
    needs the whole picture to propose a fraction adjustment."""
    per_split = {}
    all_passed = True

    for split_name in decision_bearing_splits:
        frame = frames[split_name]
        if split_name == FINAL_EVAL_SPLIT:
            assert_final_eval_access_allowed(
                purpose="support_check", caller="02b_build_sixway_splits"
            )
        support = patient_level_support(frame, PRIMARY_ENDPOINT_LABELS)
        passed, failures = check_support_rule(support, min_positive, min_negative)
        all_passed = all_passed and passed
        per_split[split_name] = {
            "passed": passed,
            "failures": failures,
            "per_label": support.to_dict("records"),
        }

    return {
        "rule": {
            "min_positive_patients": min_positive,
            "min_negative_patients": min_negative,
            "unit": "patients (image counts are supplementary only)",
            "labels": PRIMARY_ENDPOINT_LABELS,
        },
        "passed": all_passed,
        "per_split": per_split,
    }


def propose_fraction_adjustment(
    support_report: dict,
    fractions: dict[str, float],
    total_patients: int,
) -> dict:
    """Rough support-scaling estimate when frozen fractions fail the support rule.

    Computes, per failing split, the scale-up factor its fraction would need for its worst-failing
    label to clear the threshold, then proposes taking the shortfall from gen_train (the largest
    split, and the one whose marginal patient is worth least — Stage 1 LoRA training is far less
    sensitive to a few hundred patients than a support-starved evaluation split is).

    This is a PROPOSAL written to the report. The frozen fractions are never modified automatically.
    """
    proposals = {}
    for split_name, result in support_report["per_split"].items():
        if result["passed"]:
            continue
        current_fraction = fractions[split_name]
        current_patients = max(1, int(total_patients * current_fraction))

        worst_ratio = 1.0
        for failure in result["failures"]:
            for observed, required in (
                (failure["positive_patients"], failure["min_positive_patients"]),
                (failure["negative_patients"], failure["min_negative_patients"]),
            ):
                if observed < required:
                    # Scale needed so this label's count reaches the threshold.
                    ratio = required / max(observed, 1)
                    worst_ratio = max(worst_ratio, ratio)

        needed_fraction = min(1.0, current_fraction * worst_ratio)
        proposals[split_name] = {
            "current_fraction": current_fraction,
            "current_patients": current_patients,
            "limiting_scale_factor": round(worst_ratio, 3),
            "proposed_fraction": round(needed_fraction, 4),
            "additional_patients_needed": int(total_patients * (needed_fraction - current_fraction)),
        }

    total_additional = sum(p["additional_patients_needed"] for p in proposals.values())
    return {
        "estimate_type": "rough_support_scaling_estimate",
        "proposals": proposals,
        "suggested_donor_split": "gen_train",
        "total_additional_patients_needed": total_additional,
        "suggested_gen_train_fraction": round(
            fractions["gen_train"] - total_additional / max(total_patients, 1), 4
        ),
        "note": (
            "PROPOSAL ONLY — frozen fractions were not modified. Applying this requires an explicit "
            "decision to change the frozen split policy in configs/splits.yaml, and re-running with "
            "--freeze."
        ),
    }


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--namespace",
        choices=["production", "dev"],
        required=True,
        help="production = thesis splits (requires full cohort); dev = smoke-test splits",
    )
    parser.add_argument("--run-id", required=True, help="New immutable split run/version ID (for example production-v1 or dev-smoke-v3)")
    parser.add_argument(
        "--input-csv",
        default=None,
        help="Source CSV. Defaults to stage1 config's split.input_csv.",
    )
    parser.add_argument(
        "--freeze",
        action="store_true",
        help="Mark the manifest frozen. Required before Stage 5. Fails if the support check failed.",
    )
    args = parser.parse_args()

    splits_cfg = load_splits_config()
    validate_namespace_run_id(args.run_id, args.namespace)
    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    out_dir = resolve_splits_dir(args.run_id, splits_cfg)
    if out_dir.exists():
        raise SystemExit(f"Immutable split run already exists; refusing every in-place write: {out_dir}")
    out_dir.parent.mkdir(parents=True, exist_ok=True)

    fractions = {name: float(splits_cfg.fractions[name]) for name in SPLIT_NAMES}
    fraction_sum = round(sum(fractions.values()), 9)
    if fraction_sum != 1.0:
        raise SystemExit(f"Split fractions must sum to 1.0; got {fraction_sum} from {fractions}")

    input_csv = Path(args.input_csv) if args.input_csv else Path(stage1_cfg.split.input_csv)
    if not input_csv.is_file():
        raise SystemExit(f"Missing source CSV: {input_csv} (run scripts/data/00_download_dataset.py first)")

    frame = pd.read_csv(input_csv)
    schema = dataset_cfg.schema
    try:
        source_validation = validate_chexpert_frame(
            frame, dataset_cfg, Path(stage1_cfg.paths.raw_dir), require_all_images=args.namespace == "production"
        )
    except ValueError as exc:
        raise SystemExit(f"Source schema validation failed: {exc}") from exc
    frame["patient_id"] = frame[schema.path_column].apply(
        lambda value: extract_patient_id(value, schema.patient_id_regex)
    )

    namespace_cfg = splits_cfg.namespaces[args.namespace]
    cohort_check = None
    if bool(namespace_cfg.require_full_cohort):
        cohort_check = verify_full_cohort(
            frame, dataset_cfg, float(namespace_cfg.full_cohort_row_tolerance_pct)
        )
        if not cohort_check["passed"]:
            raise SystemExit(
                "Production splits require the complete intended CheXpert-v1.0-small training "
                f"cohort.\n  expected ~{cohort_check['expected_train_rows']} rows, got "
                f"{cohort_check['actual_rows']} ({cohort_check['deviation_pct']}% deviation, "
                f"tolerance {cohort_check['tolerance_pct']}%)\n"
                f"  source: {input_csv}\n"
                "Use --namespace dev for the development subset, or point split.input_csv at the "
                "full train.csv on a machine where it is available."
            )

    all_patients = set(frame["patient_id"].unique())
    vectors = patient_label_vectors(frame)
    neg_vectors = patient_label_negative_vectors(frame)
    groups = multilabel_partition_patients(vectors, neg_vectors, fractions, int(splits_cfg.split_seed))
    assert_disjoint_and_exact(groups, all_patients)

    frames = {
        name: frame[frame["patient_id"].isin(ids)].reset_index(drop=True)
        for name, ids in groups.items()
    }

    support_report = build_support_report(
        frames,
        list(splits_cfg.support_rule.decision_bearing_splits),
        int(splits_cfg.support_rule.min_positive_patients),
        int(splits_cfg.support_rule.min_negative_patients),
    )

    adjustment = None
    if not support_report["passed"]:
        adjustment = propose_fraction_adjustment(support_report, fractions, len(all_patients))

    # pathlib mkdir avoids Windows tempfile ACL inheritance that can make the staging directory
    # unreadable to the creating process in managed workspaces; uniqueness and same-filesystem
    # atomic publication are preserved.
    staging = out_dir.parent / f".{args.run_id}.staging-{uuid.uuid4().hex}"
    staging.mkdir()
    output_files = {}
    for name, split_frame in frames.items():
        if name == FINAL_EVAL_SPLIT:
            assert_final_eval_access_allowed(
                purpose="split_construction", caller="02b_build_sixway_splits"
            )
        csv_path = staging / f"{name}.csv"
        split_frame.to_csv(csv_path, index=False)
        reread = pd.read_csv(csv_path)
        if len(reread) != len(split_frame) or set(reread["patient_id"].astype(str)) != groups[name]:
            raise SystemExit(f"Staged split validation failed for {name}")
        output_files[name] = {"filename": csv_path.name, "sha256": sha256_file(csv_path), "row_count": len(reread), "patient_count": len(groups[name])}

    # Retain and mark the superseded v1 artifacts rather than deleting them (§1.5).
    legacy_manifest = Path(splits_cfg.paths.splits_root) / "split_manifest.json"
    if legacy_manifest.is_file():
        note_path = Path(splits_cfg.paths.splits_root) / SUPERSEDED_NOTE_FILENAME
        if not note_path.exists():
            note_path.write_text(SUPERSEDED_NOTE, encoding="utf-8")

    core = {
        "manifest_version": int(splits_cfg.manifest_version),
        "split_namespace": args.run_id,
        "namespace_class": args.namespace,
        "split_unit": str(splits_cfg.split_unit),
        "split_seed": int(splits_cfg.split_seed),
        "fractions": fractions,
        "input_csv_path": str(input_csv),
        "input_csv_rows": len(frame),
        "source_csv_sha256": sha256_file(input_csv),
        "source_schema_hash": source_validation["source_schema_hash"],
        "sorted_patient_set_hash": hash_dict({"patient_ids": sorted(all_patients)}, length=64),
        "output_files": output_files,
        "allocation_algorithm": "deterministic_greedy_iterative_multilabel_v1",
        "allocation_library": "internal",
        "prevalence": prevalence_report(vectors, groups),
        "num_patients_total": len(all_patients),
        "patients_per_split": {name: len(ids) for name, ids in groups.items()},
        "images_per_split": {name: len(f) for name, f in frames.items()},
        "patient_id_hash_per_split": {
            name: hash_dict({"patient_ids": sorted(ids)}, length=16) for name, ids in groups.items()
        },
        "primary_endpoint_labels": PRIMARY_ENDPOINT_LABELS,
    }

    manifest = dict(core)
    manifest["git_commit_hash"] = get_git_commit_hash()
    manifest["full_cohort_check"] = cohort_check
    manifest["support_check"] = support_report
    manifest["support_adjustment_proposal"] = adjustment
    manifest["frozen"] = False
    manifest["supersedes"] = "split_manifest.json (schema v1, three-way + derived children)"
    manifest["environment_versions"] = get_library_versions()

    if args.freeze:
        if not support_report["passed"]:
            shutil.rmtree(staging)
            print("\n" + "=" * 78, flush=True)
            print("SUPPORT CHECK FAILED — REFUSING TO FREEZE", flush=True)
            print("=" * 78, flush=True)
            for split_name, result in support_report["per_split"].items():
                if result["passed"]:
                    continue
                print(f"\n[{split_name}] {len(result['failures'])} label(s) below the frozen rule:", flush=True)
                for failure in result["failures"]:
                    print(
                        f"    {failure['label']:<32} "
                        f"pos={failure['positive_patients']:>5} (need {failure['min_positive_patients']})  "
                        f"neg={failure['negative_patients']:>5} (need {failure['min_negative_patients']})",
                        flush=True,
                    )
            if adjustment:
                print("\nRough support scaling estimate (not an optimized allocation):", flush=True)
                for split_name, proposal in adjustment["proposals"].items():
                    print(
                        f"    {split_name}: {proposal['current_fraction']} -> "
                        f"{proposal['proposed_fraction']} "
                        f"(+{proposal['additional_patients_needed']} patients)",
                        flush=True,
                    )
                print(
                    f"    donor: {adjustment['suggested_donor_split']} -> "
                    f"{adjustment['suggested_gen_train_fraction']}",
                    flush=True,
                )
                print(f"\n    {adjustment['note']}", flush=True)
            print("\nNo split run was published because the frozen support gate failed.", flush=True)
            print(
                "\nThe frozen 60/10/15/5/5/5 policy was NOT modified. Changing it is an explicit "
                "decision (docs/stages2_to_5_plan.md §1.3).",
                flush=True,
            )
            return 2
        manifest["frozen"] = True

    manifest["manifest_hash_scope"] = "full_manifest_without_manifest_hash_v2"
    manifest["manifest_hash"] = hash_dict(manifest, length=64)
    write_json(staging / SPLIT_MANIFEST_FILENAME, manifest)
    os.replace(staging, out_dir)

    print(f"Namespace:     {args.namespace}", flush=True)
    print(f"Source CSV:    {input_csv} ({len(frame)} rows)", flush=True)
    print(f"Patients:      {len(all_patients)}", flush=True)
    print(f"Seed:          {splits_cfg.split_seed}", flush=True)
    for name in SPLIT_NAMES:
        print(
            f"  {name:<24} {len(groups[name]):>7} patients  {len(frames[name]):>8} images  "
            f"({fractions[name]:.0%})",
            flush=True,
        )
    print(f"Support check: {'PASSED' if support_report['passed'] else 'FAILED'}", flush=True)
    if not support_report["passed"]:
        failing = [
            f"{split_name}({len(result['failures'])})"
            for split_name, result in support_report["per_split"].items()
            if not result["passed"]
        ]
        print(f"  failing splits: {', '.join(failing)}", flush=True)
        if args.namespace == "dev":
            print(
                "  (expected on the dev subset — recorded in the manifest, not fatal; "
                "these are NOT production-frozen thesis splits)",
                flush=True,
            )
    print(f"Frozen:        {manifest['frozen']}", flush=True)
    print(f"Manifest hash: {manifest['manifest_hash']}", flush=True)
    print(f"Wrote:         {out_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/03_preprocess_images.py`


In [ ]:
# SOURCE FILE: scripts/data/03_preprocess_images.py
#!/usr/bin/env python3
"""Safely preprocess the generation splits into square RGB JPEG images."""

from __future__ import annotations

import argparse
import json
import os
import sys
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.identifiers import sanitize_image_id  # noqa: E402
from scripts.utils.manifest import hash_dict, read_json, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import SPLIT_NAMES, read_split_manifest, resolve_splits_dir  # noqa: E402

BLANK_STD_THRESHOLD = 5.0
JPEG_QUALITY = 95
MANIFEST_VERSION = 2


def resolve_source_path(raw_dir: Path, raw_path: str) -> Path:
    value = str(raw_path).replace("\\", "/")
    for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
        if value.startswith(prefix):
            value = value[len(prefix):]
    return raw_dir / value


def validate_processed_jpeg(path: Path, resolution: int) -> tuple[bool, str | None]:
    """Fully decode a JPEG and require its encoded mode and dimensions to match the config."""
    if not path.is_file():
        return False, "missing"
    try:
        with Image.open(path) as image:
            if image.format != "JPEG":
                return False, f"format_{image.format or 'unknown'}"
            if image.mode != "RGB":
                return False, f"mode_{image.mode}"
            if image.size != (resolution, resolution):
                return False, f"size_{image.size[0]}x{image.size[1]}"
            image.load()
    except Exception as exc:
        return False, f"unreadable_{type(exc).__name__}"
    return True, None


def letterbox_resize(image: Image.Image, target_size: int) -> Image.Image:
    width, height = image.size
    scale = target_size / max(width, height)
    new_size = (max(1, round(width * scale)), max(1, round(height * scale)))
    resized = image.resize(new_size, Image.Resampling.BICUBIC)
    canvas = Image.new("RGB", (target_size, target_size), (0, 0, 0))
    canvas.paste(resized, ((target_size - new_size[0]) // 2, (target_size - new_size[1]) // 2))
    return canvas


def atomic_save_jpeg(image: Image.Image, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".tmp", dir=destination.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        image.save(temporary, format="JPEG", quality=JPEG_QUALITY)
        valid, reason = validate_processed_jpeg(temporary, image.width)
        if not valid:
            raise OSError(f"temporary JPEG validation failed: {reason}")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)


def expected_manifest(stage1_cfg, split_paths: dict[str, Path], namespace: str | None = None, split_manifest: dict | None = None) -> dict:
    data = stage1_cfg.data
    namespace = namespace or str(stage1_cfg.split.namespace)
    split_manifest = split_manifest or read_split_manifest(namespace)
    settings = {
        "resolution": int(data.resolution), "aspect_mode": str(data.aspect_mode),
        "view_filter": str(data.view_filter), "min_source_resolution": int(data.min_source_resolution),
        "horizontal_flip": bool(data.horizontal_flip), "blank_std_threshold": BLANK_STD_THRESHOLD,
        "jpeg_quality": JPEG_QUALITY,
    }
    return {
        "manifest_version": MANIFEST_VERSION,
        "preprocessing_algorithm_version": 2,
        "output_schema_version": 2,
        "split_namespace": namespace,
        "split_manifest_hash": split_manifest.get("manifest_hash"),
        "preprocessing_config_hash": hash_dict(settings, length=64),
        "output_settings": settings,
        "source_splits": {
            name: {"path": path.name, "sha256": sha256_file(path), "size_bytes": path.stat().st_size}
            for name, path in sorted(split_paths.items())
        },
    }


def check_manifest(path: Path, expected: dict, adopt_legacy: bool) -> None:
    if not path.exists():
        return
    actual = read_json(path)
    if actual == expected:
        return
    if "manifest_version" not in actual and adopt_legacy:
        legacy_settings = {k: actual.get(k) for k in expected["output_settings"] if k in actual}
        current_settings = {k: expected["output_settings"][k] for k in legacy_settings}
        if legacy_settings != current_settings:
            raise SystemExit(f"Legacy preprocessing settings conflict with the current configuration:\nold={legacy_settings}\nnew={current_settings}")
        print("WARNING: adopting a legacy preprocessing manifest with no split hashes. Existing files will still be decoded and validated before skipping.", flush=True)
        return
    raise SystemExit(
        "Existing preprocessing outputs were created with incompatible or unverifiable settings/splits.\n"
        f"Manifest: {path}\nExisting: {json.dumps(actual, indent=2, sort_keys=True)}\n"
        f"Requested: {json.dumps(expected, indent=2, sort_keys=True)}\n"
        "Use the original configuration/split files. For a legacy manifest only, review it and rerun with --adopt-legacy-manifest."
    )


def process_split(split_name, df, raw_dir, images_dir, dataset_cfg, data_cfg) -> Counter:
    out_dir = images_dir / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    schema = dataset_cfg.schema
    image_ids = [sanitize_image_id(value) for value in df[schema.path_column]]
    duplicates = [item for item, count in Counter(image_ids).items() if count > 1]
    if duplicates:
        raise SystemExit(f"{split_name}: {len(duplicates)} duplicate output image IDs; first examples: {duplicates[:5]}")

    log_path = images_dir / f"{split_name}_preprocessing_log.jsonl"
    fd, temporary_name = tempfile.mkstemp(prefix=f".{log_path.name}.", suffix=".tmp", dir=images_dir)
    counts = Counter(processed=0, skipped_existing=0, filtered=0, failed=0)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", buffering=1) as log:
            progress = tqdm(df.itertuples(index=False, name=None), total=len(df), desc=f"[{split_name}] preprocess", unit="image")
            columns = {name: index for index, name in enumerate(df.columns)}
            for values in progress:
                raw_path = values[columns[schema.path_column]]
                image_id = sanitize_image_id(raw_path)
                destination = out_dir / f"{image_id}.jpg"
                frontal_value = values[columns[schema.frontal_lateral_column]] if schema.frontal_lateral_column in columns else "Frontal"
                reason = None
                detail = None
                action = "processed"

                if str(data_cfg.view_filter) == "frontal" and not str(frontal_value).strip().lower().startswith("front"):
                    action, reason = "filtered", "filtered_lateral_view"
                else:
                    valid, validation_reason = validate_processed_jpeg(destination, int(data_cfg.resolution))
                    if valid:
                        action, reason = "skipped_existing", "validated_existing"
                    else:
                        source = resolve_source_path(raw_dir, raw_path)
                        if not source.is_file():
                            action, reason = "failed", "source_file_missing"
                        else:
                            try:
                                with Image.open(source) as source_image:
                                    gray = source_image.convert("L")
                                    width, height = gray.size
                                    if min(width, height) < int(data_cfg.min_source_resolution):
                                        action, reason = "filtered", "below_min_source_resolution"
                                    elif np.asarray(gray, dtype=np.float32).std() < BLANK_STD_THRESHOLD:
                                        action, reason = "filtered", "near_uniform_blank"
                                    else:
                                        atomic_save_jpeg(letterbox_resize(gray.convert("RGB"), int(data_cfg.resolution)), destination)
                                        detail = validation_reason and f"replaced_{validation_reason}"
                            except Exception as exc:
                                action, reason, detail = "failed", "corrupt_or_unreadable", f"{type(exc).__name__}: {exc}"

                counts[action] += 1
                record = {"image_id": image_id, "raw_path": str(raw_path), "kept": action in ("processed", "skipped_existing"), "action": action, "reason": reason, "detail": detail}
                log.write(json.dumps(record, ensure_ascii=False) + "\n")
                progress.set_postfix({key: counts[key] for key in ("processed", "skipped_existing", "filtered", "failed")}, refresh=False)
            progress.close()
            log.flush()
            os.fsync(log.fileno())
        os.replace(temporary_name, log_path)
    finally:
        Path(temporary_name).unlink(missing_ok=True)
    print(f"[{split_name}] processed={counts['processed']} skipped-existing={counts['skipped_existing']} filtered={counts['filtered']} failed={counts['failed']} log={log_path}", flush=True)
    return counts


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None, help="Explicit v2 split namespace/run (dev or production run ID)")
    parser.add_argument("--splits", nargs="+", choices=SPLIT_NAMES, default=SPLIT_NAMES)
    parser.add_argument("--adopt-legacy-manifest", action="store_true", help="Explicitly accept an old manifest lacking split hashes after checking its recorded settings")
    args = parser.parse_args()
    cfg, dataset_cfg = load_stage1_config(), load_dataset_config()
    ensure_dirs(cfg)
    namespace = args.namespace or str(cfg.split.namespace)
    split_manifest = read_split_manifest(namespace)
    split_dir = resolve_splits_dir(namespace)
    split_paths = {name: split_dir / f"{name}.csv" for name in args.splits}
    missing = [str(path) for path in split_paths.values() if not path.is_file()]
    if missing:
        raise SystemExit("Missing split files (run 02_build_patient_splits.py first): " + ", ".join(missing))
    images_dir = Path(cfg.paths.images_dir) / namespace
    images_dir.mkdir(parents=True, exist_ok=True)
    for name, path in split_paths.items():
        manifest = expected_manifest(cfg, {name: path}, namespace, split_manifest)
        manifest_path = images_dir / f"{name}_preprocessing_manifest.json"
        split_output_dir = images_dir / name
        if not manifest_path.is_file() and split_output_dir.is_dir() and any(split_output_dir.iterdir()):
            raise SystemExit(
                f"Unpinned preprocessing outputs exist at {split_output_dir} without {manifest_path}; "
                "refusing stale cross-assignment reuse. Preserve them and choose a new versioned namespace."
            )
        check_manifest(manifest_path, manifest, args.adopt_legacy_manifest)
        write_json(manifest_path, manifest)
        process_split(name, pd.read_csv(path), Path(cfg.paths.raw_dir), images_dir, dataset_cfg, cfg.data)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/04_generate_captions.py`


In [ ]:
# SOURCE FILE: scripts/data/04_generate_captions.py
#!/usr/bin/env python3
"""Generate and reconcile captions for valid, successfully preprocessed images."""

from __future__ import annotations

import argparse
import importlib
import json
import os
import sys
import tempfile
from collections import Counter
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import CaptionConfig, PATHOLOGY_COLUMNS, build_caption_variants  # noqa: E402
from scripts.utils.config import ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.identifiers import sanitize_image_id  # noqa: E402
from scripts.utils.manifest import read_json, write_json  # noqa: E402
from scripts.utils.splits import read_split_manifest, resolve_splits_dir  # noqa: E402

_preprocess = importlib.import_module("scripts.data.03_preprocess_images")
expected_manifest = _preprocess.expected_manifest
validate_processed_jpeg = _preprocess.validate_processed_jpeg


def build_caption_config(cfg) -> CaptionConfig:
    value = cfg.captions
    return CaptionConfig(value.age_bucket_width_years, value.uncertain_label_policy, value.no_finding_overrides_positives, value.num_paraphrase_variants, value.template_version)


def raw_label_vector(row: pd.Series) -> dict:
    result = {}
    for column in PATHOLOGY_COLUMNS:
        value = row.get(column)
        result[column] = None if value is None or pd.isna(value) else int(value)
    return result


def load_preprocessing_status(path: Path) -> dict[str, dict]:
    if not path.is_file():
        raise SystemExit(f"Missing preprocessing log: {path}. Complete preprocessing for this split first.")
    result = {}
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            try:
                record = json.loads(line)
            except Exception as exc:
                raise SystemExit(f"Invalid preprocessing log JSON at {path}:{line_number}: {exc}") from exc
            image_id = record.get("image_id")
            if not image_id or image_id in result:
                raise SystemExit(f"Missing/duplicate image_id in {path}:{line_number}: {image_id!r}")
            result[image_id] = record
    return result


def previous_caption_ids(path: Path) -> tuple[set[str], int]:
    ids, duplicates = set(), 0
    if path.is_file():
        with path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                image_id = json.loads(line).get("image_id")
                duplicates += image_id in ids
                ids.add(image_id)
    return ids, duplicates


def process_split(name, df, images_dir, captions_dir, schema, caption_cfg, resolution) -> Counter:
    ids = [sanitize_image_id(value) for value in df[schema.path_column]]
    duplicate_source_ids = [key for key, count in Counter(ids).items() if count > 1]
    if duplicate_source_ids:
        raise SystemExit(f"{name}: duplicate image IDs in split CSV: {duplicate_source_ids[:5]}")
    status = load_preprocessing_status(images_dir / f"{name}_preprocessing_log.jsonl")
    split_ids = set(ids)
    unknown_log_ids = set(status) - split_ids
    missing_log_ids = split_ids - set(status)
    if unknown_log_ids or missing_log_ids:
        raise SystemExit(f"{name}: stale/incomplete preprocessing log: missing={len(missing_log_ids)}, not-in-split={len(unknown_log_ids)}")

    destination = captions_dir / f"{name}_captions.jsonl"
    previous_ids, previous_duplicates = previous_caption_ids(destination)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".tmp", dir=captions_dir)
    counts = Counter(written=0, filtered_or_failed=0, missing_or_invalid=0)
    written_ids = set()
    try:
        with os.fdopen(fd, "w", encoding="utf-8", buffering=1) as output:
            for (_, row), image_id in tqdm(zip(df.iterrows(), ids), total=len(df), desc=f"[{name}] captions", unit="record"):
                preprocess_record = status[image_id]
                if not preprocess_record.get("kept", False):
                    counts["filtered_or_failed"] += 1
                    continue
                image_path = images_dir / name / f"{image_id}.jpg"
                valid, reason = validate_processed_jpeg(image_path, resolution)
                if not valid:
                    counts["missing_or_invalid"] += 1
                    print(f"[{name}] WARNING: excluding {image_id}: processed image is {reason}", flush=True)
                    continue
                record = {
                    "image_id": image_id, "split": name,
                    "image_relpath": (Path(name) / f"{image_id}.jpg").as_posix(),
                    "patient_id": row.get("patient_id"),
                    "caption_variants": build_caption_variants(row.to_dict(), caption_cfg),
                    "raw_labels": raw_label_vector(row), "sex": row.get(schema.sex_column),
                    "age": row.get(schema.age_column), "frontal_lateral": row.get(schema.frontal_lateral_column),
                    "ap_pa": row.get(schema.ap_pa_column),
                }
                output.write(json.dumps(record, ensure_ascii=False, default=lambda value: None if pd.isna(value) else str(value)) + "\n")
                written_ids.add(image_id)
                counts["written"] += 1
            output.flush()
            os.fsync(output.fileno())
        os.replace(temporary_name, destination)
    finally:
        Path(temporary_name).unlink(missing_ok=True)
    stale = previous_ids - written_ids
    missing_previous = written_ids - previous_ids if previous_ids else set()
    print(f"[{name}] written={counts['written']} filtered/failed={counts['filtered_or_failed']} missing/invalid={counts['missing_or_invalid']} prior-duplicates={previous_duplicates} stale-prior={len(stale)} new-vs-prior={len(missing_previous)} -> {destination}", flush=True)
    return counts


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--splits", nargs="+", default=["gen_train", "gen_val"])
    args = parser.parse_args()
    cfg, dataset_cfg = load_stage1_config(), load_dataset_config()
    ensure_dirs(cfg)
    namespace = args.namespace or str(cfg.split.namespace)
    split_manifest = read_split_manifest(namespace)
    split_paths = {name: resolve_splits_dir(namespace) / f"{name}.csv" for name in args.splits}
    missing = [str(path) for path in split_paths.values() if not path.is_file()]
    if missing:
        raise SystemExit("Missing split files: " + ", ".join(missing))
    images_dir = Path(cfg.paths.images_dir) / namespace
    captions_dir = Path(cfg.paths.captions_dir) / namespace
    captions_dir.mkdir(parents=True, exist_ok=True)
    for name, path in split_paths.items():
        expected = expected_manifest(cfg, {name: path}, namespace, split_manifest)
        manifest_path = images_dir / f"{name}_preprocessing_manifest.json"
        if not manifest_path.is_file() or read_json(manifest_path) != expected:
            raise SystemExit(f"Preprocessing provenance for {name} does not match the current namespace/settings. Rerun preprocessing first.")
        process_split(name, pd.read_csv(path), images_dir, captions_dir, dataset_cfg.schema, build_caption_config(cfg), int(cfg.data.resolution))
    write_json(captions_dir / "caption_template_version.json", {
        "template_version": cfg.captions.template_version,
        "num_paraphrase_variants": cfg.captions.num_paraphrase_variants,
        "age_bucket_width_years": cfg.captions.age_bucket_width_years,
        "uncertain_label_policy": cfg.captions.uncertain_label_policy,
        "no_finding_overrides_positives": cfg.captions.no_finding_overrides_positives,
        "preprocessing_manifests": {name: read_json(images_dir / f"{name}_preprocessing_manifest.json") for name in args.splits},
        "split_namespace": namespace,
        "split_manifest_hash": split_manifest.get("manifest_hash"),
    })
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


# Stage 1 — SDXL LoRA training


## `scripts/train/train_lora_sdxl.py`


In [ ]:
# SOURCE FILE: scripts/train/train_lora_sdxl.py
#!/usr/bin/env python3
"""SDXL + LoRA fine-tuning on CheXpert (docs/stage1_plan.md §3, §8).

Adapted from diffusers' text_to_image_lora_sdxl reference pattern, with:
  - UNet-only LoRA (self+cross attention projections), text encoders frozen
  - bf16 mixed precision, PyTorch SDPA attention, gradient checkpointing
  - one-time VAE latent + text embedding caching (captions are deterministic-from-labels)
  - EMA of the LoRA adapter weights
  - resumable checkpointing via Accelerate save_state/load_state, all paths under PROJECT_ROOT
    (expected to be the persistent volume on RunPod — see configs/stage1_lora_sdxl.yaml)
  - TensorBoard (+ optional Weights & Biases) logging

Run directly, or via scripts/train/launch_resumable.sh which handles the resume-on-restart logic.

Usage:
    accelerate launch --config_file configs/accelerate_config.yaml scripts/train/train_lora_sdxl.py \\
        [--resume-from CHECKPOINT_DIR] [--run-id RUN_ID] [override.key=value ...]
"""

from __future__ import annotations

import argparse
import hashlib
import json
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
from omegaconf import OmegaConf
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import select_variant_index  # noqa: E402
from scripts.utils.config import ensure_dirs, load_stage1_config  # noqa: E402
from scripts.utils.manifest import build_checkpoint_metadata, make_run_id, read_json, write_json  # noqa: E402
from scripts.utils.seed import set_seed  # noqa: E402


# --------------------------------------------------------------------------------------
# Model / cache construction
# --------------------------------------------------------------------------------------

def tracker_hparams(cfg) -> dict[str, int | float | str | bool]:
    """Flatten a resolved OmegaConf config into TensorBoard-safe scalar hparams."""
    resolved = OmegaConf.to_container(cfg, resolve=True)
    flattened: dict[str, int | float | str | bool] = {}

    def visit(prefix: str, value) -> None:
        if isinstance(value, dict):
            for key, child in value.items():
                visit(f"{prefix}.{key}" if prefix else str(key), child)
        elif isinstance(value, (list, tuple)):
            flattened[prefix] = json.dumps(value, sort_keys=True)
        elif value is None:
            flattened[prefix] = "null"
        elif isinstance(value, (int, float, str, bool)):
            flattened[prefix] = value
        else:
            flattened[prefix] = str(value)

    visit("", resolved)
    return flattened

def build_models(cfg, device_dtype: torch.dtype):
    from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
    from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer
    from peft import LoraConfig

    model_id = cfg.model.base_model_id
    cache_dir = cfg.paths.hf_cache_dir
    revision = cfg.model.revision if cfg.model.revision else None

    tokenizer_one = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer", revision=revision, cache_dir=cache_dir)
    tokenizer_two = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer_2", revision=revision, cache_dir=cache_dir)
    noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler", revision=revision, cache_dir=cache_dir)

    text_encoder_one = CLIPTextModel.from_pretrained(
        model_id, subfolder="text_encoder", revision=revision, cache_dir=cache_dir, torch_dtype=device_dtype
    )
    text_encoder_two = CLIPTextModelWithProjection.from_pretrained(
        model_id, subfolder="text_encoder_2", revision=revision, cache_dir=cache_dir, torch_dtype=device_dtype
    )
    vae = AutoencoderKL.from_pretrained(
        model_id, subfolder="vae", revision=revision, cache_dir=cache_dir,
        torch_dtype=torch.float32 if cfg.model.vae_upcast_fp32 else device_dtype,
    )
    unet = UNet2DConditionModel.from_pretrained(
        model_id, subfolder="unet", revision=revision, cache_dir=cache_dir, torch_dtype=device_dtype
    )

    for model in (vae, text_encoder_one, text_encoder_two, unet):
        model.requires_grad_(False)

    if cfg.lora.train_text_encoders:
        raise NotImplementedError(
            "lora.train_text_encoders=true is a documented fallback (docs/stage1_plan.md §3) "
            "for weak rare-label conditioning, not implemented in v1. Set it back to false, or "
            "implement CLIPText LoRA attachment here before enabling it."
        )

    lora_config = LoraConfig(
        r=cfg.lora.rank,
        lora_alpha=cfg.lora.alpha,
        lora_dropout=cfg.lora.dropout,
        init_lora_weights="gaussian",
        target_modules=list(cfg.lora.target_modules),
    )
    unet.add_adapter(lora_config)
    for p in unet.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    if cfg.training.gradient_checkpointing:
        unet.enable_gradient_checkpointing()

    return {
        "tokenizer_one": tokenizer_one,
        "tokenizer_two": tokenizer_two,
        "text_encoder_one": text_encoder_one,
        "text_encoder_two": text_encoder_two,
        "vae": vae,
        "unet": unet,
        "noise_scheduler": noise_scheduler,
    }


def _load_captions(captions_path: Path) -> list[dict]:
    records = []
    seen = set()
    with open(captions_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                record = json.loads(line)
                image_id = record.get("image_id")
                if not image_id or image_id in seen:
                    raise ValueError(f"Missing or duplicate image_id {image_id!r} in {captions_path}")
                if not record.get("caption_variants"):
                    raise ValueError(f"Caption record {image_id!r} has no variants")
                seen.add(image_id)
                records.append(record)
    return records


def validate_training_inputs(cfg) -> None:
    """Fail before model downloads when captions/preprocessing no longer match the config."""
    namespace = str(cfg.split.namespace)
    metadata_path = Path(cfg.paths.captions_dir) / namespace / "caption_template_version.json"
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Missing caption metadata: {metadata_path}; run 04_generate_captions.py")
    metadata = read_json(metadata_path)
    expected_caption = {
        "template_version": cfg.captions.template_version,
        "num_paraphrase_variants": cfg.captions.num_paraphrase_variants,
        "age_bucket_width_years": cfg.captions.age_bucket_width_years,
        "uncertain_label_policy": cfg.captions.uncertain_label_policy,
        "no_finding_overrides_positives": cfg.captions.no_finding_overrides_positives,
    }
    mismatches = {key: (metadata.get(key), value) for key, value in expected_caption.items() if metadata.get(key) != value}
    if metadata.get("split_namespace") != namespace:
        mismatches["split_namespace"] = (metadata.get("split_namespace"), namespace)
    from scripts.utils.splits import read_split_manifest, resolve_splits_dir
    split_manifest = read_split_manifest(namespace)
    if metadata.get("split_manifest_hash") != split_manifest.get("manifest_hash"):
        mismatches["split_manifest_hash"] = (metadata.get("split_manifest_hash"), split_manifest.get("manifest_hash"))
    preprocessing_by_split = metadata.get("preprocessing_manifests", {})
    preprocessing = preprocessing_by_split.get("gen_train", {})
    output_settings = preprocessing.get("output_settings", {})
    if output_settings.get("resolution") != cfg.data.resolution:
        mismatches["preprocessing_resolution"] = (output_settings.get("resolution"), cfg.data.resolution)
    for split_name in ("gen_train", "gen_val"):
        split_path = resolve_splits_dir(namespace) / f"{split_name}.csv"
        recorded = preprocessing_by_split.get(split_name, {}).get("source_splits", {}).get(split_name, {}).get("sha256")
        current = hashlib.sha256(split_path.read_bytes()).hexdigest() if split_path.is_file() else None
        if recorded != current:
            mismatches[f"{split_name}_sha256"] = (recorded, current)
    if mismatches:
        raise ValueError(f"Caption/preprocessing provenance is stale or incompatible: {mismatches}. Rerun preprocessing and captions.")


def build_or_load_cache(cfg, models, split_name: str, device: torch.device, max_samples: int | None = None):
    """One-time VAE latent + text embedding cache per split (docs/stage1_plan.md §8).

    Latents are cached per image (as encoder-distribution moments, so a fresh sample is still
    drawn each epoch). Text embeddings are cached per unique caption string across all
    paraphrase variants of all images, since many images share the low-cardinality caption
    vocabulary.
    """
    namespace = str(cfg.split.namespace)
    images_dir = Path(cfg.paths.images_dir) / namespace
    captions_path = Path(cfg.paths.captions_dir) / namespace / f"{split_name}_captions.jsonl"
    cache_dir = Path(cfg.paths.processed_dir) / "cache" / namespace
    cache_dir.mkdir(parents=True, exist_ok=True)
    records = _load_captions(captions_path)
    if max_samples is not None:
        if max_samples < 1:
            raise ValueError("--max-samples-per-split must be positive")
        records = records[:max_samples]
    if not records:
        raise FileNotFoundError(
            f"No caption records found at {captions_path} — run 03_preprocess_images.py and "
            "04_generate_captions.py first."
        )

    resolution = cfg.data.resolution
    record_identity = json.dumps([record["image_id"] for record in records], separators=(",", ":"))
    captions_hash = hashlib.sha256((hashlib.sha256(captions_path.read_bytes()).hexdigest() + record_identity).encode()).hexdigest()[:12]
    model_key = hashlib.sha256(f"{cfg.model.base_model_id}:{cfg.model.revision}:{resolution}".encode()).hexdigest()[:8]
    cache_key = f"{captions_hash}_{model_key}"
    latent_cache_path = cache_dir / f"{split_name}_latent_cache_{cache_key}.pt"
    prompt_cache_path = cache_dir / f"{split_name}_prompt_cache_{cache_key}.pt"

    for record in records:
        image_path = images_dir / record["image_relpath"]
        if not image_path.is_file():
            raise FileNotFoundError(f"Caption record {record['image_id']} references missing image: {image_path}")

    if not latent_cache_path.exists():
        from PIL import Image
        from torchvision.transforms import functional as TF

        vae = models["vae"].to(device, dtype=torch.float32).eval()
        moments = torch.empty((len(records), 8, resolution // 8, resolution // 8), dtype=torch.float16)
        with torch.inference_mode():
            for i, rec in enumerate(tqdm(records, desc=f"[{split_name}] VAE latent cache")):
                image_path = images_dir / rec["image_relpath"]
                with Image.open(image_path) as im:
                    pixels = TF.normalize(TF.to_tensor(im.convert("RGB")), [0.5] * 3, [0.5] * 3)
                params = vae.encode(pixels.unsqueeze(0).to(device, torch.float32)).latent_dist.parameters
                moments[i] = params[0].cpu().half()
        torch.save(
            {
                "image_ids": [r["image_id"] for r in records],
                "moments": moments,
                "scaling_factor": vae.config.scaling_factor,
                "target_size": resolution,
            },
            latent_cache_path,
        )
        vae.to("cpu")
        del moments
        torch.cuda.empty_cache()
    else:
        print(f"[{split_name}] using existing latent cache: {latent_cache_path}")

    if not prompt_cache_path.exists():
        text_dtype = {
            "fp32": torch.float32,
            "fp16": torch.float16,
            "bf16": torch.bfloat16,
        }[str(cfg.training.precision).lower()]
        # Some SDXL checkpoints contain mixed stored dtypes (notably text_projection).  Cast the
        # complete frozen encoders explicitly so their hidden states and projection weights agree.
        text_encoder_one = models["text_encoder_one"].to(device=device, dtype=text_dtype).eval()
        text_encoder_two = models["text_encoder_two"].to(device=device, dtype=text_dtype).eval()
        tokenizer_one, tokenizer_two = models["tokenizer_one"], models["tokenizer_two"]

        unique_captions = sorted({c for r in records for c in r["caption_variants"]})
        prompt_cache: dict[str, dict] = {}
        with torch.inference_mode():
            for caption in tqdm(unique_captions, desc=f"[{split_name}] text embedding cache"):
                ids1 = tokenizer_one(caption, padding="max_length", truncation=True, max_length=77, return_tensors="pt").input_ids.to(device)
                ids2 = tokenizer_two(caption, padding="max_length", truncation=True, max_length=77, return_tensors="pt").input_ids.to(device)
                o1 = text_encoder_one(ids1, output_hidden_states=True, return_dict=True)
                o2 = text_encoder_two(ids2, output_hidden_states=True, return_dict=True)
                prompt_cache[caption] = {
                    "prompt_embeds": torch.cat([o1.hidden_states[-2], o2.hidden_states[-2]], dim=-1)[0].cpu().half(),
                    "pooled_embeds": o2.text_embeds[0].cpu().half(),
                }
        torch.save(prompt_cache, prompt_cache_path)
        text_encoder_one.to("cpu")
        text_encoder_two.to("cpu")
        del prompt_cache
        torch.cuda.empty_cache()
    else:
        print(f"[{split_name}] using existing prompt cache: {prompt_cache_path}")

    return latent_cache_path, prompt_cache_path, records


class CachedSDXLDataset(Dataset):
    """Reads precomputed latents + text embeddings; draws a fresh VAE-latent sample and a
    deterministic (per image id + epoch) caption-variant selection each access."""

    def __init__(self, latent_cache_path: Path, prompt_cache_path: Path, records: list[dict], target_size: int):
        self.latent_cache = torch.load(latent_cache_path, map_location="cpu", weights_only=False)
        self.prompt_cache = torch.load(prompt_cache_path, map_location="cpu", weights_only=False)
        self.records = records
        self.target_size = target_size
        self.epoch = 0
        expected_ids = [record["image_id"] for record in records]
        if self.latent_cache.get("image_ids") != expected_ids:
            raise ValueError("Latent cache image order does not match the caption records; refusing stale cache")
        if self.latent_cache.get("target_size") != target_size:
            raise ValueError("Latent cache resolution does not match the configured target size")

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        rec = self.records[index]
        moments = self.latent_cache["moments"][index]
        mean, logvar = moments[:4].float(), moments[4:].float().clamp(-30, 20)
        std = torch.exp(0.5 * logvar)
        latent = (mean + std * torch.randn_like(mean)) * self.latent_cache["scaling_factor"]

        variant_idx = select_variant_index(rec["image_id"], self.epoch, len(rec["caption_variants"]))
        caption = rec["caption_variants"][variant_idx]
        prompt = self.prompt_cache[caption]

        time_ids = torch.tensor(
            [self.target_size, self.target_size, 0, 0, self.target_size, self.target_size], dtype=torch.float16
        )
        return {
            "latents": latent.half(),
            "prompt_embeds": prompt["prompt_embeds"],
            "pooled_embeds": prompt["pooled_embeds"],
            "time_ids": time_ids,
        }


# --------------------------------------------------------------------------------------
# EMA
# --------------------------------------------------------------------------------------

class LoraEMA:
    """EMA over the UNet's LoRA parameters only (docs/stage1_plan.md §8: cheap since LoRA
    weights are small; base/frozen weights are never touched)."""

    def __init__(self, unet, decay: float):
        self.decay = decay
        self.shadow = {
            name: param.detach().clone().float()
            for name, param in unet.named_parameters()
            if param.requires_grad
        }

    @torch.no_grad()
    def update(self, unet) -> None:
        for name, param in unet.named_parameters():
            if not param.requires_grad:
                continue
            self.shadow[name].mul_(self.decay).add_(param.detach().float(), alpha=1 - self.decay)

    @torch.no_grad()
    def copy_to(self, unet) -> None:
        for name, param in unet.named_parameters():
            if param.requires_grad and name in self.shadow:
                param.data.copy_(self.shadow[name].to(param.dtype))

    def state_dict(self) -> dict:
        return {"decay": self.decay, "shadow": {k: v.cpu() for k, v in self.shadow.items()}}

    def load_state_dict(self, state: dict) -> None:
        self.decay = float(state["decay"])
        if set(state["shadow"]) != set(self.shadow):
            raise ValueError("EMA checkpoint parameters do not match the current LoRA adapter")
        self.shadow = {name: value.float() for name, value in state["shadow"].items()}


# --------------------------------------------------------------------------------------
# Checkpointing
# --------------------------------------------------------------------------------------

def save_checkpoint(accelerator, unet, ema, cfg, run_dir: Path, step: int, epoch: int, split_manifest_hash: str, seed: int, keep_last_n: int) -> None:
    from diffusers.utils import convert_state_dict_to_diffusers
    from peft.utils import get_peft_model_state_dict
    from diffusers import StableDiffusionXLPipeline

    accelerator.wait_for_everyone()
    if not accelerator.is_main_process:
        return

    full_checkpoint_dir = run_dir / f"checkpoint-{step}"
    accelerator.save_state(str(full_checkpoint_dir))

    lora_dir = run_dir / "lora_weights" / f"step_{step}"
    raw_unet = accelerator.unwrap_model(unet)
    state = convert_state_dict_to_diffusers(get_peft_model_state_dict(raw_unet))
    StableDiffusionXLPipeline.save_lora_weights(str(lora_dir), unet_lora_layers=state, safe_serialization=True)

    if ema is not None:
        torch.save(ema.state_dict(), lora_dir / "ema_shadow.pt")

    from scripts.utils.manifest import sha256_file
    from scripts.utils.splits import read_split_manifest, resolve_splits_dir
    namespace = str(cfg.split.namespace)
    manifest = read_split_manifest(namespace)
    split_dir = resolve_splits_dir(namespace)
    source_csv = Path(manifest["input_csv_path"])
    metadata = build_checkpoint_metadata(
        step=step,
        epoch=epoch,
        config=dict(cfg),
        split_manifest_hash=split_manifest_hash,
        seed=seed,
        extra={
            "split_namespace": namespace,
            "split_manifest_version": manifest["manifest_version"],
            "source_csv_hash": manifest.get("source_csv_sha256") or (sha256_file(source_csv) if source_csv.is_file() else None),
            "gen_train_csv_hash": sha256_file(split_dir / "gen_train.csv"),
            "gen_val_csv_hash": sha256_file(split_dir / "gen_val.csv"),
        },
    )
    write_json(full_checkpoint_dir / "metadata.json", metadata)
    write_json(lora_dir / "metadata.json", metadata)

    write_json(run_dir / "latest.json", {"checkpoint_dir": str(full_checkpoint_dir), "step": step, "epoch": epoch})

    # Retention: keep only the last N full resumable checkpoints; lora_weights/* is kept in full.
    full_checkpoints = sorted(
        [p for p in run_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]),
    )
    for old in full_checkpoints[:-keep_last_n]:
        import shutil

        shutil.rmtree(old, ignore_errors=True)

    print(f"Saved checkpoint at step {step} -> {full_checkpoint_dir} (+ lora_weights snapshot)")


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------

def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--resume-from", type=str, default=None, help="Full checkpoint directory to resume from")
    parser.add_argument("--run-id", type=str, default=None, help="Existing run_id to continue writing into")
    parser.add_argument("--max-samples-per-split", type=int, default=None, help="Diagnostic/smoke-test limit; uses separate caches")
    parser.add_argument("overrides", nargs="*", help="OmegaConf dotlist overrides, e.g. training.train_batch_size=4")
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    from accelerate import Accelerator
    from accelerate.utils import ProjectConfiguration

    cfg = load_stage1_config(overrides=args.overrides)
    ensure_dirs(cfg)
    validate_training_inputs(cfg)
    set_seed(cfg.run.seed)

    checkpoints_root = Path(cfg.paths.checkpoints_dir)
    run_id = args.run_id or make_run_id(dict(cfg))
    run_dir = checkpoints_root / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    write_json(checkpoints_root / "latest_run.json", {"run_id": run_id})

    from scripts.utils.manifest import sha256_file
    from scripts.utils.splits import read_split_manifest, resolve_splits_dir
    namespace = str(cfg.split.namespace)
    split_manifest = read_split_manifest(namespace)
    split_manifest_hash = split_manifest["manifest_hash"]
    split_dir = resolve_splits_dir(namespace)
    source_csv = Path(split_manifest["input_csv_path"])
    stage1_provenance = {
        "split_namespace": namespace,
        "split_manifest_version": split_manifest["manifest_version"],
        "split_manifest_hash": split_manifest_hash,
        "source_csv_hash": split_manifest.get("source_csv_sha256") or (sha256_file(source_csv) if source_csv.is_file() else None),
        "gen_train_csv_hash": sha256_file(split_dir / "gen_train.csv"),
        "gen_val_csv_hash": sha256_file(split_dir / "gen_val.csv"),
    }

    log_dir = Path(cfg.paths.logs_dir) / run_id
    trackers = []
    if cfg.logging.tensorboard:
        trackers.append("tensorboard")
    if cfg.logging.wandb.enabled:
        trackers.append("wandb")

    precision = str(cfg.training.precision).lower()
    accelerate_precision = "no" if precision == "fp32" else precision
    if accelerate_precision not in {"no", "fp16", "bf16"}:
        raise ValueError(
            f"Unsupported training.precision={cfg.training.precision!r}; "
            "expected one of: fp32, fp16, bf16"
        )

    accelerator = Accelerator(
        gradient_accumulation_steps=cfg.training.gradient_accumulation_steps,
        mixed_precision=accelerate_precision,
        log_with=trackers or None,
        project_config=ProjectConfiguration(project_dir=str(run_dir), logging_dir=str(log_dir)),
    )
    if trackers:
        accelerator.init_trackers("stage1_lora_sdxl", config=tracker_hparams(cfg))

    weight_dtype = {
        "fp32": torch.float32,
        "fp16": torch.float16,
        "bf16": torch.bfloat16,
    }[precision]
    models = build_models(cfg, weight_dtype)

    train_latent_cache, train_prompt_cache, train_records = build_or_load_cache(cfg, models, "gen_train", accelerator.device, args.max_samples_per_split)
    val_latent_cache, val_prompt_cache, val_records = build_or_load_cache(cfg, models, "gen_val", accelerator.device, args.max_samples_per_split)

    train_dataset = CachedSDXLDataset(train_latent_cache, train_prompt_cache, train_records, cfg.data.resolution)
    val_dataset = CachedSDXLDataset(val_latent_cache, val_prompt_cache, val_records, cfg.data.resolution)
    train_loader = DataLoader(train_dataset, batch_size=cfg.training.train_batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.training.train_batch_size, shuffle=False, num_workers=0, pin_memory=True)

    unet = models["unet"]
    trainable_params = [p for p in unet.parameters() if p.requires_grad]

    optimizer = None
    if cfg.optimizer.name == "adamw_8bit":
        try:
            import bitsandbytes as bnb

            optimizer = bnb.optim.AdamW8bit(trainable_params, lr=cfg.optimizer.learning_rate, weight_decay=cfg.optimizer.weight_decay)
        except Exception as e:
            print(f"bitsandbytes AdamW8bit unavailable ({e}); falling back to torch.optim.AdamW (docs/stage1_plan.md §3/§12).")
    if optimizer is None:
        optimizer = torch.optim.AdamW(trainable_params, lr=cfg.optimizer.learning_rate, weight_decay=cfg.optimizer.weight_decay)

    from diffusers.optimization import get_scheduler

    lr_scheduler = get_scheduler(
        cfg.optimizer.lr_scheduler,
        optimizer=optimizer,
        num_warmup_steps=cfg.optimizer.lr_warmup_steps,
        num_training_steps=cfg.training.max_train_steps,
    )

    unet, optimizer, train_loader, val_loader, lr_scheduler = accelerator.prepare(
        unet, optimizer, train_loader, val_loader, lr_scheduler
    )

    ema = LoraEMA(accelerator.unwrap_model(unet), cfg.training.ema.decay) if cfg.training.ema.enabled else None
    if ema is not None:
        accelerator.register_for_checkpointing(ema)

    global_step = 0
    if args.resume_from:
        resumed_metadata = read_json(Path(args.resume_from) / "metadata.json")
        for key, value in stage1_provenance.items():
            if resumed_metadata.get(key) != value:
                raise ValueError(f"Checkpoint {key} differs from current data ({resumed_metadata.get(key)!r} != {value!r}); refusing unsafe resume")
        accelerator.load_state(args.resume_from)
        global_step = resumed_metadata["step"]
        print(f"Resumed from {args.resume_from} at step {global_step}")

    noise_scheduler = models["noise_scheduler"]
    progress = tqdm(total=cfg.training.max_train_steps, initial=global_step, disable=not accelerator.is_local_main_process)
    epoch = global_step // max(1, len(train_loader))

    unet.train()
    while global_step < cfg.training.max_train_steps:
        train_dataset.set_epoch(epoch)
        for batch in train_loader:
            with accelerator.accumulate(unet):
                latents = batch["latents"].to(accelerator.device, weight_dtype)
                noise = torch.randn_like(latents)
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device
                ).long()
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                added_cond_kwargs = {
                    "text_embeds": batch["pooled_embeds"].to(accelerator.device, weight_dtype),
                    "time_ids": batch["time_ids"].to(accelerator.device, weight_dtype),
                }
                model_pred = unet(
                    noisy_latents,
                    timesteps,
                    encoder_hidden_states=batch["prompt_embeds"].to(accelerator.device, weight_dtype),
                    added_cond_kwargs=added_cond_kwargs,
                    return_dict=False,
                )[0]
                target = (
                    noise if noise_scheduler.config.prediction_type == "epsilon"
                    else noise_scheduler.get_velocity(latents, noise, timesteps)
                )
                loss = F.mse_loss(model_pred.float(), target.float())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(trainable_params, cfg.optimizer.max_grad_norm)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1
                progress.update(1)
                progress.set_postfix(loss=f"{loss.item():.4f}")
                if ema is not None:
                    ema.update(accelerator.unwrap_model(unet))

                if trackers:
                    accelerator.log({"train/loss": loss.item(), "train/lr": lr_scheduler.get_last_lr()[0]}, step=global_step)

                if global_step % cfg.checkpointing.save_every_n_steps == 0:
                    save_checkpoint(
                        accelerator, unet, ema, cfg, run_dir, global_step, epoch,
                        split_manifest_hash, cfg.run.seed, cfg.checkpointing.keep_last_n_full_checkpoints,
                    )

                if global_step % cfg.validation.val_every_n_steps == 0:
                    val_loss = run_validation(accelerator, unet, val_loader, noise_scheduler, weight_dtype)
                    if trackers:
                        accelerator.log({"val/loss": val_loss}, step=global_step)
                    print(f"[step {global_step}] val_loss={val_loss:.4f}")
                    unet.train()

            if global_step >= cfg.training.max_train_steps:
                break
        epoch += 1

    progress.close()
    save_checkpoint(
        accelerator, unet, ema, cfg, run_dir, global_step, epoch,
        split_manifest_hash, cfg.run.seed, cfg.checkpointing.keep_last_n_full_checkpoints,
    )
    final_dir = run_dir / "final"
    from diffusers.utils import convert_state_dict_to_diffusers
    from peft.utils import get_peft_model_state_dict
    from diffusers import StableDiffusionXLPipeline

    if accelerator.is_main_process:
        raw_unet = accelerator.unwrap_model(unet)
        if ema is not None:
            ema.copy_to(raw_unet)
        state = convert_state_dict_to_diffusers(get_peft_model_state_dict(raw_unet))
        StableDiffusionXLPipeline.save_lora_weights(str(final_dir), unet_lora_layers=state, safe_serialization=True)
        write_json(
            final_dir / "metadata.json",
            build_checkpoint_metadata(global_step, epoch, dict(cfg), split_manifest_hash, cfg.run.seed,
                                      extra={"ema_applied": ema is not None, **stage1_provenance}),
        )
        print(f"Final LoRA weights (EMA applied: {ema is not None}) -> {final_dir}")

    if trackers:
        accelerator.end_training()

    return 0


@torch.no_grad()
def run_validation(accelerator, unet, val_loader, noise_scheduler, weight_dtype) -> float:
    unet.eval()
    total_loss = 0.0
    n_batches = 0
    for batch in val_loader:
        latents = batch["latents"].to(accelerator.device, weight_dtype)
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        added_cond_kwargs = {
            "text_embeds": batch["pooled_embeds"].to(accelerator.device, weight_dtype),
            "time_ids": batch["time_ids"].to(accelerator.device, weight_dtype),
        }
        model_pred = unet(
            noisy_latents, timesteps, encoder_hidden_states=batch["prompt_embeds"].to(accelerator.device, weight_dtype),
            added_cond_kwargs=added_cond_kwargs, return_dict=False,
        )[0]
        target = noise if noise_scheduler.config.prediction_type == "epsilon" else noise_scheduler.get_velocity(latents, noise, timesteps)
        loss = F.mse_loss(model_pred.float(), target.float())
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


if __name__ == "__main__":
    raise SystemExit(main())


# Stage 2 — Synthetic generation


## `scripts/generate/01_sample_label_recipes.py`


In [ ]:
# SOURCE FILE: scripts/generate/01_sample_label_recipes.py
#!/usr/bin/env python3
"""Stage 2a — sample the synthetic label-recipe table (docs/stages2_to_5_plan.md §3).

Produces the `intended_label_vector` table Stage 2b generates from. These are conditioning
intents, NOT ground truth: whether the resulting pixels actually agree is measured separately by
ASISM's agreement signal (§4.5).

Recipe validity is an auditable procedure, not an assertion (§3):
  1. empirical co-occurrence mined from gen_train at >= min_support_patients;
  2. medical-rule overrides, both directions (allow-list and block-list);
  3. No Finding recipes are the ALL-ZERO vector over GENERATION_TARGET_LABELS (the primary
     disease labels plus any excluded-from-primary-for-support-only labels, plan §3 revision
     note), mutually exclusive with any positive pathology;
  4. Support Devices is a context attribute only — it never enters the disease vector;
  5. no -1 (uncertain) intent is ever encoded;
  6. frontal-only, matching Stage 1's view_filter;
  7. per-label quotas oversample rare classes subject to the support rule.

EVERY accept/reject decision is written to recipe_decisions.jsonl with its reason — nothing is
silently dropped, mirroring Stage 1's preprocessing-log discipline.

Usage:
    python scripts/generate/01_sample_label_recipes.py [--namespace dev] [--limit N]
"""

from __future__ import annotations

import argparse
import json
import os
import random
import sys
import tempfile
from collections import Counter
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.labels import (  # noqa: E402
    GENERATION_TARGET_LABELS,
    normalize_label,
    patient_level_support,
)
from scripts.utils.artifact_contracts import config_sha256, current_code_identity_hash, namespace_identity, stage2_paths  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import resolve_splits_dir  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402

DEVICE_LABEL = "Support Devices"


def load_stage2_config():
    return load_named_config("stage2_generation.yaml", "stage2")


def mine_cooccurrence(frame: pd.DataFrame, min_support_patients: int) -> dict[frozenset, int]:
    """Count how many distinct real PATIENTS exhibit each positive-label combination.

    Patient-level (not image-level) so a single patient with many studies cannot manufacture
    apparent support for a combination on their own.
    """
    combo_patients: dict[frozenset, set[str]] = {}

    for row in frame.to_dict("records"):
        positives = frozenset(
            label
            for label in GENERATION_TARGET_LABELS
            if normalize_label(row.get(label)) == 1
        )
        if not positives:
            continue
        combo_patients.setdefault(positives, set()).add(row["patient_id"])

    return {
        combo: len(patients)
        for combo, patients in combo_patients.items()
        if len(patients) >= min_support_patients
    }


def compute_label_quotas(support: pd.DataFrame, recipe_cfg) -> dict[str, int]:
    """Per-label target counts, oversampling rare labels relative to real prevalence (§3.7).

    target = base_quota * (median_prevalence / prevalence) ** rarity_exponent, clipped.
    rarity_exponent=0 reduces this to a flat base_quota for every label.
    """
    prevalence = {}
    for row in support.to_dict("records"):
        total = max(row["positive_patients"] + row["negative_patients"], 1)
        prevalence[row["label"]] = max(row["positive_patients"] / total, 1e-6)

    values = sorted(prevalence.values())
    median = values[len(values) // 2] if values else 1.0

    quotas = {}
    for label, rate in prevalence.items():
        scale = (median / rate) ** float(recipe_cfg.rarity_exponent)
        target = int(round(float(recipe_cfg.base_quota) * scale))
        quotas[label] = int(
            min(max(target, int(recipe_cfg.min_per_label)), int(recipe_cfg.max_per_label))
        )
    return quotas


def blocked_by_medical_rule(combo: frozenset, block_rules: list[list[str]]) -> str | None:
    """A block rule fires when the recipe contains ALL labels in the rule — i.e. the rule describes
    a contradictory co-presentation, not merely one suspicious label."""
    for rule in block_rules:
        rule_set = frozenset(rule)
        if rule_set and rule_set.issubset(combo):
            return f"medical_rule_block:{sorted(rule_set)}"
    return None


def build_recipes(
    frame: pd.DataFrame,
    config,
    seed: int,
) -> tuple[list[dict], list[dict], dict]:
    """Return (accepted_recipes, decisions, stats). `decisions` records every candidate considered
    with an explicit accept/reject reason."""
    recipe_cfg = config.recipes
    rng = random.Random(seed)

    support = patient_level_support(frame, GENERATION_TARGET_LABELS)
    quotas = compute_label_quotas(support, recipe_cfg)
    cooccurrence = mine_cooccurrence(frame, int(recipe_cfg.min_support_patients))

    allow_rules = [frozenset(rule) for rule in recipe_cfg.medical_rule_allow]
    block_rules = [list(rule) for rule in recipe_cfg.medical_rule_block]
    max_positive = int(recipe_cfg.max_positive_labels)

    # Candidate pool: mined combinations within the size cap, plus every single label (so a rare
    # label is never unreachable just because it rarely co-occurs), plus explicit allow-rules.
    candidates: dict[frozenset, str] = {}
    for combo in cooccurrence:
        if len(combo) <= max_positive:
            candidates[combo] = "empirical_cooccurrence"
    for label in GENERATION_TARGET_LABELS:
        candidates.setdefault(frozenset({label}), "single_label_guaranteed")
    for rule in allow_rules:
        if rule:
            candidates[rule] = "medical_rule_allow"

    decisions: list[dict] = []
    accepted_combos: list[tuple[frozenset, str]] = []

    for combo, origin in sorted(candidates.items(), key=lambda item: (len(item[0]), sorted(item[0]))):
        block_reason = blocked_by_medical_rule(combo, block_rules)
        if block_reason and origin != "medical_rule_allow":
            decisions.append(
                {
                    "combination": sorted(combo),
                    "origin": origin,
                    "accepted": False,
                    "reason": block_reason,
                    "support_patients": cooccurrence.get(combo, 0),
                }
            )
            continue
        if len(combo) > max_positive:
            decisions.append(
                {
                    "combination": sorted(combo),
                    "origin": origin,
                    "accepted": False,
                    "reason": f"exceeds_max_positive_labels:{max_positive}",
                    "support_patients": cooccurrence.get(combo, 0),
                }
            )
            continue

        support_patients = cooccurrence.get(combo, 0)
        if origin == "single_label_guaranteed" and support_patients == 0:
            reason = "accepted_single_label_below_support_but_required_for_coverage"
        elif origin == "medical_rule_allow":
            reason = "accepted_medical_rule_allow"
        else:
            reason = "accepted_empirical_support"

        decisions.append(
            {
                "combination": sorted(combo),
                "origin": origin,
                "accepted": True,
                "reason": reason,
                "support_patients": support_patients,
            }
        )
        accepted_combos.append((combo, reason))

    if not accepted_combos:
        raise SystemExit(
            "No label combinations survived recipe validation — check recipes.min_support_patients "
            "and medical_rule_block in configs/stage2_generation.yaml."
        )

    singles = [c for c, _ in accepted_combos if len(c) == 1]
    multis = [c for c, _ in accepted_combos if len(c) > 1]

    total_disease_target = sum(quotas.values())
    no_finding_share = float(recipe_cfg.no_finding_proportion)
    total_target = int(round(total_disease_target / max(1.0 - no_finding_share, 1e-6)))
    no_finding_target = total_target - total_disease_target

    age_buckets = list(recipe_cfg.age_buckets)
    sexes = list(recipe_cfg.sexes)
    device_rate = float(recipe_cfg.support_devices_proportion)
    single_share = float(recipe_cfg.single_label_proportion)

    recipes: list[dict] = []
    remaining = dict(quotas)
    index = 0

    def emit(combo: frozenset) -> None:
        nonlocal index
        age = rng.choice(age_buckets)
        sex = rng.choice(sexes)
        has_device = rng.random() < device_rate
        intended = {label: (1 if label in combo else 0) for label in GENERATION_TARGET_LABELS}
        recipe_id = f"recipe_{index:07d}"
        recipes.append(
            {
                "recipe_id": recipe_id,
                "intended_label_vector": json.dumps(intended, sort_keys=True),
                "positive_labels": json.dumps(sorted(combo)),
                "num_positive_labels": len(combo),
                "is_no_finding": len(combo) == 0,
                "age_bucket_start": age,
                "sex": sex,
                "support_devices": int(has_device),
                "view": "Frontal",
            }
        )
        index += 1

    # Disease recipes: draw until every label's quota is met, respecting the single/multi mix.
    guard = 0
    max_iterations = total_disease_target * 20 + 1000
    while any(count > 0 for count in remaining.values()) and guard < max_iterations:
        guard += 1
        use_single = rng.random() < single_share or not multis
        pool = singles if use_single else multis
        # Prefer combinations that still serve an unmet quota.
        hungry = [c for c in pool if any(remaining.get(label, 0) > 0 for label in c)]
        combo = rng.choice(hungry) if hungry else rng.choice(pool)
        emit(combo)
        for label in combo:
            if label in remaining:
                remaining[label] = max(0, remaining[label] - 1)

    for _ in range(max(0, no_finding_target)):
        emit(frozenset())

    rng.shuffle(recipes)
    for position, recipe in enumerate(recipes):
        recipe["recipe_id"] = f"recipe_{position:07d}"

    stats = {
        "num_recipes": len(recipes),
        "num_disease_recipes": sum(1 for r in recipes if not r["is_no_finding"]),
        "num_no_finding_recipes": sum(1 for r in recipes if r["is_no_finding"]),
        "num_candidate_combinations": len(candidates),
        "num_accepted_combinations": len(accepted_combos),
        "num_rejected_combinations": sum(1 for d in decisions if not d["accepted"]),
        "per_label_quota": quotas,
        "per_label_emitted": {
            label: sum(
                1
                for r in recipes
                if json.loads(r["intended_label_vector"]).get(label) == 1
            )
            for label in GENERATION_TARGET_LABELS
        },
        "single_vs_multi": dict(
            Counter("single" if r["num_positive_labels"] == 1 else
                    ("no_finding" if r["is_no_finding"] else "multi") for r in recipes)
        ),
        "quota_satisfied": all(count == 0 for count in remaining.values()),
    }
    return recipes, decisions, stats


def atomic_write_jsonl(path: Path, records: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_name, path)
    finally:
        Path(temp_name).unlink(missing_ok=True)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    try:
        frame.to_csv(temp_name, index=False)
        os.replace(temp_name, path)
    finally:
        Path(temp_name).unlink(missing_ok=True)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None, help="Override split_namespace from config")
    parser.add_argument("--limit", type=int, default=None, help="Cap recipe count (smoke testing)")
    args = parser.parse_args()

    config = load_stage2_config()
    namespace = args.namespace or str(config.split_namespace)

    provenance = namespace_identity(namespace)
    gen_train = load_split(
        "gen_train", namespace, purpose="schema_validation", caller="01_sample_label_recipes"
    )

    recipes, decisions, stats = build_recipes(gen_train, config, seed=int(config.generation.seed))

    if args.limit is not None and len(recipes) > args.limit:
        recipes = recipes[: args.limit]
        for position, recipe in enumerate(recipes):
            recipe["recipe_id"] = f"recipe_{position:07d}"
        # Recompute every count that describes the emitted set, so the manifest can never record
        # a mix that contradicts num_recipes.
        stats["limited_to"] = args.limit
        stats["quota_satisfied"] = False  # truncation invalidates the quota guarantee
        stats["num_recipes"] = len(recipes)
        stats["num_disease_recipes"] = sum(1 for r in recipes if not r["is_no_finding"])
        stats["num_no_finding_recipes"] = sum(1 for r in recipes if r["is_no_finding"])
        stats["per_label_emitted"] = {
            label: sum(
                1 for r in recipes if json.loads(r["intended_label_vector"]).get(label) == 1
            )
            for label in GENERATION_TARGET_LABELS
        }
        stats["single_vs_multi"] = dict(
            Counter(
                "single" if r["num_positive_labels"] == 1
                else ("no_finding" if r["is_no_finding"] else "multi")
                for r in recipes
            )
        )

    paths = stage2_paths(config, namespace)
    recipes_path = paths["recipes_path"]
    decisions_path = paths["recipe_decisions_path"]

    atomic_write_csv(recipes_path, pd.DataFrame(recipes))
    atomic_write_jsonl(decisions_path, decisions)

    recipe_config_block = {
        "recipes": OmegaConf.to_container(config.recipes, resolve=True),
        "split_provenance": provenance,
    }
    manifest = {
        "stage": "stage2_recipes",
        "recipes_path": str(recipes_path),
        "decisions_path": str(decisions_path),
        "schema_version": 2,
        "config_hash": hash_dict(recipe_config_block, length=64),
        "recipe_config_sha256": config_sha256(config.recipes),
        "git_commit_hash": get_git_commit_hash(),
        "code_identity_sha256": current_code_identity_hash(),
        "split_provenance": provenance,
        "split_namespace": namespace,
        "namespace_class": provenance["namespace_class"],
        "split_manifest_hash": provenance["split_manifest_hash"],
        "source_gen_train_csv_sha256": sha256_file(resolve_splits_dir(namespace) / "gen_train.csv"),
        "recipes_csv_sha256": sha256_file(recipes_path),
        "decisions_jsonl_sha256": sha256_file(decisions_path),
        "recipe_config": recipe_config_block["recipes"],
        "stats": stats,
    }
    write_json(paths["recipes_manifest"], manifest)

    print(f"Split namespace:    {namespace} (manifest {provenance['split_manifest_hash']})", flush=True)
    print(f"Candidate combos:   {stats['num_candidate_combinations']}", flush=True)
    print(f"  accepted:         {stats['num_accepted_combinations']}", flush=True)
    print(f"  rejected:         {stats['num_rejected_combinations']}", flush=True)
    print(f"Recipes emitted:    {stats['num_recipes']}", flush=True)
    print(f"  disease:          {stats['num_disease_recipes']}", flush=True)
    print(f"  no-finding:       {stats['num_no_finding_recipes']}", flush=True)
    print(f"  mix:              {stats['single_vs_multi']}", flush=True)
    print(f"  quotas satisfied: {stats['quota_satisfied']}", flush=True)
    print(f"Wrote:              {recipes_path}", flush=True)
    print(f"Decisions log:      {decisions_path}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/generate/02_generate_synthetic_images.py`


In [ ]:
# SOURCE FILE: scripts/generate/02_generate_synthetic_images.py
#!/usr/bin/env python3
"""Stage 2b — generate synthetic CXRs from the recipe table (docs/stages2_to_5_plan.md §3).

Loads base SDXL + the explicitly-pinned Stage 1 LoRA weights and generates one image per recipe.
Captions come from scripts/utils/caption_builder.py VERBATIM — the same module Stage 1 trained
with, because any train/generation phrasing drift degrades conditioning fidelity
(docs/stage1_plan.md §11).

Resumable and idempotent on the same terms as 03_preprocess_images.py: an existing output is
skipped only after being decoded and validated (correct format, mode, and resolution), never on
mere existence; every write is atomic (temp file + os.replace) so an interrupted pod cannot leave a
half-written JPEG.

PILOT GATE (§3, FROZEN): `--mode full` refuses to start unless a pilot_approval_manifest.json
exists, records a completed pilot, records passing automatic checks, and carries an explicit human
approval record. Approval is a separate deliberate command; this script never self-approves.

Usage:
    python scripts/generate/02_generate_synthetic_images.py --mode pilot
    python scripts/generate/02_generate_synthetic_images.py --approve-pilot --reviewer "Walaa"
    python scripts/generate/02_generate_synthetic_images.py --mode full
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import CaptionConfig, build_caption  # noqa: E402
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.experiment_registry import ExperimentRun  # noqa: E402
from scripts.utils.labels import GENERATION_TARGET_LABELS  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    ArtifactContractError, config_sha256, current_code_identity_hash, namespace_identity,
    require_manifest_fields, stage2_paths,
)
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json, sha256_directory, sha256_file, write_json  # noqa: E402

SCHEMA_VERSION = 2


def load_stage2_config():
    return load_named_config("stage2_generation.yaml", "stage2")


def derive_seed(base_seed: int, recipe_id: str) -> int:
    """Deterministic per-image seed: same recipe id always regenerates the same image, so a resumed
    run reproduces exactly what an uninterrupted run would have produced."""
    digest = hashlib.sha256(f"{base_seed}:{recipe_id}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


def recipe_to_caption_row(recipe: dict) -> dict:
    """Map a recipe row into the dict shape caption_builder expects.

    Support Devices is injected here as a CONTEXT attribute (it drives the device clause) while
    staying out of the primary disease vector — §3.4.
    """
    intended = json.loads(recipe["intended_label_vector"])
    row = {label: intended.get(label, 0) for label in GENERATION_TARGET_LABELS}
    row["No Finding"] = 1 if bool(recipe["is_no_finding"]) else 0
    row["Support Devices"] = int(recipe["support_devices"])
    row["Age"] = int(recipe["age_bucket_start"])
    row["Sex"] = str(recipe["sex"])
    row["Frontal/Lateral"] = str(recipe["view"])
    row["AP/PA"] = "AP"
    return row


def validate_generated_image(path: Path, resolution: int) -> tuple[bool, str | None]:
    """Fully decode and check an existing output. Existence alone is never sufficient."""
    if not path.is_file():
        return False, "missing"
    try:
        with Image.open(path) as image:
            if image.format != "JPEG":
                return False, f"format_{image.format or 'unknown'}"
            if image.mode != "RGB":
                return False, f"mode_{image.mode}"
            if image.size != (resolution, resolution):
                return False, f"size_{image.size[0]}x{image.size[1]}"
            image.load()
    except Exception as exc:
        return False, f"unreadable_{type(exc).__name__}"
    return True, None


def atomic_save_jpeg(image: Image.Image, destination: Path, quality: int = 95) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".tmp", dir=destination.parent)
    os.close(fd)
    temp_path = Path(temp_name)
    try:
        image.save(temp_path, format="JPEG", quality=quality)
        os.replace(temp_path, destination)
    finally:
        temp_path.unlink(missing_ok=True)


def check_upstream_ready(config) -> Path:
    """Validate Stage 2's real upstream gate (an explicit, existing Stage 1 LoRA checkpoint) using
    only config and the filesystem — before importing the GPU stack, so a missing checkpoint fails
    with a clear message rather than an unrelated ImportError."""
    lora_dir = config.checkpoint.lora_weights_dir
    if not lora_dir:
        raise SystemExit(
            "UPSTREAM GATE: checkpoint.lora_weights_dir is not set in configs/stage2_generation.yaml.\n"
            "Stage 2 requires an EXPLICIT Stage 1 LoRA checkpoint (never 'latest') so generation is "
            "reproducible — see docs/stages2_to_5_plan.md §3.\n"
            "Set it to e.g. checkpoints/stage1_lora_sdxl/<run_id>/lora_weights/step_20000"
        )
    lora_path = Path(lora_dir)
    if not lora_path.is_dir():
        raise SystemExit(
            f"UPSTREAM GATE: LoRA weights directory not found: {lora_path}\n"
            "Stage 1 must produce a checkpoint before Stage 2 can generate."
        )
    return lora_path


def validate_generation_inputs(config, namespace: str, paths: dict[str, Path]) -> dict:
    if not paths["recipes_path"].is_file():
        legacy = Path(config.paths.synthetic_root) / "label_recipes.csv"
        if legacy.is_file():
            raise ArtifactContractError(
                f"Refusing stale unnamespaced Stage 2 recipes at {legacy}. Regenerate recipes into {paths['root']} for split run {namespace!r}."
            )
        raise ArtifactContractError(f"Missing namespaced recipe table: {paths['recipes_path']}")
    identity = namespace_identity(namespace)
    expected = {
        "schema_version": 2,
        "split_namespace": namespace,
        "namespace_class": identity["namespace_class"],
        "split_manifest_hash": identity["split_manifest_hash"],
        "recipe_config_sha256": config_sha256(config.recipes),
        "recipes_csv_sha256": sha256_file(paths["recipes_path"]),
        "code_identity_sha256": current_code_identity_hash(),
    }
    recipe_manifest = require_manifest_fields(paths["recipes_manifest"], expected, "Stage 2 recipe")
    lora_path = check_upstream_ready(config)
    lora_hash = sha256_directory(lora_path)
    metadata_path = lora_path / "metadata.json"
    if not metadata_path.is_file():
        raise ArtifactContractError(f"LoRA checkpoint has no metadata.json: {lora_path}")
    metadata = read_json(metadata_path)
    for key, value in (("split_namespace", namespace), ("split_manifest_hash", identity["split_manifest_hash"])):
        if metadata.get(key) != value:
            raise ArtifactContractError(f"LoRA checkpoint {key} mismatch: expected {value!r}, got {metadata.get(key)!r}")
    return {**identity, "recipes_csv_sha256": expected["recipes_csv_sha256"],
            "recipes_manifest_sha256": sha256_file(paths["recipes_manifest"]),
            "lora_checkpoint_sha256": lora_hash, "lora_metadata_sha256": sha256_file(metadata_path),
            "generation_config_sha256": config_sha256(config.generation),
            "stage2_config_sha256": config_sha256(config), "code_identity_sha256": current_code_identity_hash()}


def build_pipeline(config, stage1_cfg):
    """Load SDXL + the pinned LoRA weights. Imported lazily so the pilot gate and manifest logic
    remain testable on a machine with no GPU stack installed."""
    lora_path = check_upstream_ready(config)

    try:
        import torch
        from diffusers import DPMSolverMultistepScheduler, StableDiffusionXLPipeline
    except ImportError as exc:
        raise SystemExit(
            f"UPSTREAM GATE: the generation stack is not installed here ({exc}).\n"
            "Stage 2 generation is GPU work — run it on the RunPod pod after:\n"
            "  pip install -r environment/requirements.txt"
        ) from exc

    # Keep every SDXL component and the injected LoRA adapter in the same dtype.
    # The pinned SDXL checkpoint/LoRA path is float16-compatible; forcing bfloat16 here can
    # leave projection weights in fp16 and fail at inference with Half != BFloat16.
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    pipeline = StableDiffusionXLPipeline.from_pretrained(
        config.checkpoint.base_model_id,
        revision=config.checkpoint.revision,
        torch_dtype=dtype,
        cache_dir=str(stage1_cfg.paths.hf_cache_dir),
    )
    pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
    pipeline.load_lora_weights(str(lora_path))
    if torch.cuda.is_available():
        pipeline = pipeline.to(device="cuda", dtype=dtype)
    pipeline.set_progress_bar_config(disable=True)
    return pipeline


def run_generation(
    recipes: pd.DataFrame,
    config,
    stage1_cfg,
    images_dir: Path,
    manifest_path: Path,
    mode: str,
    artifact_provenance: dict,
) -> dict:
    """Generate images for `recipes`, skipping already-valid outputs. Returns counters."""
    resolution = int(config.generation.resolution)
    caption_config = CaptionConfig(
        age_bucket_width_years=int(stage1_cfg.captions.age_bucket_width_years),
        uncertain_label_policy=str(stage1_cfg.captions.uncertain_label_policy),
        no_finding_overrides_positives=bool(stage1_cfg.captions.no_finding_overrides_positives),
        num_paraphrase_variants=int(stage1_cfg.captions.num_paraphrase_variants),
        template_version=str(stage1_cfg.captions.template_version),
    )

    # Determine what still needs generating BEFORE loading the model, so a fully-complete resume
    # costs nothing and never touches the GPU.
    existing_rows: dict[str, dict] = {}
    if manifest_path.is_file():
        for line_number, line in enumerate(manifest_path.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            row = json.loads(line)
            image_id = str(row.get("image_id", ""))
            if not image_id or image_id in existing_rows:
                raise SystemExit(f"Generation manifest has a missing/duplicate image_id at line {line_number}: {manifest_path}")
            for key, expected in artifact_provenance.items():
                if row.get(key) != expected:
                    raise SystemExit(f"Generation manifest provenance mismatch for {image_id} at {key}; refusing stale reuse")
            existing_rows[image_id] = row

    pending = []
    retained_rows = []
    counts = {"generated": 0, "skipped_valid": 0, "failed": 0}
    for recipe in recipes.to_dict("records"):
        destination = images_dir / f"{recipe['recipe_id']}.jpg"
        valid, _ = validate_generated_image(destination, resolution)
        existing = existing_rows.get(str(recipe["recipe_id"]))
        if valid and existing is None:
            raise SystemExit(
                f"Unregistered generated image exists at {destination}; refusing to treat a stale/unpinned image as complete"
            )
        if valid:
            counts["skipped_valid"] += 1
            retained_rows.append(existing)
        else:
            pending.append(recipe)

    unknown = sorted(set(existing_rows) - set(recipes["recipe_id"].astype(str)))
    if unknown:
        raise SystemExit(f"Generation manifest contains {len(unknown)} IDs absent from the current recipe set: {unknown[:5]}")

    # Remove rows whose image failed validation before appending replacements. Publication is
    # atomic, so an interruption cannot create duplicate manifest rows on the next resume.
    if manifest_path.is_file() and len(retained_rows) != len(existing_rows):
        fd, temp_name = tempfile.mkstemp(prefix=f".{manifest_path.name}.", suffix=".tmp", dir=manifest_path.parent)
        os.close(fd)
        try:
            with open(temp_name, "w", encoding="utf-8") as handle:
                for row in retained_rows:
                    handle.write(json.dumps(row, ensure_ascii=False) + "\n")
            os.replace(temp_name, manifest_path)
        finally:
            Path(temp_name).unlink(missing_ok=True)

    print(
        f"[{mode}] {counts['skipped_valid']} already valid, {len(pending)} to generate",
        flush=True,
    )
    if not pending:
        return counts

    pipeline = build_pipeline(config, stage1_cfg)
    import torch  # available by construction once build_pipeline() returned

    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    generation_config_hash = hash_dict(OmegaConf.to_container(config.generation, resolve=True))
    base_seed = int(config.generation.seed)

    with open(manifest_path, "a", encoding="utf-8", buffering=1) as manifest_handle:
        progress = tqdm(pending, desc=f"[{mode}] generate", unit="image")
        for recipe in progress:
            recipe_id = recipe["recipe_id"]
            destination = images_dir / f"{recipe_id}.jpg"
            caption_row = recipe_to_caption_row(recipe)
            caption = build_caption(caption_row, caption_config, variant_index=0)
            seed = derive_seed(base_seed, recipe_id)

            try:
                generator = torch.Generator(
                    device="cuda" if torch.cuda.is_available() else "cpu"
                ).manual_seed(seed)
                result = pipeline(
                    prompt=caption,
                    negative_prompt=str(config.generation.negative_prompt) or None,
                    num_inference_steps=int(config.generation.num_inference_steps),
                    guidance_scale=float(config.generation.guidance_scale),
                    height=resolution,
                    width=resolution,
                    generator=generator,
                )
                image = result.images[0].convert("RGB")
                atomic_save_jpeg(image, destination)

                valid, reason = validate_generated_image(destination, resolution)
                if not valid:
                    counts["failed"] += 1
                    progress.set_postfix(counts, refresh=False)
                    continue

                counts["generated"] += 1
                manifest_handle.write(
                    json.dumps(
                        {
                            "image_id": recipe_id,
                            "schema_version": SCHEMA_VERSION,
                            "intended_label_vector": json.loads(recipe["intended_label_vector"]),
                            "is_no_finding": bool(recipe["is_no_finding"]),
                            "support_devices": int(recipe["support_devices"]),
                            "caption": caption,
                            "seed": seed,
                            "generation_config_hash": generation_config_hash,
                            "lora_weights_dir": str(config.checkpoint.lora_weights_dir),
                            **artifact_provenance,
                            "mode": mode,
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )
            except Exception as exc:  # noqa: BLE001 - one bad recipe must not kill a long run
                counts["failed"] += 1
                print(f"[{mode}] FAILED {recipe_id}: {type(exc).__name__}: {exc}", flush=True)
            progress.set_postfix(counts, refresh=False)
        progress.close()

    return counts


def run_pilot_checks(images_dir: Path, recipe_ids: list[str], config) -> dict:
    """Automatic artifact checks the pilot must pass before it is eligible for approval."""
    checks_cfg = config.pilot.checks
    resolution = int(config.generation.resolution)

    total = len(recipe_ids)
    valid = 0
    wrong_resolution = 0
    not_rgb = 0
    near_uniform = 0
    missing = 0

    for recipe_id in recipe_ids:
        path = images_dir / f"{recipe_id}.jpg"
        if not path.is_file():
            missing += 1
            continue
        try:
            with Image.open(path) as image:
                if image.mode != "RGB":
                    not_rgb += 1
                if image.size != (resolution, resolution):
                    wrong_resolution += 1
                array = np.asarray(image.convert("L"), dtype=np.float32)
                if array.std() < float(checks_cfg.blank_std_threshold):
                    near_uniform += 1
                valid += 1
        except Exception:
            missing += 1

    valid_fraction = valid / total if total else 0.0
    near_uniform_fraction = near_uniform / total if total else 1.0

    results = {
        "total": total,
        "valid": valid,
        "missing_or_unreadable": missing,
        "wrong_resolution": wrong_resolution,
        "not_rgb": not_rgb,
        "near_uniform": near_uniform,
        "valid_fraction": round(valid_fraction, 4),
        "near_uniform_fraction": round(near_uniform_fraction, 4),
    }
    passed = (
        valid_fraction >= float(checks_cfg.min_valid_image_fraction)
        and near_uniform_fraction <= float(checks_cfg.max_near_uniform_fraction)
        and (wrong_resolution == 0 if bool(checks_cfg.require_correct_resolution) else True)
        and (not_rgb == 0 if bool(checks_cfg.require_rgb) else True)
    )
    results["passed"] = passed
    results["thresholds"] = OmegaConf.to_container(checks_cfg, resolve=True)
    return results


def approve_pilot(config, paths: dict[str, Path], provenance: dict, reviewer: str, notes: str) -> int:
    """Record explicit human approval. Refuses if the pilot did not run or its checks did not pass.

    This is deliberately a separate command a person runs after looking at the images — the
    generation script never approves its own output.
    """
    approval_path = paths["pilot_approval_manifest"]
    if not approval_path.is_file():
        raise SystemExit(
            f"No pilot manifest at {approval_path}. Run --mode pilot first, review the images, "
            "then approve."
        )
    manifest = read_json(approval_path)
    if manifest.get("split_provenance") != provenance:
        raise SystemExit("Pilot provenance no longer matches the current split/recipe/checkpoint/config; rerun the pilot")
    pilot_manifest = paths["pilot_manifest"]
    if not pilot_manifest.is_file() or manifest.get("pilot_manifest_sha256") != sha256_file(pilot_manifest):
        raise SystemExit("Pilot generation manifest is missing or changed; approval refused")
    if not manifest.get("pilot_completed", False):
        raise SystemExit("Pilot did not complete — cannot approve.")
    if not manifest.get("checks", {}).get("passed", False):
        raise SystemExit(
            "Pilot automatic checks did not pass — cannot approve.\n"
            f"{json.dumps(manifest.get('checks', {}), indent=2)}"
        )

    manifest["approval"] = {
        "approved": True,
        "reviewer": reviewer,
        "notes": notes,
        "approved_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(approval_path, manifest)
    print(f"Pilot APPROVED by {reviewer} at {manifest['approval']['approved_at_utc']}", flush=True)
    print(f"Manifest: {approval_path}", flush=True)
    print("Full generation is now unblocked.", flush=True)
    return 0


def enforce_pilot_gate(config, paths: dict[str, Path], provenance: dict) -> dict:
    """FROZEN gate (§3): full generation refuses to start without a completed, checked, and
    explicitly approved pilot."""
    approval_path = paths["pilot_approval_manifest"]
    if not approval_path.is_file():
        raise SystemExit(
            "PILOT GATE: full generation refused.\n"
            f"  No pilot approval manifest at {approval_path}\n"
            "  Run:  python scripts/generate/02_generate_synthetic_images.py --mode pilot\n"
            "  Then review the pilot images and approve with --approve-pilot --reviewer <name>."
        )
    manifest = read_json(approval_path)
    if manifest.get("split_provenance") != provenance:
        raise SystemExit("PILOT GATE: approval provenance differs from current production inputs")
    if manifest.get("pilot_manifest_sha256") != sha256_file(paths["pilot_manifest"]):
        raise SystemExit("PILOT GATE: pilot manifest hash mismatch")
    if not manifest.get("pilot_completed", False):
        raise SystemExit("PILOT GATE: full generation refused — pilot did not complete.")
    if not manifest.get("checks", {}).get("passed", False):
        raise SystemExit(
            "PILOT GATE: full generation refused — pilot automatic checks did not pass.\n"
            f"{json.dumps(manifest.get('checks', {}), indent=2)}"
        )
    if not manifest.get("approval", {}).get("approved", False):
        raise SystemExit(
            "PILOT GATE: full generation refused — pilot completed and passed checks, but carries "
            "no explicit approval record.\n"
            "  Approve with: --approve-pilot --reviewer <name>"
        )
    return manifest


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=["pilot", "full"], default=None)
    parser.add_argument("--approve-pilot", action="store_true", help="Record explicit pilot approval")
    parser.add_argument("--reviewer", default=None, help="Required with --approve-pilot")
    parser.add_argument("--notes", default="", help="Optional reviewer notes")
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_stage2_config()
    stage1_cfg = load_stage1_config()
    namespace = args.namespace or str(config.split_namespace)
    paths = stage2_paths(config, namespace)

    if args.approve_pilot:
        if not args.reviewer:
            raise SystemExit("--approve-pilot requires --reviewer <name>")
        provenance = validate_generation_inputs(config, namespace, paths)
        return approve_pilot(config, paths, provenance, args.reviewer, args.notes)

    if not args.mode:
        raise SystemExit("--mode {pilot,full} is required (or use --approve-pilot)")

    paths = stage2_paths(config, namespace)
    provenance = validate_generation_inputs(config, namespace, paths)
    recipes_path = paths["recipes_path"]
    recipes = pd.read_csv(recipes_path)

    if args.mode == "pilot":
        images_dir = paths["pilot_images"]
        # Deterministic spanning subset: evenly strided so the pilot covers the recipe space
        # rather than sampling one corner of it.
        pilot_n = min(int(config.pilot.num_images), len(recipes))
        stride = max(1, len(recipes) // pilot_n)
        selected = recipes.iloc[::stride].head(pilot_n).reset_index(drop=True)
        manifest_path = paths["pilot_manifest"]
    else:
        enforce_pilot_gate(config, paths, provenance)
        images_dir = paths["images_dir"]
        selected = recipes
        manifest_path = paths["manifest_path"]

    images_dir.mkdir(parents=True, exist_ok=True)

    registry_config = {
        "stage2": OmegaConf.to_container(config, resolve=True),
        "split_provenance": provenance,
        "mode": args.mode,
    }
    with ExperimentRun(
        stage=f"stage2_generation_{args.mode}",
        config=registry_config,
        dataset_version=provenance["split_manifest_hash"],
    ) as run:
        counts = run_generation(
            selected, config, stage1_cfg, images_dir, manifest_path, args.mode, provenance
        )
        run.set_checkpoint_path(images_dir)
        run.set_metrics(counts)

    if args.mode == "pilot":
        checks = run_pilot_checks(images_dir, selected["recipe_id"].tolist(), config)
        approval_path = paths["pilot_approval_manifest"]
        existing_approval = {}
        if approval_path.is_file():
            existing_approval = read_json(approval_path).get("approval", {})
        write_json(
            approval_path,
            {
                "schema_version": SCHEMA_VERSION,
                "pilot_completed": True,
                "pilot_generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "num_pilot_images": len(selected),
                "counts": counts,
                "checks": checks,
                "images_dir": str(images_dir),
                "split_provenance": provenance,
                "pilot_manifest_sha256": sha256_file(manifest_path),
                "git_commit_hash": get_git_commit_hash(),
                # Approval is NEVER auto-granted here; a prior approval is preserved but a new
                # pilot run invalidates it by resetting to unapproved unless it already existed
                # for this same manifest.
                "approval": existing_approval if existing_approval else {"approved": False},
            },
        )
        print(f"\nPilot checks: {'PASSED' if checks['passed'] else 'FAILED'}", flush=True)
        print(json.dumps(checks, indent=2), flush=True)
        print(f"\nManifest: {approval_path}", flush=True)
        if checks["passed"]:
            print(
                "\nNext: review the pilot images yourself, then run:\n"
                "  python scripts/generate/02_generate_synthetic_images.py "
                "--approve-pilot --reviewer <your name>",
                flush=True,
            )
        return 0 if checks["passed"] else 3

    print(f"\n[full] generated={counts['generated']} skipped_valid={counts['skipped_valid']} "
          f"failed={counts['failed']}", flush=True)
    print(f"Images:   {images_dir}", flush=True)
    print(f"Manifest: {manifest_path}", flush=True)
    write_json(paths["generation_provenance"], provenance)
    if counts["failed"] == 0:
        manifest_rows = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        actual_ids = [str(row.get("image_id")) for row in manifest_rows]
        expected_ids = selected["recipe_id"].astype(str).tolist()
        if len(actual_ids) != len(set(actual_ids)) or set(actual_ids) != set(expected_ids):
            raise SystemExit("Generation completion refused: manifest IDs are duplicate, missing, or not the exact recipe partition")
        write_json(paths["generation_completion"], {**provenance, "status": "complete",
                   "generation_manifest_sha256": sha256_file(manifest_path),
                   "num_rows": len(manifest_rows), "image_id_set_sha256": hash_dict({"image_ids": sorted(actual_ids)}, length=64)})
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


# Stage 3 — ASISM


## `scripts/asism/01_compute_signals.py`


In [ ]:
# SOURCE FILE: scripts/asism/01_compute_signals.py
#!/usr/bin/env python3
"""Stage 3a — compute the five ASISM signals (docs/stages2_to_5_plan.md §4.1-§4.5).

Writes FIVE independent versioned parquet artifacts, one per signal:
    similarity_scores.parquet  iqa_scores.parquet  uncertainty_scores.parquet
    explainability_scores.parquet  agreement_scores.parquet

Independent artifacts (rather than one shared file) so a signal can be recomputed, audited, or
EXCLUDED by the Go/No-Go gate (§4.6) without disturbing the others — an excluded-but-valid signal's
artifact is retained for auditability.

IQA needs no model and runs anywhere. The other four query the frozen auxiliary classifier (§2) or
an embedding encoder, and fail cleanly at that upstream gate when it is absent.

Usage:
    python scripts/asism/01_compute_signals.py --signal all
    python scripts/asism/01_compute_signals.py --signal iqa      # no GPU required
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.signals import (  # noqa: E402
    compute_agreement_scores,
    compute_iqa_scores,
    compute_similarity_scores,
    compute_uncertainty_scores,
    aggregate_pathology_overlaps,
    expected_region_for,
    region_overlap_score,
    write_score_artifact,
)
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS, normalize_label  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    auxiliary_checkpoint_path, config_sha256, current_code_identity_hash, namespace_identity,
    require_generation_complete, stage3_paths,
)
from scripts.utils.manifest import get_git_commit_hash, read_json, sha256_file  # noqa: E402
from scripts.utils.splits import load_split  # noqa: E402

ALL_SIGNALS = ["similarity", "iqa", "uncertainty", "explainability", "agreement"]


def load_config():
    return load_named_config("stage3_asism.yaml", "stage3")


def load_synthetic_manifest(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: missing Stage 2 generation manifest at {path}\n"
            "Run scripts/generate/02_generate_synthetic_images.py --mode full first."
        )
    rows = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    if not rows:
        raise SystemExit(f"UPSTREAM GATE: generation manifest {path} is empty.")
    return pd.DataFrame(rows)


def validate_generation_manifest_rows(manifest: pd.DataFrame, generation: dict) -> None:
    if manifest["image_id"].astype(str).duplicated().any() or len(manifest) != int(generation.get("num_rows", -1)):
        raise SystemExit("Stage 2 generation manifest has duplicate IDs or disagrees with its completion row count")
    keys = (
        "split_namespace", "namespace_class", "split_manifest_hash", "recipes_csv_sha256",
        "recipes_manifest_sha256", "lora_checkpoint_sha256", "lora_metadata_sha256",
        "generation_config_sha256", "stage2_config_sha256", "code_identity_sha256",
    )
    for key in keys:
        if key not in manifest.columns or not manifest[key].map(lambda value: value == generation.get(key)).all():
            raise SystemExit(f"Stage 2 generation row provenance mismatch at {key}; ASISM refused")


def load_auxiliary_classifier(config, namespace: str, expected_provenance: dict):
    """Load the frozen auxiliary classifier (§2). Fails cleanly if Stage 3's prerequisite is absent."""
    from scripts.utils.classifier import build_model, require_torch

    resume_path = auxiliary_checkpoint_path(config, namespace)
    checkpoint_path = resume_path.with_name(f"{resume_path.stem}.best{resume_path.suffix}")
    if not checkpoint_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: auxiliary classifier not found at {checkpoint_path}\n"
            "It is a Stage 3 prerequisite (docs/stages2_to_5_plan.md §2). Train it first:\n"
            "  python scripts/classify/00_train_auxiliary_classifier.py"
        )
    torch = require_torch()
    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    checkpoint_provenance = payload.get("provenance", {})
    for key in ("split_namespace", "split_manifest_hash", "code_identity_sha256"):
        expected = expected_provenance.get(key)
        if checkpoint_provenance.get(key) != expected:
            raise SystemExit(f"Auxiliary best-checkpoint provenance mismatch at {key}: expected {expected!r}, got {checkpoint_provenance.get(key)!r}")
    aux_manifest_path = resume_path.with_suffix(".manifest.json")
    if not aux_manifest_path.is_file():
        raise SystemExit(f"Auxiliary checkpoint manifest missing: {aux_manifest_path}")
    aux_manifest = read_json(aux_manifest_path)
    if aux_manifest.get("best_checkpoint_sha256") != sha256_file(checkpoint_path):
        raise SystemExit("Auxiliary best-checkpoint hash differs from its manifest")
    model_config = payload.get("model_config", {})
    model = build_model(len(CLASSIFIER_TARGET_LABELS), float(model_config["dropout_p"]),
                        str(model_config["pretrained_source"]))
    model.load_state_dict(payload["model_state"])
    model.eval()
    return model, int(model_config["resolution"])


def synthetic_records(manifest: pd.DataFrame, images_dir: Path) -> list[dict]:
    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    records = []
    for row in manifest.to_dict("records"):
        labels = dict(row["intended_label_vector"])
        labels["No Finding"] = 1 if row.get("is_no_finding") else 0
        labels["Support Devices"] = int(row.get("support_devices", 0))
        records.append(
            {
                "image_id": row["image_id"],
                "image_path": str(images_dir / f"{row['image_id']}.jpg"),
                "labels": {label: labels.get(label) for label in CLASSIFIER_TARGET_LABELS},
                "is_synthetic": True,
            }
        )
    return records


def run_iqa(manifest, images_dir, config, provenance, scores_dir) -> None:
    rows = []
    for image_id in tqdm(manifest["image_id"], desc="iqa", unit="img"):
        scores = compute_iqa_scores(images_dir / f"{image_id}.jpg", config)
        scores["image_id"] = image_id
        rows.append(scores)
    frame = pd.DataFrame(rows)
    columns = ["image_id"] + [c for c in frame.columns if c != "image_id"]
    write_score_artifact(frame[columns], scores_dir / "iqa_scores.parquet", "iqa", provenance)
    print(f"iqa: {len(frame)} rows, {int(frame['iqa_valid'].sum())} valid", flush=True)


def run_uncertainty_and_agreement(
    manifest, images_dir, config, provenance, scores_dir, which: list[str]
) -> None:
    """Both signals come from the same auxiliary-classifier forward passes, so they share the
    (expensive) MC-Dropout inference — but they are written as SEPARATE artifacts and are gated
    independently (§4.0, §4.6). They are never merged into a single score."""
    from scripts.utils.classifier import predict_probabilities

    model, resolution = load_auxiliary_classifier(config, provenance["split_namespace"], provenance)
    records = synthetic_records(manifest, images_dir)
    passes = int(config.signals.uncertainty.mc_dropout_passes)

    mean_all, std_all = predict_probabilities(
        model, records, resolution, mc_dropout_passes=passes
    )

    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    primary_idx = [CLASSIFIER_TARGET_LABELS.index(label) for label in PRIMARY_ENDPOINT_LABELS]
    mean_primary = mean_all[:, primary_idx]
    std_primary = std_all[:, primary_idx]

    if "uncertainty" in which:
        frame = compute_uncertainty_scores(std_primary, mean_primary, config)
        frame.insert(0, "image_id", manifest["image_id"].values)
        write_score_artifact(
            frame, scores_dir / "uncertainty_scores.parquet", "uncertainty", provenance
        )
        print(f"uncertainty: {len(frame)} rows, bands={frame['uncertainty_band'].value_counts().to_dict()}", flush=True)

    if "agreement" in which:
        intended = [dict(v) for v in manifest["intended_label_vector"]]
        frame = compute_agreement_scores(mean_primary, intended, config)
        frame.insert(0, "image_id", manifest["image_id"].values)
        write_score_artifact(
            frame, scores_dir / "agreement_scores.parquet", "agreement", provenance
        )
        print(f"agreement: {len(frame)} rows, mean={frame['agreement_score'].mean():.4f}", flush=True)


def run_explainability(manifest, images_dir, config, provenance, scores_dir) -> None:
    """Grad-CAM overlap with the expected region for each recipe's intended pathologies.

    An explainability-PLAUSIBILITY signal, not proof of diagnostic correctness (§4.4).
    """
    from scripts.utils.classifier import CXRRecordDataset, require_torch

    torch = require_torch()
    model, resolution = load_auxiliary_classifier(config, provenance["split_namespace"], provenance)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    records = synthetic_records(manifest, images_dir)
    dataset = CXRRecordDataset(records, resolution=resolution)

    # Hook the last dense block: standard Grad-CAM target for DenseNet.
    activations, gradients = {}, {}
    target_layer = model.features.denseblock4

    def forward_hook(_module, _input, output):
        activations["value"] = output.detach()

    def backward_hook(_module, _grad_in, grad_out):
        gradients["value"] = grad_out[0].detach()

    handle_f = target_layer.register_forward_hook(forward_hook)
    handle_b = target_layer.register_full_backward_hook(backward_hook)

    rows = []
    try:
        for index in tqdm(range(len(dataset)), desc="explainability", unit="img"):
            item = dataset[index]
            image = item["image"].unsqueeze(0).to(device)
            intended = dict(manifest.iloc[index]["intended_label_vector"])
            positives = [l for l in PRIMARY_ENDPOINT_LABELS if int(intended.get(l, 0)) == 1]
            target_labels = positives or ["__max_disease__"]
            per_label_overlap = {}
            cam_maxima = []
            for target_label in target_labels:
                model.zero_grad(set_to_none=True)
                logits = model(image)
                if target_label == "__max_disease__":
                    disease_idx = [CLASSIFIER_TARGET_LABELS.index(l) for l in PRIMARY_ENDPOINT_LABELS]
                    score = logits[0, disease_idx].max()
                    region = expected_region_for({}, config)
                else:
                    score = logits[0, CLASSIFIER_TARGET_LABELS.index(target_label)]
                    single_intent = {label: int(label == target_label) for label in PRIMARY_ENDPOINT_LABELS}
                    region = expected_region_for(single_intent, config)
                score.backward()
                weights = gradients["value"].mean(dim=(2, 3), keepdim=True)
                cam = torch.relu((weights * activations["value"]).sum(dim=1)).squeeze(0).cpu().numpy()
                per_label_overlap[target_label] = region_overlap_score(cam, region)
                cam_maxima.append(float(cam.max()))
            # Frozen aggregation: every intended positive contributes equally; mean overlap is the
            # primary score and the minimum is retained to expose a missed co-pathology.
            overlap, minimum_overlap = aggregate_pathology_overlaps(per_label_overlap)
            rows.append(
                {
                    "image_id": manifest.iloc[index]["image_id"],
                    "explainability_region_overlap": overlap,
                    "explainability_min_pathology_overlap": minimum_overlap,
                    "explainability_target_labels": json.dumps(target_labels),
                    "explainability_per_label_overlap": json.dumps(per_label_overlap, sort_keys=True),
                    "explainability_aggregation": "mean_over_all_intended_positive_pathologies",
                    "explainability_cam_max": max(cam_maxima),
                    "explainability_is_plausibility_signal_only": True,
                }
            )
    finally:
        handle_f.remove()
        handle_b.remove()

    frame = pd.DataFrame(rows)
    write_score_artifact(
        frame, scores_dir / "explainability_scores.parquet", "explainability", provenance
    )
    print(f"explainability: {len(frame)} rows, mean overlap={frame['explainability_region_overlap'].mean():.4f}", flush=True)


def run_similarity(manifest, images_dir, config, provenance, scores_dir, namespace) -> None:
    """DINOv2 embeddings for synthetic images and a stratified real-reference sample."""
    from scripts.utils.classifier import require_torch
    from scripts.utils.identifiers import sanitize_image_id

    torch = require_torch()
    try:
        import timm
    except ImportError as exc:
        raise SystemExit(
            f"UPSTREAM GATE: the similarity encoder needs `timm` ({exc}).\n"
            "Install it on the pod: pip install timm"
        ) from exc

    stage1_cfg = load_stage1_config()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Never let timm resolve a mutable default checkpoint. Download the exact reviewed revision,
    # then load it into the explicitly named architecture.
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file as load_safetensors

    similarity_cfg = config.signals.similarity
    weights_path = hf_hub_download(
        repo_id=str(similarity_cfg.weights_repo_id),
        filename=str(similarity_cfg.weights_filename),
        revision=str(similarity_cfg.weights_revision),
    )
    encoder = timm.create_model(str(similarity_cfg.encoder), pretrained=False, num_classes=0)
    incompatible = encoder.load_state_dict(load_safetensors(weights_path), strict=False)
    unexpected = list(incompatible.unexpected_keys)
    disallowed_missing = [key for key in incompatible.missing_keys if not key.startswith("head.")]
    if unexpected or disallowed_missing:
        raise RuntimeError(
            "Pinned DINOv2 checkpoint is incompatible with the configured architecture: "
            f"missing={disallowed_missing}, unexpected={unexpected}"
        )
    encoder = encoder.eval().to(device)
    data_config = timm.data.resolve_model_data_config(encoder)
    transform = timm.data.create_transform(**data_config, is_training=False)

    from PIL import Image

    def embed(paths: list[Path]) -> np.ndarray:
        outputs = []
        batch_size = int(config.signals.similarity.batch_size)
        for start in tqdm(range(0, len(paths), batch_size), desc="embed", unit="batch"):
            batch = []
            for path in paths[start : start + batch_size]:
                with Image.open(path) as image:
                    batch.append(transform(image.convert("RGB")))
            with torch.no_grad():
                outputs.append(encoder(torch.stack(batch).to(device)).cpu().numpy())
        return np.concatenate(outputs, axis=0)

    # Real reference pool: a stratified sample of gen_train (the generator's own training data is
    # the correct realism reference; classifier_train is reserved for A-E).
    real_frame = load_split("gen_train", namespace, purpose="schema_validation", caller="asism_similarity")
    per_label = int(config.signals.similarity.reference_sample_per_label)
    real_images_dir = Path(stage1_cfg.paths.images_dir) / namespace / "gen_train"

    selected_rows, seen = [], set()
    for label in PRIMARY_ENDPOINT_LABELS:
        subset = real_frame[real_frame[label].map(normalize_label) == 1].head(per_label)
        for row in subset.to_dict("records"):
            key = row["Path"]
            if key not in seen:
                seen.add(key)
                selected_rows.append(row)
    # Plus negatives, so the class-agnostic fallback tier is not all-pathology.
    for row in real_frame.head(per_label).to_dict("records"):
        if row["Path"] not in seen:
            seen.add(row["Path"])
            selected_rows.append(row)

    reference_paths, reference_vectors = [], []
    for row in selected_rows:
        path = real_images_dir / f"{sanitize_image_id(row['Path'])}.jpg"
        if not path.is_file():
            continue
        reference_paths.append(path)
        reference_vectors.append(
            frozenset(l for l in PRIMARY_ENDPOINT_LABELS if normalize_label(row.get(l)) == 1)
        )

    if not reference_paths:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed gen_train images under {real_images_dir}; "
            "Stage 1 preprocessing must run before the similarity signal."
        )

    synthetic_paths = [images_dir / f"{image_id}.jpg" for image_id in manifest["image_id"]]
    print(f"similarity: embedding {len(synthetic_paths)} synthetic, {len(reference_paths)} real", flush=True)

    frame = compute_similarity_scores(
        embed(synthetic_paths),
        embed(reference_paths),
        reference_vectors,
        [dict(v) for v in manifest["intended_label_vector"]],
        config,
    )
    frame.insert(0, "image_id", manifest["image_id"].values)
    write_score_artifact(frame, scores_dir / "similarity_scores.parquet", "similarity", provenance)
    print(
        f"similarity: {len(frame)} rows, near-duplicates flagged="
        f"{int(frame['novelty_is_near_duplicate'].sum())}",
        flush=True,
    )


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--signal", default="all", choices=ALL_SIGNALS + ["all"])
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_config()
    namespace = args.namespace or str(config.split_namespace)
    identity = namespace_identity(namespace)

    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")

    stage2_artifacts, generation = require_generation_complete(stage2_cfg, namespace)
    images_dir = stage2_artifacts["images_dir"]
    manifest = load_synthetic_manifest(stage2_artifacts["manifest_path"])
    validate_generation_manifest_rows(manifest, generation)
    scores_dir = stage3_paths(config, namespace)["scores_dir"]
    scores_dir.mkdir(parents=True, exist_ok=True)

    artifact_provenance = {
        **identity,
        "asism_config_sha256": config_sha256(config),
        "generation_manifest_sha256": sha256_file(stage2_artifacts["manifest_path"]),
        "generation_completion_sha256": sha256_file(stage2_artifacts["generation_completion"]),
        "lora_checkpoint_sha256": generation["lora_checkpoint_sha256"],
        "code_identity_sha256": current_code_identity_hash(),
        "git_commit_hash": get_git_commit_hash(),
        "n_synthetic_images": len(manifest),
    }

    which = ALL_SIGNALS if args.signal == "all" else [args.signal]

    if "iqa" in which:
        run_iqa(manifest, images_dir, config, artifact_provenance, scores_dir)
    if {"uncertainty", "agreement"} & set(which):
        run_uncertainty_and_agreement(
            manifest, images_dir, config, artifact_provenance, scores_dir, which
        )
    if "explainability" in which:
        run_explainability(manifest, images_dir, config, artifact_provenance, scores_dir)
    if "similarity" in which:
        run_similarity(manifest, images_dir, config, artifact_provenance, scores_dir, namespace)

    print(f"\nArtifacts -> {scores_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/02_gonogo.py`


In [ ]:
# SOURCE FILE: scripts/asism/02_gonogo.py
#!/usr/bin/env python3
"""Stage 3b — ASISM component Go/No-Go gate (docs/stages2_to_5_plan.md §4.6).

Runs BEFORE weights/thresholds are tuned and frozen. Each of the five signals is checked for
technical validity, score directionality, numerical stability, reproducibility, missing-output
rate, redundancy with the other signals, and downstream usefulness.

The rule this script exists to enforce: **a signal that fails its gate is not forced into the final
selector merely because it was implemented.** Outcomes are one of:

    include        - passed everything; eligible for the frozen selector
    ablation_only  - technically valid but weak or near-duplicate of another signal; retained for
                     the leave-one-signal-out analysis, not weighted into the selector
    exclude        - technically invalid or unstable; removed from the selector, artifact retained
                     for audit if it is at least readable, exclusion + reason recorded

Evidence comes from asism_tuning_heldout-derived artifacts only. final_eval_heldout is never read.

Usage:
    python scripts/asism/02_gonogo.py
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, read_json, write_json  # noqa: E402
from scripts.utils.artifact_contracts import asism_score_provenance, require_score_artifact, stage3_paths  # noqa: E402

# Each signal's headline score column, and whether higher is better.
PRIMARY_SCORE_COLUMN = {
    "similarity": ("similarity_knn_mean", True),
    "iqa": ("iqa_composite", True),
    "uncertainty": ("uncertainty_mean_std", None),   # None = no a-priori direction (§4.3)
    "explainability": ("explainability_region_overlap", True),
    "agreement": ("agreement_score", True),
}

ARTIFACT_FILENAME = {name: f"{name}_scores.parquet" for name in PRIMARY_SCORE_COLUMN}


def check_technical_validity(frame: pd.DataFrame, signal: str) -> dict:
    column, _ = PRIMARY_SCORE_COLUMN[signal]
    has_ids = "image_id" in frame.columns
    has_score = column in frame.columns
    return {
        "passed": bool(has_ids and has_score and len(frame) > 0),
        "has_image_id": has_ids,
        "has_primary_score_column": has_score,
        "n_rows": len(frame),
        "primary_score_column": column,
    }


def check_numerical_stability(values: np.ndarray, config) -> dict:
    finite = np.isfinite(values)
    n_unique = int(len(np.unique(values[finite]))) if finite.any() else 0
    min_unique = int(config.gonogo.min_unique_values)
    non_finite_fraction = float(1.0 - finite.mean()) if len(values) else 1.0
    return {
        "passed": bool(non_finite_fraction == 0.0 and n_unique >= min_unique),
        "non_finite_fraction": non_finite_fraction,
        "n_unique_values": n_unique,
        "min_unique_required": min_unique,
        "std": float(np.std(values[finite])) if finite.any() else float("nan"),
        "note": (
            "A near-constant score carries no selection information even if it computes cleanly."
        ),
    }


def check_missing_rate(frame: pd.DataFrame, signal: str, config) -> dict:
    column, _ = PRIMARY_SCORE_COLUMN[signal]
    values = frame[column].to_numpy(dtype=np.float64, na_value=np.nan)
    missing_fraction = float(np.isnan(values).mean()) if len(values) else 1.0
    # IQA additionally reports explicit per-image validity.
    if signal == "iqa" and "iqa_valid" in frame.columns:
        missing_fraction = float(1.0 - frame["iqa_valid"].astype(bool).mean())
    threshold = float(config.gonogo.max_missing_fraction)
    return {
        "passed": bool(missing_fraction <= threshold),
        "missing_fraction": missing_fraction,
        "max_allowed": threshold,
    }


def check_directionality(frame: pd.DataFrame, signal: str) -> dict:
    """Does the score move the way the methodology claims it does?

    Each check uses an internal contrast whose expected direction is known a priori, rather than
    assuming the sign is right because the formula looks right.
    """
    result = {"passed": None, "method": "", "detail": {}}

    if signal == "iqa":
        # Images the domain checks flagged as near-uniform must score BELOW unflagged ones.
        if "iqa_is_near_uniform" in frame.columns and frame["iqa_is_near_uniform"].any():
            flagged = frame.loc[frame["iqa_is_near_uniform"].astype(bool), "iqa_composite"].mean()
            clean = frame.loc[~frame["iqa_is_near_uniform"].astype(bool), "iqa_composite"].mean()
            result.update(
                passed=bool(flagged < clean),
                method="near_uniform_images_score_lower_than_clean",
                detail={"flagged_mean": float(flagged), "clean_mean": float(clean)},
            )
        else:
            result.update(
                passed=True,
                method="no_flagged_images_in_sample",
                detail={"note": "No near-uniform images present; directionality not contradicted."},
            )

    elif signal == "agreement":
        # No-Finding recipes and disease recipes must not be degenerate: the score must vary, and
        # the unintended-penalty must actually reduce the score where it fires.
        if "agreement_n_confident_unintended" in frame.columns:
            penalized = frame["agreement_n_confident_unintended"] > 0
            if penalized.any() and (~penalized).any():
                with_penalty = frame.loc[penalized, "agreement_score"].mean()
                without = frame.loc[~penalized, "agreement_score"].mean()
                result.update(
                    passed=bool(with_penalty < without),
                    method="confident_unintended_predictions_lower_the_score",
                    detail={"penalized_mean": float(with_penalty), "clean_mean": float(without)},
                )
            else:
                result.update(passed=True, method="penalty_never_fired_in_sample", detail={})
        else:
            result.update(passed=False, method="missing_penalty_column", detail={})

    elif signal == "similarity":
        # Flagged near-duplicates must receive no novelty credit.
        if "novelty_is_near_duplicate" in frame.columns and frame["novelty_is_near_duplicate"].any():
            flagged_novelty = frame.loc[
                frame["novelty_is_near_duplicate"].astype(bool), "novelty_score"
            ].max()
            result.update(
                passed=bool(flagged_novelty == 0.0),
                method="memorized_images_receive_zero_novelty",
                detail={"max_novelty_among_flagged": float(flagged_novelty)},
            )
        else:
            result.update(passed=True, method="no_near_duplicates_in_sample", detail={})

    elif signal == "explainability":
        # Overlap is a fraction of activation mass: it must lie in [0, 1].
        values = frame["explainability_region_overlap"].dropna()
        in_range = bool(((values >= 0.0) & (values <= 1.0)).all()) if len(values) else False
        result.update(
            passed=in_range,
            method="region_overlap_is_a_valid_fraction",
            detail={"min": float(values.min()) if len(values) else None,
                    "max": float(values.max()) if len(values) else None},
        )

    elif signal == "uncertainty":
        # §4.3 forbids an a-priori direction: high uncertainty is NOT defined as bad. The check is
        # that the bands are populated and ordered, not that the score correlates with quality.
        if "uncertainty_band" in frame.columns:
            counts = frame["uncertainty_band"].value_counts().to_dict()
            result.update(
                passed=bool(len(counts) >= 2),
                method="bands_are_populated_no_direction_asserted",
                detail={"band_counts": {str(k): int(v) for k, v in counts.items()}},
            )
        else:
            result.update(passed=False, method="missing_band_column", detail={})

    return result


def check_reproducibility(scores_dir: Path, signal: str, config) -> dict:
    """Compare the artifact against its recorded provenance fingerprint.

    Full recomputation is a GPU-cost decision for the operator; what is enforced here is that the
    artifact carries the provenance needed to reproduce it and that its row count matches.
    """
    sidecar = scores_dir / f"{signal}_scores.provenance.json"
    if not sidecar.is_file():
        return {"passed": False, "reason": "missing_provenance_sidecar"}
    provenance = read_json(sidecar)
    required = {"schema_version", "signal", "n_rows", "asism_config_sha256", "git_commit_hash"}
    missing = sorted(required - set(provenance))
    return {
        "passed": bool(not missing),
        "missing_provenance_fields": missing,
        "recorded_n_rows": provenance.get("n_rows"),
        "asism_config_sha256": provenance.get("asism_config_sha256"),
    }


def check_redundancy(merged: pd.DataFrame, signal: str, config) -> dict:
    """Correlation of this signal's headline score with the other signals'.

    A signal that is a near-linear restatement of another adds little independent information, and
    weighting both would double-count the same evidence.
    """
    column, _ = PRIMARY_SCORE_COLUMN[signal]
    threshold = float(config.gonogo.redundancy_abs_correlation_max)

    correlations = {}
    for other, (other_column, _) in PRIMARY_SCORE_COLUMN.items():
        if other == signal or other_column not in merged.columns or column not in merged.columns:
            continue
        pair = merged[[column, other_column]].dropna()
        if len(pair) < 10 or pair[column].std() == 0 or pair[other_column].std() == 0:
            continue
        correlations[other] = float(pair[column].corr(pair[other_column]))

    worst = max(correlations.items(), key=lambda item: abs(item[1]), default=(None, 0.0))
    return {
        "passed": bool(abs(worst[1]) < threshold),
        "correlations": correlations,
        "most_correlated_with": worst[0],
        "max_abs_correlation": abs(worst[1]),
        "threshold": threshold,
    }


def check_usefulness(merged: pd.DataFrame, signal: str) -> dict:
    """Does including this signal change which images a simple equal-weight selector would keep?

    Compares the top-50% selection under an equal-weight composite WITH and WITHOUT this signal. A
    signal that changes nothing about the selection cannot influence downstream results, whatever
    its intrinsic quality.
    """
    available = [
        (name, col)
        for name, (col, direction) in PRIMARY_SCORE_COLUMN.items()
        if col in merged.columns and direction is True
    ]
    if len(available) < 2:
        return {"passed": True, "reason": "insufficient_signals_to_compare", "jaccard": None}

    def composite(names: list[str]) -> pd.Series:
        parts = []
        for _, col in [(n, c) for n, c in available if n in names]:
            values = merged[col].astype(float)
            spread = values.max() - values.min()
            parts.append((values - values.min()) / spread if spread > 0 else values * 0.0)
        return sum(parts) / max(len(parts), 1)

    all_names = [name for name, _ in available]
    if signal not in all_names:
        return {"passed": True, "reason": "signal_has_no_directional_score", "jaccard": None}
    without_names = [name for name in all_names if name != signal]

    n_keep = max(1, len(merged) // 2)
    with_top = set(composite(all_names).nlargest(n_keep).index)
    without_top = set(composite(without_names).nlargest(n_keep).index)

    intersection = len(with_top & without_top)
    union = len(with_top | without_top)
    jaccard = intersection / union if union else 1.0

    return {
        "passed": bool(jaccard < 0.99),
        "jaccard_overlap_of_selection": jaccard,
        "n_selected": n_keep,
        "note": "Jaccard ~1.0 means the signal does not change the selection at all.",
    }


def decide(checks: dict) -> tuple[str, str]:
    """Map check results to one of include / ablation_only / exclude (§4.6)."""
    hard_failures = [
        name
        for name in ("technical_validity", "numerical_stability", "missing_rate", "reproducibility")
        if checks[name].get("passed") is False
    ]
    if hard_failures:
        return "exclude", f"failed hard checks: {', '.join(hard_failures)}"

    if checks["directionality"].get("passed") is False:
        return "exclude", "score direction contradicts the methodology's stated meaning"

    soft_failures = []
    if checks["redundancy"].get("passed") is False:
        soft_failures.append(
            f"redundant with {checks['redundancy'].get('most_correlated_with')} "
            f"(|r|={checks['redundancy'].get('max_abs_correlation'):.3f})"
        )
    if checks["usefulness"].get("passed") is False:
        soft_failures.append("does not change selection")

    if soft_failures:
        return "ablation_only", "; ".join(soft_failures)
    return "include", "passed all checks"


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--scores-dir", default=None)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_named_config("stage3_asism.yaml", "stage3")

    namespace = args.namespace or str(config.split_namespace)
    stage2_config = load_named_config("stage2_generation.yaml", "stage2")
    expected_provenance = asism_score_provenance(config, stage2_config, namespace)
    paths = stage3_paths(config, namespace)
    scores_dir = Path(args.scores_dir) if args.scores_dir else paths["scores_dir"]

    frames: dict[str, pd.DataFrame] = {}
    absent: list[str] = []
    for signal, filename in ARTIFACT_FILENAME.items():
        path = scores_dir / filename
        if path.is_file():
            require_score_artifact(path, signal, expected_provenance)
            frames[signal] = pd.read_parquet(path)
        else:
            absent.append(signal)

    if not frames:
        raise SystemExit(
            f"UPSTREAM GATE: no ASISM score artifacts found in {scores_dir}\n"
            "Run scripts/asism/01_compute_signals.py first."
        )

    merged = None
    for signal, frame in frames.items():
        column, _ = PRIMARY_SCORE_COLUMN[signal]
        subset = frame[["image_id", column]] if column in frame.columns else frame[["image_id"]]
        merged = subset if merged is None else merged.merge(subset, on="image_id", how="outer")
    merged = merged.set_index("image_id")

    report = {
        "stage": "asism_gonogo",
        "scores_dir": str(scores_dir),
        "git_commit_hash": get_git_commit_hash(),
        "signals_present": sorted(frames),
        "signals_absent": sorted(absent),
        "evidence_split": "asism_tuning_heldout only; final_eval_heldout never read",
        "per_signal": {},
    }

    for signal, frame in frames.items():
        column, _ = PRIMARY_SCORE_COLUMN[signal]
        values = (
            frame[column].to_numpy(dtype=np.float64, na_value=np.nan)
            if column in frame.columns
            else np.array([])
        )

        checks = {
            "technical_validity": check_technical_validity(frame, signal),
            "numerical_stability": check_numerical_stability(values, config),
            "missing_rate": check_missing_rate(frame, signal, config),
            "reproducibility": check_reproducibility(scores_dir, signal, config),
            "directionality": check_directionality(frame, signal),
            "redundancy": check_redundancy(merged, signal, config),
            "usefulness": check_usefulness(merged, signal),
        }
        outcome, reason = decide(checks)
        report["per_signal"][signal] = {
            "outcome": outcome,
            "reason": reason,
            "checks": checks,
            "artifact_retained_for_audit": True,
        }

    for signal in absent:
        report["per_signal"][signal] = {
            "outcome": "exclude",
            "reason": "artifact not produced",
            "checks": {},
            "artifact_retained_for_audit": False,
        }

    included = sorted(s for s, r in report["per_signal"].items() if r["outcome"] == "include")
    ablation = sorted(s for s, r in report["per_signal"].items() if r["outcome"] == "ablation_only")
    excluded = sorted(s for s, r in report["per_signal"].items() if r["outcome"] == "exclude")

    report["surviving_signals"] = included
    report["ablation_only_signals"] = ablation
    report["excluded_signals"] = excluded
    report["asism_variant_status"] = (
        "primary" if len(included) >= 3
        else "reduced_variant — fewer than 3 signals survived; the resulting selector is reported "
             "as an alternative/ablation rather than the primary method (§4.6)"
    )

    report.update(expected_provenance)
    write_json(paths["gonogo_report"], report)

    print("ASISM Go/No-Go", flush=True)
    print("=" * 60, flush=True)
    for signal in sorted(report["per_signal"]):
        entry = report["per_signal"][signal]
        print(f"  {signal:<16} {entry['outcome']:<14} {entry['reason']}", flush=True)
    print("=" * 60, flush=True)
    print(f"  include:       {included}", flush=True)
    print(f"  ablation only: {ablation}", flush=True)
    print(f"  exclude:       {excluded}", flush=True)
    print(f"  variant:       {report['asism_variant_status']}", flush=True)
    print(f"\nReport -> {config.paths.gonogo_report}", flush=True)

    if not included:
        print(
            "\nNo signal survived the gate — the selector cannot be frozen. Investigate the "
            "failing checks before tuning.",
            flush=True,
        )
        return 2
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/03_tune_freeze_select.py`


In [ ]:
# SOURCE FILE: scripts/asism/03_tune_freeze_select.py
#!/usr/bin/env python3
"""Stage 3c — ASISM bounded tuning, freeze, and selection (docs/stages2_to_5_plan.md §4.7, §4.8).

Three phases, each independently invocable:

  --phase estimate   Compute the proxy-run count and GPU-hour estimate and apply the compute-budget
                     gate. Runs no training. This is the gate that must pass BEFORE any results are
                     seen — if the estimate is over budget, the search space is reduced here, never
                     after observing performance.

  --phase tune       Two-stage bounded search (coarse screening -> shortlist validation). Proxy
                     trains on classifier_train + the ACTUALLY-CONSTRUCTED candidate synthetic
                     subset and evaluates on asism_tuning_heldout ONLY. Fixed optimizer-step budget,
                     no candidate-specific early stopping. classifier_val is not used to rank
                     candidates. final_eval_heldout is never touched. Resumable at trial granularity.

  --phase select     Apply the frozen configuration: quality floor FIRST, then class quota. Writes
                     selected_manifest.jsonl (condition C's data) and rejected_log.jsonl with a
                     reason per rejected image.

Usage:
    python scripts/asism/03_tune_freeze_select.py --phase estimate
    python scripts/asism/03_tune_freeze_select.py --phase tune
    python scripts/asism/03_tune_freeze_select.py --phase select
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import tempfile
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.signals import SCHEMA_VERSION  # noqa: E402
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS, normalize_label  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json, write_frozen_json, write_json  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    asism_score_provenance, require_score_artifact, require_generation_complete,
    stage2_paths, stage3_paths,
)

# Directional signals eligible for the weighted composite (uncertainty is banded, not directional).
DIRECTIONAL_SCORES = {
    "similarity": "similarity_knn_mean",
    "iqa": "iqa_composite",
    "explainability": "explainability_region_overlap",
    "agreement": "agreement_score",
}


def load_config():
    return load_named_config("stage3_asism.yaml", "stage3")


def compute_budget_estimate(config) -> dict:
    """Pre-execution accounting (§4.7). Must be computed and checked before any trial runs."""
    tuning = config.tuning
    coarse_runs = int(tuning.coarse.trials) * int(tuning.coarse.folds) * int(tuning.coarse.seeds)
    shortlist_runs = (
        int(tuning.shortlist.size) * int(tuning.shortlist.folds) * int(tuning.shortlist.seeds)
    )
    total_runs = coarse_runs + shortlist_runs
    hours_each = float(tuning.compute_budget.hours_per_proxy_run_estimate)
    estimated_hours = total_runs * hours_each
    max_hours = float(tuning.compute_budget.max_gpu_hours)

    return {
        "coarse_runs": coarse_runs,
        "shortlist_runs": shortlist_runs,
        "total_proxy_runs": total_runs,
        "hours_per_proxy_run_estimate": hours_each,
        "estimated_gpu_hours": round(estimated_hours, 3),
        "max_gpu_hours": max_hours,
        "within_budget": bool(estimated_hours <= max_hours),
        "formula": (
            "total = coarse_trials*coarse_folds*coarse_seeds "
            "+ shortlist_size*shortlist_folds*shortlist_seeds"
        ),
    }


def enforce_compute_budget(config) -> dict:
    estimate = compute_budget_estimate(config)
    if not estimate["within_budget"]:
        raise SystemExit(
            "COMPUTE BUDGET GATE: the ASISM search exceeds its declared budget.\n"
            f"  estimated: {estimate['estimated_gpu_hours']} GPU-hours "
            f"({estimate['total_proxy_runs']} proxy runs x "
            f"{estimate['hours_per_proxy_run_estimate']}h)\n"
            f"  budget:    {estimate['max_gpu_hours']} GPU-hours\n\n"
            "Reduce the search space in configs/stage3_asism.yaml (tuning.coarse.trials, "
            "shortlist.size, folds, or seeds) BEFORE running.\n"
            "Reducing it after seeing results would let the outcome influence the search design "
            "(docs/stages2_to_5_plan.md §4.7)."
        )
    return estimate


def load_merged_scores(config, surviving: list[str]) -> pd.DataFrame:
    """Merge the surviving signals' artifacts by image_id (§4.0)."""
    scores_dir = Path(config.paths.scores_dir)
    merged = None
    for signal in surviving:
        path = scores_dir / f"{signal}_scores.parquet"
        if not path.is_file():
            raise SystemExit(f"Surviving signal {signal!r} has no artifact at {path}")
        require_score_artifact(path, signal, config._expected_score_provenance)
        frame = pd.read_parquet(path)
        merged = frame if merged is None else merged.merge(frame, on="image_id", how="inner")

    # Uncertainty is always merged when present: §4.8 uses its band even though it is not part of
    # the directional composite.
    uncertainty_path = scores_dir / "uncertainty_scores.parquet"
    if uncertainty_path.is_file() and "uncertainty_band" not in (merged.columns if merged is not None else []):
        require_score_artifact(uncertainty_path, "uncertainty", config._expected_score_provenance)
        uncertainty = pd.read_parquet(uncertainty_path)
        merged = uncertainty if merged is None else merged.merge(uncertainty, on="image_id", how="left")

    if merged is None or merged.empty:
        raise SystemExit("No ASISM scores available to merge.")
    return merged


def normalize_series(values: pd.Series) -> pd.Series:
    values = values.astype(float)
    low, high = values.min(), values.max()
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - low) / (high - low)


def composite_score(frame: pd.DataFrame, weights: dict[str, float]) -> pd.Series:
    total_weight = sum(weights.values())
    if total_weight <= 0:
        return pd.Series(np.zeros(len(frame)), index=frame.index)
    score = pd.Series(np.zeros(len(frame)), index=frame.index)
    for signal, weight in weights.items():
        column = DIRECTIONAL_SCORES.get(signal)
        if weight <= 0 or column is None or column not in frame.columns:
            continue
        score = score + weight * normalize_series(frame[column])
    return score / total_weight


def candidate_weight_grid(surviving: list[str], n_trials: int, seed: int) -> list[dict[str, float]]:
    """Coarse grid over weights for the directional surviving signals (§4.7 Stage A).

    Deliberately coarse: fine-grained optimization against asism_tuning_heldout is exactly the
    overfitting risk the bounded-search design exists to limit.
    """
    directional = [s for s in surviving if s in DIRECTIONAL_SCORES]
    if not directional:
        return [{}]

    levels = [0.0, 0.5, 1.0]
    combinations = [
        dict(zip(directional, values))
        for values in product(levels, repeat=len(directional))
        if sum(values) > 0
    ]
    equal = {signal: 1.0 for signal in directional}
    required = [equal]
    required += [{name: (1.0 if name == signal else 0.0) for name in directional} for signal in directional]
    if "uncertainty" in surviving:
        # Uncertainty is intentionally banded rather than treated as monotonically good/bad. This
        # baseline therefore ranks only by the frozen preferred-band bonus inside class quotas.
        required.append({**{name: 0.0 for name in directional}, "__uncertainty_band_only__": 1.0})
    if len(directional) > 1:
        required += [{name: (0.0 if name == omitted else 1.0) for name in directional} for omitted in directional]
    # De-duplicate while retaining deterministic baseline order.
    required = list({tuple(sorted(item.items())): item for item in required}.values())
    if n_trials < len(required):
        raise ValueError(f"coarse.trials={n_trials} cannot cover {len(required)} required equal/single-signal/leave-one-out baselines")

    rng = np.random.default_rng(seed)
    if len(combinations) > n_trials:
        rest = [c for c in combinations if c not in required]
        chosen = rng.choice(len(rest), size=n_trials - len(required), replace=False)
        combinations = required + [rest[i] for i in chosen]
    else:
        combinations = required + [c for c in combinations if c not in required]
    return combinations[:n_trials]


def select_images(
    frame: pd.DataFrame,
    weights: dict[str, float],
    config,
    intended_by_id: dict[str, dict],
) -> tuple[list[str], list[dict]]:
    """§4.8 ordered policy: quality floor FIRST, then per-label quota.

    A poor image is never accepted merely because its pathology is rare — rarity changes how many
    images compete for a quota, never whether the floor applies.
    """
    selection_cfg = config.selection
    scored = frame.copy()
    scored["composite"] = composite_score(scored, weights)

    rejected: list[dict] = []

    # ---- Step 1: absolute quality floor, applied to everything.
    floor_value = np.percentile(scored["composite"], float(selection_cfg.quality_floor_percentile))
    below_floor = scored["composite"] < floor_value
    for image_id in scored.loc[below_floor, "image_id"]:
        rejected.append({"image_id": image_id, "reason": "below_quality_floor"})

    # Memorized near-duplicates are rejected outright: they add no new information and carry a
    # patient-privacy risk that no quota should be able to override.
    eligible = scored.loc[~below_floor].copy()
    if "novelty_is_near_duplicate" in eligible.columns:
        memorized = eligible["novelty_is_near_duplicate"].astype(bool)
        for image_id in eligible.loc[memorized, "image_id"]:
            rejected.append({"image_id": image_id, "reason": "near_duplicate_memorization"})
        eligible = eligible.loc[~memorized]

    # ---- Step 2: per-label quota among floor-passing images.
    preference = str(selection_cfg.uncertainty_band_preference)
    if preference != "any" and "uncertainty_band" in eligible.columns:
        # Preference re-ranks within the quota; it never overrides the floor.
        eligible["band_bonus"] = (eligible["uncertainty_band"] == preference).astype(float) * 0.05
        eligible["composite"] = eligible["composite"] + eligible["band_bonus"]

    min_per_label = int(selection_cfg.min_accepted_per_label)
    max_per_label = int(selection_cfg.max_accepted_per_label)

    accepted: set[str] = set()
    per_label_counts: dict[str, int] = {}

    for label in PRIMARY_ENDPOINT_LABELS:
        label_ids = [
            image_id
            for image_id in eligible["image_id"]
            if int(intended_by_id.get(image_id, {}).get(label, 0)) == 1
        ]
        if not label_ids:
            per_label_counts[label] = 0
            continue
        subset = eligible[eligible["image_id"].isin(label_ids)].nlargest(max_per_label, "composite")
        chosen = subset["image_id"].tolist()
        accepted.update(chosen)
        per_label_counts[label] = len(chosen)

    # No-Finding recipes compete as their own group.
    no_finding_ids = [
        image_id
        for image_id in eligible["image_id"]
        if sum(int(v) for v in intended_by_id.get(image_id, {}).values()) == 0
    ]
    if no_finding_ids:
        subset = eligible[eligible["image_id"].isin(no_finding_ids)].nlargest(max_per_label, "composite")
        accepted.update(subset["image_id"].tolist())
        per_label_counts["__no_finding__"] = len(subset)

    for image_id in eligible["image_id"]:
        if image_id not in accepted:
            rejected.append({"image_id": image_id, "reason": "did_not_win_class_quota"})

    starved = {
        label: count
        for label, count in per_label_counts.items()
        if 0 < count < min_per_label
    }
    if starved:
        print(
            f"  NOTE: {len(starved)} label(s) below min_accepted_per_label={min_per_label}: "
            f"{starved}\n  (not back-filled — the quality floor is absolute, §4.8)",
            flush=True,
        )

    return sorted(accepted), rejected


def build_intended_lookup(config) -> dict[str, dict]:
    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    manifest_path = stage2_paths(stage2_cfg, str(config.split_namespace))["manifest_path"]
    if not manifest_path.is_file():
        raise SystemExit(f"UPSTREAM GATE: missing generation manifest {manifest_path}")
    lookup = {}
    with open(manifest_path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                entry = json.loads(line)
                lookup[entry["image_id"]] = entry["intended_label_vector"]
    return lookup


def run_proxy_trial(
    weights: dict[str, float],
    merged: pd.DataFrame,
    intended_by_id: dict[str, dict],
    config,
    namespace: str,
    seed: int,
    fold: int,
    trial_key: str,
) -> float:
    """One proxy evaluation (§4.7): train on classifier_train + candidate subset, evaluate on
    asism_tuning_heldout. Returns macro-AUROC over the primary endpoint labels."""
    from scripts.utils.classifier import (
        TrainingBudget,
        evaluate_classifier,
        records_from_split,
        records_from_synthetic_manifest,
        train_classifier,
    )

    stage1_cfg = load_stage1_config()
    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    tuning = config.tuning

    selected_ids, _ = select_images(merged, weights, config, intended_by_id)

    # Real component is classifier_train — NOT gen_train (§4.7 frozen data flow).
    real_frame = load_split(
        "classifier_train", namespace, purpose="schema_validation", caller="asism_proxy"
    )
    real_records = records_from_split(
        real_frame, Path(stage1_cfg.paths.images_dir) / namespace / "classifier_train"
    )
    s2paths = stage2_paths(stage2_cfg, namespace)
    synthetic_records = records_from_synthetic_manifest(s2paths["manifest_path"], s2paths["images_dir"], selected_ids)

    eval_frame = load_split(
        "asism_tuning_heldout", namespace, purpose="schema_validation", caller="asism_proxy"
    )
    fold_manifest_hash = None
    if fold > 0:
        fold_manifest = read_json(Path(config.paths.fold_manifest))
        fold_key = str(fold - 1)
        if fold_key not in fold_manifest["folds"]:
            raise SystemExit(f"ASISM fold {fold} is absent from the frozen fold manifest")
        patient_ids = set(fold_manifest["folds"][fold_key]["patient_ids"])
        eval_frame = eval_frame[eval_frame["patient_id"].astype(str).isin(patient_ids)].copy()
        fold_manifest_hash = fold_manifest["fold_manifest_hash"]
    eval_records = records_from_split(
        eval_frame, Path(stage1_cfg.paths.images_dir) / namespace / "asism_tuning_heldout"
    )

    budget = TrainingBudget(
        max_steps=int(tuning.coarse.proxy_max_steps) if fold == 0 else int(tuning.shortlist.proxy_max_steps),
        batch_size=int(tuning.proxy.batch_size),
        learning_rate=float(tuning.proxy.learning_rate),
        seed=seed,
        eval_every_n_steps=0,  # no candidate-specific early stopping (§4.7)
    )
    checkpoint_path = Path(config.paths.asism_dir) / "proxy_checkpoints" / f"{hash_dict({'trial_key': trial_key}, length=32)}.pt"
    checkpoint_provenance = {
        "split_manifest_hash": split_provenance(namespace)["split_manifest_hash"],
        "split_namespace": namespace, "dataset_id": trial_key,
        "condition_or_draw_id": trial_key, "config_hash": hash_dict({"weights": weights, "budget": budget.__dict__}, length=64),
        "label_policy": "chexpert_mask_uncertain_v1", "fold_manifest_hash": fold_manifest_hash,
        "code_identity_sha256": config._expected_score_provenance["code_identity_sha256"],
    }
    model, _, _ = train_classifier(
        train_records=real_records + synthetic_records,
        val_records=[],
        budget=budget,
        dropout_p=float(tuning.proxy.dropout_p),
        pretrained_source=str(tuning.proxy.pretrained_source),
        resolution=int(tuning.proxy.resolution),
        progress_desc=f"proxy s{seed}f{fold}",
        checkpoint_path=checkpoint_path,
        resume_from=checkpoint_path if checkpoint_path.is_file() else None,
        checkpoint_metadata=checkpoint_provenance,
    )
    metrics = evaluate_classifier(model, eval_records, int(tuning.proxy.resolution))
    return float(metrics["macro_auroc"]), fold_manifest_hash


def append_search_log(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", buffering=1) as handle:
        handle.write(json.dumps(record, sort_keys=True, default=str) + "\n")


def completed_trials(path: Path, expected_provenance: dict) -> dict[str, float]:
    """Resume at trial granularity (§4.7): an interrupted search re-reads its own log."""
    if not path.is_file():
        return {}
    done = {}
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                entry = json.loads(line)
                for key, value in expected_provenance.items():
                    if entry.get(key) != value:
                        raise SystemExit(f"Stale ASISM search log at {path}:{key}: expected {value!r}, got {entry.get(key)!r}")
                done[entry["trial_key"]] = entry["proxy_macro_auroc"]
    return done


def phase_tune(config, namespace: str) -> int:
    estimate = enforce_compute_budget(config)

    gonogo_path = Path(config.paths.gonogo_report)
    if not gonogo_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: Go/No-Go report not found at {gonogo_path}\n"
            "Signals must clear the gate before tuning (§4.6): python scripts/asism/02_gonogo.py"
        )
    gonogo = read_json(gonogo_path)
    for key, value in config._expected_score_provenance.items():
        if gonogo.get(key) != value:
            raise SystemExit(f"Go/No-Go provenance mismatch at {key}: expected {value!r}, got {gonogo.get(key)!r}")
    surviving = list(gonogo["surviving_signals"])
    if not surviving:
        raise SystemExit("No signals survived the Go/No-Go gate — nothing to tune.")

    merged = load_merged_scores(config, surviving)
    intended_by_id = build_intended_lookup(config)
    search_log = Path(config.paths.search_log)
    done = completed_trials(search_log, dict(config._expected_score_provenance))

    tuning = config.tuning
    from scripts.utils.splits import freeze_patient_folds
    fold_path = Path(config.paths.fold_manifest)
    if not fold_path.exists():
        fold_frame = load_split("asism_tuning_heldout", namespace, purpose="schema_validation", caller="asism_fold_freeze")
        freeze_patient_folds(fold_path, fold_frame["patient_id"].astype(str).tolist(), int(tuning.shortlist.folds), int(tuning.search_seed), split_provenance(namespace)["split_manifest_hash"])
    fold_manifest = read_json(fold_path)
    grid = candidate_weight_grid(surviving, int(tuning.coarse.trials), int(tuning.search_seed))

    print(f"Compute estimate: {estimate['total_proxy_runs']} runs, "
          f"{estimate['estimated_gpu_hours']}h (budget {estimate['max_gpu_hours']}h)", flush=True)
    print(f"Surviving signals: {surviving}", flush=True)
    print(f"\nStage A — coarse screening ({len(grid)} candidates)", flush=True)

    coarse_results = []
    for index, weights in enumerate(grid):
        key = f"coarse:{index}:{hash_dict(weights)}"
        if key in done:
            score = done[key]
            print(f"  [{index}] resumed  {weights} -> {score:.4f}", flush=True)
        else:
            score, _ = run_proxy_trial(
                weights, merged, intended_by_id, config, namespace,
                seed=int(tuning.search_seed), fold=0, trial_key=key,
            )
            append_search_log(search_log, {
                "trial_key": key, "phase": "coarse", "weights": weights,
                "proxy_macro_auroc": score, "seed": int(tuning.search_seed), "fold": 0,
                **dict(config._expected_score_provenance),
            })
            print(f"  [{index}] {weights} -> {score:.4f}", flush=True)
        coarse_results.append({"weights": weights, "score": score})

    coarse_results.sort(key=lambda item: item["score"], reverse=True)
    shortlist = coarse_results[: int(tuning.shortlist.size)]
    print(f"\nStage B — shortlist validation ({len(shortlist)} candidates)", flush=True)

    shortlist_results = []
    for index, candidate in enumerate(shortlist):
        scores = []
        for fold in range(1, int(tuning.shortlist.folds) + 1):
            for seed_offset in range(int(tuning.shortlist.seeds)):
                seed = int(tuning.search_seed) + seed_offset
                key = f"shortlist:{index}:{fold}:{seed}:{hash_dict(candidate['weights'])}"
                if key in done:
                    scores.append(done[key])
                    continue
                score, fold_hash = run_proxy_trial(
                    candidate["weights"], merged, intended_by_id, config, namespace,
                    seed=seed, fold=fold, trial_key=key,
                )
                append_search_log(search_log, {
                    "trial_key": key, "phase": "shortlist", "weights": candidate["weights"],
                    "proxy_macro_auroc": score, "seed": seed, "fold": fold,
                    "fold_manifest_hash": fold_hash,
                    **dict(config._expected_score_provenance),
                })
                scores.append(score)
        mean_score = float(np.mean(scores))
        shortlist_results.append({
            "weights": candidate["weights"], "mean_score": mean_score,
            "std_score": float(np.std(scores)), "n_runs": len(scores),
        })
        print(f"  [{index}] {candidate['weights']} -> {mean_score:.4f} "
              f"(+/-{np.std(scores):.4f}, n={len(scores)})", flush=True)

    # Frozen selection rule: best mean; ties within the noise band resolve to the SIMPLER
    # configuration (fewer active signals, then smaller total weight).
    band = float(tuning.tie_noise_band)
    finite_results = [item for item in shortlist_results if np.isfinite(item["mean_score"])]
    selection_fallback = None
    if finite_results:
        best_score = max(item["mean_score"] for item in finite_results)
        tied = [item for item in finite_results if best_score - item["mean_score"] <= band]
    elif os.environ.get("THESIS_SMOKE_MODE") == "1":
        # A tiny fixture fold can contain only one class for every label, making macro-AUROC
        # undefined. Keep the engineering smoke moving with the already-finite coarse result;
        # production remains fail-closed below.
        best_coarse = max(coarse_results, key=lambda item: item["score"])
        tied = [{**shortlist_results[0], "mean_score": float(best_coarse["score"])}]
        selection_fallback = "smoke_only_all_shortlist_aurocs_undefined_used_best_coarse_score"
    else:
        raise SystemExit(
            "ASISM tuning produced no finite shortlist macro-AUROC values. Increase held-out "
            "class support or revise the predeclared fold plan; no production winner was frozen."
        )
    winner = min(
        tied,
        key=lambda item: (
            sum(1 for w in item["weights"].values() if w > 0),
            sum(item["weights"].values()),
        ),
    )

    frozen = {
        "schema_version": SCHEMA_VERSION,
        "stage": "asism_frozen",
        "surviving_signals": surviving,
        "ablation_only_signals": gonogo.get("ablation_only_signals", []),
        "excluded_signals": gonogo.get("excluded_signals", []),
        "asism_variant_status": gonogo.get("asism_variant_status"),
        "frozen_weights": winner["weights"],
        "selection_policy": OmegaConf.to_container(config.selection, resolve=True),
        "tie_break_rule": "best mean proxy macro-AUROC; ties within noise band -> simpler config",
        "tie_noise_band": band,
        "n_tied_candidates": len(tied),
        "selection_fallback": selection_fallback,
        "winner_mean_proxy_macro_auroc": winner["mean_score"],
        "shortlist_results": shortlist_results,
        "coarse_results": coarse_results,
        "required_baselines": {
            "equal_weight": [item for item in coarse_results if len(set(item["weights"].values())) == 1 and any(item["weights"].values())],
            "individual_signal": [item for item in coarse_results if sum(value > 0 for value in item["weights"].values()) == 1],
            "leave_one_signal_out": [item for item in coarse_results if sum(value == 0 for value in item["weights"].values()) == 1],
            "uncertainty_band_only": [item for item in coarse_results if item["weights"].get("__uncertainty_band_only__", 0) > 0],
        },
        "compute_accounting": estimate,
        "proxy_data_flow": {
            "train_real": "classifier_train",
            "train_synthetic": "candidate subset (actually constructed)",
            "evaluate": "asism_tuning_heldout",
            "classifier_val_used_for_ranking": False,
            "final_eval_heldout_used": False,
        },
        "search_log": str(search_log),
        "fold_manifest_hash": fold_manifest["fold_manifest_hash"],
        "split_provenance": split_provenance(namespace),
        "git_commit_hash": get_git_commit_hash(),
        "frozen": True,
        **dict(config._expected_score_provenance),
    }
    write_frozen_json(Path(config.paths.frozen_manifest), frozen)

    print(f"\nFROZEN weights: {winner['weights']}", flush=True)
    print(f"  proxy macro-AUROC: {winner['mean_score']:.4f}", flush=True)
    print(f"  manifest -> {config.paths.frozen_manifest}", flush=True)
    return 0


def phase_select(config, namespace: str) -> int:
    frozen_path = Path(config.paths.frozen_manifest)
    if not frozen_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: ASISM is not frozen ({frozen_path} missing).\n"
            "Run --phase tune first."
        )
    frozen = read_json(frozen_path)
    for key, value in config._expected_score_provenance.items():
        if frozen.get(key) != value:
            raise SystemExit(f"Frozen ASISM provenance mismatch at {key}: expected {value!r}, got {frozen.get(key)!r}")
    if not frozen.get("frozen"):
        raise SystemExit("ASISM manifest exists but is not marked frozen.")

    merged = load_merged_scores(config, frozen["surviving_signals"])
    intended_by_id = build_intended_lookup(config)
    selected, rejected = select_images(merged, frozen["frozen_weights"], config, intended_by_id)

    selected_path = Path(config.paths.selected_manifest)
    rejected_path = Path(config.paths.rejected_log)
    selected_path.parent.mkdir(parents=True, exist_ok=True)

    def atomic_jsonl(path: Path, records: list[dict]) -> None:
        fd, temp = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as handle:
                for record in records:
                    handle.write(json.dumps(record, sort_keys=True) + "\n")
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temp, path)
        finally:
            Path(temp).unlink(missing_ok=True)

    atomic_jsonl(
        selected_path,
        [
            {
                "image_id": image_id,
                "intended_label_vector": intended_by_id.get(image_id, {}),
                "asism_frozen_weights": frozen["frozen_weights"],
                "asism_manifest_hash": hash_dict(frozen["frozen_weights"]),
            }
            for image_id in selected
        ],
    )
    atomic_jsonl(rejected_path, rejected)

    reasons = pd.Series([r["reason"] for r in rejected]).value_counts().to_dict() if rejected else {}
    write_json(
        selected_path.with_name("selection_summary.json"),
        {
            "n_candidates": len(merged),
            "n_selected": len(selected),
            "n_rejected": len(rejected),
            "rejection_reasons": reasons,
            "frozen_weights": frozen["frozen_weights"],
            "selection_policy": frozen["selection_policy"],
            "split_provenance": split_provenance(namespace),
            "git_commit_hash": get_git_commit_hash(),
        },
    )

    print(f"Candidates: {len(merged)}", flush=True)
    print(f"Selected:   {len(selected)}", flush=True)
    print(f"Rejected:   {len(rejected)}  {reasons}", flush=True)
    print(f"-> {selected_path}", flush=True)
    return 0


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--phase", choices=["estimate", "tune", "select"], required=True)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_config()
    namespace = args.namespace or str(config.split_namespace)
    stage2_config = load_named_config("stage2_generation.yaml", "stage2")
    expected_score_provenance = asism_score_provenance(config, stage2_config, namespace)
    config.split_namespace = namespace
    paths = stage3_paths(config, namespace)
    for key, value in paths.items():
        if key in config.paths:
            config.paths[key] = str(value)
    config._expected_score_provenance = expected_score_provenance

    if args.phase == "estimate":
        estimate = compute_budget_estimate(config)
        print(json.dumps(estimate, indent=2), flush=True)
        if not estimate["within_budget"]:
            print(
                "\nOVER BUDGET — reduce the search space in configs/stage3_asism.yaml before "
                "running --phase tune.",
                flush=True,
            )
            return 2
        print("\nWithin budget. Proceed: --phase tune", flush=True)
        return 0

    if args.phase == "tune":
        return phase_tune(config, namespace)
    return phase_select(config, namespace)


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/signals.py`


In [ ]:
# SOURCE FILE: scripts/asism/signals.py
"""The five ASISM signals (docs/stages2_to_5_plan.md §4.1-§4.5).

Each signal computes a per-image score set written to its own versioned parquet artifact. All five
are subject to the same Go/No-Go gate (§4.6) — implementation is not admission.

The IQA signal is deliberately pure NumPy/PIL: it is the one signal that needs no model and no GPU,
so it stays runnable and testable locally, which is what lets the whole selection/ranking layer be
developed off the pod.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS, intended_vector_to_labels

SCHEMA_VERSION = 2


def aggregate_pathology_overlaps(per_label: dict[str, float]) -> tuple[float, float]:
    """Frozen multi-label Grad-CAM rule: equal-weight mean, with minimum retained as a guardrail."""
    finite = [float(value) for value in per_label.values() if np.isfinite(value)]
    if not finite:
        return float("nan"), float("nan")
    return float(np.mean(finite)), float(min(finite))


# ----------------------------------------------------------------------------------------------
# §4.2 Image Quality Assessment — no model, no GPU
# ----------------------------------------------------------------------------------------------

def _laplacian_variance(gray: np.ndarray) -> float:
    """Variance of the Laplacian: the standard focus/blur measure. Low variance = few sharp edges."""
    kernel = np.array([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]])
    padded = np.pad(gray, 1, mode="edge")
    response = np.zeros_like(gray, dtype=np.float64)
    for dy in range(3):
        for dx in range(3):
            weight = kernel[dy, dx]
            if weight != 0.0:
                response += weight * padded[dy : dy + gray.shape[0], dx : dx + gray.shape[1]]
    return float(response.var())


def compute_iqa_scores(image_path: Path, config) -> dict:
    """Multi-faceted technical quality (§4.2).

    Generic no-reference IQA is represented by the sharpness/contrast statistics; the domain checks
    (border artifact, clipping, near-uniformity, grayscale consistency, anatomical completeness
    proxy) are what the signal actually leans on. Whether the generic part adds anything beyond the
    domain part is settled by ablation, not assumed.
    """
    iqa_cfg = config.signals.iqa
    try:
        with Image.open(image_path) as image:
            rgb = image.convert("RGB")
            array = np.asarray(rgb, dtype=np.float32)
            gray = np.asarray(rgb.convert("L"), dtype=np.float32)
    except Exception as exc:
        return {"iqa_valid": False, "iqa_error": f"{type(exc).__name__}: {exc}"}

    height, width = gray.shape
    std = float(gray.std())
    mean = float(gray.mean())

    # Grayscale consistency: a real CXR is achromatic. Channel divergence indicates colour drift
    # from the RGB generation path.
    channel_spread = float(np.mean(np.std(array, axis=2)))

    # Clipping: mass piled at the extremes of the dynamic range.
    clipped_low = float(np.mean(gray <= 1.0))
    clipped_high = float(np.mean(gray >= 254.0))

    # Border artifact: letterbox residue or a dead frame around the anatomy.
    border_width = max(1, int(min(height, width) * 0.06))
    border = np.concatenate(
        [
            gray[:border_width, :].ravel(),
            gray[-border_width:, :].ravel(),
            gray[:, :border_width].ravel(),
            gray[:, -border_width:].ravel(),
        ]
    )
    border_uniform_fraction = float(np.mean(np.abs(border - border.mean()) < 2.0))

    # Anatomical completeness proxy: a plausible CXR has substantially more signal in the central
    # region than at the frame edge. A near-empty centre means the anatomy is missing or collapsed.
    centre = gray[
        int(height * 0.25) : int(height * 0.75), int(width * 0.25) : int(width * 0.75)
    ]
    centre_energy_ratio = float(centre.std() / (std + 1e-6))

    sharpness = _laplacian_variance(gray)

    # Histogram abnormality: distance from a smooth unimodal shape. A spiky/degenerate histogram
    # indicates posterization or a broken decode.
    histogram, _ = np.histogram(gray, bins=64, range=(0, 255))
    histogram = histogram / max(histogram.sum(), 1)
    histogram_entropy = float(-np.sum(histogram[histogram > 0] * np.log(histogram[histogram > 0])))

    is_near_uniform = std < float(iqa_cfg.blank_std_threshold)
    is_blurry = sharpness < float(iqa_cfg.laplacian_blur_threshold)
    is_low_contrast = std < float(iqa_cfg.low_contrast_std)
    has_border_artifact = border_uniform_fraction > float(iqa_cfg.border_uniform_fraction)

    # Composite in [0, 1]: 1.0 is clean. Each detected defect subtracts a fixed amount, so the
    # composite degrades gracefully rather than being dominated by one continuous term.
    penalties = (
        0.40 * float(is_near_uniform)
        + 0.20 * float(is_blurry)
        + 0.10 * float(is_low_contrast)
        + 0.10 * float(has_border_artifact)
        + 0.10 * float(clipped_low + clipped_high > 0.20)
        + 0.10 * float(channel_spread > 2.0)
    )
    composite = float(max(0.0, 1.0 - penalties))

    return {
        "iqa_valid": True,
        "iqa_composite": composite,
        "iqa_sharpness": sharpness,
        "iqa_contrast_std": std,
        "iqa_mean_intensity": mean,
        "iqa_channel_spread": channel_spread,
        "iqa_clipped_low_fraction": clipped_low,
        "iqa_clipped_high_fraction": clipped_high,
        "iqa_border_uniform_fraction": border_uniform_fraction,
        "iqa_centre_energy_ratio": centre_energy_ratio,
        "iqa_histogram_entropy": histogram_entropy,
        "iqa_is_near_uniform": bool(is_near_uniform),
        "iqa_is_blurry": bool(is_blurry),
        "iqa_is_low_contrast": bool(is_low_contrast),
        "iqa_has_border_artifact": bool(has_border_artifact),
    }


# ----------------------------------------------------------------------------------------------
# §4.5 Intended-label agreement
# ----------------------------------------------------------------------------------------------

def compute_agreement_scores(
    probabilities: np.ndarray,
    intended_vectors: list[dict],
    config,
) -> pd.DataFrame:
    """Proxy label-consistency between the generator's intent and the auxiliary classifier's read.

        agreement = mean P(intended positive labels)
                  - penalty_weight * mean P(confidently predicted UNintended labels)

    For a No Finding recipe the intended positive set is empty, so the first term is defined as
    (1 - mean probability over all 11 primary labels): agreement is high exactly when the classifier
    sees no disease, which is what that recipe asked for.

    NOT clinical proof. The generator and the classifier can share a bias and agree while both are
    wrong; this measures consistency between two models, nothing more (§4.5).
    """
    agreement_cfg = config.signals.agreement
    threshold = float(agreement_cfg.unintended_confidence_threshold)
    penalty_weight = float(agreement_cfg.penalty_weight)

    label_index = {label: position for position, label in enumerate(PRIMARY_ENDPOINT_LABELS)}
    rows = []

    for row_index, intended in enumerate(intended_vectors):
        probs = probabilities[row_index]
        intended_positive = intended_vector_to_labels(intended)
        unintended = [label for label in PRIMARY_ENDPOINT_LABELS if label not in intended_positive]

        if intended_positive:
            intended_mean = float(
                np.mean([probs[label_index[label]] for label in intended_positive])
            )
            is_no_finding = False
        else:
            # No Finding recipe: "intended" outcome is uniformly low disease probability.
            intended_mean = float(
                1.0 - np.mean([probs[label_index[label]] for label in PRIMARY_ENDPOINT_LABELS])
            )
            is_no_finding = True

        unintended_probs = [probs[label_index[label]] for label in unintended]
        confident_unintended = [p for p in unintended_probs if p >= threshold]
        unintended_penalty = float(np.mean(confident_unintended)) if confident_unintended else 0.0

        rows.append(
            {
                "agreement_score": intended_mean - penalty_weight * unintended_penalty,
                "agreement_intended_mean_prob": intended_mean,
                "agreement_unintended_penalty": unintended_penalty,
                "agreement_n_intended_positive": len(intended_positive),
                "agreement_n_confident_unintended": len(confident_unintended),
                "agreement_is_no_finding_recipe": is_no_finding,
            }
        )

    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------------------
# §4.3 Uncertainty
# ----------------------------------------------------------------------------------------------

def compute_uncertainty_scores(
    probability_std: np.ndarray,
    probability_mean: np.ndarray,
    config,
) -> pd.DataFrame:
    """MC-Dropout predictive spread over the 11 primary labels, banded.

    Bands are reported, NOT converted into a penalty here: moderate uncertainty may be the most
    informative band (TSynD's premise, §4.3), so whether a band is good or bad is a selection-policy
    decision made later in §4.8, not baked into the signal.
    """
    uncertainty_cfg = config.signals.uncertainty
    low_max = float(uncertainty_cfg.low_band_max)
    moderate_max = float(uncertainty_cfg.moderate_band_max)

    mean_std = probability_std.mean(axis=1)
    max_std = probability_std.max(axis=1)
    # Predictive entropy of the mean probabilities: a second, complementary uncertainty view.
    clipped = np.clip(probability_mean, 1e-6, 1 - 1e-6)
    entropy = -(clipped * np.log(clipped) + (1 - clipped) * np.log(1 - clipped)).mean(axis=1)

    bands = np.where(mean_std <= low_max, "low", np.where(mean_std <= moderate_max, "moderate", "extreme"))

    return pd.DataFrame(
        {
            "uncertainty_mean_std": mean_std,
            "uncertainty_max_std": max_std,
            "uncertainty_entropy": entropy,
            "uncertainty_band": bands,
        }
    )


# ----------------------------------------------------------------------------------------------
# §4.1 Similarity
# ----------------------------------------------------------------------------------------------

def hierarchical_reference_indices(
    intended: dict,
    reference_label_vectors: list[frozenset],
    min_exact: int,
) -> tuple[np.ndarray, str]:
    """Pick real-reference rows for one synthetic image, walking the §4.1 hierarchy.

    Tier 1 exact intended-vector match -> Tier 2 nearest multilabel neighbourhood (Jaccard) ->
    Tier 3 pathology prototypes (any shared positive) -> Tier 4 class-agnostic realism fallback.
    The fallback is always available, so no image goes unscored for lack of an exact-match pool —
    which is precisely the failure mode a rigid exact-match requirement would cause for rare
    combinations.
    """
    intended_set = frozenset(intended_vector_to_labels(intended))

    exact = np.array(
        [i for i, combo in enumerate(reference_label_vectors) if combo == intended_set], dtype=int
    )
    if len(exact) >= min_exact:
        return exact, "tier1_exact"

    if intended_set:
        similarities = np.array(
            [
                len(intended_set & combo) / max(len(intended_set | combo), 1)
                for combo in reference_label_vectors
            ]
        )
        neighbourhood = np.where(similarities >= 0.5)[0]
        if len(neighbourhood) >= min_exact:
            return neighbourhood, "tier2_neighbourhood"

        prototype = np.array(
            [i for i, combo in enumerate(reference_label_vectors) if intended_set & combo], dtype=int
        )
        if len(prototype) >= min_exact:
            return prototype, "tier3_prototype"

    return np.arange(len(reference_label_vectors)), "tier4_class_agnostic"


def compute_similarity_scores(
    synthetic_embeddings: np.ndarray,
    reference_embeddings: np.ndarray,
    reference_label_vectors: list[frozenset],
    intended_vectors: list[dict],
    config,
) -> pd.DataFrame:
    """k-NN cosine similarity to real references, plus a SEPARATE memorization flag.

    Fidelity and memorization are reported as distinct columns on purpose: an image that is nearly
    identical to one specific training image is a memorization red flag, not a fidelity success, and
    collapsing the two would make the selector reward exactly what it should catch (§4.1).
    """
    similarity_cfg = config.signals.similarity
    k = int(similarity_cfg.k_neighbors)
    min_exact = int(similarity_cfg.min_exact_match_references)
    near_dup_similarity = float(similarity_cfg.near_duplicate_similarity)
    near_dup_spread = float(similarity_cfg.near_duplicate_spread)

    synthetic_norm = synthetic_embeddings / (
        np.linalg.norm(synthetic_embeddings, axis=1, keepdims=True) + 1e-8
    )
    reference_norm = reference_embeddings / (
        np.linalg.norm(reference_embeddings, axis=1, keepdims=True) + 1e-8
    )

    rows = []
    for row_index in range(len(synthetic_norm)):
        indices, tier = hierarchical_reference_indices(
            intended_vectors[row_index], reference_label_vectors, min_exact
        )
        similarities = reference_norm[indices] @ synthetic_norm[row_index]

        effective_k = min(k, len(similarities))
        top_k = np.sort(similarities)[-effective_k:][::-1]
        top1 = float(top_k[0])
        knn_mean = float(top_k.mean())
        # Gap between the single closest reference and the rest of the neighbourhood.
        spread = float(top_k[0] - top_k[-1]) if effective_k > 1 else 0.0
        # How far top1 stands out from the remaining neighbours specifically.
        rest_mean = float(top_k[1:].mean()) if effective_k > 1 else top1
        top1_gap = top1 - rest_mean

        # Memorization = the image is near-identical to SOME real training image. top1 crossing the
        # threshold is sufficient on its own; a large top1_gap additionally indicates it collapsed
        # onto ONE specific image rather than onto a cluster of near-identical real images.
        # (Requiring a SMALL spread here would be backwards: an image memorizing one reference is
        # by construction far from the others, so its spread is large.)
        is_near_duplicate = bool(top1 >= near_dup_similarity)
        collapsed_onto_single = bool(is_near_duplicate and top1_gap >= near_dup_spread)

        rows.append(
            {
                "similarity_knn_mean": knn_mean,
                "similarity_top1": top1,
                "similarity_topk_spread": spread,
                "similarity_reference_tier": tier,
                "similarity_n_references": int(len(indices)),
                "similarity_k_used": effective_k,
                "similarity_top1_gap": top1_gap,
                "novelty_is_near_duplicate": is_near_duplicate,
                "novelty_collapsed_onto_single_reference": collapsed_onto_single,
                # Novelty score: high when the image resembles the class without being a near-copy
                # of a specific training image. Memorized images get no novelty credit.
                "novelty_score": float(knn_mean * (1.0 - float(is_near_duplicate))),
            }
        )

    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------------------
# §4.4 Explainability
# ----------------------------------------------------------------------------------------------

def region_overlap_score(cam: np.ndarray, region: tuple[float, float, float, float]) -> float:
    """Fraction of total Grad-CAM activation mass falling inside the expected region.

    An explainability-PLAUSIBILITY signal, never proof of diagnostic correctness (§4.4). High
    overlap means the classifier looked where the pathology usually appears; it says nothing about
    whether the pathology is actually present.
    """
    height, width = cam.shape
    x0, y0, x1, y1 = region
    col0, col1 = int(x0 * width), int(x1 * width)
    row0, row1 = int(y0 * height), int(y1 * height)

    cam = np.maximum(cam, 0.0)
    total = float(cam.sum())
    if total <= 0.0:
        return float("nan")
    return float(cam[row0:row1, col0:col1].sum() / total)


def expected_region_for(intended: dict, config) -> tuple[float, float, float, float]:
    """Pathology-specific expected region, or the explicit baseline box for No Finding recipes."""
    explain_cfg = config.signals.explainability
    positives = intended_vector_to_labels(intended)
    if str(explain_cfg.region_mode) == "baseline" or not positives:
        return tuple(explain_cfg.baseline_box)

    regions = [
        tuple(explain_cfg.pathology_regions[label])
        for label in positives
        if label in explain_cfg.pathology_regions
    ]
    if not regions:
        return tuple(explain_cfg.baseline_box)

    # Union of the per-pathology regions: a multi-label recipe may legitimately activate anywhere
    # its constituent pathologies are expected.
    return (
        min(r[0] for r in regions),
        min(r[1] for r in regions),
        max(r[2] for r in regions),
        max(r[3] for r in regions),
    )


def write_score_artifact(
    frame: pd.DataFrame,
    path: Path,
    signal_name: str,
    provenance: dict,
) -> None:
    """Atomic, versioned parquet write with an embedded provenance block (§4.0)."""
    import os
    import tempfile

    path.parent.mkdir(parents=True, exist_ok=True)
    frame = frame.copy()
    frame.attrs["schema_version"] = SCHEMA_VERSION
    frame.attrs["signal"] = signal_name

    sidecar = path.with_suffix(".provenance.json")
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    try:
        frame.to_parquet(temp_name, index=False)
        os.replace(temp_name, path)
    finally:
        Path(temp_name).unlink(missing_ok=True)

    from scripts.utils.manifest import sha256_file, write_json
    write_json(sidecar, {
        "schema_version": SCHEMA_VERSION, "signal": signal_name, "n_rows": len(frame),
        "columns": list(frame.columns), "parquet_sha256": sha256_file(path), **provenance,
    })


# Stage 4 — Classifier training


## `scripts/classify/00_train_auxiliary_classifier.py`


In [ ]:
# SOURCE FILE: scripts/classify/00_train_auxiliary_classifier.py
#!/usr/bin/env python3
"""Train the auxiliary real-only reference classifier (docs/stages2_to_5_plan.md §2).

This is an ASISM PREREQUISITE, not condition A. It exists so Stage 3's uncertainty (§4.3),
explainability (§4.4), and agreement (§4.5) signals have a real-data classifier to query. Condition
A is a separate experiment trained later, from scratch, on classifier_train under §7's frozen
protocol.

Trained on gen_train, validated on gen_val — never classifier_train/classifier_val (those are
reserved unspent for A-E) and never a heldout split. Using the generator's own splits for a scorer
of the generator's output is the point: it keeps the entire classifier development pool free.

Initialization is leakage-safe by construction: CheXpert-pretrained weights are REJECTED in
scripts/utils/classifier.build_model(), because their training patients cannot be shown disjoint
from final_eval_heldout.

Usage:
    python scripts/classify/00_train_auxiliary_classifier.py [--namespace dev] [--max-steps N]
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.classifier import (  # noqa: E402
    TrainingBudget,
    records_from_split,
    require_torch,
    train_classifier,
)
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.experiment_registry import ExperimentRun  # noqa: E402
from scripts.utils.artifact_contracts import auxiliary_checkpoint_path, current_code_identity_hash, namespace_identity  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--max-steps", type=int, default=4000)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    config = load_named_config("stage3_asism.yaml", "stage3")
    stage1_cfg = load_stage1_config()

    namespace = args.namespace or str(config.split_namespace)
    provenance = namespace_identity(namespace)

    images_dir = Path(stage1_cfg.paths.images_dir) / namespace
    train_frame = load_split("gen_train", namespace, purpose="schema_validation", caller="aux_clf")
    val_frame = load_split("gen_val", namespace, purpose="schema_validation", caller="aux_clf")

    train_records = records_from_split(train_frame, images_dir / "gen_train")
    val_records = records_from_split(val_frame, images_dir / "gen_val")

    if not train_records:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed gen_train images found under {images_dir / 'gen_train'}.\n"
            "Stage 1 preprocessing (scripts/data/03_preprocess_images.py) must run first."
        )
    if not val_records:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed gen_val images found under {images_dir / 'gen_val'}."
        )

    require_torch()

    checkpoint_path = auxiliary_checkpoint_path(config, namespace)
    budget = TrainingBudget(
        max_steps=args.max_steps,
        batch_size=args.batch_size,
        seed=args.seed,
        eval_every_n_steps=max(100, args.max_steps // 10),
    )

    registry_config = {
        "auxiliary_classifier": OmegaConf.to_container(config.auxiliary_classifier, resolve=True),
        "budget": budget.__dict__,
        "split_provenance": provenance,
    }

    with ExperimentRun(
        stage="auxiliary_reference_classifier",
        config=registry_config,
        dataset_version=provenance["split_manifest_hash"],
    ) as run:
        model, accounting, history = train_classifier(
            train_records=train_records,
            val_records=val_records,
            budget=budget,
            dropout_p=float(config.auxiliary_classifier.dropout_p),
            pretrained_source=str(config.auxiliary_classifier.pretrained_source),
            resolution=int(config.auxiliary_classifier.resolution),
            progress_desc="aux-classifier",
            checkpoint_path=checkpoint_path,
            resume_from=checkpoint_path if checkpoint_path.is_file() else None,
            checkpoint_metadata={
                "split_manifest_hash": provenance["split_manifest_hash"],
                "split_namespace": namespace,
                "dataset_id": "auxiliary_gen_train_gen_val",
                "config_hash": hash_dict(registry_config),
                "label_policy": "chexpert_mask_uncertain_v1",
                "condition_or_draw_id": "auxiliary",
                "code_version": get_git_commit_hash(),
                "code_identity_sha256": current_code_identity_hash(),
            },
        )
        best = max(history, key=lambda entry: entry["macro_auroc"]) if history else {}
        accounting.config_hash = hash_dict(registry_config)

        run.set_checkpoint_path(checkpoint_path)
        run.set_metrics({"best_macro_auroc": best.get("macro_auroc"), **accounting.to_dict()})

    write_json(
        checkpoint_path.with_suffix(".manifest.json"),
        {
            "stage": "auxiliary_reference_classifier",
            "role": "ASISM prerequisite scorer — NOT condition A",
            "trained_on": "gen_train",
            "validated_on": "gen_val",
            "never_touched": ["classifier_train", "classifier_val", "asism_tuning_heldout", "final_eval_heldout"],
            "pretrained_source": str(config.auxiliary_classifier.pretrained_source),
            "pretrained_provenance": {
                "note": "ImageNet weights via torchvision; no CheXpert patients involved.",
                "chexpert_pretrained_rejected": True,
            },
            "split_provenance": provenance,
            "accounting": accounting.to_dict(),
            "eval_history": history,
            "config_hash": hash_dict(registry_config),
            "resume_checkpoint": str(checkpoint_path),
            "resume_checkpoint_sha256": sha256_file(checkpoint_path),
            "best_checkpoint": accounting.extra.get("best_checkpoint_path"),
            "best_checkpoint_sha256": sha256_file(accounting.extra["best_checkpoint_path"]),
            "code_identity_sha256": current_code_identity_hash(),
            "git_commit_hash": get_git_commit_hash(),
        },
    )

    print(f"Auxiliary classifier -> {checkpoint_path}", flush=True)
    if history:
        print(f"Best gen_val macro-AUROC: {max(e['macro_auroc'] for e in history):.4f}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/classify/01_train_conditions.py`


In [ ]:
# SOURCE FILE: scripts/classify/01_train_conditions.py
#!/usr/bin/env python3
"""Stage 4 — train conditions A-E (docs/stages2_to_5_plan.md §7).

    A  real only                    classifier_train
    B  real + ALL synthetic         classifier_train + every Stage 2 image
    C  real + ASISM-selected        classifier_train + the frozen ASISM selection
    D  real + matched-random        classifier_train + FIVE independent matched draws
    E  synthetic only               every Stage 2 image

Condition D is the primary control: each of its five draws matches C on total synthetic count,
per-label marginal positive counts (within a frozen tolerance), and single-vs-multi-label
proportion. Without it, any C-over-A gain could be explained by "adding N more images of roughly
this class mix" rather than by ASISM's selection quality.

Fairness (§7.1): every run — all conditions, all draws, all seeds — uses the SAME optimizer-step
budget, batch size, augmentation, and checkpoint-selection rule. All model selection is on
classifier_val; final_eval_heldout is never touched here.

Usage:
    python scripts/classify/01_train_conditions.py --condition A
    python scripts/classify/01_train_conditions.py --condition D          # all 5 draws x 3 seeds
    python scripts/classify/01_train_conditions.py --condition all
    python scripts/classify/01_train_conditions.py --plan-only            # run/cost accounting
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.experiment_registry import ExperimentRun  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, get_library_versions, hash_dict, read_json, sha256_file, write_frozen_json, write_json  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    current_code_identity_hash, namespace_identity, require_generation_complete,
    require_manifest_fields, stage2_paths, stage3_paths, stage4_paths,
)

CONDITIONS = ["A", "B", "C", "D", "E"]


def load_configs():
    stage4 = load_named_config("stage4_classifier.yaml", "stage4")
    stage3 = load_named_config("stage3_asism.yaml", "stage3")
    stage2 = load_named_config("stage2_generation.yaml", "stage2")
    return stage4, stage3, stage2


def load_all_synthetic_ids(stage2_cfg) -> list[str]:
    path = Path(stage2_cfg.paths.manifest_path)
    if not path.is_file():
        raise SystemExit(f"UPSTREAM GATE: missing generation manifest {path}")
    ids = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                ids.append(json.loads(line)["image_id"])
    return ids


def load_selected_ids(stage3_cfg) -> list[str]:
    path = Path(stage3_cfg.paths.selected_manifest)
    if not path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: ASISM selection not found at {path}\n"
            "Run: python scripts/asism/03_tune_freeze_select.py --phase select"
        )
    ids = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                ids.append(json.loads(line)["image_id"])
    return ids


def intended_lookup(stage2_cfg) -> dict[str, dict]:
    lookup = {}
    with open(Path(stage2_cfg.paths.manifest_path), encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                entry = json.loads(line)
                lookup[entry["image_id"]] = entry["intended_label_vector"]
    return lookup


def profile_of(image_ids: list[str], intended: dict[str, dict]) -> dict:
    """Class-distribution profile used to match condition D against condition C."""
    per_label = {label: 0 for label in PRIMARY_ENDPOINT_LABELS}
    single, multi, none = 0, 0, 0
    for image_id in image_ids:
        vector = intended.get(image_id, {})
        positives = [l for l in PRIMARY_ENDPOINT_LABELS if int(vector.get(l, 0)) == 1]
        for label in positives:
            per_label[label] += 1
        if len(positives) == 0:
            none += 1
        elif len(positives) == 1:
            single += 1
        else:
            multi += 1
    total = max(len(image_ids), 1)
    return {
        "total": len(image_ids),
        "per_label_positive": per_label,
        "single_label": single,
        "multi_label": multi,
        "no_finding": none,
        "single_fraction": single / total,
        "multi_fraction": multi / total,
    }


def build_matched_random_draw(
    all_ids: list[str],
    target_profile: dict,
    intended: dict[str, dict],
    seed: int,
    config,
) -> tuple[list[str], dict]:
    """One matched-random draw for condition D.

    Greedy per-label matching against the target's marginal positive counts, then a top-up to the
    exact total. Joint label-vector matching is ATTEMPTED (by preferring candidates whose exact
    vector the target also contains) but never allowed to make matching infeasible — a strict joint
    match is unachievable for rare combinations and would silently shrink the draw.

    Residual imbalance is measured and returned, never hidden.
    """
    matching = config.condition_d.matching
    rng = np.random.default_rng(seed)
    available = [i for i in all_ids]
    rng.shuffle(available)

    target_counts = dict(target_profile["per_label_positive"])
    target_total = int(target_profile["total"])

    by_vector: dict[tuple, list[str]] = {}
    for image_id in available:
        vector = intended.get(image_id, {})
        key = tuple(sorted(l for l in PRIMARY_ENDPOINT_LABELS if int(vector.get(l, 0)) == 1))
        by_vector.setdefault(key, []).append(image_id)

    chosen: list[str] = []
    chosen_set: set[str] = set()
    counts = {label: 0 for label in PRIMARY_ENDPOINT_LABELS}

    if bool(matching.match_per_label_marginal):
        for label in sorted(PRIMARY_ENDPOINT_LABELS, key=lambda l: target_counts[l]):
            need = target_counts[label]
            if need <= 0:
                continue
            candidates = [
                image_id
                for image_id in available
                if image_id not in chosen_set
                and int(intended.get(image_id, {}).get(label, 0)) == 1
            ]
            for image_id in candidates:
                if counts[label] >= need or len(chosen) >= target_total:
                    break
                chosen.append(image_id)
                chosen_set.add(image_id)
                for other in PRIMARY_ENDPOINT_LABELS:
                    if int(intended.get(image_id, {}).get(other, 0)) == 1:
                        counts[other] += 1

    # Top up to the exact total, preferring No-Finding vectors if the target has them.
    if len(chosen) < target_total:
        remaining = [i for i in available if i not in chosen_set]
        need_no_finding = target_profile["no_finding"] - sum(
            1 for i in chosen if not any(
                int(intended.get(i, {}).get(l, 0)) == 1 for l in PRIMARY_ENDPOINT_LABELS
            )
        )
        no_finding_pool = [
            i for i in remaining
            if not any(int(intended.get(i, {}).get(l, 0)) == 1 for l in PRIMARY_ENDPOINT_LABELS)
        ]
        for image_id in no_finding_pool[: max(0, need_no_finding)]:
            if len(chosen) >= target_total:
                break
            chosen.append(image_id)
            chosen_set.add(image_id)
        for image_id in remaining:
            if len(chosen) >= target_total:
                break
            if image_id not in chosen_set:
                chosen.append(image_id)
                chosen_set.add(image_id)

    chosen = chosen[:target_total]

    # Deterministic repair: hill-climb swaps against the frozen marginal and single/multi target.
    # No reseeding/retry is used; ties follow the seeded `available` order above.
    def positive_set(image_id):
        return frozenset(label for label in PRIMARY_ENDPOINT_LABELS if int(intended.get(image_id, {}).get(label, 0)) == 1)
    chosen_vectors = [positive_set(image_id) for image_id in chosen]
    current_counts = {label: sum(label in vector for vector in chosen_vectors) for label in PRIMARY_ENDPOINT_LABELS}
    current_single = sum(len(vector) == 1 for vector in chosen_vectors)
    current_multi = sum(len(vector) > 1 for vector in chosen_vectors)
    def mismatch_counts(label_counts, single, multi):
        label_error = sum(abs(label_counts[label] - target_counts[label]) / max(target_counts[label], 1) for label in PRIMARY_ENDPOINT_LABELS)
        shape_error = abs(single / max(target_total, 1) - target_profile["single_fraction"]) + abs(multi / max(target_total, 1) - target_profile["multi_fraction"])
        return label_error + shape_error
    current_score = mismatch_counts(current_counts, current_single, current_multi)
    outside = [image_id for image_id in available if image_id not in chosen_set][:500]
    for _ in range(min(100, max(target_total, 1))):
        best = None
        for chosen_index, old_id in list(enumerate(chosen))[:500]:
            old_vector = chosen_vectors[chosen_index]
            for new_id in outside:
                new_vector = positive_set(new_id)
                trial_counts = {label: current_counts[label] - (label in old_vector) + (label in new_vector) for label in PRIMARY_ENDPOINT_LABELS}
                trial_single = current_single - (len(old_vector) == 1) + (len(new_vector) == 1)
                trial_multi = current_multi - (len(old_vector) > 1) + (len(new_vector) > 1)
                score = mismatch_counts(trial_counts, trial_single, trial_multi)
                if score + 1e-12 < current_score and (best is None or score < best[0]):
                    best = (score, chosen_index, old_id, new_id, new_vector, trial_counts, trial_single, trial_multi)
        if best is None:
            break
        current_score, chosen_index, old_id, new_id, new_vector, current_counts, current_single, current_multi = best
        chosen[chosen_index] = new_id
        chosen_vectors[chosen_index] = new_vector
        outside.remove(new_id)
        outside.append(old_id)
        chosen_set.remove(old_id)
        chosen_set.add(new_id)
    achieved = profile_of(chosen, intended)

    tolerance = float(matching.per_label_tolerance_fraction)
    residual = {}
    within_tolerance = True
    for label in PRIMARY_ENDPOINT_LABELS:
        target_count = target_counts[label]
        achieved_count = achieved["per_label_positive"][label]
        allowed = max(1.0, tolerance * max(target_count, 1))
        delta = achieved_count - target_count
        residual[label] = {
            "target": target_count,
            "achieved": achieved_count,
            "delta": delta,
            "within_tolerance": bool(abs(delta) <= allowed),
        }
        if abs(delta) > allowed:
            within_tolerance = False

    single_delta = abs(achieved["single_fraction"] - target_profile["single_fraction"])
    single_ok = single_delta <= float(matching.single_multi_tolerance_fraction)

    valid_pool = set(all_ids)
    constraints = {
        "exact_total": achieved["total"] == target_total,
        "per_label_marginals": within_tolerance,
        "single_multi_proportion": single_ok,
        "no_duplicate_ids": len(chosen) == len(set(chosen)),
        "all_ids_in_valid_pool": set(chosen).issubset(valid_pool),
    }
    unsatisfied = [name for name, passed in constraints.items() if not passed]
    return chosen, {
        "seed": seed,
        "target_total": target_total,
        "achieved_total": achieved["total"],
        "count_matched_exactly": bool(achieved["total"] == target_total),
        "per_label_residual": residual,
        "all_labels_within_tolerance": within_tolerance,
        "single_fraction_target": target_profile["single_fraction"],
        "single_fraction_achieved": achieved["single_fraction"],
        "single_fraction_delta": single_delta,
        "single_multi_within_tolerance": bool(single_ok),
        "joint_vector_matching_attempted": bool(matching.attempt_joint_vector_matching),
        "note": "Residual imbalance is recorded, not hidden; exact joint matching is not forced.",
        "constraints": constraints,
        "passed": not unsatisfied,
        "unsatisfied_constraints": unsatisfied,
        "draw_id_hash": hash_dict({"seed": seed, "ids": chosen}, length=64),
    }


def training_plan(stage4_cfg) -> dict:
    seeds = list(stage4_cfg.seeds)
    n_draws = int(stage4_cfg.condition_d.n_draws)
    all_runs = {
        "A": len(seeds), "B": len(seeds), "C": len(seeds),
        "D": n_draws * len(seeds), "E": len(seeds),
    }
    enabled = list(stage4_cfg.get("conditions", CONDITIONS))
    runs = {condition: all_runs[condition] for condition in enabled}
    total = sum(runs.values())
    return {
        "runs_per_condition": runs,
        "total_runs": total,
        "seeds": seeds,
        "n_d_draws": n_draws,
        "optimizer_steps_per_run": int(stage4_cfg.training.max_steps),
        "total_optimizer_steps": total * int(stage4_cfg.training.max_steps),
        "formula": "sum of enabled per-condition runs; D contributes n_draws x |seeds|",
    }


def build_records(condition: str, draw_ids: list[str] | None, cfgs, namespace: str):
    from scripts.utils.classifier import records_from_split, records_from_synthetic_manifest

    stage4_cfg, stage3_cfg, stage2_cfg = cfgs
    stage1_cfg = load_stage1_config()
    images_dir = Path(stage1_cfg.paths.images_dir) / namespace

    real_records = []
    if condition != "E":
        frame = load_split(
            "classifier_train", namespace, purpose="schema_validation", caller="stage4"
        )
        real_records = records_from_split(frame, images_dir / "classifier_train")
        if not real_records:
            raise SystemExit(
                f"UPSTREAM GATE: no preprocessed classifier_train images under "
                f"{images_dir / 'classifier_train'}"
            )

    synthetic_ids: list[str] | None = None
    if condition in ("B", "E"):
        synthetic_ids = None  # all
    elif condition == "C":
        synthetic_ids = load_selected_ids(stage3_cfg)
    elif condition == "D":
        synthetic_ids = draw_ids

    synthetic_records = []
    if condition != "A":
        synthetic_records = records_from_synthetic_manifest(
            Path(stage2_cfg.paths.manifest_path),
            Path(stage2_cfg.paths.images_dir),
            synthetic_ids,
        )
        if not synthetic_records:
            raise SystemExit(
                f"UPSTREAM GATE: condition {condition} needs synthetic images but none were found."
            )

    val_frame = load_split("classifier_val", namespace, purpose="schema_validation", caller="stage4")
    val_records = records_from_split(val_frame, images_dir / "classifier_val")
    if not val_records:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed classifier_val images under "
            f"{images_dir / 'classifier_val'}"
        )

    return real_records + synthetic_records, val_records


def run_one(condition: str, seed: int, draw_index: int | None, draw_ids, cfgs, namespace: str) -> dict:
    from scripts.utils.classifier import TrainingBudget, train_classifier

    stage4_cfg, _, _ = cfgs
    train_records, val_records = build_records(condition, draw_ids, cfgs, namespace)

    tag = f"{condition}" + (f"_draw{draw_index}" if draw_index is not None else "") + f"_seed{seed}"
    checkpoint_path = Path(stage4_cfg.paths.checkpoints_dir) / f"condition_{tag}.pt"

    budget = TrainingBudget(
        max_steps=int(stage4_cfg.training.max_steps),
        batch_size=int(stage4_cfg.training.batch_size),
        learning_rate=float(stage4_cfg.training.learning_rate),
        weight_decay=float(stage4_cfg.training.weight_decay),
        eval_every_n_steps=int(stage4_cfg.training.eval_every_n_steps),
        seed=seed,
    )

    config_block = {
        "stage4": OmegaConf.to_container(stage4_cfg, resolve=True),
        "condition": condition,
        "draw_index": draw_index,
        "seed": seed,
    }

    with ExperimentRun(
        stage=f"stage4_condition_{condition}",
        config=config_block,
        dataset_version=split_provenance(namespace)["split_manifest_hash"],
    ) as run:
        model, accounting, history = train_classifier(
            train_records=train_records,
            val_records=val_records,
            budget=budget,
            dropout_p=float(stage4_cfg.model.dropout_p),
            pretrained_source=str(stage4_cfg.model.pretrained_source),
            resolution=int(stage4_cfg.model.resolution),
            progress_desc=f"cond-{tag}",
            checkpoint_path=checkpoint_path,
            resume_from=checkpoint_path if checkpoint_path.is_file() else None,
            checkpoint_metadata={
                "split_manifest_hash": split_provenance(namespace)["split_manifest_hash"],
                "split_namespace": namespace, "dataset_id": tag,
                "condition_or_draw_id": f"{condition}:{draw_index}",
                "config_hash": hash_dict(config_block), "label_policy": "chexpert_mask_uncertain_v1",
                "code_version": get_git_commit_hash(), "code_identity_sha256": current_code_identity_hash(),
            },
        )
        accounting.dataset_id = tag
        accounting.config_hash = hash_dict(config_block)
        accounting.sampling_policy = str(stage4_cfg.training.sampling_policy)
        best = max(history, key=lambda entry: entry["macro_auroc"]) if history else {}
        run.set_checkpoint_path(checkpoint_path)
        run.set_metrics({"best_val_macro_auroc": best.get("macro_auroc"), **accounting.to_dict()})

    return {
        "tag": tag,
        "condition": condition,
        "draw_index": draw_index,
        "seed": seed,
        "checkpoint": accounting.extra.get("best_checkpoint_path", str(checkpoint_path)),
        "resume_checkpoint": str(checkpoint_path),
        "best_val_macro_auroc": best.get("macro_auroc"),
        "accounting": accounting.to_dict(),
        "eval_history": history,
    }


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--condition", choices=CONDITIONS + ["all"], default="all")
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--plan-only", action="store_true", help="Print run/cost accounting and exit")
    args = parser.parse_args()

    cfgs = load_configs()
    stage4_cfg, stage3_cfg, stage2_cfg = cfgs
    namespace = args.namespace or str(stage4_cfg.split_namespace)
    stage4_cfg.split_namespace = namespace
    for key, value in stage4_paths(stage4_cfg, namespace).items():
        stage4_cfg.paths[key] = str(value)
    for key, value in stage3_paths(stage3_cfg, namespace).items():
        if key in stage3_cfg.paths:
            stage3_cfg.paths[key] = str(value)
    for key, value in stage2_paths(stage2_cfg, namespace).items():
        if key in stage2_cfg.paths:
            stage2_cfg.paths[key] = str(value)

    plan = training_plan(stage4_cfg)
    print("Stage 4 run plan (§7 frozen seed policy)", flush=True)
    print(json.dumps(plan, indent=2), flush=True)
    if args.plan_only:
        return 0

    identity = namespace_identity(namespace)
    _, generation_completion = require_generation_complete(stage2_cfg, namespace)
    require_manifest_fields(Path(stage3_cfg.paths.frozen_manifest), {
        "split_namespace": namespace, "namespace_class": identity["namespace_class"],
        "split_manifest_hash": identity["split_manifest_hash"],
        "generation_manifest_sha256": generation_completion["generation_manifest_sha256"],
        "code_identity_sha256": current_code_identity_hash(), "frozen": True,
    }, "frozen ASISM")

    results_dir = Path(stage4_cfg.paths.results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    # Freeze the experiment protocol before any training run starts (§9 step 11).
    protocol = {
        "stage": "stage4_frozen_protocol",
        "frozen": True,
        "split_provenance": split_provenance(namespace),
        "model": OmegaConf.to_container(stage4_cfg.model, resolve=True),
        "training": OmegaConf.to_container(stage4_cfg.training, resolve=True),
        "seeds": list(stage4_cfg.seeds),
        "condition_d": OmegaConf.to_container(stage4_cfg.condition_d, resolve=True),
        "threshold_policy": OmegaConf.to_container(stage4_cfg.threshold_policy, resolve=True),
        "label_policy": {
            "classifier_target_labels": 14,
            "primary_endpoint_labels": PRIMARY_ENDPOINT_LABELS,
            "no_finding_in_primary": False,
            "support_devices_in_primary": False,
        },
        "uncertainty_policy": "raw preserved; -1 and blank masked in loss and metrics",
        "fairness_protocol": "equal optimizer steps across A-E and all D draws",
        "primary_endpoint": f"macro-AUROC over the {len(PRIMARY_ENDPOINT_LABELS)} primary labels on final_eval_heldout",
        "primary_comparison": "C vs D",
        "multiplicity": {"confirmatory": "holm_bonferroni", "exploratory": "benjamini_hochberg"},
        "run_plan": plan,
        "git_commit_hash": get_git_commit_hash(),
        "code_identity_sha256": current_code_identity_hash(),
        "environment_versions": get_library_versions(),
        "serialized_config": OmegaConf.to_container(stage4_cfg, resolve=True),
    }
    protocol_path = Path(stage4_cfg.paths.protocol_manifest)
    if protocol_path.exists():
        existing = read_json(protocol_path)
        if existing != protocol:
            raise SystemExit("Frozen experiment protocol already exists with different content; use a new versioned protocol/run ID")
    else:
        write_frozen_json(protocol_path, protocol)
    freeze_components = {
        Path(stage4_cfg.paths.label_policy_manifest): protocol["label_policy"],
        Path(stage4_cfg.paths.threshold_policy_manifest): protocol["threshold_policy"],
        Path(stage4_cfg.paths.seed_plan_manifest): {"seeds": protocol["seeds"], "run_plan": plan},
        Path(stage4_cfg.paths.d_matching_policy_manifest): protocol["condition_d"],
    }
    for artifact_path, payload in freeze_components.items():
        frozen_payload = {"schema_version": 1, "protocol_manifest_hash": sha256_file(protocol_path),
                          "config": payload, "environment_versions": protocol.get("environment_versions", {})}
        if artifact_path.exists():
            existing = read_json(artifact_path)
            if existing != {**frozen_payload, "frozen": True}:
                raise SystemExit(f"Frozen methodology artifact differs and cannot be overwritten: {artifact_path}")
        else:
            write_frozen_json(artifact_path, frozen_payload)
    print(f"\nFrozen protocol -> {stage4_cfg.paths.protocol_manifest}", flush=True)

    enabled_conditions = list(stage4_cfg.get("conditions", CONDITIONS))
    if args.condition != "all" and args.condition not in enabled_conditions:
        raise SystemExit(f"Condition {args.condition} is disabled by this explicit configuration: {enabled_conditions}")
    conditions = enabled_conditions if args.condition == "all" else [args.condition]
    seeds = list(stage4_cfg.seeds)
    all_results = []
    completed_dir = results_dir / "completed_runs"
    completed_dir.mkdir(exist_ok=True)

    def execute_or_resume(condition, seed, draw_index=None, ids=None):
        tag = f"{condition}" + (f"_draw{draw_index}" if draw_index is not None else "") + f"_seed{seed}"
        expected_run_identity = {
            "split_manifest_hash": split_provenance(namespace)["split_manifest_hash"],
            "split_namespace": namespace, "tag": tag,
            "code_identity_sha256": current_code_identity_hash(),
        }
        record_path = completed_dir / f"{tag}.json"
        if record_path.is_file():
            record = read_json(record_path)
            checkpoint = Path(record.get("checkpoint", ""))
            history_hash = hash_dict({"history": record.get("result", {}).get("eval_history", [])}, length=64)
            compatible = all(record.get(key) == value for key, value in expected_run_identity.items())
            if (record.get("status") == "complete" and compatible and checkpoint.is_file()
                    and record.get("checkpoint_hash") == sha256_file(checkpoint)
                    and record.get("metric_history_hash") == history_hash):
                return record["result"]
            raise SystemExit(f"Invalid/corrupt completed-run record for {tag}; refusing silent reuse: {record_path}")
        result = run_one(condition, seed, draw_index, ids, cfgs, namespace)
        checkpoint = Path(result["checkpoint"])
        history_hash = hash_dict({"history": result["eval_history"]}, length=64)
        completion = {"schema_version": 2, "status": "complete", **expected_run_identity, "checkpoint": str(checkpoint),
                      "checkpoint_hash": sha256_file(checkpoint), "metric_history_hash": history_hash,
                      "accounting": result["accounting"], "split_provenance": split_provenance(namespace), "result": result}
        write_json(record_path, completion)
        return result

    # Condition D's draws are matched against condition C's actual selection.
    draw_specs: list[tuple[int, list[str], dict]] = []
    if "D" in conditions:
        intended = intended_lookup(stage2_cfg)
        selected_ids = load_selected_ids(stage3_cfg)
        target_profile = profile_of(selected_ids, intended)
        all_ids = load_all_synthetic_ids(stage2_cfg)
        pool = [i for i in all_ids if i not in set(selected_ids)] or all_ids

        for draw_index, draw_seed in enumerate(list(stage4_cfg.condition_d.draw_seeds)):
            ids, report = build_matched_random_draw(
                pool, target_profile, intended, draw_seed, stage4_cfg
            )
            draw_specs.append((draw_index, ids, report))
            if not report["passed"]:
                raise SystemExit(
                    f"Condition D matching infeasible for draw {draw_index}, seed {draw_seed}; "
                    f"refusing training. Unsatisfied constraints: {report['unsatisfied_constraints']}; "
                    f"details={json.dumps(report, sort_keys=True)}"
                )
            write_json(results_dir / f"condition_d_draw_{draw_index}.json", {"draw_index": draw_index, "image_ids": ids, **report})

        write_json(
            results_dir / "condition_d_matching_report.json",
            {
                "target_profile_from_condition_C": target_profile,
                "n_draws": len(draw_specs),
                "draws": [report for _, _, report in draw_specs],
            },
        )
        print(f"Condition D: {len(draw_specs)} matched draws built", flush=True)
        for _, _, report in draw_specs:
            print(
                f"  seed {report['seed']}: n={report['achieved_total']}/{report['target_total']} "
                f"labels_within_tolerance={report['all_labels_within_tolerance']}",
                flush=True,
            )

    for condition in conditions:
        if condition == "D":
            for draw_index, ids, _ in draw_specs:
                for seed in seeds:
                    all_results.append(execute_or_resume(condition, seed, draw_index, ids))
        else:
            for seed in seeds:
                all_results.append(execute_or_resume(condition, seed))

    write_json(
        results_dir / "stage4_training_results.json",
        {
            "results": all_results,
            "run_plan": plan,
            "protocol_manifest": str(stage4_cfg.paths.protocol_manifest),
            "git_commit_hash": get_git_commit_hash(),
        },
    )
    print(f"\n{len(all_results)} runs complete -> {results_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


# Stage 5 — Evaluation


## `scripts/eval/compare_conditions.py`


In [ ]:
# SOURCE FILE: scripts/eval/compare_conditions.py
#!/usr/bin/env python3
"""Stage 5 analysis — condition comparison, statistics, and result tables (§8).

Reads only the prediction parquet files written by stage5_evaluate.py. No torch, no GPU: the whole
analysis layer runs locally, which is what keeps interpretation and figure iteration off the pod.

Implements the frozen statistical policy (§8.2):
  - patient-level paired bootstrap for effect sizes and 95% CIs;
  - Holm-Bonferroni for the CONFIRMATORY family (primary endpoint / primary comparison);
  - Benjamini-Hochberg FDR for EXPLORATORY analyses, labelled as such everywhere;
  - effect sizes reported alongside every p-value.

Condition D is summarised ACROSS its five draws (mean and spread), never as a single draw.

Usage:
    python scripts/eval/compare_conditions.py --run-dir outputs/stage5/<run_id>
"""

from __future__ import annotations

import argparse
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.labels import (  # noqa: E402
    CLASSIFIER_TARGET_LABELS,
    PRIMARY_ENDPOINT_LABELS,
    build_label_arrays,
)
from scripts.utils.manifest import read_json, sha256_file, write_json  # noqa: E402
from scripts.utils.metrics import (  # noqa: E402
    auroc,
    benjamini_hochberg,
    full_metric_suite,
    holm_bonferroni,
    paired_bootstrap_difference,
    patient_level_bootstrap,
)

TAG_PATTERN = re.compile(r"^(?P<condition>[A-E])(?:_draw(?P<draw>\d+))?_seed(?P<seed>\d+)$")

# The CONFIRMATORY family, fixed before results are seen (§8.1/§8.2). Everything else is
# exploratory and corrected with FDR.
CONFIRMATORY_COMPARISONS = [("C", "D")]
SECONDARY_COMPARISONS = [("A", "B"), ("A", "C"), ("A", "E"), ("B", "C")]


def parse_tag(tag: str) -> dict | None:
    match = TAG_PATTERN.match(tag)
    if not match:
        return None
    return {
        "condition": match.group("condition"),
        "draw": int(match.group("draw")) if match.group("draw") is not None else None,
        "seed": int(match.group("seed")),
    }


def load_predictions(run_dir: Path) -> tuple[dict[str, list[pd.DataFrame]], pd.DataFrame]:
    predictions_dir = run_dir / "predictions"
    if not predictions_dir.is_dir():
        raise SystemExit(f"No predictions directory at {predictions_dir}")

    by_condition: dict[str, list[pd.DataFrame]] = defaultdict(list)
    reference = None
    for path in sorted(predictions_dir.glob("*.parquet")):
        info = parse_tag(path.stem)
        if info is None:
            continue
        sidecar_path = path.with_suffix(".complete.json")
        if not sidecar_path.is_file():
            raise SystemExit(f"Prediction completion sidecar missing: {sidecar_path}")
        sidecar = read_json(sidecar_path)
        if sidecar.get("prediction_sha256") != sha256_file(path):
            raise SystemExit(f"Prediction hash mismatch: {path}")
        frame = pd.read_parquet(path)
        frame.attrs.update(info)
        by_condition[info["condition"]].append(frame)
        reference = frame if reference is None else reference

    if reference is None:
        raise SystemExit(f"No prediction files found in {predictions_dir}")
    return by_condition, reference


def build_truth(run_dir: Path, reference: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Ground-truth arrays for the evaluation rows, aligned to the prediction row order."""
    manifest = read_json(run_dir / "final_eval_run_manifest.json")
    namespace = manifest["namespace"]

    # Non-outcome access: we already hold the outcome-bearing predictions; this reloads the labels
    # for metric computation inside the same declared run.
    from scripts.utils.splits import load_split

    frame = load_split(
        "final_eval_heldout",
        namespace,
        purpose="load_labels_for_evaluation",
        final_eval_run_id=manifest["final_eval_run_id"],
        final_eval_context_path=run_dir / "final_eval_registration.json",
        caller="compare_conditions",
    )
    from scripts.utils.identifiers import sanitize_image_id

    frame["image_id"] = frame["Path"].apply(sanitize_image_id)
    aligned = reference[["image_id"]].merge(frame, on="image_id", how="left")

    for label in CLASSIFIER_TARGET_LABELS:
        if label not in aligned.columns:
            aligned[label] = np.nan
    raw, targets, masks = build_label_arrays(aligned[CLASSIFIER_TARGET_LABELS], CLASSIFIER_TARGET_LABELS)
    return raw, targets, masks


def macro_auroc_for(probabilities: np.ndarray, targets: np.ndarray, masks: np.ndarray,
                    rows: np.ndarray) -> float:
    """Masked macro-AUROC over the 11 primary labels for a subset of rows."""
    values = []
    for label in PRIMARY_ENDPOINT_LABELS:
        index = CLASSIFIER_TARGET_LABELS.index(label)
        mask = masks[rows, index].astype(bool)
        if mask.sum() == 0:
            continue
        y_true = targets[rows][mask, index].astype(int)
        y_score = probabilities[rows][mask, index]
        if (y_true == 1).sum() == 0 or (y_true == 0).sum() == 0:
            continue
        value = auroc(y_score, y_true)
        if not np.isnan(value):
            values.append(value)
    return float(np.mean(values)) if values else float("nan")


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--bootstrap-resamples", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    run_dir = Path(args.run_dir)
    by_condition, reference = load_predictions(run_dir)
    _, targets, masks = build_truth(run_dir, reference)
    patient_ids = reference["patient_id"].to_numpy()

    label_columns = CLASSIFIER_TARGET_LABELS

    # Per-condition mean probabilities across seeds (and, for D, across draws AND seeds).
    condition_probabilities: dict[str, np.ndarray] = {}
    condition_spread: dict[str, dict] = {}
    for condition, frames in by_condition.items():
        stacked = np.stack([frame[label_columns].to_numpy() for frame in frames], axis=0)
        condition_probabilities[condition] = stacked.mean(axis=0)

        per_run_macro = []
        for frame in frames:
            value = macro_auroc_for(
                frame[label_columns].to_numpy(), targets, masks, np.arange(len(frame))
            )
            per_run_macro.append(
                {"draw": frame.attrs.get("draw"), "seed": frame.attrs.get("seed"),
                 "macro_auroc": value}
            )
        values = [entry["macro_auroc"] for entry in per_run_macro if not np.isnan(entry["macro_auroc"])]
        condition_spread[condition] = {
            "n_runs": len(frames),
            "per_run": per_run_macro,
            "mean_macro_auroc": float(np.mean(values)) if values else float("nan"),
            "std_macro_auroc": float(np.std(values, ddof=1)) if len(values) > 1 else float("nan"),
            "min_macro_auroc": float(np.min(values)) if values else float("nan"),
            "max_macro_auroc": float(np.max(values)) if values else float("nan"),
        }

    # Full metric suite per condition.
    per_condition_metrics = {}
    for condition, probabilities in condition_probabilities.items():
        thresholds_path = next(
            (run_dir / "predictions").glob(f"{condition}*seed*.thresholds.json"), None
        )
        thresholds = read_json(thresholds_path)["thresholds"] if thresholds_path else {}
        suite = full_metric_suite(probabilities, targets, masks, thresholds, label_columns)

        bootstrap = patient_level_bootstrap(
            patient_ids,
            lambda rows, p=probabilities: macro_auroc_for(p, targets, masks, rows),
            n_resamples=args.bootstrap_resamples,
            seed=args.seed,
        )
        suite["macro_auroc_bootstrap"] = bootstrap
        suite["across_run_spread"] = condition_spread[condition]
        per_condition_metrics[condition] = suite

    # Paired comparisons.
    def compare(left: str, right: str) -> dict | None:
        if left not in condition_probabilities or right not in condition_probabilities:
            return None
        return paired_bootstrap_difference(
            patient_ids,
            lambda rows: macro_auroc_for(condition_probabilities[left], targets, masks, rows),
            lambda rows: macro_auroc_for(condition_probabilities[right], targets, masks, rows),
            n_resamples=args.bootstrap_resamples,
            seed=args.seed,
        )

    confirmatory, exploratory = {}, {}
    for left, right in CONFIRMATORY_COMPARISONS:
        result = compare(left, right)
        if result:
            confirmatory[f"{left}_vs_{right}"] = result
    for left, right in SECONDARY_COMPARISONS:
        result = compare(left, right)
        if result:
            exploratory[f"{left}_vs_{right}"] = result

    # Per-label exploratory comparisons for the primary comparison pair.
    per_label_exploratory = {}
    if "C" in condition_probabilities and "D" in condition_probabilities:
        for label in PRIMARY_ENDPOINT_LABELS:
            index = label_columns.index(label)

            def label_auroc(rows, probabilities, idx=index):
                mask = masks[rows, idx].astype(bool)
                if mask.sum() == 0:
                    return float("nan")
                y_true = targets[rows][mask, idx].astype(int)
                if (y_true == 1).sum() == 0 or (y_true == 0).sum() == 0:
                    return float("nan")
                return auroc(probabilities[rows][mask, idx], y_true)

            result = paired_bootstrap_difference(
                patient_ids,
                lambda rows: label_auroc(rows, condition_probabilities["C"]),
                lambda rows: label_auroc(rows, condition_probabilities["D"]),
                n_resamples=max(200, args.bootstrap_resamples // 5),
                seed=args.seed,
            )
            per_label_exploratory[f"C_vs_D::{label}"] = result

    confirmatory_corrected = holm_bonferroni(
        {name: result["p_value"] for name, result in confirmatory.items()}
    )
    exploratory_corrected = benjamini_hochberg(
        {
            **{name: result["p_value"] for name, result in exploratory.items()},
            **{name: result["p_value"] for name, result in per_label_exploratory.items()},
        }
    )

    report = {
        "run_dir": str(run_dir),
        "primary_endpoint": f"macro-AUROC over the {len(PRIMARY_ENDPOINT_LABELS)} primary disease labels",
        "primary_comparison": "C vs D",
        "n_eval_rows": len(reference),
        "n_eval_patients": int(pd.Series(patient_ids).nunique()),
        "per_condition": per_condition_metrics,
        "confirmatory": {
            name: {**result, "correction": confirmatory_corrected.get(name, {})}
            for name, result in confirmatory.items()
        },
        "exploratory": {
            name: {**result, "correction": exploratory_corrected.get(name, {})}
            for name, result in {**exploratory, **per_label_exploratory}.items()
        },
        "statistical_policy": {
            "resampling_unit": "patient",
            "confirmatory_correction": "holm_bonferroni",
            "exploratory_correction": "benjamini_hochberg_fdr",
            "effect_sizes_reported": True,
            "policy_frozen_before_results": True,
        },
    }
    write_json(run_dir / "stage5_comparison_report.json", report)

    # Human-readable tables.
    rows = []
    for condition in sorted(per_condition_metrics):
        suite = per_condition_metrics[condition]
        bootstrap = suite["macro_auroc_bootstrap"]
        spread = suite["across_run_spread"]
        rows.append(
            {
                "condition": condition,
                "macro_auroc": round(suite["macro_auroc_primary"], 4),
                "ci_lower": round(bootstrap["ci_lower"], 4),
                "ci_upper": round(bootstrap["ci_upper"], 4),
                "across_run_mean": round(spread["mean_macro_auroc"], 4),
                "across_run_std": (
                    round(spread["std_macro_auroc"], 4)
                    if not np.isnan(spread["std_macro_auroc"]) else None
                ),
                "n_runs": spread["n_runs"],
                "macro_auprc": round(suite["macro_auprc_primary"], 4),
                "micro_auroc": round(suite["micro_auroc_primary"], 4),
                "mean_brier": round(suite["mean_brier_primary"], 4),
                "mean_ece": round(suite["mean_ece_primary"], 4),
            }
        )
    condition_table = pd.DataFrame(rows)
    condition_table.to_csv(run_dir / "table_conditions.csv", index=False)

    comparison_rows = []
    for family, source, corrected in (
        ("confirmatory", confirmatory, confirmatory_corrected),
        ("exploratory", {**exploratory, **per_label_exploratory}, exploratory_corrected),
    ):
        for name, result in source.items():
            correction = corrected.get(name, {})
            comparison_rows.append(
                {
                    "comparison": name,
                    "family": family,
                    "effect_size": round(result["effect_size"], 4),
                    "ci_lower": round(result["ci_lower"], 4),
                    "ci_upper": round(result["ci_upper"], 4),
                    "p_value": round(result["p_value"], 5),
                    "adjusted_p": (
                        round(correction.get("adjusted_p_value"), 5)
                        if correction.get("adjusted_p_value") is not None
                        and not np.isnan(correction.get("adjusted_p_value", np.nan))
                        else None
                    ),
                    "significant": correction.get("rejected"),
                    "correction": correction.get("method"),
                }
            )
    comparison_table = pd.DataFrame(comparison_rows)
    comparison_table.to_csv(run_dir / "table_comparisons.csv", index=False)

    print(
        f"\nCondition summary (primary endpoint: macro-AUROC over {len(PRIMARY_ENDPOINT_LABELS)} disease labels)",
        flush=True,
    )
    print(condition_table.to_string(index=False), flush=True)
    print("\nComparisons", flush=True)
    print(comparison_table.to_string(index=False), flush=True)
    print(f"\nReport -> {run_dir / 'stage5_comparison_report.json'}", flush=True)
    print(f"Tables -> {run_dir / 'table_conditions.csv'}, {run_dir / 'table_comparisons.csv'}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/eval/compute_fid_clipscore.py`


In [ ]:
# SOURCE FILE: scripts/eval/compute_fid_clipscore.py
#!/usr/bin/env python3
"""FID and CLIP-score trend for a generated probe sample set (docs/stage1_plan.md §9).

Two FID variants are computed:
  - Standard Inception-based FID (torchmetrics) — field-standard, comparable to other work, but a
    poor fit for grayscale medical images (Inception features are ImageNet-domain).
  - Domain FID using torchxrayvision's DenseNet121 features — the more literature-appropriate
    signal for this project (docs/stage1_plan.md: "FID doesn't reflect diagnostic content").

Both are tracked ONLY as relative trends across checkpoints, not absolute quality bars (plan §9).
The real reference set is gen_val (held out from training but touched by Stage 1) —
classifier_heldout must never be read here, since it is reserved untouched for Stage 4/5.

Usage:
    python scripts/eval/compute_fid_clipscore.py --generated-dir outputs/stage1_samples/<checkpoint_name>
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_stage1_config  # noqa: E402
from scripts.utils.manifest import read_json, write_json  # noqa: E402


def load_images_as_uint8_tensor(paths: list[Path], size: int) -> torch.Tensor:
    from torchvision.transforms import functional as TF

    tensors = []
    for p in paths:
        with Image.open(p) as im:
            im = im.convert("RGB").resize((size, size))
            tensors.append(TF.pil_to_tensor(im))
    return torch.stack(tensors)


def frechet_distance(mu1: np.ndarray, sigma1: np.ndarray, mu2: np.ndarray, sigma2: np.ndarray) -> float:
    from scipy import linalg

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean))


def compute_domain_fid(real_paths: list[Path], generated_paths: list[Path], device: torch.device) -> float:
    import torchxrayvision as xrv
    import torchvision.transforms as T

    model = xrv.models.DenseNet(weights="densenet121-res224-all").to(device).eval()
    transform = T.Compose([T.Resize((224, 224))])

    def extract_features(paths: list[Path]) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for p in paths:
                with Image.open(p) as im:
                    arr = np.array(im.convert("L"), dtype=np.float32)
                arr = xrv.datasets.normalize(arr, 255)
                tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(device)
                tensor = transform(tensor)
                features = model.features(tensor)
                feats.append(features.mean(dim=[2, 3]).squeeze(0).cpu().numpy())
        return np.stack(feats)

    real_feats = extract_features(real_paths)
    gen_feats = extract_features(generated_paths)

    mu1, sigma1 = real_feats.mean(axis=0), np.cov(real_feats, rowvar=False)
    mu2, sigma2 = gen_feats.mean(axis=0), np.cov(gen_feats, rowvar=False)
    return frechet_distance(mu1, sigma1, mu2, sigma2)


def compute_inception_fid(real_paths: list[Path], generated_paths: list[Path], device: torch.device) -> float:
    from torchmetrics.image.fid import FrechetInceptionDistance

    fid = FrechetInceptionDistance(normalize=False).to(device)
    fid.update(load_images_as_uint8_tensor(real_paths, 299).to(device), real=True)
    fid.update(load_images_as_uint8_tensor(generated_paths, 299).to(device), real=False)
    return float(fid.compute().item())


def compute_clip_score(generated_paths: list[Path], prompts: list[str], device: torch.device) -> float:
    from torchmetrics.multimodal.clip_score import CLIPScore

    metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(device)
    images = load_images_as_uint8_tensor(generated_paths, 224).to(device)
    score = metric(images, prompts)
    return float(score.item())


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--generated-dir", type=str, required=True, help="Directory with probe_manifest.json (from generate_probe_samples.py)")
    parser.add_argument("--num-real-reference", type=int, default=200)
    args = parser.parse_args()

    cfg = load_stage1_config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    generated_dir = Path(args.generated_dir)
    manifest = read_json(generated_dir / "probe_manifest.json")
    generated_paths = [Path(r["image_path"]) for r in manifest["records"]]
    prompts = [r["prompt"] for r in manifest["records"]]

    val_images_dir = Path(cfg.paths.images_dir) / "gen_val"
    real_paths = sorted(val_images_dir.glob("*.jpg"))[: args.num_real_reference]
    if not real_paths:
        print(f"No real reference images found under {val_images_dir} — run the preprocessing pipeline first.")
        return 1

    results = {"generated_dir": str(generated_dir), "num_real_reference": len(real_paths)}

    if cfg.validation.clip_score:
        results["clip_score"] = compute_clip_score(generated_paths, prompts, device)
        print(f"CLIP score: {results['clip_score']:.4f}")

    if cfg.validation.fid.inception:
        try:
            results["fid_inception"] = compute_inception_fid(real_paths, generated_paths, device)
            print(f"FID (Inception): {results['fid_inception']:.2f}")
        except Exception as e:
            print(f"Inception FID failed ({e}); skipping.")

    if cfg.validation.fid.domain_torchxrayvision:
        try:
            results["fid_domain_torchxrayvision"] = compute_domain_fid(real_paths, generated_paths, device)
            print(f"FID (domain, torchxrayvision): {results['fid_domain_torchxrayvision']:.2f}")
        except Exception as e:
            print(f"Domain FID failed ({e}); skipping.")

    write_json(generated_dir / "fid_clipscore_results.json", results)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/eval/generate_probe_samples.py`


In [ ]:
# SOURCE FILE: scripts/eval/generate_probe_samples.py
#!/usr/bin/env python3
"""Generate a fixed-seed probe-prompt sample grid from a Stage 1 LoRA checkpoint
(docs/stage1_plan.md §9 — qualitative plausibility + informal directional-conditioning check).

The probe prompt set is fixed and regenerated with the same seeds at every checkpoint, so samples
are directly comparable checkpoint-to-checkpoint: one prompt per pathology positive, one no-finding
prompt, and a few multi-label combinations. Prompts are built with scripts/utils/caption_builder.py
— the same module used at training time — so probe conditioning matches train-time phrasing exactly.

Usage:
    python scripts/eval/generate_probe_samples.py --checkpoint checkpoints/stage1_lora_sdxl/<run_id>/final
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import CaptionConfig, PATHOLOGY_COLUMNS, build_caption  # noqa: E402
from scripts.utils.config import load_stage1_config  # noqa: E402
from scripts.utils.manifest import write_json  # noqa: E402

DEVICE_COLUMN = "Support Devices"
NO_FINDING_COLUMN = "No Finding"

MULTI_LABEL_COMBOS = [
    ["Cardiomegaly", "Pleural Effusion"],
    ["Atelectasis", "Consolidation"],
    ["Edema", "Pneumonia"],
]


def build_probe_rows(caption_config: CaptionConfig) -> list[dict]:
    """One synthetic label row per single-pathology positive, one No Finding, a few multi-label
    combos. Age/sex are fixed to a neutral reference point so only the finding clause varies."""
    base_row = {"Sex": "Male", "Age": 50, "Frontal/Lateral": "Frontal", "AP/PA": "AP"}
    rows = []

    no_finding_row = dict(base_row, **{NO_FINDING_COLUMN: 1})
    rows.append({"name": "no_finding", "row": no_finding_row})

    for col in PATHOLOGY_COLUMNS:
        if col in (NO_FINDING_COLUMN, DEVICE_COLUMN):
            continue
        row = dict(base_row, **{c: 0 for c in PATHOLOGY_COLUMNS})
        row[col] = 1
        rows.append({"name": col.replace(" ", "_").lower(), "row": row})

    for combo in MULTI_LABEL_COMBOS:
        row = dict(base_row, **{c: 0 for c in PATHOLOGY_COLUMNS})
        for col in combo:
            row[col] = 1
        rows.append({"name": "_".join(c.replace(" ", "").lower() for c in combo), "row": row})

    return rows


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=str, required=True, help="LoRA weights dir (e.g. .../lora_weights/step_1000 or .../final)")
    parser.add_argument("--steps", type=int, default=30)
    parser.add_argument("--guidance-scale", type=float, default=5.0)
    args = parser.parse_args()

    cfg = load_stage1_config()
    caption_config = CaptionConfig(
        age_bucket_width_years=cfg.captions.age_bucket_width_years,
        uncertain_label_policy=cfg.captions.uncertain_label_policy,
        no_finding_overrides_positives=cfg.captions.no_finding_overrides_positives,
        num_paraphrase_variants=cfg.captions.num_paraphrase_variants,
        template_version=cfg.captions.template_version,
    )

    checkpoint_path = Path(args.checkpoint)
    # e.g. ".../lora_weights/step_1000" -> "lora_weights_step_1000", ".../final" -> "final"
    checkpoint_name = "_".join(checkpoint_path.parts[-2:]) if checkpoint_path.parent.name == "lora_weights" else checkpoint_path.name
    out_dir = Path(cfg.paths.outputs_dir) / checkpoint_name
    out_dir.mkdir(parents=True, exist_ok=True)

    from diffusers import StableDiffusionXLPipeline

    pipeline = StableDiffusionXLPipeline.from_pretrained(
        cfg.model.base_model_id, torch_dtype=torch.bfloat16, cache_dir=cfg.paths.hf_cache_dir
    )
    pipeline.load_lora_weights(str(checkpoint_path))
    pipeline.to("cuda" if torch.cuda.is_available() else "cpu")
    pipeline.set_progress_bar_config(disable=False)

    negative_prompt = "text, watermark, labels, color image, cropped anatomy, deformed chest, low quality"
    probe_rows = build_probe_rows(caption_config)

    records = []
    for i, probe in enumerate(probe_rows):
        prompt = build_caption(probe["row"], caption_config, variant_index=0)
        seed = cfg.validation.probe_seed + i
        generator = torch.Generator(device=pipeline.device).manual_seed(seed)
        image = pipeline(
            prompt=prompt,
            negative_prompt=negative_prompt,
            height=cfg.data.resolution,
            width=cfg.data.resolution,
            num_inference_steps=args.steps,
            guidance_scale=args.guidance_scale,
            generator=generator,
        ).images[0]
        image_path = out_dir / f"{probe['name']}.png"
        image.save(image_path)
        records.append({"name": probe["name"], "prompt": prompt, "seed": seed, "image_path": str(image_path)})
        print(f"[{i + 1}/{len(probe_rows)}] {probe['name']}: {prompt}")

    write_json(out_dir / "probe_manifest.json", {"checkpoint": str(checkpoint_path), "records": records})
    print(f"Wrote {len(records)} probe samples -> {out_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/eval/stage5_evaluate.py`


In [ ]:
# SOURCE FILE: scripts/eval/stage5_evaluate.py
#!/usr/bin/env python3
"""Stage 5 — protected final evaluation on final_eval_heldout (docs/stages2_to_5_plan.md §8).

Refuses to run unless every precondition in §8 holds:
  - the production split manifest exists AND is frozen AND passed its support check;
  - the recorded split hash matches the current manifest;
  - the frozen experiment-protocol manifest exists (Stage 4);
  - the ASISM freeze manifest exists (Stage 3);
  - the required A-E/D checkpoints exist, with their hashes recorded;
  - the classification-threshold policy is frozen;
  - an explicit --final-eval-run-id is supplied.

The run id is what unlocks outcome-bearing access to final_eval_heldout through
scripts/utils/splits.assert_final_eval_access_allowed(). No other code path can obtain it.

TECHNICAL RESUME vs. METHODOLOGICAL RE-EVALUATION (§8.4): predictions are written incrementally,
and an interrupted run may resume from the same frozen artifacts and complete the same partial
prediction files without altering the protocol. That is an engineering property. What is NOT
permitted is inspecting results and then re-running under a changed method — a genuine
methodological error requires returning to the development stage, re-freezing, and declaring a NEW
run id, which produces a separate result rather than overwriting this one.

Usage:
    python scripts/eval/stage5_evaluate.py --final-eval-run-id stage5-2026-08-07-a --namespace production
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import tempfile
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import CLASSIFIER_TARGET_LABELS, build_label_arrays  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import (  # noqa: E402
    load_split,
    read_split_manifest,
    require_frozen_production_split_run,
    split_provenance,
)
from scripts.utils.artifact_contracts import current_code_identity_hash, stage3_paths, stage4_paths  # noqa: E402


def file_hash(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()[:16]


def require_smoke_final_eval_authorization(namespace: str, manifest: dict, project_root: Path) -> dict:
    """Authorize fixture or real-data engineering smoke without weakening production guards.

    The historical fixture smoke keeps its existing marker.  A real-data smoke requires a
    namespace- and split-hash-bound marker, so a marker created for one development split cannot
    unlock another split (and a production-class namespace is always refused).
    """
    if manifest.get("namespace_class") != "dev":
        raise SystemExit("Smoke Stage 5 requires a dev-class split")

    if namespace == "dev-smoke-v1" and (project_root / "SMOKE_ONLY.json").is_file():
        return {"kind": "fixture", "upstream_code_identity_sha256": current_code_identity_hash()}

    marker_path = project_root / "REAL_DATA_SMOKE_ONLY.json"
    if not marker_path.is_file():
        raise SystemExit(
            "Real-data smoke Stage 5 requires REAL_DATA_SMOKE_ONLY.json in PROJECT_ROOT"
        )
    marker = read_json(marker_path)
    expected = {
        "schema_version": 1,
        "purpose": "real-data-engineering-smoke-only",
        "namespace": namespace,
        "split_manifest_hash": manifest.get("manifest_hash"),
        "not_scientific_evidence": True,
    }
    mismatches = {
        key: {"expected": value, "actual": marker.get(key)}
        for key, value in expected.items()
        if marker.get(key) != value
    }
    if mismatches:
        raise SystemExit(f"Real-data smoke authorization marker mismatch: {mismatches}")
    upstream_identity = marker.get("upstream_code_identity_sha256")
    if not isinstance(upstream_identity, str) or len(upstream_identity) != 64:
        raise SystemExit(
            "Real-data smoke authorization marker requires a 64-character "
            "upstream_code_identity_sha256"
        )
    return {"kind": "real_data", "upstream_code_identity_sha256": upstream_identity}


def enforce_preconditions(namespace: str, run_id: str) -> dict:
    """All of §8's execution guards. Any failure aborts before final_eval_heldout is opened."""
    failures: list[str] = []
    evidence: dict = {"final_eval_run_id": run_id, "namespace": namespace}

    if not run_id:
        failures.append("no --final-eval-run-id supplied")

    # 1-2. Production splits frozen, support check passed, hash recorded.
    try:
        smoke_mode = os.environ.get("THESIS_SMOKE_MODE") == "1"
        if smoke_mode:
            manifest = read_split_manifest(namespace)
            smoke_authorization = require_smoke_final_eval_authorization(
                namespace, manifest, Path(os.environ.get("PROJECT_ROOT", "."))
            )
            evidence["smoke_only_not_scientific_evidence"] = True
            evidence["smoke_kind"] = smoke_authorization["kind"]
            evidence["upstream_code_identity_sha256"] = smoke_authorization[
                "upstream_code_identity_sha256"
            ]
        else:
            manifest = require_frozen_production_split_run(namespace)
        evidence["split_manifest_hash"] = manifest.get("manifest_hash")
        evidence["split_support_check_passed"] = manifest.get("support_check", {}).get("passed")
        evidence["split_frozen"] = manifest.get("frozen")
    except SystemExit as exc:
        failures.append(f"split manifest: {exc}")

    expected_code_identity = evidence.get(
        "upstream_code_identity_sha256", current_code_identity_hash()
    )

    stage4_cfg = load_named_config("stage4_classifier.yaml", "stage4")
    stage3_cfg = load_named_config("stage3_asism.yaml", "stage3")
    for key, value in stage4_paths(stage4_cfg, namespace).items():
        stage4_cfg.paths[key] = str(value)
    for key, value in stage3_paths(stage3_cfg, namespace).items():
        if key in stage3_cfg.paths:
            stage3_cfg.paths[key] = str(value)

    # 3. Frozen experiment protocol.
    protocol_path = Path(stage4_cfg.paths.protocol_manifest)
    if not protocol_path.is_file():
        failures.append(f"frozen experiment-protocol manifest missing: {protocol_path}")
    else:
        protocol = read_json(protocol_path)
        if not protocol.get("frozen"):
            failures.append("experiment-protocol manifest exists but is not marked frozen")
        evidence["protocol_manifest_hash"] = sha256_file(protocol_path)
        if protocol.get("split_provenance", {}).get("split_manifest_hash") != evidence.get("split_manifest_hash"):
            failures.append("experiment protocol split hash differs from the requested frozen split")
        if protocol.get("code_identity_sha256") != expected_code_identity:
            failures.append("experiment protocol code identity differs from the current source tree")
        # 6. Threshold policy frozen.
        if not protocol.get("threshold_policy"):
            failures.append("classification-threshold policy is not recorded in the frozen protocol")
        else:
            evidence["threshold_policy"] = protocol["threshold_policy"]
        frozen_component_hashes = {}
        for key in ("label_policy_manifest", "threshold_policy_manifest", "seed_plan_manifest", "d_matching_policy_manifest"):
            component_path = Path(stage4_cfg.paths[key])
            if not component_path.is_file():
                failures.append(f"frozen methodology component missing: {component_path}")
                continue
            component = read_json(component_path)
            if not component.get("frozen") or component.get("protocol_manifest_hash") != evidence["protocol_manifest_hash"]:
                failures.append(f"frozen methodology component incompatible: {component_path}")
            frozen_component_hashes[key] = sha256_file(component_path)
        evidence["frozen_component_hashes"] = frozen_component_hashes

    # 4. ASISM freeze manifest.
    asism_path = Path(stage3_cfg.paths.frozen_manifest)
    if not asism_path.is_file():
        failures.append(f"ASISM freeze manifest missing: {asism_path}")
    else:
        asism = read_json(asism_path)
        if not asism.get("frozen"):
            failures.append("ASISM manifest exists but is not marked frozen")
        evidence["asism_manifest_hash"] = sha256_file(asism_path)
        if asism.get("split_manifest_hash") != evidence.get("split_manifest_hash"):
            failures.append("ASISM frozen manifest split hash differs from final split")
        if asism.get("code_identity_sha256") != expected_code_identity:
            failures.append("ASISM frozen manifest code identity differs from current source tree")
        evidence["asism_surviving_signals"] = asism.get("surviving_signals")

    # 5. Required checkpoints.
    results_path = Path(stage4_cfg.paths.results_dir) / "stage4_training_results.json"
    if not results_path.is_file():
        failures.append(f"Stage 4 training results missing: {results_path}")
    else:
        results = read_json(results_path)["results"]
        import torch
        checkpoint_hashes = {}
        missing = []
        for entry in results:
            path = Path(entry["checkpoint"])
            if path.is_file():
                checkpoint_hashes[entry["tag"]] = sha256_file(path)
                payload = torch.load(path, map_location="cpu", weights_only=False)
                provenance = payload.get("provenance", {})
                if payload.get("checkpoint_role") != "best_validation_model":
                    failures.append(f"checkpoint is not a persisted best-validation model: {entry['tag']}")
                if provenance.get("split_manifest_hash") != evidence.get("split_manifest_hash"):
                    failures.append(f"checkpoint split provenance mismatch: {entry['tag']}")
                if provenance.get("code_identity_sha256") != expected_code_identity:
                    failures.append(f"checkpoint code identity mismatch: {entry['tag']}")
            else:
                missing.append(entry["tag"])
        if missing:
            failures.append(f"{len(missing)} required checkpoint(s) missing: {missing[:5]}")
        expected = read_json(results_path).get("run_plan", {}).get("total_runs")
        if expected and len(results) != expected:
            failures.append(f"expected {expected} runs, found {len(results)}")
        evidence["checkpoint_hashes"] = checkpoint_hashes
        evidence["n_checkpoints"] = len(checkpoint_hashes)

    if failures:
        raise SystemExit(
            "STAGE 5 REFUSED — preconditions not met (docs/stages2_to_5_plan.md §8):\n"
            + "\n".join(f"  - {failure}" for failure in failures)
            + "\n\nfinal_eval_heldout was NOT opened."
        )
    return evidence


def predict_condition(entry: dict, records: list[dict], stage4_cfg) -> np.ndarray:
    from scripts.utils.classifier import build_model, predict_probabilities, require_torch

    torch = require_torch()
    payload = torch.load(entry["checkpoint"], map_location="cpu", weights_only=False)
    model = build_model(
        len(CLASSIFIER_TARGET_LABELS),
        float(stage4_cfg.model.dropout_p),
        str(stage4_cfg.model.pretrained_source),
    )
    model.load_state_dict(payload["model_state"])
    probabilities, _ = predict_probabilities(
        model, records, int(stage4_cfg.model.resolution)
    )
    return probabilities


def select_thresholds(entry: dict, namespace: str, stage4_cfg) -> dict[str, float]:
    """Thresholds chosen on classifier_val ONLY (frozen policy), never on final_eval_heldout."""
    from scripts.utils.classifier import records_from_split
    from scripts.utils.metrics import sensitivity_specificity_f1

    stage1_cfg = load_stage1_config()
    val_frame = load_split("classifier_val", namespace, purpose="schema_validation", caller="stage5")
    val_records = records_from_split(
        val_frame, Path(stage1_cfg.paths.images_dir) / namespace / "classifier_val"
    )
    probabilities = predict_condition(entry, val_records, stage4_cfg)

    frame = pd.DataFrame([record["labels"] for record in val_records])
    for label in CLASSIFIER_TARGET_LABELS:
        if label not in frame.columns:
            frame[label] = np.nan
    _, targets, masks = build_label_arrays(frame[CLASSIFIER_TARGET_LABELS], CLASSIFIER_TARGET_LABELS)

    thresholds = {}
    for index, label in enumerate(CLASSIFIER_TARGET_LABELS):
        mask = masks[:, index].astype(bool)
        if mask.sum() == 0:
            thresholds[label] = 0.5
            continue
        y_true = targets[mask, index].astype(int)
        y_score = probabilities[mask, index]
        best_threshold, best_j = 0.5, -np.inf
        for candidate in np.linspace(0.05, 0.95, 19):
            sensitivity, specificity, _ = sensitivity_specificity_f1(y_score, y_true, candidate)
            if np.isnan(sensitivity) or np.isnan(specificity):
                continue
            youden_j = sensitivity + specificity - 1
            if youden_j > best_j:
                best_j, best_threshold = youden_j, float(candidate)
        thresholds[label] = best_threshold
    return thresholds


def validate_prediction_frame(frame: pd.DataFrame, expected_image_ids: list[str], expected_patient_ids: list[str]) -> None:
    required = ["image_id", "patient_id", *CLASSIFIER_TARGET_LABELS]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"prediction schema missing columns: {missing}")
    if len(frame) != len(expected_image_ids):
        raise ValueError(f"wrong prediction row count: {len(frame)} != {len(expected_image_ids)}")
    if frame["image_id"].duplicated().any():
        raise ValueError("duplicate prediction image IDs")
    if set(frame["image_id"].astype(str)) != set(map(str, expected_image_ids)):
        raise ValueError("prediction image-ID set mismatch")
    if set(frame["patient_id"].astype(str)) != set(map(str, expected_patient_ids)):
        raise ValueError("prediction patient-ID set mismatch")


def publish_predictions_atomic(path: Path, frame: pd.DataFrame, provenance: dict, expected_image_ids: list[str], expected_patient_ids: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp.parquet", dir=path.parent)
    os.close(fd)
    temporary = Path(temp_name)
    try:
        frame.to_parquet(temporary, index=False)
        validate_prediction_frame(pd.read_parquet(temporary), expected_image_ids, expected_patient_ids)
        os.replace(temporary, path)
        sidecar = {"schema_version": 1, "status": "complete", "prediction_sha256": sha256_file(path),
                   "row_count": len(frame), "image_id_set_hash": hash_dict({"ids": sorted(map(str, expected_image_ids))}, length=64),
                   "patient_id_set_hash": hash_dict({"ids": sorted(set(map(str, expected_patient_ids)))}, length=64), **provenance}
        write_json(path.with_suffix(".complete.json"), sidecar)
    finally:
        temporary.unlink(missing_ok=True)


def validate_completed_predictions(path: Path, expected_provenance: dict, expected_image_ids: list[str], expected_patient_ids: list[str]) -> None:
    sidecar_path = path.with_suffix(".complete.json")
    if not path.is_file() or not sidecar_path.is_file():
        raise ValueError("prediction file and completion sidecar are both required")
    sidecar = read_json(sidecar_path)
    if sidecar.get("prediction_sha256") != sha256_file(path):
        raise ValueError("prediction file hash mismatch/corruption")
    mismatch = {k: (sidecar.get(k), v) for k, v in expected_provenance.items() if sidecar.get(k) != v}
    if mismatch:
        raise ValueError(f"prediction provenance mismatch: {mismatch}")
    validate_prediction_frame(pd.read_parquet(path), expected_image_ids, expected_patient_ids)


def register_final_evaluation(context_path: Path, registration: dict) -> dict:
    """Register a new run or validate an identical technical resume."""
    identity_keys = (
        "final_eval_run_id", "namespace", "split_manifest_hash", "asism_manifest_hash",
        "protocol_manifest_hash", "checkpoint_hashes", "threshold_policy_hash",
        "code_git_version", "code_identity_sha256",
    )
    if context_path.exists():
        existing = read_json(context_path)
        for key in identity_keys:
            if existing.get(key) != registration.get(key):
                raise SystemExit(f"Registered final-evaluation context is incompatible at {key}; technical resume refused")
        return existing
    for prior_path in context_path.parent.parent.glob("*/final_eval_registration.json"):
        prior = read_json(prior_path)
        if prior.get("split_manifest_hash") == registration["split_manifest_hash"] and prior.get("status") in {"outcome_accessed", "complete"}:
            raise SystemExit("Final outcomes were already accessed under another run ID; a corrected run requires a new linked protocol freeze")
    write_json(context_path, registration)
    return registration


def mark_final_outcome_access(context_path: Path) -> dict:
    registration = read_json(context_path)
    registration["status"] = "outcome_accessed"
    registration["first_outcome_access_at_utc"] = registration.get("first_outcome_access_at_utc") or datetime.now(timezone.utc).isoformat()
    write_json(context_path, registration)
    return registration


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--final-eval-run-id", required=True)
    parser.add_argument("--namespace", default="production")
    parser.add_argument("--bootstrap-resamples", type=int, default=1000)
    args = parser.parse_args()

    run_id = args.final_eval_run_id
    namespace = args.namespace

    print("Stage 5 precondition checks...", flush=True)
    evidence = enforce_preconditions(namespace, run_id)
    print("  all preconditions satisfied", flush=True)

    stage4_cfg = load_named_config("stage4_classifier.yaml", "stage4")
    for key, value in stage4_paths(stage4_cfg, namespace).items():
        stage4_cfg.paths[key] = str(value)
    stage1_cfg = load_stage1_config()

    output_dir = Path(stage4_cfg.paths.project_root) / "outputs" / "stage5" / namespace / run_id
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions_dir = output_dir / "predictions"
    predictions_dir.mkdir(exist_ok=True)

    results = read_json(Path(stage4_cfg.paths.results_dir) / "stage4_training_results.json")["results"]
    # Freeze concrete classifier_val-selected values before any outcome-bearing final-data access.
    thresholds_path = output_dir / "frozen_threshold_values.json"
    if not thresholds_path.exists():
        threshold_values = {entry["tag"]: select_thresholds(entry, namespace, stage4_cfg) for entry in results}
        write_json(thresholds_path, {"frozen": True, "selected_on": "classifier_val", "values": threshold_values,
                                    "policy": dict(stage4_cfg.threshold_policy)})
    threshold_hash = sha256_file(thresholds_path)

    context_path = output_dir / "final_eval_registration.json"
    registration = {
        "final_eval_run_id": run_id, "namespace": namespace, "split_manifest_hash": evidence["split_manifest_hash"],
        "asism_manifest_hash": evidence["asism_manifest_hash"], "protocol_manifest_hash": evidence["protocol_manifest_hash"],
        "checkpoint_hashes": evidence["checkpoint_hashes"], "threshold_policy_hash": threshold_hash,
        "code_git_version": get_git_commit_hash(), "code_identity_sha256": current_code_identity_hash(),
        "registered_at_utc": datetime.now(timezone.utc).isoformat(), "status": "registered",
    }
    registration = register_final_evaluation(context_path, registration)

    # This is the ONLY outcome-bearing read of final_eval_heldout in the codebase, and it is
    # unlocked solely by the explicit run id (§1.6).
    from scripts.utils.classifier import records_from_split

    eval_frame = load_split(
        "final_eval_heldout",
        namespace,
        purpose="load_labels_for_evaluation",
        final_eval_run_id=run_id,
        final_eval_context_path=context_path,
        caller="stage5_evaluate",
    )
    registration = mark_final_outcome_access(context_path)
    eval_records = records_from_split(
        eval_frame, Path(stage1_cfg.paths.images_dir) / namespace / "final_eval_heldout"
    )
    if not eval_records:
        raise SystemExit(
            "UPSTREAM GATE: no preprocessed final_eval_heldout images found. "
            "Stage 1 preprocessing must have run for this split."
        )
    print(f"  final_eval_heldout: {len(eval_records)} images, "
          f"{eval_frame['patient_id'].nunique()} patients", flush=True)

    # Technical resume: per-condition prediction files are written incrementally and reused.
    for entry in results:
        prediction_path = predictions_dir / f"{entry['tag']}.parquet"
        prediction_provenance = {
            "checkpoint_hash": evidence["checkpoint_hashes"][entry["tag"]],
            "split_hash": evidence["split_manifest_hash"], "protocol_hash": evidence["protocol_manifest_hash"],
            "threshold_hash": threshold_hash, "code_version": get_git_commit_hash(),
        }
        if prediction_path.is_file():
            try:
                validate_completed_predictions(prediction_path, prediction_provenance,
                                               [r["image_id"] for r in eval_records], [r["patient_id"] for r in eval_records])
            except Exception as exc:
                raise SystemExit(f"Existing predictions for {entry['tag']} are partial/corrupt/incompatible; refusing reuse: {exc}") from exc
            print(f"  [{entry['tag']}] safely resumed from validated predictions", flush=True)
            continue
        print(f"  [{entry['tag']}] predicting...", flush=True)
        probabilities = predict_condition(entry, eval_records, stage4_cfg)
        frame = pd.DataFrame(probabilities, columns=CLASSIFIER_TARGET_LABELS)
        frame.insert(0, "image_id", [record["image_id"] for record in eval_records])
        frame.insert(1, "patient_id", [record["patient_id"] for record in eval_records])
        publish_predictions_atomic(prediction_path, frame, prediction_provenance,
                                   [r["image_id"] for r in eval_records], [r["patient_id"] for r in eval_records])

    write_json(
        output_dir / "final_eval_run_manifest.json",
        {
            "final_eval_run_id": run_id,
            "run_at_utc": datetime.now(timezone.utc).isoformat(),
            "namespace": namespace,
            "preconditions": evidence,
            "n_eval_images": len(eval_records),
            "n_eval_patients": int(eval_frame["patient_id"].nunique()),
            "predictions_dir": str(predictions_dir),
            "split_provenance": split_provenance(namespace),
            "git_commit_hash": get_git_commit_hash(),
            "protocol_note": (
                "One frozen final-evaluation protocol. Technical resume is permitted; "
                "methodological re-evaluation is not. A genuine methodological error requires "
                "re-freezing and a NEW run id, reported alongside this one."
            ),
        },
    )
    registration["status"] = "complete"
    registration["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(context_path, registration)

    print(f"\nPredictions written -> {predictions_dir}", flush=True)
    print("Next (no GPU required):", flush=True)
    print(f"  python scripts/eval/compare_conditions.py --run-dir {output_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


# End-to-end smoke pipeline


## `scripts/smoke/00_build_fixture.py`


In [ ]:
# SOURCE FILE: scripts/smoke/00_build_fixture.py
#!/usr/bin/env python3
"""Create the deterministic synthetic CheXpert-shaped input used only by dev-smoke-v1."""
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.labels import CLASSIFIER_TARGET_LABELS


def main() -> int:
    root = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
    if root.name != "dev-smoke-v1":
        raise SystemExit(f"Smoke fixture PROJECT_ROOT must end in dev-smoke-v1, got {root}")
    raw = root / "data" / "chexpert" / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    rows = []
    for index in range(60):
        patient = f"patient{index:05d}"
        relative = Path("CheXpert-v1.0-small") / "train" / patient / "study1" / "view1_frontal.jpg"
        # Runtime resolvers strip the CheXpert-v1.0-small prefix before joining raw_dir.
        destination = raw / Path(*relative.parts[1:])
        destination.parent.mkdir(parents=True, exist_ok=True)
        yy, xx = np.mgrid[:256, :256]
        array = np.clip(25 + (xx + yy) * 0.25 + (index % 7) * 4, 0, 255).astype(np.uint8)
        Image.fromarray(array, mode="L").save(destination, quality=92)
        row = {"Path": relative.as_posix(), "Sex": "Male" if index % 2 else "Female",
               "Age": 35 + index % 45, "Frontal/Lateral": "Frontal", "AP/PA": "PA"}
        for label_index, label in enumerate(CLASSIFIER_TARGET_LABELS):
            row[label] = int((index + label_index) % 3 == 0)
        row["No Finding"] = int(not any(row[label] for label in CLASSIFIER_TARGET_LABELS[1:13]))
        rows.append(row)
    pd.DataFrame(rows).to_csv(raw / "train.csv", index=False)
    pd.DataFrame(rows[:4]).to_csv(raw / "valid.csv", index=False)
    marker = root / "SMOKE_ONLY.json"
    marker.write_text('{"namespace":"dev-smoke-v1","scientific_evidence":false}\n', encoding="utf-8")
    print(f"Smoke fixture ready: {root} (60 patients; never production evidence)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/smoke/run_smoke_pipeline.py`


In [ ]:
# SOURCE FILE: scripts/smoke/run_smoke_pipeline.py
#!/usr/bin/env python3
"""Resumable Stage 1-5 fixture pipeline; outputs are permanently marked non-scientific."""
from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parents[2]
NAMESPACE = "dev-smoke-v1"


def require(path: Path, label: str) -> None:
    if not path.exists():
        raise SystemExit(f"SMOKE GATE FAILED: missing {label}: {path}")
    print(f"  validated {label}: {path}", flush=True)


def require_successful_preprocessing(root: Path, split: str) -> None:
    log = root / f"{split}_preprocessing_log.jsonl"
    require(log, f"{split} preprocessing log")
    rows = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()]
    failed = [row for row in rows if row.get("action") == "failed"]
    kept = [row for row in rows if row.get("kept")]
    if failed or not kept:
        raise SystemExit(f"SMOKE GATE FAILED: {split} has failed={len(failed)}, kept={len(kept)}")


def run(command: list[str], env: dict[str, str]) -> None:
    print("\n+ " + subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=REPO, env=env, check=True)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--phase", choices=["local", "runpod", "all"], default="all")
    parser.add_argument("--workspace", default=str(REPO / "outputs" / "smoke" / NAMESPACE))
    parser.add_argument("--approve-smoke-pilot", action="store_true",
                        help="Explicitly approve fixture pilot after visual review; never approves production")
    args = parser.parse_args()
    workspace = Path(args.workspace).resolve()
    if workspace.name != NAMESPACE or "production" in str(workspace).lower():
        raise SystemExit("Smoke workspace must end in dev-smoke-v1 and must not contain 'production'")
    workspace.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PROJECT_ROOT": str(workspace), "THESIS_CONFIG_OVERLAY": str(REPO / "configs" / "smoke_e2e.yaml"),
           "THESIS_SMOKE_MODE": "1", "PYTHONUNBUFFERED": "1"}
    py = sys.executable

    if args.phase in {"local", "all"}:
        run([py, "scripts/smoke/00_build_fixture.py"], env)
        split_manifest = workspace / "data/chexpert/processed/splits" / NAMESPACE / "split_manifest_v2.json"
        if not split_manifest.is_file():
            run([py, "scripts/data/02b_build_sixway_splits.py", "--namespace", "dev", "--run-id", NAMESPACE], env)
        require(split_manifest, "six-way split manifest")
        run([py, "scripts/data/03_preprocess_images.py", "--namespace", NAMESPACE], env)
        for split in ("gen_train", "gen_val", "classifier_train", "classifier_val", "asism_tuning_heldout", "final_eval_heldout"):
            require(workspace / "data/chexpert/processed/images_768" / NAMESPACE / f"{split}_preprocessing_manifest.json", f"{split} preprocessing manifest")
            require_successful_preprocessing(workspace / "data/chexpert/processed/images_768" / NAMESPACE, split)
        run([py, "scripts/data/04_generate_captions.py", "--namespace", NAMESPACE, "--splits", "gen_train", "gen_val"], env)
        for split in ("gen_train", "gen_val"):
            caption_path = workspace / "data/chexpert/processed/captions" / NAMESPACE / f"{split}_captions.jsonl"
            require(caption_path, f"{split} captions")
            if not caption_path.read_text(encoding="utf-8").strip():
                raise SystemExit(f"SMOKE GATE FAILED: empty captions at {caption_path}")
        print("\nLOCAL SMOKE PHASE COMPLETE. No thesis data was accessed.", flush=True)
        if args.phase == "local":
            return 0

    require(workspace / "SMOKE_ONLY.json", "fixture-only marker")
    split_manifest = workspace / "data/chexpert/processed/splits" / NAMESPACE / "split_manifest_v2.json"
    require(split_manifest, "split manifest")

    stage1_run = workspace / "checkpoints/stage1_lora_sdxl/smoke-stage1-v1"
    final_lora = stage1_run / "final"
    if not (final_lora / "metadata.json").is_file():
        command = ["accelerate", "launch", "--config_file", "configs/accelerate_config.yaml",
                   "scripts/train/train_lora_sdxl.py", "--run-id", "smoke-stage1-v1"]
        latest = stage1_run / "latest.json"
        if latest.is_file():
            checkpoint_dir = json.loads(latest.read_text(encoding="utf-8"))["checkpoint_dir"]
            command += ["--resume-from", checkpoint_dir]
        run(command, env)
    require(final_lora / "metadata.json", "Stage 1 final LoRA metadata")
    env["SMOKE_LORA_DIR"] = str(final_lora)

    run([py, "scripts/generate/01_sample_label_recipes.py", "--namespace", NAMESPACE, "--limit", "4"], env)
    synthetic = workspace / "data/chexpert/synthetic" / NAMESPACE
    require(synthetic / "recipes_manifest.json", "Stage 2 recipe manifest")
    run([py, "scripts/generate/02_generate_synthetic_images.py", "--mode", "pilot", "--namespace", NAMESPACE], env)
    require(synthetic / "pilot/pilot_approval_manifest.json", "pilot checks")
    if not args.approve_smoke_pilot:
        print("\nSTOP: inspect the four fixture pilot images, then rerun this same command with --approve-smoke-pilot.", flush=True)
        return 3
    run([py, "scripts/generate/02_generate_synthetic_images.py", "--approve-pilot", "--reviewer", "smoke-fixture-review",
         "--notes", "Fixture-only pipeline validation", "--namespace", NAMESPACE], env)
    run([py, "scripts/generate/02_generate_synthetic_images.py", "--mode", "full", "--namespace", NAMESPACE], env)
    require(synthetic / "generation_complete.json", "Stage 2 completion")

    run([py, "scripts/classify/00_train_auxiliary_classifier.py", "--namespace", NAMESPACE, "--max-steps", "2", "--batch-size", "2"], env)
    require(workspace / "checkpoints/auxiliary_classifier" / NAMESPACE / "auxiliary_classifier.best.pt", "auxiliary best checkpoint")
    run([py, "scripts/asism/01_compute_signals.py", "--signal", "all", "--namespace", NAMESPACE], env)
    for signal in ("similarity", "iqa", "uncertainty", "explainability", "agreement"):
        require(synthetic / "scores" / f"{signal}_scores.provenance.json", f"{signal} provenance")
    run([py, "scripts/asism/02_gonogo.py", "--namespace", NAMESPACE], env)
    require(synthetic / "asism/gonogo_report.json", "Go/No-Go report")
    run([py, "scripts/asism/03_tune_freeze_select.py", "--phase", "estimate", "--namespace", NAMESPACE], env)
    run([py, "scripts/asism/03_tune_freeze_select.py", "--phase", "tune", "--namespace", NAMESPACE], env)
    run([py, "scripts/asism/03_tune_freeze_select.py", "--phase", "select", "--namespace", NAMESPACE], env)
    require(synthetic / "asism/asism_frozen_manifest.json", "frozen ASISM baseline manifest")

    run([py, "scripts/classify/01_train_conditions.py", "--condition", "all", "--namespace", NAMESPACE], env)
    require(workspace / "outputs/stage4" / NAMESPACE / "stage4_training_results.json", "Stage 4 A/B/C results")
    run([py, "scripts/eval/stage5_evaluate.py", "--final-eval-run-id", "smoke-final-v1", "--namespace", NAMESPACE,
         "--bootstrap-resamples", "20"], env)
    stage5 = workspace / "outputs/stage5" / NAMESPACE / "smoke-final-v1"
    require(stage5 / "final_eval_run_manifest.json", "Stage 5 smoke manifest")
    run([py, "scripts/eval/compare_conditions.py", "--run-dir", str(stage5), "--bootstrap-resamples", "20"], env)
    require(stage5 / "stage5_comparison_report.json", "Stage 5 comparison report")
    print("\nEND-TO-END SMOKE COMPLETE — FIXTURE ENGINEERING EVIDENCE ONLY, NEVER THESIS RESULTS.", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


# Configuration files

These YAML files are the project's auditable configuration layer.

## `configs/accelerate_config.yaml`

```yaml
# Generated by `accelerate config` conventions, checked in so training is reproducible without
# an interactive setup step on a fresh RunPod pod. Single A40, single process — no multi-GPU/
# multi-node config needed (plan §8).
compute_environment: LOCAL_MACHINE
distributed_type: "NO"
downcast_bf16: "no"
gpu_ids: "0"
machine_rank: 0
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

```

## `configs/dataset_config.yaml`

```yaml
# CheXpert-v1.0-small schema and acquisition reference. Kept separate from stage1_lora_sdxl.yaml
# because this describes the *dataset itself*, not a tunable pipeline choice — it should stay
# stable even as training hyperparameters change.

source:
  provider: "kaggle"
  # Downloaded via kagglehub.dataset_download() by scripts/data/00_download_dataset.py.
  kaggle_dataset_slug: "ashery/chexpert"
  expected_train_rows: 224316
  expected_valid_rows: 234

schema:
  path_column: "Path"
  patient_id_regex: "patient\\d+"   # extracted from Path, e.g. CheXpert-v1.0-small/train/patient00001/...
  sex_column: "Sex"
  age_column: "Age"
  frontal_lateral_column: "Frontal/Lateral"
  ap_pa_column: "AP/PA"
  # All 14 pathology label columns, ternary-valued in {-1, 0, 1, <blank>}. Order here is the
  # canonical serialization order used by scripts/utils/caption_builder.py.
  pathology_columns:
    - "No Finding"
    - "Enlarged Cardiomediastinum"
    - "Cardiomegaly"
    - "Lung Opacity"
    - "Lung Lesion"
    - "Edema"
    - "Consolidation"
    - "Pneumonia"
    - "Atelectasis"
    - "Pneumothorax"
    - "Pleural Effusion"
    - "Pleural Other"
    - "Fracture"
    - "Support Devices"
  device_column: "Support Devices"  # handled separately from pathology findings in captions (plan §7)

expected_directory_structure: |
  data/chexpert/raw/
    train.csv
    valid.csv
    train/
      patient00001/
        study1/
          view1_frontal.jpg
          ...
    valid/
      patient64541/
        ...

integrity_checks:
  # Executed by scripts/data/01_verify_download.py — see docs/stage1_plan.md §6.
  - "train.csv row count within 1% of expected_train_rows"
  - "valid.csv row count matches expected_valid_rows"
  - "all pathology_columns present with values in {-1, 0, 1, NaN}"
  - "spot-check N random images open correctly and have plausible dimensions"
  - "per-label positive-rate roughly consistent with published CheXpert statistics"

```

## `configs/smoke_e2e.yaml`

```yaml
# Explicit fixture-only overrides. Activated only through THESIS_CONFIG_OVERLAY.
smoke_only: true
namespace: dev-smoke-v1

dataset:
  source:
    expected_train_rows: 60
    expected_valid_rows: 4

splits:
  namespaces:
    dev:
      require_full_cohort: false

stage1:
  split:
    namespace: dev-smoke-v1
  data:
    resolution: 64
    min_source_resolution: 32
  lora:
    rank: 2
    alpha: 2
  optimizer:
    name: adamw
    lr_warmup_steps: 0
  training:
    precision: fp32
    train_batch_size: 1
    gradient_accumulation_steps: 1
    gradient_checkpointing: false
    cache_latents_and_embeddings: false
    max_train_steps: 2
    min_train_steps: 0
    ema:
      enabled: false
  checkpointing:
    save_every_n_steps: 1
    keep_last_n_full_checkpoints: 2
  validation:
    val_every_n_steps: 1

stage2:
  split_namespace: dev-smoke-v1
  checkpoint:
    lora_weights_dir: ${oc.env:SMOKE_LORA_DIR,null}
  generation:
    resolution: 64
    num_inference_steps: 4
    guidance_scale: 3.0
    batch_size: 1
    seed: 20260807
  recipes:
    min_support_patients: 1
    base_quota: 1
    min_per_label: 1
    max_per_label: 1
    age_buckets: [40, 60]
  pilot:
    num_images: 4
    checks:
      min_valid_image_fraction: 1.0
      max_near_uniform_fraction: 1.0

stage3:
  split_namespace: dev-smoke-v1
  auxiliary_classifier:
    resolution: 64
    dropout_p: 0.2
    pretrained_source: random
  signals:
    similarity:
      batch_size: 2
      k_neighbors: 2
      reference_sample_per_label: 2
      min_exact_match_references: 1
    uncertainty:
      mc_dropout_passes: 2
  gonogo:
    min_unique_values: 2
    directionality_probe_size: 4
  tuning:
    coarse:
      trials: 10
      folds: 1
      seeds: 1
      proxy_max_steps: 1
    shortlist:
      size: 1
      folds: 2
      seeds: 1
      proxy_max_steps: 1
    proxy:
      batch_size: 2
      resolution: 64
      pretrained_source: random
    compute_budget:
      max_gpu_hours: 1.0
      hours_per_proxy_run_estimate: 0.01
  selection:
    quality_floor_percentile: 0
    target_synthetic_to_real_ratio: 0.25
    min_accepted_per_label: 0
    max_accepted_per_label: 4

stage4:
  split_namespace: dev-smoke-v1
  conditions: [A, B, C]
  model:
    pretrained_source: random
    resolution: 64
  training:
    max_steps: 2
    batch_size: 2
    eval_every_n_steps: 1
  seeds: [42]
  condition_d:
    n_draws: 0
    draw_seeds: []

```

## `configs/splits.yaml`

```yaml
# Six-way patient-level split policy (docs/stages2_to_5_plan.md §1) — FROZEN v1.
#
# Supersedes the schema-v1 three-way split in configs/stage1_lora_sdxl.yaml `split:` (which built
# gen_train/gen_val/classifier_heldout and derived asism_tuning_heldout/final_eval_heldout as
# children of classifier_heldout). That architecture left no population for a classifier
# development split. See docs/stages2_to_5_plan.md §1.1 for why deriving one from gen_train was
# rejected.
#
# The v1 artifacts are retained and marked superseded; they are never read by Stage 2-5 code.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  raw_dir: ${paths.data_root}/raw
  processed_dir: ${paths.data_root}/processed
  # Namespaced so a dev-subset split can never overwrite a production split (plan §1.4).
  splits_root: ${paths.processed_dir}/splits

manifest_version: 2

# Six DIRECT top-level partitions from one deterministic patient list and one seed.
# No split is derived from another. Must sum to exactly 1.0 (asserted at build time).
#
# Revised from the original 60/10/15/5/5/5 (docs/stages2_to_5_plan.md §1.2 revision note): on the
# full production CheXpert cohort, the original 5% decision-bearing fractions left "Lung Lesion"
# and "Atelectasis" short of the frozen ≥50-negative-patient support rule (§1.3) in
# classifier_val/asism_tuning_heldout/final_eval_heldout. Raising those three splits to 9% (taken
# from gen_train) clears both with margin. "Pleural Other" cannot be fixed by any fraction choice
# (only ~100 patients dataset-wide carry a confident negative label for it) and is excluded from
# PRIMARY_ENDPOINT_LABELS instead (scripts/utils/labels.py) rather than distorting the split policy
# further for an unfixable label.
fractions:
  gen_train: 0.48             # Stage 1 LoRA training; ASISM real-reference pool; auxiliary classifier
  gen_val: 0.10               # Stage 1 monitoring; auxiliary classifier validation
  classifier_train: 0.15      # A-E real training component; ASISM proxy real component
  classifier_val: 0.09        # A-E model selection ONLY (early stop, checkpoint, threshold, hparams)
  asism_tuning_heldout: 0.09  # ASISM Go/No-Go evidence + proxy-search evaluation
  final_eval_heldout: 0.09    # Stage 5 only

split_seed: 42
split_unit: "patient_id"      # never image level — CheXpert patients have multiple studies

# Minimum-support rule (plan §1.3) — FROZEN. Patient counts, not image counts, decide eligibility.
# Applied to every label in the primary endpoint set, in every split where that label drives a
# decision or a metric.
support_rule:
  min_positive_patients: 50
  min_negative_patients: 50
  decision_bearing_splits:
    - classifier_train
    - classifier_val
    - asism_tuning_heldout
    - final_eval_heldout

# Production namespace requires the source CSV to match the full-cohort integrity contract in
# configs/dataset_config.yaml (source.expected_train_rows) within this tolerance. The dev namespace
# skips that requirement and marks rare-label support insufficiency in its manifest instead.
namespaces:
  production:
    require_full_cohort: true
    full_cohort_row_tolerance_pct: 1.0
  dev:
    require_full_cohort: false

```

## `configs/stage1_lora_sdxl.yaml`

```yaml
# Stage 1 — SDXL + LoRA fine-tuning on CheXpert. Single source of truth for tunables.
# See docs/stage1_plan.md for the full rationale behind every value here.
#
# All paths are relative to PROJECT_ROOT, which defaults to the current working directory.
# On RunPod, set PROJECT_ROOT=/workspace/chest-synth-thesis (or wherever this repo is cloned on
# the persistent volume) before launching any script, so nothing resolves onto the ephemeral
# container disk.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  raw_dir: ${paths.data_root}/raw
  processed_dir: ${paths.data_root}/processed
  images_dir: ${paths.processed_dir}/images_768
  splits_dir: ${paths.processed_dir}/splits
  captions_dir: ${paths.processed_dir}/captions
  checkpoints_dir: ${paths.project_root}/checkpoints/stage1_lora_sdxl
  logs_dir: ${paths.project_root}/logs/stage1_lora_sdxl
  outputs_dir: ${paths.project_root}/outputs/stage1_samples
  hf_cache_dir: ${paths.project_root}/.cache/huggingface

run:
  # run_id is composed at launch time as "<timestamp>_<git_sha>_<config_hash>" by scripts/utils/manifest.py.
  # Leave null here; set explicitly only when resuming a specific prior run.
  run_id: null
  seed: 42

dev_subset:
  # Fast-iteration development subset (docs/stage1_plan.md's "small subset" made concrete/configurable).
  # This ONLY exists to validate the pipeline quickly; the full CheXpert dataset generates final
  # thesis results. Never hardcode a sample size in code — this is the single source of truth for it.
  enabled: false            # 01b_build_dev_subset.py no-ops (prints a message) when false.
  target_size: 10000        # approximate output image count; configurable (5k/10k/20k/...).
  seed: 42                  # independent of split_seed so subset draws can vary without perturbing
                             # the 70/10/20 patient-splitting logic.
  prevalence_tolerance_pct: 5.0   # report-only threshold; deviations beyond this are WARNINGs, not
                                   # fatal — the subset only needs to be "reasonably representative."
  output_csv: ${paths.processed_dir}/dev_subset/train_subset.csv
  manifest_path: ${paths.processed_dir}/dev_subset/dev_subset_manifest.json
  prevalence_report_path: ${paths.processed_dir}/dev_subset/prevalence_report.json

model:
  base_model_id: "stabilityai/stable-diffusion-xl-base-1.0"
  revision: "462165984030d82259a11f4367a4eed129e94a7b"
  vae_upcast_fp32: true  # avoids the known SDXL-VAE fp16 NaN/black-image bug (plan §2)
  use_refiner: false

lora:
  rank: 32
  alpha: 32
  dropout: 0.0
  target_modules: ["to_q", "to_k", "to_v", "to_out.0"]  # UNet self+cross attention projections only
  train_text_encoders: false  # documented fallback: set true (+ text_encoder_lr) if rare-label conditioning is weak

optimizer:
  name: "adamw_8bit"        # falls back to "adamw" automatically if bitsandbytes/CUDA mismatch is detected
  learning_rate: 1.0e-4
  lr_scheduler: "cosine"
  lr_warmup_steps: 750
  weight_decay: 1.0e-2
  max_grad_norm: 1.0

data:
  view_filter: "frontal"   # "frontal" | "both" — lateral images are flagged, not deleted, when filtered
  resolution: 768
  aspect_mode: "letterbox" # aspect-preserving resize + minimal pad; never non-uniform stretch, never center-crop
  normalize_mean: 0.5
  normalize_std: 0.5
  horizontal_flip: false   # MUST stay false — flipping inverts CXR left/right anatomy
  min_source_resolution: 200  # px, below this an image is flagged/dropped as low quality

split:
  # Explicit schema-v2 six-way split namespace. There is deliberately no fallback to the
  # superseded root-level schema-v1 files.
  namespace: dev-smoke-v1
  manifest_version: 2
  input_csv: ${paths.raw_dir}/train.csv  # source CSV for patient splitting. Point this at
                                          # dev_subset.output_csv (below) to build gen_train/gen_val/
                                          # classifier_heldout from the dev subset instead of the full
                                          # ~224k-image CheXpert train.csv — a config-only switch, no
                                          # code change required in 02_build_patient_splits.py or any
                                          # downstream script.
  gen_train_fraction: 0.70
  gen_val_fraction: 0.10
  classifier_heldout_fraction: 0.20
  split_seed: 42
  split_unit: "patient_id"  # never split at image level — CheXpert patients have multiple studies
  # classifier_heldout is further split so ASISM's weight/threshold search (Stage 3) never tunes
  # against the same data Stage 5 reports its headline numbers on (leakage/optimistic-bias risk).
  asism_tuning_fraction: 0.35   # fraction of classifier_heldout patients used only by Stage 3's search
  heldout_split_seed: 43        # separate seed from split_seed so this sub-split can be reseeded
                                 # independently of the top-level gen_train/gen_val/heldout split

captions:
  template_version: "v1"
  num_paraphrase_variants: 4
  age_bucket_width_years: 10
  uncertain_label_policy: "omit"  # -1 labels are omitted from the finding clause, not asserted present/absent
  no_finding_overrides_positives: true  # treat "No Finding"==1 as authoritative over spurious co-positive labels

training:
  precision: "bf16"
  attention_backend: "sdpa"  # PyTorch 2.x scaled_dot_product_attention; xformers intentionally not used (plan §5)
  train_batch_size: 8
  gradient_accumulation_steps: 2
  gradient_checkpointing: true
  cache_latents_and_embeddings: true
  ema:
    enabled: true
    decay: 0.9999
  max_train_steps: 30000    # upper bound; actual stop driven by val-loss plateau + stable probe samples (plan §9)
  min_train_steps: 15000

checkpointing:
  save_every_n_steps: 1000
  keep_last_n_full_checkpoints: 3
  keep_all_lora_weight_snapshots: true

validation:
  val_every_n_steps: 1000
  probe_seed: 123
  fid:
    inception: true
    domain_torchxrayvision: true
  clip_score: true

logging:
  tensorboard: true
  wandb:
    enabled: false  # set true + WANDB_API_KEY env var to enable the optional supplementary tracker
    project: "chexpert-sdxl-lora-stage1"

```

## `configs/stage2_generation.yaml`

```yaml
# Stage 2 — synthetic generation (docs/stages2_to_5_plan.md §3).
#
# Conditioning labels are `intended_label_vector` — what the generator was ASKED for, never
# "ground truth". Stage 3 §4.5 measures whether the pixels agree; nothing here establishes it.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  synthetic_root: ${paths.data_root}/synthetic
  images_dir: ${paths.synthetic_root}/images
  recipes_path: ${paths.synthetic_root}/label_recipes.csv
  recipe_decisions_path: ${paths.synthetic_root}/recipe_decisions.jsonl
  manifest_path: ${paths.synthetic_root}/generation_manifest.jsonl
  pilot_dir: ${paths.synthetic_root}/pilot
  pilot_approval_manifest: ${paths.synthetic_root}/pilot/pilot_approval_manifest.json

split_namespace: "dev-smoke-v1"  # immutable run ID; production IDs use production-<version>

checkpoint:
  # Explicit LoRA checkpoint — NOT "latest". Generation must be reproducible and decoupled from
  # any training still in progress (docs/stage1_plan.md §11 handoff contract).
  lora_weights_dir: null      # e.g. checkpoints/stage1_lora_sdxl/<run_id>/lora_weights/step_20000
  base_model_id: "stabilityai/stable-diffusion-xl-base-1.0"
  revision: "462165984030d82259a11f4367a4eed129e94a7b"

generation:
  resolution: 768             # must match Stage 1 data.resolution
  num_inference_steps: 40
  guidance_scale: 7.0
  scheduler: "DPMSolverMultistep"
  batch_size: 4
  seed: 20260807              # base seed; per-image seed is derived deterministically from image_id
  negative_prompt: ""

recipes:
  # Empirical co-occurrence mining from gen_train (§3.1)
  min_support_patients: 25          # a label combination must occur in >= this many real patients
  # Target recipe mix (§3)
  single_label_proportion: 0.55     # fraction of disease recipes carrying exactly one positive label
  multi_label_proportion: 0.45
  no_finding_proportion: 0.20       # fraction of ALL recipes that are the all-zero (No Finding) vector
  max_positive_labels: 3            # cap on simultaneous positive disease labels per recipe

  # Per-label generation quota, oversampling rare classes relative to real prevalence (§3.7).
  # target_count = base_quota * (median_prevalence / label_prevalence) ** rarity_exponent,
  # clipped to [min_per_label, max_per_label]. rarity_exponent=0 disables oversampling.
  base_quota: 400
  rarity_exponent: 0.5
  min_per_label: 100
  max_per_label: 1500

  # Demographic coverage: recipes are sampled across these buckets rather than left incidental.
  age_buckets: [20, 30, 40, 50, 60, 70, 80]
  sexes: ["Male", "Female"]
  balance_demographics: true

  # Support Devices is carried as a context attribute only. It never affects the primary
  # disease-agreement score (§4.5) unless secondary_agreement is explicitly enabled in Stage 3.
  support_devices_proportion: 0.35

  # Medical-rule overrides (§3.2). Both directions, both auditable.
  # allow: clinically plausible combinations permitted even below min_support_patients.
  # block: combinations judged likely CheXpert NLP-extraction noise, rejected even if observed.
  medical_rule_allow: []
  medical_rule_block: []

pilot:
  num_images: 48              # small spanning batch for manual visual review before full generation
  # Automatic artifact checks the pilot must pass before it can be approved.
  checks:
    min_valid_image_fraction: 0.95
    require_correct_resolution: true
    require_rgb: true
    max_near_uniform_fraction: 0.02   # near-blank outputs indicate a broken generation config
    blank_std_threshold: 5.0

```

## `configs/stage3_asism.yaml`

```yaml
# Stage 3 — ASISM (docs/stages2_to_5_plan.md §4).
#
# Five independent signals, each written to its own versioned parquet artifact, each subject to the
# SAME Go/No-Go gate (§4.6). No signal is admitted to the frozen selector merely because it was
# implemented.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  synthetic_root: ${paths.data_root}/synthetic
  scores_dir: ${paths.synthetic_root}/scores
  asism_dir: ${paths.synthetic_root}/asism
  gonogo_report: ${paths.synthetic_root}/asism/gonogo_report.json
  search_log: ${paths.synthetic_root}/asism/proxy_search_log.jsonl
  frozen_manifest: ${paths.synthetic_root}/asism/asism_frozen_manifest.json
  fold_manifest: ${paths.synthetic_root}/asism/asism_patient_folds.json
  selected_manifest: ${paths.synthetic_root}/asism/selected_manifest.jsonl
  rejected_log: ${paths.synthetic_root}/asism/rejected_log.jsonl

split_namespace: "dev-smoke-v1"

auxiliary_classifier:
  checkpoint_path: ${paths.project_root}/checkpoints/auxiliary_classifier/auxiliary_classifier.pt
  resolution: 320
  dropout_p: 0.2
  pretrained_source: "imagenet"   # CheXpert-pretrained is rejected in code (§2)

signals:
  similarity:
    encoder: "vit_small_patch14_dinov2"  # architecture; pinned weights are declared below
    weights_repo_id: "timm/vit_small_patch14_dinov2.lvd142m"
    weights_filename: "model.safetensors"
    weights_revision: "936966a8732c5442c9def5d126f2cc4ad4243dba"
    batch_size: 32
    k_neighbors: 15               # k-NN aggregation, never single-nearest-neighbor
    reference_sample_per_label: 400
    min_exact_match_references: 25    # below this, fall back down the hierarchy (§4.1)
    # Memorization flag: high single-NN similarity with low rank-k spread means the image is
    # suspiciously close to ONE real training image rather than to the class in general.
    near_duplicate_similarity: 0.95
    near_duplicate_spread: 0.02

  iqa:
    # Generic NR-IQA is a BASELINE component; the domain checks below carry the signal (§4.2).
    blank_std_threshold: 5.0
    clipping_percentile: 0.995
    low_contrast_std: 15.0
    border_uniform_fraction: 0.30
    laplacian_blur_threshold: 40.0

  uncertainty:
    mc_dropout_passes: 20
    # Band edges on predictive std over the 11 primary labels (§4.3). Moderate uncertainty is NOT
    # penalized: it may be the informative band (TSynD's premise).
    low_band_max: 0.05
    moderate_band_max: 0.15

  explainability:
    method: "gradcam"
    # Fixed central-thorax box is an explicit BASELINE (§4.4). Pathology-specific expected regions
    # are relative (x0, y0, x1, y1) fractions of the image.
    region_mode: "pathology_specific"
    baseline_box: [0.20, 0.15, 0.80, 0.85]
    pathology_regions:
      Cardiomegaly: [0.30, 0.40, 0.75, 0.85]
      Enlarged Cardiomediastinum: [0.30, 0.30, 0.75, 0.80]
      Pleural Effusion: [0.10, 0.60, 0.90, 0.95]
      Pneumothorax: [0.05, 0.15, 0.95, 0.70]
      Edema: [0.15, 0.25, 0.85, 0.80]
      Consolidation: [0.10, 0.25, 0.90, 0.85]
      Pneumonia: [0.10, 0.25, 0.90, 0.85]
      Atelectasis: [0.10, 0.30, 0.90, 0.90]
      Lung Opacity: [0.10, 0.20, 0.90, 0.85]
      Lung Lesion: [0.10, 0.15, 0.90, 0.85]
      Pleural Other: [0.05, 0.50, 0.95, 0.95]
      Fracture: [0.05, 0.10, 0.95, 0.60]

  agreement:
    # agreement = mean P(intended positives) - penalty * mean P(confident unintended positives)
    # For a No Finding recipe (all-zero intended vector) this reduces to: all 12 probabilities low.
    unintended_confidence_threshold: 0.5
    penalty_weight: 0.5
    # Support Devices stays OUT of the primary disease-agreement score (§3.4, §4.5).
    secondary_agreement:
      enabled: false
      labels: ["Support Devices"]

gonogo:
  # A signal must clear every applicable check to enter the frozen selector (§4.6).
  max_missing_fraction: 0.05
  min_unique_values: 20            # a near-constant score carries no information
  directionality_probe_size: 200
  redundancy_abs_correlation_max: 0.95   # above this, a signal is near-duplicate of another
  reproducibility_tolerance: 1.0e-6

tuning:
  # Two-stage bounded search (§4.7). Proxy trains on classifier_train + candidate synthetic subset,
  # evaluates on asism_tuning_heldout ONLY. classifier_val is NOT used to rank candidates.
  coarse:
    trials: 12
    folds: 1
    seeds: 1
    proxy_max_steps: 300
  shortlist:
    size: 4
    folds: 3
    seeds: 2
    proxy_max_steps: 800
  proxy:
    batch_size: 32
    learning_rate: 1.0e-4
    resolution: 320
    dropout_p: 0.2
    pretrained_source: "imagenet"
  # Compute-budget gate: if the pre-execution estimate exceeds this, the search space must be
  # reduced BEFORE any results are seen — never adjusted afterward.
  compute_budget:
    max_gpu_hours: 8.0
    hours_per_proxy_run_estimate: 0.15
  # Ties within this band resolve to the SIMPLER configuration (fewer signals, less extreme weights).
  tie_noise_band: 0.002
  search_seed: 42

selection:
  # Ordered policy (§4.8): quality floor FIRST, then class quota. Rarity never bypasses the floor.
  quality_floor_percentile: 25      # reject the bottom N% by composite score, absolutely
  target_synthetic_to_real_ratio: 1.0
  min_accepted_per_label: 50
  max_accepted_per_label: 2000
  uncertainty_band_preference: "moderate"   # "any" | "low" | "moderate"

```

## `configs/stage4_classifier.yaml`

```yaml
# Stage 4 — conditions A-E (docs/stages2_to_5_plan.md §7). FROZEN protocol.
#
# Architecture, initialization, label policy, uncertainty policy, optimizer family, batch size,
# augmentation, checkpoint-selection rule, validation rule, and threshold-selection rule are
# IDENTICAL across all conditions. Only the training data differs.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  checkpoints_dir: ${paths.project_root}/checkpoints/stage4_conditions
  results_dir: ${paths.project_root}/outputs/stage4
  protocol_manifest: ${paths.project_root}/outputs/stage4/frozen_experiment_protocol.json
  label_policy_manifest: ${paths.project_root}/outputs/stage4/frozen_label_policy.json
  threshold_policy_manifest: ${paths.project_root}/outputs/stage4/frozen_threshold_policy.json
  seed_plan_manifest: ${paths.project_root}/outputs/stage4/frozen_seed_run_plan.json
  d_matching_policy_manifest: ${paths.project_root}/outputs/stage4/frozen_d_matching_policy.json

split_namespace: "dev-smoke-v1"
conditions: [A, B, C, D, E]

model:
  architecture: "densenet121"
  pretrained_source: "imagenet"    # CheXpert-pretrained is rejected in code (§2)
  dropout_p: 0.2
  resolution: 320

# FROZEN fairness protocol (§7.1): equal OPTIMIZER STEPS across A-E and every D draw.
# Not equal epochs — at fixed epochs a larger dataset silently receives more gradient updates,
# conflating "more data" with "more training".
training:
  max_steps: 6000
  batch_size: 32
  learning_rate: 1.0e-4
  weight_decay: 1.0e-4
  eval_every_n_steps: 500
  sampling_policy: "shuffled_concat"   # recorded per run; identical across conditions
  augmentation: "none"                 # horizontal flip is forbidden: it inverts CXR anatomy

# FROZEN seed policy (§7): A/B/C/E get 3 seeds each; EVERY D draw gets the same 3 seeds.
# Total runs = 4*3 + 5*3 = 27.
seeds: [42, 43, 44]

condition_d:
  n_draws: 5                    # FROZEN — exactly five independent matched-random draws
  draw_seeds: [101, 102, 103, 104, 105]
  # Each draw must match condition C on:
  matching:
    match_total_count: true
    match_per_label_marginal: true
    per_label_tolerance_fraction: 0.10   # frozen tolerance on per-label positive counts
    match_single_multi_proportion: true
    single_multi_tolerance_fraction: 0.10
    attempt_joint_vector_matching: true  # only where support permits; never made infeasible

# Classification-threshold selection policy (frozen before Stage 5).
threshold_policy:
  method: "max_youden_j"       # chosen on classifier_val ONLY, never on final_eval_heldout
  selection_split: "classifier_val"

```